In [ ]:
import numpy as np
import scipy
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import scienceplots
plt.style.use('science')

orangish_color = '#FC9272'
bluish_color = '#1C9099'
greenish_color = '#2ca25f'

ticklabelsize = 26
axeslabelsize = 27
titlesize = 30
linewidth = 4

plt.rcParams['image.cmap'] = 'OrRd'


# def set_matplotlib_style():
#     plt.style.use(["science", "grid"])
#     import matplotlib
# #     matplotlib.use("pgf")
#     matplotlib.rcParams.update(
#         {
# #             "pgf.texsystem": "pdflatex",
#             "font.family": "serif",
#             "text.usetex": True,
# #             "pgf.rcfonts": False,
#             # "font.size": 20,
#         }
#     )

# set_matplotlib_style()

def set_axes_style(ax):
    width = 0.5
    ax.tick_params(axis='both', which='minor', length=3)
    ax.tick_params(axis='both', which='major', length=6)
    ax.spines["left"].set_linewidth(width)
    ax.spines["bottom"].set_linewidth(width)
    ax.spines["right"].set_linewidth(width)
    ax.spines["top"].set_linewidth(width)
#     ax.tick_params(width=width)#, labelsize=ticklabelsize)

def save_fig(filename):
    plt.savefig(
        filename, format='pdf', metadata={'CreationDate': None}
    )

# Code from https://www.steven-braun.com/blog/2021/matplotlib-viz/
textwidth = 7.03058
aspect_ratio = 6/6
scale = 1.0
width = textwidth * scale
height = width * aspect_ratio

In [ ]:
from scipy import stats
from sklearn.metrics import r2_score
from utils import unnormalize_time, unnormalize_mag
from mtan_utils import mean_squared_error
from constants import agn_list, stars_list, sn_list

time_in_hrs = True

In [ ]:
def plot_lc(
    df_alerts, name, title_suffix='', fid_column='fid', magpsf_column='magpsf',
    jd_column='jd', sigmapsf_column='sigmapsf', finkclass_column='finkclass',
    objectId_column='objectId', figsize=(15, 6), make_first_time_zero=True,
    annotate_finkclass=False, show_corrected_mags=False
):
    """Plots photometry for the given name (objectId) from the alerts dataframe.

    Parameters
    ----------
    name: str
        objectID
    """
    if show_corrected_mags:
        magpsf_column, sigmapsf_column = 'magpsf_corr', 'sigmapsf_corr'

    # Accumulate all alerts for the provided objectId
    pdf = df_alerts[df_alerts[objectId_column] == name].sort_values(by=jd_column)

    if len(pdf) == 0:
        raise ValueError(f'No alerts exist for objectId = {name}!')

    fig, ax = plt.subplots(figsize=figsize)

    # Colors to plot
    colordic = {1: bluish_color, 2: orangish_color}

    # Labels of ZTF filters
    filtdic = {1: 'g', 2: 'r'}

    if make_first_time_zero:
        min_t = pdf[jd_column].min()

    for filt in np.unique(pdf[fid_column]):
        # select data from one filter at a time
        maskFilt = pdf[fid_column] == filt

        t = pdf[maskFilt][jd_column] - min_t if make_first_time_zero else pdf[maskFilt][jd_column]

        ax.errorbar(
            t,
            pdf[maskFilt][magpsf_column],
            pdf[maskFilt][sigmapsf_column],
            ls = '', marker='o', color=colordic[filt], label=filtdic[filt]
        )

        ax.errorbar(
            t,
            pdf[maskFilt][magpsf_column],
            pdf[maskFilt][sigmapsf_column],
            ls='', marker='^', color=colordic[filt]
        )

        if (finkclass_column is not None) and (annotate_finkclass):
            _offset_x = (t.max() - t.min()) / 200
            _offset_y = (pdf[maskFilt][magpsf_column].max() - pdf[maskFilt][magpsf_column].min()) / 100
            for x, y, string in zip(t, pdf[maskFilt][magpsf_column], pdf[maskFilt][finkclass_column]):
                ax.text(
                    x+_offset_x, y+_offset_y, string,
                    color=colordic[filt]
                )

    ax.invert_yaxis()
    legend = ax.legend(fancybox=False, edgecolor='black')#, fontsize=ticklabelsize)
    legend.get_frame().set_linewidth(0.5)
    ax.set_title(f'{pdf[objectId_column].unique()[0]}'+': '+title_suffix)#, fontsize=titlesize)
    ax.set_xlabel('Modified Julian Date' if not make_first_time_zero else 'time (days)')#, fontsize=axeslabelsize)
    ax.set_ylabel('Magnitude')#, fontsize=axeslabelsize)

    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    return ax
    # msg = """
    # - Circles (●) with error bars show valid alerts that pass the Fink quality cuts.
    # - Upper triangles with errors (▲), represent alert measurements that do not satisfy Fink quality cuts, but are nevetheless contained in the history of valid alerts and used by classifiers.
    # - Lower triangles (▽), represent 5-sigma mag limit in difference image based on PSF-fit photometry contained in the history of valid alerts.
    # """
    # print(msg)

def plot_lc_normalized_data(observed_data, observed_mask, observed_tp, title=None):
    """Just like plot_lc() but plots normalized data
    (i.e., after all preprocessing on magnitudes and times),
    just before passing them as input to the model.

    combined_data must be a tuple of (observed_data, observed_mask, time) coming from a
    dataloader (since it must be normalized).

    observed_data shape: [1, lc_length, dim]  # 1 is the batch size.
    observed_mask shape: [1, lc_length, dim]
    observed_tp shape: [1, lc_length, 1]

    Below is an example code to do that:
    ```
    dim = 2
    test_loader = torch.load('test_dataloader.pth')
    batch = next(iter(test_loader))
    data = batch[0]

    # `index` below controls which image in this batch should be shown.
    index = 0
    assert index <= batch_len - 1

    batch_len = data.shape[0]
    observed_data = data[index, :, :dim].cpu().detach().numpy()
    observed_mask = data[index, :, dim:2 * dim].cpu().detach().numpy()
    observed_tp = data[index, :, -1].cpu().detach().numpy()
    ```
    """
    dim = 2
    # Replace values that are unobserved with zero.
    observed_data[observed_mask == 0] = np.nan

    # Colors to plot
    colordic = {1: bluish_color, 2: orangish_color}

    # Labels of ZTF filters
    filtdic = {1: 'g', 2: 'r'}

    fig, ax = plt.subplots(1, 1, figsize=(15, 6))
    for i in range(dim):
        ax.plot(
            observed_tp.squeeze(), observed_data[:, i],
            ls = '', marker='o', color=colordic[i+1], label=filtdic[i+1]
        )

    for a in ax:
        a.tick_params(axis='x')#, labelsize=ticklabelsize)
        a.tick_params(axis='y')#, labelsize=ticklabelsize)

    ax.invert_yaxis()
    legend = ax.legend(fancybox=False, edgecolor='black')#, fontsize=ticklabelsize)
    legend.get_frame().set_linewidth(0.5)
    ax.set_title(f'{title}')#, fontsize=titlesize)
    ax.set_xlabel('Normalized time')#, fontsize=axeslabelsize)
    ax.set_ylabel('Normalized magnitude')#, fontsize=axeslabelsize)

#     ax.tick_params(axis='x', labelsize=ticklabelsize)
#     ax.tick_params(axis='y', labelsize=ticklabelsize)

    set_axes_style(ax)

    plt.show()

from matplotlib.ticker import MaxNLocator, MultipleLocator

import warnings
def plot_decoded_lc(
    decoded_lcs, key, df_alerts, num_filters=2, objId=None,
    unnormalize_lcs=True, min_time=None, max_time=None,
    min_max_mags=None, prob_plot=True, whichclass=None,
    also_show_sigmapsf_and_rmse=True, show_only_rmse=True,
    fid_column='fid', sigmapsf_column='sigmapsf', objectId_column='objectId',
    show_separate_bands=True, fig_width=15, fig_height=12,
    show_legend=True, legend_loc='best', magpsf_column='magpsf', show_nothing=False,
    ax=None, show_separate_bands_wspace=0.05, show_pred=True, show_corrected_mags=False
):
    """This function assumes batch size=1 is used while storing the decoded_lcs.
    min_max_mags is a 2D numpy array. Each row contains [object ID, min_mag, max_mag].

    Note: if `show_corrected_mags` is True, it's assumed that `decoded_lcs` contains
    decoded light curves corresponding to corrected magnitudes.
    """
    if show_corrected_mags:
        magpsf_column, sigmapsf_column = 'magpsf_corr', 'sigmapsf_corr'

    pred_x, observed_data, observed_mask, observed_times = decoded_lcs[key]

    print(pred_x.shape, observed_data.shape, observed_mask.shape, observed_times.shape)

#     try:
    assert np.all(observed_times[:, 0] == observed_times[:, 1])  # Because we simply repeated the dimension in the evaluation_unsupervised.py code.
#     except:
#         a_clean = np.nan_to_num(observed_times[:, 0], nan=0.0)
#         b_clean = np.nan_to_num(observed_times[:, 1], nan=0.0)
#         assert np.array_equal(
#             np.nan_to_num(observed_times[:, 0], nan=0.0),
#             np.nan_to_num(observed_times[:, 1], nan=0.0)
#         )

    observed_times = observed_times[:, 0]
    # Set to NaN to ensure non-essential time values for this lc are not shown.
    observed_times[observed_times == 0.0] = np.nan
    # Recall that we use make_first_time_zero=True in prepare_data (see get_lc), so the first time value will be zero
    # But we don't want that first time value to be set to NaN. So we undo that below.
    observed_times[0] = 0.0

    if unnormalize_lcs:
        if min_max_mags is None:
            raise ValueError(
                "min_max_mags must be provided if unnormalize_lcs is True!"
            )
        if objId is None:
            raise ValueError("objId must be provided if unnormalize_lcs is True!")

        # Extract the row corresponding to the min/max of the object ID.
        row = min_max_mags[np.where(min_max_mags[:,0] == objId)].squeeze()

        if min_time is None or max_time is None:
            raise ValueError(
                "min_time and max_time must be provided if unnormalize_lcs is True!"
            )

        # Since we normalize times using min/max of whole train data,
        # min_time and max_time must represent that.
        observed_times = unnormalize_time(observed_times, min_time, max_time)
        # Since we normalize mags using min/max of the light curve itself,
        # min_mag and max_mag must represent that.
        observed_data = unnormalize_mag(
            observed_data, observed_mask, float(row[1]), float(row[2])
        )
        # Pass a mask of all ones for prediction because for
        # interpolation all points can be considered observed.
        pred_x = unnormalize_mag(
            pred_x, np.ones_like(observed_mask), float(row[1]), float(row[2])
        )

#     c1 = ['#2b8cbe', '#a6bddb']  # dark and then light blue
#     c2 = ['#f03b20', '#feb24c']  # dark and then light orange

    colors = [bluish_color, orangish_color]

#     colors = [c1, c2]

    if also_show_sigmapsf_and_rmse:
#         pdf = df_alerts[df_alerts[objectId_column] == objId].compute()
        compute_attr = getattr(df_alerts, 'compute', None)
        if callable(compute_attr):
            pdf = df_alerts[df_alerts[objectId_column] == objId].compute()
        else:
            pdf = df_alerts[df_alerts[objectId_column] == objId]

    if show_separate_bands:
        fig, ax = plt.subplots(1, 2, figsize=(fig_width, fig_height))
        plt.subplots_adjust(wspace=show_separate_bands_wspace, hspace=0.)
        for _i, a in enumerate(ax):
            a.set_ylim(
                np.floor(np.nanmin(observed_data[:, _i])),
                np.ceil(np.nanmax(observed_data[:, _i]))
            )
            a.yaxis.set_major_locator(MaxNLocator(nbins=3, integer=True))
    else:
        if ax is None:
            fig = plt.figure(figsize=(fig_width, fig_height))
            ax = plt.gca()
        ax.set_ylim(
            np.floor(np.nanmin(observed_data)),
            np.ceil(np.nanmax(observed_data))
        )
        ax.yaxis.set_major_locator(MaxNLocator(nbins=3, integer=True))
        ax.xaxis.set_major_locator(MaxNLocator(nbins=4, integer=True))

    if show_separate_bands:
        ax[0].sharex(ax[1])

    if whichclass is not None:
        if show_separate_bands:
            pass
#             ax[0].set_title(objId + f' - {whichclass}')#, fontsize=titlesize)
        else:
            ax.set_title(objId + f' - {whichclass}')#, fontsize=titlesize)
    else:
        if show_separate_bands:
            pass
#             ax[0].set_title(objId)#, fontsize=titlesize)
        else:
            ax.set_title(objId)#, fontsize=titlesize)

    if not show_separate_bands:
        offset_text = 0.15

    for i in range(num_filters):
        obs_to_show = observed_data[:, i].copy()

        # If `unnormalize_mag` was True, unobserved mags would have been
        # set to nan. So instead set them to zero to be able to calculate
        # metrics and not give nan as output.
        obs_to_show[observed_mask[:, i] == 0.0] = 0.

        rmse = np.sqrt(
            mean_squared_error(obs_to_show, pred_x[:, i], observed_mask[:, i])
        )

        # Set unobserved values to NaN so that they don't show up in the plot.
        obs_to_show[observed_mask[:, i] == 0.0] = np.nan

        resid = obs_to_show - pred_x[:, i]
        #resid_without_nan = resid[~np.isnan(resid)]

        # For calculating R2 score:
        # obs_not_nan_mask = ~np.isnan(obs_to_show)
        # r2score = r2_score(obs_to_show[obs_not_nan_mask], pred_x[:, i][obs_not_nan_mask])

        if show_separate_bands:
            ax[i].plot(observed_times/24 if time_in_hrs else observed_times, obs_to_show, ls = '', marker='o', c=colors[i], label=r'$m_{\mathrm{obs}}$', markersize=5, alpha=0.75)
            if show_pred:
                ax[i].plot(observed_times/24 if time_in_hrs else observed_times, pred_x[:, i], c=colors[i], label=r'$m_{\mathrm{pred}}$', linewidth=2, alpha=0.75)
#             ax[i].scatter(observed_times/24 if time_in_hrs else observed_times, resid, c='black', label=r'$m_{\mathrm{obs}} - m_{\mathrm{pred}}$', alpha=0.8, s=300)
        else:
            ax.plot(observed_times/24 if time_in_hrs else observed_times, obs_to_show, ls = '', marker='o', c=colors[i], label= None, markersize=5, alpha=0.75)
            if show_pred:
                ax.plot(observed_times/24 if time_in_hrs else observed_times, pred_x[:, i], c=colors[i], label=None, linewidth=2, alpha=0.75)
#             ax[1].scatter(observed_times/24 if time_in_hrs else observed_times, resid, c=colors[i], label=None, alpha=0.8, s=300)

        band = r'$g$' if i == 0 else r'$r$'
        if also_show_sigmapsf_and_rmse:
            maskFilt = pdf[fid_column] == (i+1)
            assert len(pdf[maskFilt]) == observed_mask[:, i].sum()
#             avg_sigmapsf = pdf[maskFilt][sigmapsf_column].mean()

            sigmapsf_rms = np.sqrt(np.mean(pdf[maskFilt][sigmapsf_column] ** 2))
#             rmse_n = rmse / sigmapsf_rms

#             with warnings.catch_warnings():
#                 warnings.simplefilter("ignore")
#                 # Ensure this point-by-point comparison is of the same corresponding obs magnitudes.
#                 assert np.all(
#                     obs_to_show[~np.isnan(resid)] == pdf.sort_values(by='jd')[maskFilt][magpsf_column]
#                 )
#                 z = resid[~np.isnan(resid)] / pdf.sort_values(by='jd')[maskFilt][sigmapsf_column]

#             chi2 = np.sum(z**2)
#             chi2_red = chi2 / len(z)

            if show_nothing:
                text = ''
            elif show_only_rmse:
                text = band + ': ' + r'$\mathrm{RMSE} = $'+ f' {rmse:.2f}'
            else:
                text = band + ': ' + r'$\mathrm{RMSE}, \sigma_{\mathrm{psf, rms}} = $'+ f' {rmse:.2f}, {sigmapsf_rms:.2f}'

            if show_separate_bands:
                ax[i].text(
                    0.7 if not show_only_rmse else 0.8, 0.9, text,
                    horizontalalignment='center', verticalalignment='center',
                    transform=ax[0, i].transAxes, #fontsize=30
                )
            else:
                ax.text(
                    0.7 if not show_only_rmse else 0.8, 0.9-offset_text if i == 1 else 0.9, text,
                    horizontalalignment='center', verticalalignment='center',
                    transform=ax.transAxes, #fontsize=30
                )
        else:
            pass
#             if show_separate_bands:
#                 ax[0, i].text(
#                     0.8, 0.9, r'$\mathrm{RMSE} = $'+ f'{rmse:.3f}',
#                     horizontalalignment='center', verticalalignment='center',
#                     transform=ax[0, i].transAxes, fontsize=30
#                 )
#             else:
#                 ax[0].text(
#                     0.8, 0.9-offset_text if i == 1 else 0.9, band + r': $\mathrm{RMSE} = $'+ f'{rmse:.3f}',
#                     horizontalalignment='center', verticalalignment='center',
#                     transform=ax[0].transAxes, fontsize=30
#                 )

#         if prob_plot:
#             ins = ax[1, i].inset_axes([0.05,0.05,0.15,0.15])
#     #         ins.hist(resid)
#             stats.probplot(resid, plot=ins)
#             ins.set_xticks([])
#             ins.set_yticks([])
#             ins.set_ylabel('')
#             ins.set_xlabel('')
#             ins.set_title('Probability plot', fontsize=titlesize)
#             ins.spines[['right', 'top', 'left']].set_visible(False)

#         ax[1, i].axhline(y=5*stats.median_abs_deviation(resid[~np.isnan(resid)]), linestyle='--', c='black', alpha=0.8)
#         ax[1, i].axhline(y=-5*stats.median_abs_deviation(resid[~np.isnan(resid)]), linestyle='--', c='black', alpha=0.8)
#         if show_separate_bands:
#             ax[1, i].axhline(y=0, c='black', alpha=0.8, linestyle='dashed')
#         else:
#             ax[1].axhline(y=0, c='black', alpha=0.8, linestyle='dashed')

        # Set x-axis limits based on its lc only so that we get more detailed understanding of one lc.
        last_non_nan_obs_idx = (~np.isnan(obs_to_show)).cumsum().argmax()
#         ax[0, i].set_xlim([-1, last_non_nan_obs_idx+1])
#         ax[1, i].set_xlim([-1, last_non_nan_obs_idx+1])

        if unnormalize_lcs:
            if show_separate_bands:
#                 ax[i].set_ylabel(r'$m$')#, fontsize=axeslabelsize)
                ax[i].set_xlabel('Time (days)')#, fontsize=axeslabelsize)
#                 ax[1, i].set_ylabel(r'$m_{\mathrm{obs}} - m_{\mathrm{pred}}$')#, fontsize=axeslabelsize)
#                 ax[1, i].set_xlabel(r'$t$ (days)')#, fontsize=axeslabelsize)
            else:
                ax.set_ylabel(r'$m$')#, fontsize=axeslabelsize)
                ax.set_xlabel('Time (days)')#, fontsize=axeslabelsize)
#                 ax[1].set_ylabel(r'$m_{\mathrm{obs}} - m_{\mathrm{pred}}$')#, fontsize=axeslabelsize)
#                 ax[1].set_xlabel(r'$t$ (days)')#, fontsize=axeslabelsize)
        else:
            if show_separate_bands:
                ax[i].set_ylabel(r'$m_{\mathrm{n}}$')#, fontsize=axeslabelsize)
                ax[i].set_xlabel(r'$t_{\mathrm{n}}$ (days)')#, fontsize=axeslabelsize)
#                 ax[1, i].set_ylabel(r'$m_{\mathrm{obs, n}} - m_{\mathrm{pred, n}}$')#, fontsize=axeslabelsize)
#                 ax[1, i].set_xlabel(r'$t_{\mathrm{n}}$ (days)')#, fontsize=axeslabelsize)
            else:
                ax.set_ylabel(r'$m_{\mathrm{n}}$')#, fontsize=axeslabelsize)
                ax.set_xlabel(r'$t_{\mathrm{n}}$ (days)')#, fontsize=axeslabelsize)
#                 ax[1].set_ylabel(r'$m_{\mathrm{obs, n}} - m_{\mathrm{pred, n}}$')#, fontsize=axeslabelsize)
#                 ax[1].set_xlabel(r'$t_{\mathrm{n}}$ (days)')#, fontsize=axeslabelsize)

#         if i == 0:
#             ax[0, i].set_title(key+r'; $g$ filter', fontsize=titlesize)
#         elif i == 1:
#             ax[0, i].set_title(key+r'; $r$ filter', fontsize=titlesize)

        if show_separate_bands:
            pass
#             if i == 0:
#                 ax[i].set_title(r'$g$')#, fontsize=titlesize)
#             elif i == 1:
#                 ax[i].set_title(r'$r$')#, fontsize=titlesize)

        if show_separate_bands:
            ax[i].invert_yaxis()
#             ax[1, i].invert_yaxis()
        else:
            if i == 0:
                ax.invert_yaxis()
#                 ax[1].invert_yaxis()

    if show_separate_bands:
        for a in ax:
            a.tick_params(axis='x')#, labelsize=ticklabelsize, width=2, length=10)
            a.tick_params(axis='y')#, labelsize=ticklabelsize, width=2, length=10)
    else:
        ax.tick_params(axis='x')#, labelsize=ticklabelsize, width=2, length=10)
        ax.tick_params(axis='y')#, labelsize=ticklabelsize, width=2, length=10)

    if show_legend:
        if show_separate_bands:
            legend = ax[1].legend(fancybox=False, edgecolor='black')#, fontsize=ticklabelsize)
            legend.get_frame().set_linewidth(0.5)
            for a in ax:
                set_axes_style(a)
        else:
            g_patch = mpatches.Patch(color=colors[0], label=r'$g$')
            r_patch = mpatches.Patch(color=colors[1], label=r'$r$')

            legend = ax.legend(
                handles=[g_patch, r_patch], #fontsize=ticklabelsize,
                loc=legend_loc, fancybox=False, edgecolor='black'
            )
            legend.get_frame().set_linewidth(0.5)
#         set_axes_style(ax)

#     # Remove residual plot
#     if show_separate_bands:
#         fig.delaxes(ax[1, 0])
#         fig.delaxes(ax[1, 1])
#     else:
#         fig.delaxes(ax[1])

    return ax

def quick_plot_decoded_lc(
    decoded_lcs_test, df_alerts, objectId, min_max_mags,
    fig_width=15, fig_height=12,
    show_legend=True, legend_loc='best',
    whichclass=None, show_only_rmse=False,
):
    """
    Plot interpolated light curve using objectId

    Requires `fine_classes` variable to be defined.
    See one of the above cells.

    If whichclass is None, will use `fine_classes`
    which must be globally defined.
    """
    test_currObjId_index = np.where(test_objIds == objectId)
    key = list(decoded_lcs_test.keys())[test_currObjId_index[0][0]]
    ax = plot_decoded_lc(
        decoded_lcs_test, key=key, df_alerts=df_alerts, objId=objectId,
        unnormalize_lcs=True, min_time=min_time, max_time=max_time,
        min_max_mags=min_max_mags, prob_plot=False,
        whichclass=fine_classes[test_currObjId_index[0][0]] if whichclass is None else whichclass,
        also_show_sigmapsf_and_rmse=False, show_only_rmse=show_only_rmse,
        show_separate_bands=False, fig_width=fig_width, fig_height=fig_height, show_legend=show_legend,
        legend_loc=legend_loc
    )

    return ax

def get_residuals_from_decoded_lc(
    pdfs_by_objId,
    decoded_lcs, key, num_filters=2, objId=None,
    unnormalize_lcs=True, min_time=None, max_time=None,
    min_max_mags=None, prob_plot=True, whichclass=None, fid_column='fid',
    sigmapsf_column='sigmapsf', objectId_column='objectId',
    magpsf_column='magpsf',
):
    pred_x, observed_data, observed_mask, observed_times = decoded_lcs[key]

#     print(pred_x.shape, observed_data.shape, observed_mask.shape, observed_times.shape)

    assert np.all(observed_times[:, 0] == observed_times[:, 1])  # Because we simply repeated the dimension in the evaluation_unsupervised.py code.
    observed_times = observed_times[:, 0]
    # Set to NaN to ensure non-essential time values for this lc are not shown.
    observed_times[observed_times == 0.0] = np.nan
    # Recall that we use make_first_time_zero=True in prepare_data (see get_lc), so the first time value will be zero
    # But we don't want that first time value to be set to NaN. So we undo that below.
    observed_times[0] = 0.0

    if unnormalize_lcs:
        if min_max_mags is None:
            raise ValueError(
                "min_max_mags must be provided if unnormalize_lcs is True!"
            )
        if objId is None:
            raise ValueError("objId must be provided if unnormalize_lcs is True!")

        # Extract the row corresponding to the min/max of the object ID.
        row = min_max_mags[np.where(min_max_mags[:,0] == objId)].squeeze()

        if min_time is None or max_time is None:
            raise ValueError(
                "min_time and max_time must be provided if unnormalize_lcs is True!"
            )

        # Since we normalize times using min/max of whole train data,
        # min_time and max_time must represent that.
        observed_times = unnormalize_time(observed_times, min_time, max_time)
        # Since we normalize mags using min/max of the light curve itself,
        # min_mag and max_mag must represent that.
        observed_data = unnormalize_mag(
            observed_data, observed_mask, float(row[1]), float(row[2])
        )
        # Pass a mask of all ones for prediction because for
        # interpolation all points can be considered observed.
        pred_x = unnormalize_mag(
            pred_x, np.ones_like(observed_mask), float(row[1]), float(row[2])
        )

        # pdf = df_alerts[df_alerts[objectId_column] == objId].compute()
        pdf = pdfs_by_objId[objId]


    residuals, sigmapsf_list = [], []
    for i in range(num_filters):
        obs_to_show = observed_data[:, i].copy()

        # If `unnormalize_mag` was True, unobserved mags would have been
        # set to nan. So instead set them to zero to be able to calculate
        # metrics and not give nan as output.
        obs_to_show[observed_mask[:, i] == 0.0] = 0.

        rmse = np.sqrt(
            mean_squared_error(obs_to_show, pred_x[:, i], observed_mask[:, i])
        )

        # Set unobserved values to NaN so that they don't show up in the plot.
        obs_to_show[observed_mask[:, i] == 0.0] = np.nan

        resid = obs_to_show - pred_x[:, i]
        resid_without_nan = resid[~np.isnan(resid)]

        maskFilt = pdf[fid_column] == (i+1)
        assert len(pdf[maskFilt]) == observed_mask[:, i].sum() == len(resid_without_nan)

        sigmapsf = pdf[maskFilt][sigmapsf_column]

        residuals.append(resid_without_nan)
        sigmapsf_list.append(sigmapsf)

    return np.hstack(residuals), np.hstack(sigmapsf_list)

from utils import normalize_time_values, get_lc

In [ ]:
# NOTE: Need to change min_time and max_time if a different dataset is used.

# NOTE: Check value of min_time and max_time when running main_processing.py - the value for the test set must match what's below here.
# max_time: calculated across the test dataset using the SN+AGN data (`ftransfer_ztf_2024-05-27_433174_copy`).
# min_time: calculated across the test dataset. This will be zero since we set first time to zero.
train_min_max_times = np.load('train_min_max_times.npy')
min_time, max_time = train_min_max_times[0], train_min_max_times[1]
min_max_mags = np.load('min_max_mags.npy')

In [ ]:
############################################################################################
# Each row in `total_objIds` and `total_common_finkclasses` correspond to the same objectId.
############################################################################################

test_outputs_condensed = np.load('evaluate_test_outputs_condensed.npy')
total_objIds = np.load('total_objIds.npy')
# total_objIds_encoded = np.load('total_objIds_encoded.npy')
total_common_finkclasses = np.load('total_common_finkclasses.npy', allow_pickle=True)
train_objIds = np.load('train_objIds.npy')
val_objIds = np.load('val_objIds.npy')
test_objIds = np.load('evaluate_test_objIds_dataloader.npy')  # it's order matches test_outputs_condensed
test_outputs_condensed.shape, total_objIds.shape, total_common_finkclasses.shape, train_objIds.shape, val_objIds.shape, test_objIds.shape

In [ ]:
np.any(test_outputs_condensed == -999), np.any(np.isnan(test_outputs_condensed))

In [ ]:
import dask.dataframe as dd

df_alerts = dd.read_parquet(
    'alerts_processed_ftransfer_ztf_2025_05_31_430518.parquet',
    columns=[
                'objectId', 'finkclass', 'tnsclass',
                 'fid', 'magpsf', 'sigmapsf', 'jd'
            ]
)
df_alerts.shape

In [ ]:
# Read the raw alerts.
# NOTE: This will be a large dataframe depending on how many alerts were used and will require large RAM.
# That's why we use dask to read it here.
df_alerts_raw = dd.read_parquet(
    'ftransfer_ztf_2025-05-31_430518/',
    columns=['objectId', 'finkclass', 'tnsclass']
)

In [ ]:
_x = df_alerts_raw['finkclass'].compute().value_counts()
print(_x, _x.sum())

In [ ]:
_x = df_alerts_raw['tnsclass'].compute().value_counts()
print(_x, _x.sum())

In [ ]:
df_alerts['finkclass'].compute().value_counts()

In [ ]:
df_alerts['tnsclass'].compute().value_counts()

In [ ]:
objIds_custom_agn_sn = df_alerts_raw['objectId'].unique()

# Get those alerts having finkclass in agn_list and tnsclass in sn_list at the same time.
# df_alerts_raw_subset = df_alerts_raw[(df_alerts_raw['finkclass'].isin(agn_list)) & (df_alerts_raw['tnsclass'].isin(sn_list))]

print(f'no. of transients in raw set of alerts = {len(objIds_custom_agn_sn)}')

# Get only those transients that are present in the final (processed) alerts dataframe since we only use those.
# objIds_custom_agn_sn = set(objIds_custom_agn_sn).intersection(df_alerts['objectId'].unique())
# print(f'no. of transients after removing those only in the raw alerts (not in the processed alerts)  = {len(objIds_custom_agn_sn)}')
# objIds_custom_agn_sn_final = []
# for o in df_alerts['objectId'].unique():
#     # Get the most common finkclass for all alerts of the current transient.
#     curr_transient_alerts = df_alerts_raw_subset[df_alerts_raw_subset['objectId'] == o]
#     s = curr_transient_alerts.shape
#     print(s[0].compute(), s[1])
#     if (
#         len(
#             curr_transient_alerts[curr_transient_alerts['finkclass'].isin(agn_list)]
#         ) >= 1
#         and
#         len(
#             curr_transient_alerts[curr_transient_alerts['tnsclass'].isin(sn_list)]
#         ) >= 1
#     ):
#         objIds_custom_agn_sn_final.append(o)

In [ ]:
df_alerts_pandas = df_alerts.compute()

In [ ]:
test_finkclasses = []
for tobjId in test_objIds:
    idx = np.where(total_objIds == tobjId)[0][0]
#     if tobjId in objIds_custom_agn_sn_final:
#         test_finkclasses.append(np.array(['agn_sn']))
#     else:
    test_finkclasses.append(total_common_finkclasses[idx])

test_finkclasses = np.array(test_finkclasses)

train_finkclasses = []
for tobjId in train_objIds:
    idx = np.where(total_objIds == tobjId)[0][0]
#     if tobjId in objIds_custom_agn_sn_final:
#         train_finkclasses.append(np.array(['agn_sn']))
#     else:
    train_finkclasses.append(total_common_finkclasses[idx])

train_finkclasses = np.array(train_finkclasses)

val_finkclasses = []
for tobjId in val_objIds:
    idx = np.where(total_objIds == tobjId)[0][0]
#     if tobjId in objIds_custom_agn_sn_final:
#         val_finkclasses.append(np.array(['agn_sn']))
#     else:
    val_finkclasses.append(total_common_finkclasses[idx])

val_finkclasses = np.array(val_finkclasses)

print(len(test_finkclasses), len(train_finkclasses), len(val_finkclasses))

test_finkclasses_flatten = test_finkclasses.flatten()

fig, ax = plt.subplots(1, 1, figsize=(16, 5))
test_finkclasses_stats = pd.Series(
    test_finkclasses_flatten[test_finkclasses_flatten != '0'].flatten()  # !=0 because those were added just for apdding.
).value_counts()
test_finkclasses_stats.plot(kind='bar', ax=ax, rot=0)
ax.set_yscale('log')
ax.set_title('Test', fontsize=titlesize)
ax.tick_params(axis='x', labelsize=ticklabelsize, width=2, length=10)
ax.tick_params(axis='y', labelsize=ticklabelsize, width=2, length=10)
plt.show()

print(test_finkclasses_stats)

train_finkclasses_flatten = train_finkclasses.flatten()

fig, ax = plt.subplots(1, 1, figsize=(16, 5))
train_finkclasses_stats = pd.Series(
    train_finkclasses_flatten[train_finkclasses_flatten != '0'].flatten()  # !=0 because those were added just for apdding.
).value_counts()
train_finkclasses_stats.plot(kind='bar', ax=ax, rot=0)
ax.set_yscale('log')
ax.set_title('Train', fontsize=titlesize)
ax.tick_params(axis='x', labelsize=ticklabelsize, width=2, length=10)
ax.tick_params(axis='y', labelsize=ticklabelsize, width=2, length=10)
plt.show()

print(train_finkclasses_stats)

val_finkclasses_flatten = val_finkclasses.flatten()

fig, ax = plt.subplots(1, 1, figsize=(16, 5))
val_finkclasses_stats = pd.Series(
    val_finkclasses_flatten[val_finkclasses_flatten != '0'].flatten()  # !=0 because those were added just for apdding.
).value_counts()
val_finkclasses_stats.plot(kind='bar', ax=ax, rot=0)
ax.set_yscale('log')
ax.set_title('Validation', fontsize=titlesize)
ax.tick_params(axis='x', labelsize=ticklabelsize, width=2, length=10)
ax.tick_params(axis='y', labelsize=ticklabelsize, width=2, length=10)
plt.show()

print(val_finkclasses_stats)

In [ ]:
# test_objIds_early_SNIa = []
# for tobjId in test_objIds:
#     idx = np.where(total_objIds == tobjId)[0][0]
#     if total_common_finkclasses[idx] == 'Early SN Ia candidate':
#         test_objIds_early_SNIa.append(tobjId)

# for name in test_objIds_early_SNIa:
#     plot_lc(
#         df_alerts_pandas, name, annotate_finkclass=True
#     )

### Correction

In [ ]:
################ CORRECT AGN lIGHT CURVES ################
correct_lcs_for_agns = True

agn_indicator = (test_finkclasses == 'custom_agn').astype(bool).squeeze()
min_max_mags = np.load('min_max_mags.npy')
test_objIds = np.load('evaluate_test_objIds_dataloader.npy')
test_outputs_condensed = np.load('evaluate_test_outputs_condensed.npy')

if correct_lcs_for_agns:
    test_outputs_condensed_corrected = np.load('evaluate_test_CORRECTED_outputs_condensed.npy')
    test_objIds_corrected = np.load('evaluate_test_CORRECTED_objIds_dataloader.npy')  # it's order matches test_outputs_condensed
#     min_max_mags_corrected = np.load('min_max_mags_ftransfer_ztf_2025_05_31_430518_CORRECTED.npy')

    assert np.all(test_objIds == test_objIds_corrected)
    # Start from original, replace only AGN rows with corrected
    test_outputs_condensed_corrected_agn = test_outputs_condensed.copy()
    test_objIds_corrected_agn = test_objIds.copy()
#     min_max_mags_corrected_agn = min_max_mags.copy()
    test_outputs_condensed_corrected_agn[agn_indicator] = test_outputs_condensed_corrected[agn_indicator]
#     min_max_mags_corrected_agn[agn_indicator] = min_max_mags_corrected[agn_indicator]

    assert np.all(test_objIds_corrected == test_objIds)
    assert np.all(test_objIds_corrected_agn == test_objIds)
    np.testing.assert_array_equal(test_outputs_condensed_corrected_agn[agn_indicator], test_outputs_condensed_corrected[agn_indicator])
    np.testing.assert_array_equal(test_outputs_condensed_corrected_agn[~agn_indicator], test_outputs_condensed[~agn_indicator])

#     np.testing.assert_array_equal(min_max_mags_corrected_agn[agn_indicator], min_max_mags_corrected[agn_indicator])
#     np.testing.assert_array_equal(min_max_mags_corrected_agn[~agn_indicator], min_max_mags[~agn_indicator])

    print(f"Replaced {agn_indicator.sum()} AGN rows out of {len(agn_indicator)} total")

use_corrected_agns = True
if use_corrected_agns:
    assert np.all(
        np.load('evaluate_test_objIds_dataloader.npy') == test_objIds_corrected_agn
    )
    test_objIds = test_objIds_corrected_agn
    test_outputs_condensed = test_outputs_condensed_corrected_agn
    min_max_mags = np.load('min_max_mags.npy')
    min_max_mags_corrected = np.load('min_max_mags_ftransfer_ztf_2025_05_31_430518_CORRECTED.npy')
else:
    test_objIds  = np.load('evaluate_test_objIds_dataloader.npy')
    test_outputs_condensed = np.load('evaluate_test_outputs_condensed.npy')
    min_max_mags = np.load('min_max_mags.npy') 

In [ ]:
agns_with_less_than_3months_ids = np.load('agns_with_less_than_3months_ids.npy', allow_pickle=True)
test_agns_with_less_than_3months_ids_mask = np.isin(
    test_objIds, agns_with_less_than_3months_ids
).astype(bool)
test_data_with_gte_3months_ids_mask = ~test_agns_with_less_than_3months_ids_mask
assert test_data_with_gte_3months_ids_mask.sum() + test_agns_with_less_than_3months_ids_mask.sum() == len(test_objIds)

In [ ]:
test_data_with_gte_3months_ids_mask.sum(), test_agns_with_less_than_3months_ids_mask.sum()

# 2D Projection of encoded representations

In [ ]:
(test_outputs_condensed == -99999).any()

In [ ]:
test_outputs_condensed.shape

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.axes_grid1 import make_axes_locatable
import umap
from sklearn.manifold import TSNE

# Since test_outputs_condensed is
# (num_test_examples, num_sample, max_num_ref_points_test_set, number_of_filters),
# we are averaging over the num_sample dimension.
test_outputs_condensed_condensed = test_outputs_condensed.mean(1)
# test_outputs_condensed_condensed = test_outputs_condensed_condensed[:, :6]  # Because most light curves have ~<10 datapoints. See the histogram below.
# test_outputs_condensed_condensed = test_outputs_condensed[:, 0, :, :].mean(2)

##############################################
# # Band-specific embeddings
# test_outputs_condensed_condensed_g = test_outputs_condensed_condensed.reshape(
#     test_outputs_condensed_condensed.shape[0],
#     test_outputs_condensed_condensed.shape[1],
#     0
# )
# test_outputs_condensed_condensed_r = test_outputs_condensed_condensed.reshape(
#     test_outputs_condensed_condensed.shape[0],
#     test_outputs_condensed_condensed.shape[1],
#     1
# )
# test_outputs_condensed_condensed_g = np.nan_to_num(test_outputs_condensed_condensed_g, nan=0.)
# test_outputs_condensed_condensed_r = np.nan_to_num(test_outputs_condensed_condensed_r, nan=0.)
##############################################

print(
    np.count_nonzero(test_outputs_condensed_condensed),
    test_outputs_condensed_condensed.size,
    test_outputs_condensed_condensed.shape
)

# Flatten across latent_dim dimension.
test_outputs_condensed_condensed = test_outputs_condensed_condensed.transpose(0, 2, 1).reshape(test_outputs_condensed_condensed.shape[0], -1)

# Each light curve will have different number of query points due to our approach of
# using custom query for each light curve. Thus, we set the values to nan in
# `evaluate_unsupervised.py`. Here we replace them to zero for projection visualization. 
test_outputs_condensed_condensed = np.nan_to_num(test_outputs_condensed_condensed, nan=0.)

print(
    np.count_nonzero(test_outputs_condensed_condensed),
    test_outputs_condensed_condensed.size,
    test_outputs_condensed_condensed.shape
)

agn_indicator = (test_finkclasses == 'custom_agn').astype(bool).squeeze()
# stars_indicator = (test_finkclasses == 'custom_stars').astype(int).squeeze()
# simbad_galaxies_indicator = np.apply_along_axis(lambda x: 1 if len([i for i in x if i in simbad_galaxies_list]) > 0 else 0, 1, test_finkclasses)
# cataclyv_indicator = np.apply_along_axis(lambda x: 1 if len([i for i in x if i in ['CataclyV*']]) > 0 else 0, 1, test_finkclasses)
# tns_tde_indicator = np.apply_along_axis(lambda x: 1 if len([i for i in x if i in ['TNS (TDE)']]) > 0 else 0, 1, test_finkclasses)
sn_indicator = (test_finkclasses == 'custom_sn').astype(int).squeeze()
early_sn1a_indicator = (test_finkclasses == 'Early SN Ia candidate').astype(int).squeeze()

colors = ['#1C9099', '#FC9272', 'gray']

pca = False
umap_use = True
tsne = False

# test_outputs_condensed_condensed_scaled = StandardScaler().fit_transform(test_outputs_condensed_condensed)

################# Optional #################
# def average_consecutive(matrix, x):
#     K, N = matrix.shape
#     result = []

#     for row in matrix:
#         row_avg = []
#         for i in range(0, N, x):
#             chunk = row[i:i + x]
#             row_avg.append(np.mean(chunk))
# #             row_avg.append(chunk[0])
#         result.append(row_avg)

#     return np.array(result)

# print(test_outputs_condensed_condensed.shape)
# test_outputs_condensed_condensed = average_consecutive(
#     test_outputs_condensed_condensed, 2
# )
# print(test_outputs_condensed_condensed.shape)
############################################

if pca:
    test_outputs_condensed_condensed_transformed = PCA(n_components=2).fit_transform(test_outputs_condensed_condensed)
elif umap_use:
    umap_fit = umap.UMAP(random_state=42, metric='cosine')
    test_outputs_condensed_condensed_transformed = umap_fit.fit_transform(test_outputs_condensed_condensed)
elif tsne:
    test_outputs_condensed_condensed_transformed = TSNE(random_state=42).fit_transform(test_outputs_condensed_condensed)

fig, ax = plt.subplots(1, 1, figsize=(15, 9))
if sum(agn_indicator) != 0:
    ax.scatter(
        test_outputs_condensed_condensed_transformed[:, 0][agn_indicator==1], test_outputs_condensed_condensed_transformed[:, 1][agn_indicator==1],
        c=colors[2], label='AGN', s=150, alpha=0.3
    )
if sum(sn_indicator != 0):
    ax.scatter(
        test_outputs_condensed_condensed_transformed[:, 0][sn_indicator==1], test_outputs_condensed_condensed_transformed[:, 1][sn_indicator==1],
        c=colors[0], label='SN', s=150, alpha=0.3
    )
if sum(early_sn1a_indicator != 0):
    ax.scatter(
        test_outputs_condensed_condensed_transformed[:, 0][early_sn1a_indicator==1], test_outputs_condensed_condensed_transformed[:, 1][early_sn1a_indicator==1],
        c=colors[1], label='Early SN Ia', s=150, alpha=0.3
    )
# if sum(stars_indicator) != 0:
#     ax.scatter(
#         test_outputs_condensed_condensed_transformed[:, 0][stars_indicator==1], test_outputs_condensed_condensed_transformed[:, 1][stars_indicator==1],
#         c=colors[1], label='stars_list', s=150, alpha=0.7
#     )

# ax.scatter(
#     test_outputs_condensed_condensed_transformed[:, 0][simbad_galaxies_indicator==1], test_outputs_condensed_condensed_transformed[:, 1][simbad_galaxies_indicator==1],
#     c=colors[2], label='simbad_galaxies_list', s=150, alpha=0.7
# )

# if sum(cataclyv_indicator != 0):
#     ax.scatter(
#         test_outputs_condensed_condensed_transformed[:, 0][cataclyv_indicator==1], test_outputs_condensed_condensed_transformed[:, 1][cataclyv_indicator==1],
#         c=colors[3], label='CataclyV*', s=150, alpha=0.7
#     )
# if sum(tns_tde_indicator != 0):
#     ax.scatter(
#         test_outputs_condensed_condensed_transformed[:, 0][tns_tde_indicator==1], test_outputs_condensed_condensed_transformed[:, 1][tns_tde_indicator==1],
#         c=colors[4], label='TNS (TDE)', s=150, alpha=0.7
#     )

# if sum(agn_sn_indicator != 0):
#     ax.scatter(
#         test_outputs_condensed_condensed_transformed[:, 0][agn_sn_indicator==1], test_outputs_condensed_condensed_transformed[:, 1][agn_sn_indicator==1],
#         c=colors[5], label='SNe', s=150, alpha=0.7
#     )
ax.legend(fontsize=ticklabelsize, frameon=True)
ax.set_xlabel('Projection dim. 1', fontsize=axeslabelsize)
ax.set_ylabel('Projection dim. 2', fontsize=axeslabelsize)
ax.tick_params(axis='x', labelsize=ticklabelsize)
ax.tick_params(axis='y', labelsize=ticklabelsize)
set_axes_style(ax)

In [ ]:
agn_indicator.sum(), sn_indicator.sum(), early_sn1a_indicator.sum()

In [ ]:
# df = pd.read_parquet('ftransfer_ztf_2026-02-27_723419/')
# # std per object
# obj_std = df.groupby('objectId')['magnr'].std().reset_index()

# # attach finkclass (take first per object)
# obj_class = df[['objectId', 'finkclass']].drop_duplicates()

# # merge
# obj_std = obj_std.merge(obj_class, on='objectId')

# # now aggregate per class (e.g. mean std per class)
# class_std = obj_std.groupby('finkclass')['magnr'].mean().reset_index()

# import matplotlib.pyplot as plt

# plt.figure()
# plt.bar(class_std['finkclass'], class_std['magnr'])
# plt.xlabel('finkclass')
# plt.ylabel('Mean std of magnr')
# plt.xticks(rotation=90)
# plt.tight_layout()
# plt.show()

In [ ]:
import pandas as pd
import numpy as np

# Load the necessary arrays (assuming agn_indicator is already loaded in your environment)
test_objIds = np.load('evaluate_test_objIds_dataloader.npy')

# Load the required DataFrame columns
_df_alerts_corrected = pd.read_parquet(
    'alerts_processed_ftransfer_ztf_2025_05_31_430518_CORRECTED.parquet',
    columns=['objectId', 'corrected', 'used_magpsf_corr', 'magpsf_corr']
)

###
# 1. Get unique objectIds for AGNs in your test set
test_agn_ids = set(test_objIds[agn_indicator])

# 2. Get unique objectIds in the DataFrame that have at least one magpsf_corr == 100.0
# .unique() shrinks the data size immediately
# df_ids_with_100 = set(_df_alerts_corrected.loc[_df_alerts_corrected['magpsf_corr'] == 100.0, 'objectId'])

# # 3. Find the intersection (objects that appear in BOTH sets)
# unique_agn_100_count = len(test_agn_ids & df_ids_with_100)

# print(f"Unique test-set AGNs with magpsf_corr == 100.0: {unique_agn_100_count}")

# agn_100_ids = test_agn_ids & df_ids_with_100
# print(list(agn_100_ids))
###

# ---------------------------------------------------------
# 1. Assertion: Ensure uniform used_magpsf_corr per objectId
# ---------------------------------------------------------
# .nunique() calculates unique values per group. max() must be <= 1.
# observed=True is used in case objectId is a categorical type.
unique_counts = _df_alerts_corrected.groupby('objectId', observed=True)['used_magpsf_corr'].nunique()
assert unique_counts.max() <= 1, "Assertion failed: Some objectIds have mixed used_magpsf_corr values!"

# ---------------------------------------------------------
# 2. Create the Boolean Array for all test_objIds
# ---------------------------------------------------------
# Since values are uniform per object, we can safely drop duplicates to create a fast lookup map
obj_to_used_map = (
    _df_alerts_corrected
    .drop_duplicates(subset=['objectId'])
    .set_index('objectId')['used_magpsf_corr']
)

# Map the lookup Series to the NumPy array. 
test_used_magpsf_corr_arr = pd.Series(test_objIds).map(
    obj_to_used_map).to_numpy(dtype=bool)

# ---------------------------------------------------------
# 3. Print Stats ONLY for the Test Set AGNs
# ---------------------------------------------------------
# Apply the AGN mask to get only the AGN subset
agn_used_magpsf_corr = test_used_magpsf_corr_arr[agn_indicator]

true_count = agn_used_magpsf_corr.sum()
total_count = len(agn_used_magpsf_corr)
false_count = total_count - true_count

print(f"--- Test Set AGN Stats ---")
print(f"AGNs using magpsf_corr (True):  {true_count}")
print(f"AGNs NOT using magpsf_corr (False): {false_count}")
print(f"Total Test Set AGNs:            {total_count}")

# Matching your original tuple output format for the AGN subset:
agn_stats_tuple = (
    (true_count,),   # Shape of true
    (false_count,),  # Shape of false
    (total_count,)   # Shape of total
)
print("Tuple format:", agn_stats_tuple)

In [ ]:
test_objIds[agn_indicator][~(test_used_magpsf_corr_arr[agn_indicator])]

In [ ]:
## Preparation for using Milliquas instead of SIMBAD labels for AGN
df = pd.read_parquet(
    'alerts_processed_ftransfer_ztf_2025_05_31_430518.parquet',
    columns=['objectId', 'ra', 'dec']
)

# assert np.all(
#     np.array(agn_mask).astype(float) == np.array(agn_indicator).astype(float)
# )

ra_list, dec_list, objectId_list = [], [], []
for oid in test_objIds[agn_indicator.astype(bool)]:
    subset_alerts = df[df['objectId'] == oid]
    for ii in range(len(subset_alerts)):
        ra_list.append(
            subset_alerts['ra'].iloc[ii]
        )
        dec_list.append(
            subset_alerts['dec'].iloc[ii]
        )
        objectId_list.append(
            oid
        )

# assert len(ra_list) == sum(agn_indicator)
# assert len(dec_list) == sum(agn_indicator)
np.save(
    'ra_list_simbad_agn.npy',
    np.array(ra_list)
)
np.save(
    'dec_list_simbad_agn.npy',
    np.array(dec_list)
)
np.save(
    'objectId_list_simbad_agn.npy',
    np.array(objectId_list)
)

In [ ]:
# See `simbad_to_milliquas.ipynb` for how to generate this csv.
milliquas_agn_labels = pd.read_csv('milliquas_types_list.csv')
milliquas_agn_labels['objectId'] = objectId_list
milliquas_agn_labels.head(), milliquas_agn_labels.shape

In [ ]:
milliquas_agn_labels['Type'].str.contains('Q').mean(), \
milliquas_agn_labels['Type'].str.contains('A').mean(), \
milliquas_agn_labels['Type'].str.contains('B').mean(), \
milliquas_agn_labels['Type'].str.contains('K|N').mean()

In [ ]:
# milliquas_agn_labels[milliquas_agn_labels['objectId'] == 'ZTF18acsptos']['Type']

In [ ]:
# milliquas_agn_labels[milliquas_agn_labels['objectId'] == 'ZTF19aaaoiib']['Type']

In [ ]:
def get_mode(series):
    mode = series.mode()
    if mode.size > 1:
        print('Warning: more than one mode present. Selecting first mode, which may be random.')
    return mode[0]
#     if mode.size == 1:
#         return mode[0]
#     return "x"

milliquas_agn_labels['custom_type'] = milliquas_agn_labels.groupby('objectId')['Type'].transform(get_mode)
milliquas_agn_labels.drop(columns=['Type'], inplace=True)

conditions = [
    milliquas_agn_labels['custom_type'].str.contains('Q'),
    milliquas_agn_labels['custom_type'].str.contains('A'),
    milliquas_agn_labels['custom_type'].str.contains('B'),
    milliquas_agn_labels['custom_type'].str.contains('K|N')
]
mclasses = ['QSO', 'AGN', 'BL Lac', 'Type II']

milliquas_agn_labels['custom_type'] = np.select(conditions, mclasses, default='Unknown')

milliquas_agn_labels = milliquas_agn_labels.groupby('objectId', as_index=False).first()

order_map = {v: i for i, v in enumerate(test_objIds[agn_indicator.astype(bool)])}
milliquas_agn_labels = milliquas_agn_labels.sort_values(by='objectId', key=lambda x: x.map(order_map))

milliquas_agn_labels.head()

In [ ]:
milliquas_agn_labels['custom_type'].value_counts()

In [ ]:
# df_alerts[df_alerts['finkclass'] == 'custom_agn'].compute().merge(
#     milliquas_agn_labels, on='objectId'
# )['custom_type'].value_counts(normalize=True)

In [ ]:
print(
    len(test_objIds), \
    len(milliquas_agn_labels), \
    len(test_objIds[np.array(agn_indicator).astype(bool)]), \
    len(test_objIds[(np.array(early_sn1a_indicator).astype(bool)) | (np.array(sn_indicator).astype(bool))])
)
assert len(test_objIds) == \
len(test_objIds[np.array(agn_indicator).astype(bool)]) + \
len(test_objIds[(np.array(early_sn1a_indicator).astype(bool)) | (np.array(sn_indicator).astype(bool))])

In [ ]:
################ More granular classes ################
fine_classes = []
_raw_classes = df_alerts_raw[df_alerts_raw['objectId'].isin(test_objIds)][['objectId', 'finkclass', 'tnsclass']].compute()
for i, oi in enumerate(test_objIds):
    _coarse_finkclass = df_alerts_pandas[df_alerts_pandas['objectId'] == oi]['finkclass'].mode().iloc[0]
    assert _coarse_finkclass == test_finkclasses[i]

    if _coarse_finkclass == 'custom_sn':
        _fine_class = _raw_classes[_raw_classes['objectId'] == oi]['tnsclass'].mode().iloc[0]
    elif _coarse_finkclass == 'custom_agn':
        _fine_class = _raw_classes[_raw_classes['objectId'] == oi]['finkclass'].mode().iloc[0]
    elif _coarse_finkclass == 'Early SN Ia candidate':
        _fine_class = _raw_classes[_raw_classes['objectId'] == oi]['finkclass'].mode().iloc[0]

    fine_classes.append(_fine_class)

print(pd.Series(fine_classes).value_counts())

train_fine_classes = []
_raw_classes_train = df_alerts_raw[df_alerts_raw['objectId'].isin(train_objIds)][['objectId', 'finkclass', 'tnsclass']].compute()
for i, oi in enumerate(train_objIds):
    _coarse_finkclass = df_alerts_pandas[df_alerts_pandas['objectId'] == oi]['finkclass'].mode().iloc[0]
    assert _coarse_finkclass == train_finkclasses[i]

    if _coarse_finkclass == 'custom_sn':
        _fine_class = _raw_classes_train[_raw_classes_train['objectId'] == oi]['tnsclass'].mode().iloc[0]
    elif _coarse_finkclass == 'custom_agn':
         _fine_class = _raw_classes_train[_raw_classes_train['objectId'] == oi]['finkclass'].mode().iloc[0]
    elif _coarse_finkclass == 'Early SN Ia candidate':
        _fine_class = _raw_classes_train[_raw_classes_train['objectId'] == oi]['finkclass'].mode().iloc[0]

    train_fine_classes.append(_fine_class)

print(pd.Series(train_fine_classes).value_counts())
#######################################################

In [ ]:
list(set(fine_classes).difference(train_fine_classes)), \
list(set(train_fine_classes).difference(fine_classes))

In [ ]:
fine_classes_value_counts = pd.Series(fine_classes).value_counts().rename_axis('class').reset_index(name='counts')
train_fine_classes_value_counts = pd.Series(train_fine_classes).value_counts().rename_axis('class').reset_index(name='counts')

fine_classes_value_counts.merge(train_fine_classes_value_counts, on='class', suffixes=('_test', '_train'))

In [ ]:
matplotlib.rcParams["font.size"] = "22"

In [ ]:
from matplotlib.lines import Line2D
import seaborn as sns
rgb_values = sns.color_palette("Paired", 12)
rgb_values2 = sns.color_palette("Set2")

fine_classes = pd.Series(fine_classes)
replace_map = {
    '(TNS) SLSN-I': 'SLSN',
    '(TNS) SLSN-II': 'SLSN',

    '(TNS) SN I': 'SN Ibc',

    '(TNS) SN Ia': 'SN Ia',
    '(TNS) SN Ia-91T-like': 'SN Ia',
    '(TNS) SN Ia-91bg-like': 'SN Ia',
    '(TNS) SN Ia-CSM': 'SN Ia',
    '(TNS) SN Ia-pec': 'SN Ia',
    '(TNS) SN Iax[02cx-like]': 'SN Ia',

    '(TNS) SN Ibn': 'SN Ibc',
    '(TNS) SN Ic': 'SN Ibc',
    '(TNS) SN Ic-BL': 'SN Ibc',

    '(TNS) SN II': 'SN II',
    '(TNS) SN IIP': 'SN II',
    '(TNS) SN IIb': 'SN II',
    '(TNS) SN IIn': 'SN II',

    'Early SN Ia candidate': 'Early SN Ia',

    'AGN': 'AGN',

    'BLLac': 'AGN',
    'Blazar': 'AGN',

    'LINER': 'AGN',

    'QSO': 'AGN',

    'Seyfert': 'AGN',
    'Seyfert_1': 'AGN',
    'Seyfert_2': 'AGN'
}

fine_classes_grouped = fine_classes.replace(replace_map)

color_map = {
#     'SLSN': rgb_values[-3],
    'SN Ia': rgb_values[4],
#     'SN Ibc': rgb_values2[-3],
    'SN II': rgb_values[3],
#     'Early SN Ia': rgb_values[-1],
    'AGN': rgb_values[0],
}
# assert len(color_map.keys()) == len(np.unique(fine_classes_grouped))

fig = plt.figure(figsize=(width*1.5, height))
ax = plt.gca()
ax.scatter(
    test_outputs_condensed_condensed_transformed[:, 0][(fine_classes_grouped != 'SLSN') & (fine_classes_grouped != 'Early SN Ia') & (fine_classes_grouped != 'SN Ibc')],
    test_outputs_condensed_condensed_transformed[:, 1][(fine_classes_grouped != 'SLSN') & (fine_classes_grouped != 'Early SN Ia') & (fine_classes_grouped != 'SN Ibc')],
    c=pd.Series(fine_classes_grouped[(fine_classes_grouped != 'SLSN') & (fine_classes_grouped != 'Early SN Ia') & (fine_classes_grouped != 'SN Ibc')]).map(color_map),
    s=150, alpha=0.4
)
ax.scatter(
    test_outputs_condensed_condensed_transformed[:, 0][fine_classes_grouped == 'SLSN'],
    test_outputs_condensed_condensed_transformed[:, 1][fine_classes_grouped == 'SLSN'],
    s=180, alpha=1., marker='^', color=rgb_values[-3]
)
ax.scatter(
    test_outputs_condensed_condensed_transformed[:, 0][fine_classes_grouped == 'Early SN Ia'],
    test_outputs_condensed_condensed_transformed[:, 1][fine_classes_grouped == 'Early SN Ia'],
    s=180, alpha=1., marker='v', color=rgb_values[-1]
)
ax.scatter(
    test_outputs_condensed_condensed_transformed[:, 0][fine_classes_grouped == 'SN Ibc'],
    test_outputs_condensed_condensed_transformed[:, 1][fine_classes_grouped == 'SN Ibc'],
    s=180, alpha=0.6, marker='s', color=rgb_values2[-3]
)

handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=v, label=k, markersize=20) \
    for k, v in color_map.items()
]
handles += [
    Line2D([0], [0], marker='^', color='w', markerfacecolor=rgb_values[-3], label='SLSN', markersize=20)
]
handles += [
    Line2D([0], [0], marker='v', color='w', markerfacecolor=rgb_values[-1], label='Early SN Ia', markersize=20)
]
handles += [
    Line2D([0], [0], marker='s', color='w', markerfacecolor=rgb_values2[-3], label='SN Ibc', markersize=20)
]

legend = ax.legend(
    handles=handles, bbox_to_anchor=(1, 1),
    loc='upper left', #fontsize=ticklabelsize,
    edgecolor='black'
)
legend.get_frame().set_linewidth(0.5)

ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

set_axes_style(ax)

save_fig('projection_classes.pdf')
# ===============================================================================

# import seaborn as sns

# if sum(agn_indicator) != 0:
#     ax1 = sns.jointplot(
#         x=test_outputs_condensed_condensed_transformed[:, 0][agn_indicator==1], y=test_outputs_condensed_condensed_transformed[:, 1][agn_indicator==1],
#         color=colors[0], label='AGN', kind='hex', mincnt=1
#     )
# if sum(sn_indicator != 0):
#     ax2 = sns.jointplot(
#         x=test_outputs_condensed_condensed_transformed[:, 0][sn_indicator==1], y=test_outputs_condensed_condensed_transformed[:, 1][sn_indicator==1],
#         color=colors[1], label='SN', kind='hex', mincnt=1
#     )
# if sum(early_sn1a_indicator != 0):
#     ax3 = sns.jointplot(
#         x=test_outputs_condensed_condensed_transformed[:, 0][early_sn1a_indicator==1], y=test_outputs_condensed_condensed_transformed[:, 1][early_sn1a_indicator==1],
#         color=colors[2], label='Early SN Ia'
#     )

# _min_xlim, _max_xlim, _min_ylim, _max_ylim = np.nan, np.nan, np.nan, np.nan
# _xlims = ax1.ax_joint.get_xlim(), ax2.ax_joint.get_xlim(), ax3.ax_joint.get_xlim()
# _ylims = ax1.ax_joint.get_ylim(), ax2.ax_joint.get_ylim(), ax3.ax_joint.get_ylim()
# _min_xlim = min(_xl[0] for _xl in _xlims)
# _max_xlim = max(_xl[1] for _xl in _xlims)
# _min_ylim = min(_yl[0] for _yl in _ylims)
# _max_ylim = max(_yl[1] for _yl in _ylims)

# for a in [ax1, ax2, ax3]:
#     a.ax_joint.set_xlabel('Projection dim. 1', fontsize=axeslabelsize)
#     a.ax_joint.set_ylabel('Projection dim. 2', fontsize=axeslabelsize)
#     a.ax_joint.tick_params(axis='x', labelsize=ticklabelsize)
#     a.ax_joint.tick_params(axis='y', labelsize=ticklabelsize)
#     a.ax_joint.set_xlim([_min_xlim, _max_xlim])
#     a.ax_joint.set_ylim([_min_ylim, _max_ylim])

In [ ]:
fine_classes_grouped.value_counts()

In [ ]:
from adjustText import adjust_text

In [ ]:
from matplotlib.lines import Line2D
import seaborn as sns
rgb_values = sns.color_palette("Paired", 12)
rgb_values2 = sns.color_palette("Set2")

fine_classes = pd.Series(fine_classes)

replace_map_finer = {
    '(TNS) SLSN-I': 'SLSN',
    '(TNS) SLSN-II': 'SLSN',

    '(TNS) SN I': 'SN I',

    '(TNS) SN Ia': 'SN Ia',
    '(TNS) SN Ia-91T-like': 'SN Ia',
    '(TNS) SN Ia-91bg-like': 'SN Ia',
    '(TNS) SN Ia-CSM': 'SN Ia',
    '(TNS) SN Ia-pec': 'SN Ia',
    '(TNS) SN Iax[02cx-like]': 'SN Ia',

    '(TNS) SN Ibn': 'SN Ibc',
    '(TNS) SN Ic': 'SN Ibc',
    '(TNS) SN Ic-BL': 'SN Ibc',

    '(TNS) SN II': 'SN II',
    '(TNS) SN IIP': 'SN II',
    '(TNS) SN IIb': 'SN II',
    '(TNS) SN IIn': 'SN II',

    'Early SN Ia candidate': 'Early SN Ia',

    'AGN': 'AGN',

    'BLLac': 'Blazar',
    'Blazar': 'Blazar',

    'LINER': 'LINER',

    'QSO': 'QSO',

    'Seyfert': 'Type II',
    'Seyfert_1': 'Type I',
    'Seyfert_2': 'Type II'
}

fine_classes_grouped_finer = fine_classes.replace(replace_map_finer)

assert not any(x in fine_classes_grouped_finer for x in fine_classes)

_SN_classes = ['SLSN', 'SN I', 'SN Ia', 'SN Ibc', 'SN II', 'Early SN Ia']
_AGN_classes = ['QSO', 'Blazar', 'AGN', 'Type I', 'Type II', 'LINER']

sn_color_map = dict(zip(_SN_classes, rgb_values2[:len(_SN_classes)]))
for sn_class in ['SLSN', 'Early SN Ia', 'SN Ibc', 'SN I']:
    sn_color_map.pop(sn_class)
# sn_color_map['SN Ia'] = rgb_values[4]
# sn_color_map['SN II'] = rgb_values[3]
agn_color_map = dict(zip(_AGN_classes, rgb_values2[:len(_AGN_classes)]))
for agn_class in ['Blazar', 'AGN', 'Type I', 'Type II', 'LINER']:
    agn_color_map.pop(agn_class)
sn_mask = [fcgf in _SN_classes for fcgf in fine_classes_grouped_finer]
agn_mask = [fcgf in _AGN_classes for fcgf in fine_classes_grouped_finer]

# SN
fig = plt.figure(figsize=(width*2, height))
ax = plt.gca()

# Show AGN in background first.
# ax.scatter(
#     test_outputs_condensed_condensed_transformed[agn_mask, 0],
#     test_outputs_condensed_condensed_transformed[agn_mask, 1],
#     c='gray', s=150, alpha=0.1
# )
sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[agn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[agn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2
)

ax.scatter(
    test_outputs_condensed_condensed_transformed[sn_mask, 0][(fine_classes_grouped_finer[sn_mask] != 'SLSN') & (fine_classes_grouped_finer[sn_mask] != 'Early SN Ia') & (fine_classes_grouped_finer[sn_mask] != 'SN Ibc') & (fine_classes_grouped_finer[sn_mask] != 'SN I')],
    test_outputs_condensed_condensed_transformed[sn_mask, 1][(fine_classes_grouped_finer[sn_mask] != 'SLSN') & (fine_classes_grouped_finer[sn_mask] != 'Early SN Ia') & (fine_classes_grouped_finer[sn_mask] != 'SN Ibc') & (fine_classes_grouped_finer[sn_mask] != 'SN I')],
    c=pd.Series(fine_classes_grouped_finer[sn_mask][(fine_classes_grouped_finer[sn_mask] != 'SLSN') & (fine_classes_grouped_finer[sn_mask] != 'Early SN Ia') & (fine_classes_grouped_finer[sn_mask] != 'SN Ibc') & (fine_classes_grouped_finer[sn_mask] != 'SN I')]).map(sn_color_map),
    s=150, alpha=0.4
)
ax.scatter(
    test_outputs_condensed_condensed_transformed[:, 0][fine_classes_grouped_finer == 'SLSN'],
    test_outputs_condensed_condensed_transformed[:, 1][fine_classes_grouped_finer == 'SLSN'],
    s=180, alpha=1., marker='^', color=rgb_values[-3]
)
ax.scatter(
    test_outputs_condensed_condensed_transformed[:, 0][fine_classes_grouped_finer == 'Early SN Ia'],
    test_outputs_condensed_condensed_transformed[:, 1][fine_classes_grouped_finer == 'Early SN Ia'],
    s=180, alpha=1., marker='v', color=rgb_values[-1]
)
ax.scatter(
    test_outputs_condensed_condensed_transformed[:, 0][fine_classes_grouped_finer == 'SN Ibc'],
    test_outputs_condensed_condensed_transformed[:, 1][fine_classes_grouped_finer == 'SN Ibc'],
    s=160, alpha=0.6, marker='s', color=rgb_values2[-3]
)
ax.scatter(
    test_outputs_condensed_condensed_transformed[:, 0][fine_classes_grouped_finer == 'SN I'],
    test_outputs_condensed_condensed_transformed[:, 1][fine_classes_grouped_finer == 'SN I'],
    s=160, alpha=0.6, marker='s', color=rgb_values2[-3]
)

handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=v, label=k, markersize=20) \
    for k, v in sn_color_map.items()
]
handles += [
    Line2D([0], [0], marker='s', color='w', markerfacecolor=rgb_values2[-3], label='SN Ibc', markersize=20)
]
handles += [
    Line2D([0], [0], marker='^', color='w', markerfacecolor=rgb_values[-3], label='SLSN', markersize=20)
]
handles += [
    Line2D([0], [0], marker='v', color='w', markerfacecolor=rgb_values[-1], label='Early SN Ia', markersize=20)
]
# handles += [
#     Line2D([0], [0], marker='D', color='w', alpha=0.9, markerfacecolor=rgb_values2[1], label='SN I', markersize=20)
# ]

legend = ax.legend(
    handles=handles, #bbox_to_anchor=(1, 1),
    loc='upper right', #fontsize=ticklabelsize,
    fancybox=False, edgecolor='black'
)
legend.get_frame().set_linewidth(0.5)


ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

xlim = ax.get_xlim()
ylim = ax.get_ylim()


# ######################################################################
# annotation_names = [
#     'ZTF22aaijrul',
#     'ZTF22abfyzir',
#     'ZTF21abzithf',
#     'ZTF21abujejr',
#     'ZTF22aafoqrd',
#     'ZTF21abnujlh',
#     'ZTF21abrgdug',
# #     'ZTF22aafwbey',  # SN Ia variant
# #     'ZTF22abyhvtk',
#     'ZTF22aafuind',  # SN Ia variant
# #     'ZTF22abyrovo',
#     'ZTF22aabwrov',
#     'ZTF19aaapnxn',  # extreme outlier at UMAP_x ~ 5
#     'ZTF21abidrcn',
# #     'ZTF22abeyzcl',
#     'ZTF21abmwzxt',
#     'ZTF21acafqtj',
#     'ZTF22aaayceh',
#     'ZTF22abhsxph',
#     'ZTF22aagvxjc',
#     'ZTF22aafxzfb',
#     'ZTF21abtdvpg',
#     'ZTF22aapkbkl',
#     'ZTF21abcjpnm',
# ]
color_map_ = {
    'SN Ia': sn_color_map['SN Ia'],
    'SN Ibc': rgb_values2[-3],
    'SN II': sn_color_map['SN II'],
    'SLSN': rgb_values[-3],
    'Early SN Ia': rgb_values[-1]
}
selected_SN_IDs = [
#     ('ZTF22aaijrul', 'B1'),
#     ('ZTF22abahblc', 'B1'),
#     ('ZTF22abfyzir', 'B2'),
#     ('ZTF21abzithf', 'B3'),
#     ('ZTF21abujejr', 'B4'),
#     ('ZTF21abnujlh', 'B5'),
#     ('ZTF21abrgdug', 'B6'),
    ('ZTF22aafoqrd', 'SN4'),
    ('ZTF22aabwrov', 'SN5'),
    ('ZTF19aaapnxn', 'SN6'),
    ('ZTF21abidrcn', 'SN7'),
    ('ZTF21abmwzxt', 'SN8'),
    ('ZTF21acafqtj', 'SN9'),
    ('ZTF22abhsxph', 'SN10'),
    ('ZTF22aagvxjc', 'SN11'),
    ('ZTF22aafxzfb', 'SN12'),
    ('ZTF21abtdvpg', 'SN13'),
    ('ZTF21abulpnc', 'SN14'),
#     ('ZTF21abcjpnm', 'C9'),
    ('ZTF22abfvsbt', 'SN1'),
    ('ZTF22aavctvq', 'SN2'),
    ('ZTF22aafuind', 'SN3'),
    ('ZTF22abgroeo', 'SN15'),
#     ('ZTF22aaahuly', 'SN17'),
#     ('ZTF22abieqzx', 'D3'),
#     ('ZTF21abraccv', 'D4'),
#     ('ZTF22aarwwyy', 'D5'),
#     ('ZTF21abuvury', 'D6'),
#     ('ZTF22aaafkvn', 'D7'),
    ('ZTF22aapubuy', 'SN16'),
#     ('ZTF22abcvfgs', 'E1'),
#     ('ZTF22aafumyr', 'E2'),
#     ('ZTF22aaftzcn', 'E3'),
#     ('ZTF20acbcfaa', 'E4'),
#     ('ZTF21accwovq', 'E5'),
    ('ZTF22aaacvyy', 'SN17'),
    ('ZTF20aacbwbm', 'SN18'),
#     ('ZTF21abxdlyk', 'SN20'),
#     ('ZTF22ablepod', 'SN21'),
#     ('ZTF21abqoodm', 'SN22'),
]
texts = []
for annotate_id in selected_SN_IDs:
#     if annotate_id[1][0] not in ['B', 'C', 'G']:
#         continue
    loca = np.where(test_objIds == annotate_id[0])[0]
    coord_loca = test_outputs_condensed_condensed_transformed[loca, :].squeeze()
    color = color_map_[fine_classes_grouped[loca].iloc[0]]

    texts.append(
        ax.text(
            coord_loca[0], coord_loca[1], annotate_id[1], c=color, fontsize=12
        )
    )
#     ax.annotate(
#         annotate_id,
#         test_outputs_condensed_condensed_transformed[np.where(test_objIds == annotate_id), :].squeeze(),
#         (-3, 1), 'data',
#         arrowprops=dict(arrowstyle="-|>",
#         connectionstyle="angle3", lw=1),
#         size=16, ha="center", c='blue'
#     )

texts, arrows = adjust_text(
    texts, ax=ax, only_move={'points':'y', 'texts':'y'},
    arrowprops=dict(arrowstyle="-", color='gray', lw=0.4),
    ensure_inside_axes=False,
    expand=(4, 4),
    objects=[legend, ax.yaxis, ax.xaxis],
    force_explode=(0.5, 2.)
)
for i, (arrow, text) in enumerate(zip(arrows, texts)):
    if selected_SN_IDs[i][1] == 'SN1':
        posA, posB = arrow._posA_posB
        arrow.set_positions((4, 7), None)
        text.set_position((4-0.05, 7-0.05))
    if selected_SN_IDs[i][1] == 'SN2':
        posA, posB = arrow._posA_posB
        arrow.set_positions((4, 6.5), None)
        text.set_position((4-0.05, 6.5-0.05))
    if selected_SN_IDs[i][1] == 'SN3':
        posA, posB = arrow._posA_posB
        arrow.set_positions((7, 5.2), None)
        text.set_position((7-0.05, 5.2-0.05))
    if selected_SN_IDs[i][1] == 'SN8':
        posA, posB = arrow._posA_posB
        arrow.set_positions((6.3, 5), None)
        text.set_position((6.3+0.05, 5-0.05))
    if selected_SN_IDs[i][1] == 'SN7':
        posA, posB = arrow._posA_posB
        arrow.set_positions((5.2, 4.7), None)
        text.set_position((5.2+0.05, 4.7-0.05))
    if selected_SN_IDs[i][1] == 'SN10':
        posA, posB = arrow._posA_posB
        arrow.set_positions((7, 7.5), None)
        text.set_position((7+0.05, 7.5+0.05))
    if selected_SN_IDs[i][1] == 'SN15':
        posA, posB = arrow._posA_posB
        arrow.set_positions((4.5, 4.5), None)
        text.set_position((4.5-0.05, 4.5-0.05))
    if selected_SN_IDs[i][1] == 'SN16':
        posA, posB = arrow._posA_posB
        arrow.set_positions((7.9, 6), None)
        text.set_position((7.9-0.05, 6-0.05))
    if selected_SN_IDs[i][1] == 'SN20':
        posA, posB = arrow._posA_posB
        arrow.set_positions((5.7, 9.3), None)
        text.set_position((5.7+0.05, 9.3+0.05))
    if selected_SN_IDs[i][1] == 'SN21':
        posA, posB = arrow._posA_posB
        arrow.set_positions((4.7, 9.1), None)
        text.set_position((4.7-0.05, 9.1+0.05))
    if selected_SN_IDs[i][1] == 'SN22':
        posA, posB = arrow._posA_posB
        arrow.set_positions((6.5, 8.5), None)
        text.set_position((6.5+0.05, 8.5+0.05))
#     if selected_SN_IDs[i][1] == 'SN19':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((12.1, 8.7), None)
#         text.set_position((12.1+0.05, 8.7-0.05))
#     if selected_SN_IDs[i][1] == 'B1':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((7.75, 1), None)
#         text.set_position((7.75+0.05, 1+0.05))
#     if selected_SN_IDs[i][1] == 'B2':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((6.7, 1.9), None)
#         text.set_position((6.7+0.05, 1.9+0.05))
#     if selected_SN_IDs[i][1] == 'B3':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((6.7, -1.5), None)
#         text.set_position((6.7-0.05, -1.5-0.05))
#     if selected_SN_IDs[i][1] == 'B4':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((3.4, 8.2), None)
#         text.set_position((3.4-0.05, 8.2+0.05))
#     if selected_SN_IDs[i][1] == 'B6':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((6.05, 0.39), None)
#         text.set_position((6.05-0.05, 0.39+0.05))
#     if selected_SN_IDs[i][1] == 'B8':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((4.4, -1.5), None)
#         text.set_position((4.4-0.05, -1.5-0.05))
#     if selected_SN_IDs[i][1] == 'G2':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((7.7, -0.7), None)
#         text.set_position((7.7+0.05, -0.7))
#     if selected_SN_IDs[i][1] == 'G3':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((6.4, 0.75), None)
#         text.set_position((6.4, 0.75-0.05))
#     if selected_SN_IDs[i][1] == 'C1':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((7.9, 0), None)
#         text.set_position((7.9+0.05, 0))
#     if selected_SN_IDs[i][1] == 'C2':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((7.4, 1.4), None)
#         text.set_position((7.4+0.05, 1.4+0.05))
#     if selected_SN_IDs[i][1] == 'C3':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((6.25, 1.8), None)
#         text.set_position((6.25, 1.8+0.05))
#     if selected_SN_IDs[i][1] == 'C4':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((5.8, 2), None)
#         text.set_position((5.8-0.05, 2+0.05))
#     if selected_SN_IDs[i][1] == 'C5':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((5.3, 0.1), None)
#         text.set_position((5.3+0.05, 0.1+0.05))
#     if selected_SN_IDs[i][1] == 'C6':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((3.9, 1.2), None)
#         text.set_position((3.9-0.05, 1.2-0.05))
#     if selected_SN_IDs[i][1] == 'C7':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((4, 0.7), None)
#         text.set_position((4+0.05, 0.7))
    arrow.set_mutation_scale(20)
# ######################################################################

_sn1bc_mask = [fcgf == 'SN Ibc' for fcgf in fine_classes_grouped]
_slsn_mask = [fcgf == 'SLSN' for fcgf in fine_classes_grouped]
_early_sn1a_mask = [fcgf == 'Early SN Ia' for fcgf in fine_classes_grouped]
assert sum(_sn1bc_mask) == 8
assert sum(_slsn_mask) == 7
assert sum(_early_sn1a_mask) == 3

# coords = test_outputs_condensed_condensed_transformed[_sn1bc_mask][np.argsort(test_outputs_condensed_condensed_transformed[_sn1bc_mask, 0])]
# for j, _c in enumerate(coords):
#     ax.text(_c[0]-0.02, _c[1]-0.02, f'{j+1}', fontsize=6, color='#242424')

# coords = test_outputs_condensed_condensed_transformed[_slsn_mask][np.argsort(test_outputs_condensed_condensed_transformed[_slsn_mask, 0])]
# for j, _c in enumerate(coords):
#     ax.text(_c[0]-0.02, _c[1]-0.02, f'{j+1}', fontsize=6, color='white')

# coords = test_outputs_condensed_condensed_transformed[_early_sn1a_mask][np.argsort(test_outputs_condensed_condensed_transformed[_early_sn1a_mask, 0])]
# for j, _c in enumerate(coords):
#     ax.text(_c[0]-0.02, _c[1]+0.01, f'{j+1}', fontsize=6, color='#242424')
########################################################################

set_axes_style(ax)

save_fig('projection_classes_SN.pdf')

In [ ]:
# # def unison_shuffled_copies(a, b):
# #     assert len(a) == len(b)
# #     p = np.random.permutation(len(a))
# #     return a[p], b[p]

# for rs in [42, 101, 432, 575, 1, 582, 11, 94]:
#     test_outputs_condensed_condensed_transformedS = umap.UMAP(random_state=rs, metric='cosine').fit_transform(test_outputs_condensed_condensed)
# #     b1_agn = np.where(test_objIds == 'ZTF18acegzkh')[0][0]
# #     b2_agn = np.where(test_objIds == 'ZTF18abalful')[0][0]
# #     qso_outlier = np.where(test_objIds == 'ZTF21aazltbp')[0][0]
# #     b4_agn = np.where(test_objIds == 'ZTF19aakxiwm')[0][0]
# #     b5_agn = np.where(test_objIds == 'ZTF21abcqqpo')[0][0]
# #     b6_agn = np.where(test_objIds == 'ZTF19abjascw')[0][0]
# #     b7_agn = np.where(test_objIds == 'ZTF18aakeosr')[0][0]
# #     b8_agn = np.where(test_objIds == 'ZTF17aacpvmc')[0][0]
# #     b9_agn = np.where(test_objIds == 'ZTF19aavvvdx')[0][0]
# #     b10_agn = np.where(test_objIds == 'ZTF18abuwcgq')[0][0]
# #     b11_agn = np.where(test_objIds == 'ZTF20aadvpnz')[0][0]
    
#     agn = np.where(test_objIds == 'ZTF18aaieyvr')[0][0]

#     print(
#         test_outputs_condensed_condensed_transformedS[agn, :],
# #         test_outputs_condensed_condensed_transformedS[b1_agn, :],
# #         test_outputs_condensed_condensed_transformedS[b2_agn, :],
# #         test_outputs_condensed_condensed_transformedS[qso_outlier, :],
# #         test_outputs_condensed_condensed_transformedS[b4_agn, :],
# #         test_outputs_condensed_condensed_transformedS[b5_agn, :],
# #         test_outputs_condensed_condensed_transformedS[b6_agn, :],
# #         test_outputs_condensed_condensed_transformedS[b7_agn, :],
# #         test_outputs_condensed_condensed_transformedS[b8_agn, :],
# #         test_outputs_condensed_condensed_transformedS[b9_agn, :],
# #         test_outputs_condensed_condensed_transformedS[b10_agn, :],
# #         test_outputs_condensed_condensed_transformedS[b11_agn, :],
#     )

In [ ]:
# AGN
fig, ax = plt.subplots(1, 1, figsize=(width*2, height))

# Show SN in background first
sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2
)

ax.scatter(
    test_outputs_condensed_condensed_transformed[agn_mask, 0][(fine_classes_grouped_finer[agn_mask] != 'AGN') & (fine_classes_grouped_finer[agn_mask] != 'Blazar') & (fine_classes_grouped_finer[agn_mask] != 'LINER') & (fine_classes_grouped_finer[agn_mask] != 'Type II') & (fine_classes_grouped_finer[agn_mask] != 'Type I')],
    test_outputs_condensed_condensed_transformed[agn_mask, 1][(fine_classes_grouped_finer[agn_mask] != 'AGN') & (fine_classes_grouped_finer[agn_mask] != 'Blazar') & (fine_classes_grouped_finer[agn_mask] != 'LINER') & (fine_classes_grouped_finer[agn_mask] != 'Type II') & (fine_classes_grouped_finer[agn_mask] != 'Type I')],
#     c=pd.Series(fine_classes_grouped_finer[agn_mask][(fine_classes_grouped_finer[agn_mask] != 'AGN') & (fine_classes_grouped_finer[agn_mask] != 'Blazar') & (fine_classes_grouped_finer[agn_mask] != 'LINER') & (fine_classes_grouped_finer[agn_mask] != 'Type II') & (fine_classes_grouped_finer[agn_mask] != 'Type I')]).map(agn_color_map),
    color=sn_color_map['SN Ia'],
    s=150, alpha=0.3
)
ax.scatter(
    test_outputs_condensed_condensed_transformed[:, 0][fine_classes_grouped_finer == 'Type I'],
    test_outputs_condensed_condensed_transformed[:, 1][fine_classes_grouped_finer == 'Type I'],
    s=200, alpha=0.3, color=rgb_values2[4]
)
ax.scatter(
    test_outputs_condensed_condensed_transformed[:, 0][fine_classes_grouped_finer == 'Blazar'],
    test_outputs_condensed_condensed_transformed[:, 1][fine_classes_grouped_finer == 'Blazar'],
    s=200, alpha=0.3, color=rgb_values[-1]
)
ax.scatter(
    test_outputs_condensed_condensed_transformed[:, 0][fine_classes_grouped_finer == 'AGN'],
    test_outputs_condensed_condensed_transformed[:, 1][fine_classes_grouped_finer == 'AGN'],
    s=180, alpha=0.7, marker='v', color=rgb_values2[-3]
)
ax.scatter(
    test_outputs_condensed_condensed_transformed[:, 0][fine_classes_grouped_finer == 'Type II'],
    test_outputs_condensed_condensed_transformed[:, 1][fine_classes_grouped_finer == 'Type II'],
    s=180, alpha=0.7, marker='^', color=rgb_values[-3]
)
ax.scatter(
    test_outputs_condensed_condensed_transformed[:, 0][fine_classes_grouped_finer == 'LINER'],
    test_outputs_condensed_condensed_transformed[:, 1][fine_classes_grouped_finer == 'LINER'],
    s=180, alpha=0.8, marker='s', color='gray'
)

handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=sn_color_map['SN Ia'], label='QSO', markersize=20)
]
handles += [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=rgb_values2[4], label='Type I', markersize=20)
]
handles += [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=rgb_values[-1], label='Blazar', markersize=20)
]
handles += [
    Line2D([0], [0], marker='v', color='w', markerfacecolor=rgb_values2[-3], label='AGN', markersize=20)
]
handles += [
    Line2D([0], [0], marker='^', color='w', markerfacecolor=rgb_values[-3], label='Type II', markersize=20)
]
handles += [
    Line2D([0], [0], marker='s', color='w', markerfacecolor='gray', label='LINER', markersize=20)
]

legend = ax.legend(
    handles=handles, #bbox_to_anchor=(1, 1),
    loc='upper left', #fontsize=ticklabelsize,
    fancybox=False, edgecolor='black'
)
legend.get_frame().set_linewidth(0.5)

ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

ax.set_xlim(xlim)
ax.set_ylim(ylim)
set_axes_style(ax)


color_map_ = {
    'QSO': sn_color_map['SN Ia'],
    'Type I': rgb_values2[4],
    'Blazar': rgb_values[-1],
    'AGN': rgb_values2[-3],
    'Type II': rgb_values[-3],
    'LINER': 'gray'
}
selected_AGN_IDs = [
#     ('ZTF18acegzkh', 'B1'),
#     ('ZTF18abalful', 'B2'),
    ('ZTF21aazltbp', 'B3'),
    ('ZTF19aakxiwm', 'B4'),
#     ('ZTF21abcqqpo', 'B5'),
    ('ZTF19abjascw', 'B6'),
    ('ZTF18aakeosr', 'B7'),
    ('ZTF17aacpvmc', 'B8'),
#     ('ZTF19aavvvdx', 'B9'),
#     ('ZTF18abuwcgq', 'B10'),
#     ('ZTF20aadvpnz', 'B11'),
    ('ZTF17aaadhjr', 'C1'),
    ('ZTF19aagwxlv', 'C2'),
#     ('ZTF18abvivzx', 'C3'),
#     ('ZTF18aarowgi', 'C4'),
#     ('ZTF18abxhyqv', 'C5'),
    ('ZTF18aaagyrv', 'C5'),
#     ('ZTF18abaljmt', 'C6'),
    ('ZTF18abtrubp', 'C7'),
    ('ZTF20acdcick', 'C8'),
#     ('ZTF18aalufuu', 'C9'),
#     ('ZTF18acexzbr', 'C9'),
#     ('ZTF17aaajvds', 'C10'),
    ('ZTF18abnuhuy', 'C11'),
    ('ZTF18aalufuv', 'C12'),
#     ('ZTF22aaseqhf', 'D1'),
#     ('ZTF18aajhwjb', 'D2'),
    ('ZTF19abggdjc', 'D3'),
    ('ZTF22aamptba', 'D4'),
#     ('ZTF18abezosa', 'D5'),
#     ('ZTF18acviazy', 'D6'),
    ('ZTF18acsptos', 'D7'),
    ('ZTF19aaaoiib', 'D8'),
#     ('ZTF18aasszwr', 'D9'),
    ('ZTF20abmdgdn', 'E1'),
    ('ZTF18aaieyvr', 'E2'),
#     ('ZTF19abjprez', 'E3'),
    ('ZTF19abgfgzy', 'E4'),
#     ('ZTF21aaygepu', 'E5'),
#     ('ZTF18absuhfp', 'E6'),
#     ('ZTF18adcbfig', 'E7'),
#     ('ZTF18aaowaij', 'E8'),
#     ('ZTF18acmziob', 'E9'),
]

texts = []
for annotate_id in selected_AGN_IDs:
    loca = np.where(test_objIds == annotate_id[0])[0]
    coord_loca = test_outputs_condensed_condensed_transformed[loca, :].squeeze()
    color = color_map_[fine_classes_grouped_finer[loca].iloc[0]]

    texts.append(
        ax.text(
            coord_loca[0], coord_loca[1], annotate_id[1], c=color, fontsize=12
        )
    )

texts, arrows = adjust_text(
    texts, ax=ax, only_move={'points':'y', 'texts':'y'},
    arrowprops=dict(arrowstyle="-", color='gray', lw=0.4),
    ensure_inside_axes=False,
    expand=(4, 4),
    objects=[legend, ax.yaxis, ax.xaxis],
    force_explode=(0.5, 2.)
)

for i, (arrow, text) in enumerate(zip(arrows, texts)):
    if selected_AGN_IDs[i][1] == 'B4':
        posA, posB = arrow._posA_posB
        arrow.set_positions((4.7, 8.5), None)
        text.set_position((4.7-0.05, 8.5+0.05))
    if selected_AGN_IDs[i][1] == 'B5':
        posA, posB = arrow._posA_posB
        arrow.set_positions((5.5, 8.3), None)
        text.set_position((5.5+0.05, 8.3+0.05))
    if selected_AGN_IDs[i][1] == 'B6':
        posA, posB = arrow._posA_posB
        arrow.set_positions((5.2, 6.5), None)
        text.set_position((5.2+0.05, 6.5-0.05))
    if selected_AGN_IDs[i][1] == 'B8':
        posA, posB = arrow._posA_posB
        arrow.set_positions((5.3, 8.7), None)
        text.set_position((5.3+0.05, 8.7+0.05))
    if selected_AGN_IDs[i][1] == 'B10':
        posA, posB = arrow._posA_posB
        arrow.set_positions((5.8, 6.9), None)
        text.set_position((5.8+0.05, 6.9-0.05))
    if selected_AGN_IDs[i][1] == 'B11':
        posA, posB = arrow._posA_posB
        arrow.set_positions((6.5, 6.5), None)
        text.set_position((6.5+0.05, 6.5+0.05))
    if selected_AGN_IDs[i][1] == 'C1':
        posA, posB = arrow._posA_posB
        arrow.set_positions((9.7, 5.2), None)
        text.set_position((9.7, 5.2-0.05))
    if selected_AGN_IDs[i][1] == 'C2':
        posA, posB = arrow._posA_posB
        arrow.set_positions((8.5, 6), None)
        text.set_position((8.5+0.05, 6-0.05))
    if selected_AGN_IDs[i][1] == 'C3':
        posA, posB = arrow._posA_posB
        arrow.set_positions((10, 5.1), None)
        text.set_position((10+0.05, 5.1-0.05))
    if selected_AGN_IDs[i][1] == 'C4':
        posA, posB = arrow._posA_posB
        arrow.set_positions((9.45, 5.4), None)
        text.set_position((9.45, 5.4-0.05))
    if selected_AGN_IDs[i][1] == 'C5':
        posA, posB = arrow._posA_posB
        arrow.set_positions((10.5, 5.4), None)
        text.set_position((10.5+0.05, 5.4-0.05))
    if selected_AGN_IDs[i][1] == 'C6':
        posA, posB = arrow._posA_posB
        arrow.set_positions((10.5, 5.7), None)
        text.set_position((10.5+0.05, 5.7-0.05))
    if selected_AGN_IDs[i][1] == 'C7':
        posA, posB = arrow._posA_posB
        arrow.set_positions((8.8, 5.3), None)
        text.set_position((8.8-0.05, 5.3-0.05))
    if selected_AGN_IDs[i][1] == 'C8':
        posA, posB = arrow._posA_posB
        arrow.set_positions((8.6, 5.7), None)
        text.set_position((8.6-0.05, 5.7))
    if selected_AGN_IDs[i][1] == 'C9':
        posA, posB = arrow._posA_posB
        arrow.set_positions((10.3, 5.2), None)
        text.set_position((10.3+0.05, 5.2-0.05))
    if selected_AGN_IDs[i][1] == 'C10':
        posA, posB = arrow._posA_posB
        arrow.set_positions((8.4, 6.35), None)
        text.set_position((8.4-0.05, 6.35-0.05))
    if selected_AGN_IDs[i][1] == 'C12':
        posA, posB = arrow._posA_posB
        arrow.set_positions((9.2, 5.3), None)
        text.set_position((9.2-0.05, 5.3-0.05))
    if selected_AGN_IDs[i][1] == 'D2':
        posA, posB = arrow._posA_posB
        arrow.set_positions((7.2, 11.7), None)
        text.set_position((7.2-0.05, 11.7+0.05))
    if selected_AGN_IDs[i][1] == 'D4':
        posA, posB = arrow._posA_posB
        arrow.set_positions((5.5, 9.8), None)
        text.set_position((5.5-0.05, 9.8+0.05))
    if selected_AGN_IDs[i][1] == 'D5':
        posA, posB = arrow._posA_posB
        arrow.set_positions((6, 10.5), None)
        text.set_position((6-0.05, 10.5+0.05))
    if selected_AGN_IDs[i][1] == 'D7':
        posA, posB = arrow._posA_posB
        arrow.set_positions((6, 10), None)
        text.set_position((6-0.05, 10+0.05))
    if selected_AGN_IDs[i][1] == 'D8':
        posA, posB = arrow._posA_posB
        arrow.set_positions((6.3, 10.7), None)
        text.set_position((6.3-0.05, 10.7+0.05))
    if selected_AGN_IDs[i][1] == 'E1':
        posA, posB = arrow._posA_posB
        arrow.set_positions((13.5, 11), None)
        text.set_position((13.5+0.05, 11-0.05))
    if selected_AGN_IDs[i][1] == 'E3':
        posA, posB = arrow._posA_posB
        arrow.set_positions((13.5, 7.2), None)
        text.set_position((13.5+0.05, 7.2-0.05))
    if selected_AGN_IDs[i][1] == 'E4':
        posA, posB = arrow._posA_posB
        arrow.set_positions((12.8, 11.8), None)
        text.set_position((12.8+0.05, 11.8+0.05))
    if selected_AGN_IDs[i][1] == 'E6':
        posA, posB = arrow._posA_posB
        arrow.set_positions((14.1, 9.5), None)
        text.set_position((14.1+0.05, 9.5+0.05))
    if selected_AGN_IDs[i][1] == 'E8':
        posA, posB = arrow._posA_posB
        arrow.set_positions((14.5, 8.5), None)
        text.set_position((14.5+0.05, 8.5))
    if selected_AGN_IDs[i][1] == 'E9':
        posA, posB = arrow._posA_posB
        arrow.set_positions((12, 6.3), None)
        text.set_position((12+0.05, 6.3-0.05))
    arrow.set_mutation_scale(20)

save_fig('projection_classes_AGN_SIMBAD.pdf')

In [ ]:
from matplotlib.lines import Line2D
import seaborn as sns

assert np.all(np.array(agn_mask).astype(float) == np.array(agn_indicator).astype(float))

# AGN
fig, ax = plt.subplots(1, 1, figsize=(width*2, height))

# Show SN in background first
sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2
)

########################
show_uncorrected = False
########################

qso_x = test_outputs_condensed_condensed_transformed[agn_mask, 0][milliquas_agn_labels['custom_type'] == 'QSO'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'QSO']]
qso_y = test_outputs_condensed_condensed_transformed[agn_mask, 1][milliquas_agn_labels['custom_type'] == 'QSO'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'QSO']]
qso_corr = agn_used_magpsf_corr[milliquas_agn_labels['custom_type'] == 'QSO'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'QSO']]

ax.scatter(qso_x[qso_corr], qso_y[qso_corr], color=sn_color_map['SN Ia'], s=150, alpha=0.3)
if show_uncorrected:
    ax.scatter(qso_x[~qso_corr], qso_y[~qso_corr], color=sn_color_map['SN Ia'], s=150, alpha=0.6, marker='^')

agn_x = test_outputs_condensed_condensed_transformed[agn_mask, 0][milliquas_agn_labels['custom_type'] == 'AGN'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'AGN']]
agn_y = test_outputs_condensed_condensed_transformed[agn_mask, 1][milliquas_agn_labels['custom_type'] == 'AGN'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'AGN']]
agn_corr = agn_used_magpsf_corr[milliquas_agn_labels['custom_type'] == 'AGN'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'AGN']]

ax.scatter(agn_x[agn_corr], agn_y[agn_corr], s=200, alpha=0.3, color=rgb_values2[4])
if show_uncorrected:
    ax.scatter(agn_x[~agn_corr], agn_y[~agn_corr], s=200, alpha=0.6, color=rgb_values2[4], marker='^')

bllac_x = test_outputs_condensed_condensed_transformed[agn_mask, 0][milliquas_agn_labels['custom_type'] == 'BL Lac'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'BL Lac']]
bllac_y = test_outputs_condensed_condensed_transformed[agn_mask, 1][milliquas_agn_labels['custom_type'] == 'BL Lac'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'BL Lac']]
bllac_corr = agn_used_magpsf_corr[milliquas_agn_labels['custom_type'] == 'BL Lac'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'BL Lac']]

ax.scatter(bllac_x[bllac_corr], bllac_y[bllac_corr], s=200, alpha=0.3, color=rgb_values[-1])
if show_uncorrected:
    ax.scatter(bllac_x[~bllac_corr], bllac_y[~bllac_corr], s=200, alpha=0.6, color=rgb_values[-1], marker='^')

type2_x = test_outputs_condensed_condensed_transformed[agn_mask, 0][milliquas_agn_labels['custom_type'] == 'Type II'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'Type II']]
type2_y = test_outputs_condensed_condensed_transformed[agn_mask, 1][milliquas_agn_labels['custom_type'] == 'Type II'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'Type II']]
type2_corr = agn_used_magpsf_corr[milliquas_agn_labels['custom_type'] == 'Type II'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'Type II']]

# Note: Your original Type II code used marker='^' for all. I've switched the default to 'o' 
# so the uncorrected (~) ones stand out as triangles, but feel free to swap this.
ax.scatter(type2_x[type2_corr], type2_y[type2_corr], s=180, alpha=0.7, color=rgb_values[-3]) 
if show_uncorrected:
    ax.scatter(type2_x[~type2_corr], type2_y[~type2_corr], s=180, alpha=0.6, color=rgb_values[-3], marker='^')

# ax.scatter(
#     test_outputs_condensed_condensed_transformed[agn_mask, 0][milliquas_agn_labels['custom_type'] == 'QSO'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'QSO']],
#     test_outputs_condensed_condensed_transformed[agn_mask, 1][milliquas_agn_labels['custom_type'] == 'QSO'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'QSO']],
#     color=sn_color_map['SN Ia'],
#     s=150, alpha=0.3
# )
# ax.scatter(
#     test_outputs_condensed_condensed_transformed[agn_mask, 0][milliquas_agn_labels['custom_type'] == 'AGN'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'AGN']],
#     test_outputs_condensed_condensed_transformed[agn_mask, 1][milliquas_agn_labels['custom_type'] == 'AGN'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'AGN']],
#     s=200, alpha=0.3, color=rgb_values2[4]
# )
# ax.scatter(
#     test_outputs_condensed_condensed_transformed[agn_mask, 0][milliquas_agn_labels['custom_type'] == 'BL Lac'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'BL Lac']],
#     test_outputs_condensed_condensed_transformed[agn_mask, 1][milliquas_agn_labels['custom_type'] == 'BL Lac'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'BL Lac']],
#     s=200, alpha=0.3, color=rgb_values[-1]
# )

# ax.scatter(
#     test_outputs_condensed_condensed_transformed[agn_mask, 0][milliquas_agn_labels['custom_type'] == 'Type II'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'Type II']],
#     test_outputs_condensed_condensed_transformed[agn_mask, 1][milliquas_agn_labels['custom_type'] == 'Type II'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'Type II']],
#     s=180, alpha=0.7, marker='^', color=rgb_values[-3]
# )

# ax.scatter(
#     test_outputs_condensed_condensed_transformed[agn_mask, 0][milliquas_agn_labels['custom_type'] == 'Unknown'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'Unknown']],
#     test_outputs_condensed_condensed_transformed[agn_mask, 1][milliquas_agn_labels['custom_type'] == 'Unknown'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'Unknown']],
#     s=200, alpha=0.7, color='gray'
# )

if qso_corr.sum() > 0 or show_uncorrected:
    handles = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=sn_color_map['SN Ia'], label='Quasar', markersize=20)
    ]
if agn_corr.sum() > 0 or show_uncorrected:
    handles += [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=rgb_values2[4], label='AGN', markersize=20)
    ]
if bllac_corr.sum() > 0 or show_uncorrected:
    handles += [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=rgb_values[-1], label='BL Lac', markersize=20)
    ]
# handles += [
#     Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', label='Unknown', markersize=20)
# ]
if type2_corr.sum() > 0 or show_uncorrected:
    handles += [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=rgb_values[-3], label='Type II', markersize=20)
    ]

legend = ax.legend(
    handles=handles, #bbox_to_anchor=(1, 1),
    loc='upper left', #fontsize=ticklabelsize,
    fancybox=False, edgecolor='black'
)
legend.get_frame().set_linewidth(0.5)

ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

ax.set_xlim(xlim)
ax.set_ylim(ylim)
set_axes_style(ax)


color_map_ = {
    'QSO': sn_color_map['SN Ia'],
    'AGN': rgb_values2[4],
    'BL Lac': rgb_values[-1],
    'Type II': rgb_values[-3],
#     'Unknown': 'gray'
}
selected_AGN_IDs = [
#     ('ZTF18acegzkh', 'B1'),
#     ('ZTF18abalful', 'B2'),
    ('ZTF21aazltbp', 'AGN1'),
    ('ZTF19aakxiwm', 'AGN2'),
#     ('ZTF21abcqqpo', 'B5'),
#     ('ZTF19abjascw', 'AGN3'),
    ('ZTF18aakeosr', 'AGN3'),
#     ('ZTF17aacpvmc', 'AGN5'),
#     ('ZTF19aavvvdx', 'B9'),
#     ('ZTF18abuwcgq', 'B10'),
#     ('ZTF20aadvpnz', 'B11'),
#     ('ZTF17aaadhjr', 'AGN5'),
    ('ZTF19aagwxlv', 'AGN4'),
#     ('ZTF18abvivzx', 'C3'),
#     ('ZTF18aarowgi', 'C4'),
#     ('ZTF18abxhyqv', 'C5'),
#     ('ZTF18aaagyrv', 'AGN7'),
#     ('ZTF18abaljmt', 'C6'),
    ('ZTF18abtrubp', 'AGN5'),
    ('ZTF20acdcick', 'AGN6'),
#     ('ZTF18aalufuu', 'C9'),
#     ('ZTF18acexzbr', 'C9'),
#     ('ZTF17aaajvds', 'C10'),
#     ('ZTF18abnuhuy', 'AGN11'),
    ('ZTF18aalufuv', 'AGN7'),
#     ('ZTF22aaseqhf', 'D1'),
#     ('ZTF18aajhwjb', 'D2'),
#     ('ZTF19abggdjc', 'AGN11'),
#     ('ZTF22aamptba', 'AGN14'),
#     ('ZTF18abezosa', 'D5'),
#     ('ZTF18acviazy', 'D6'),
    ('ZTF18acsptos', 'AGN8'),
    ('ZTF19aaaoiib', 'AGN9'),
#     ('ZTF18aasszwr', 'D9'),
#     ('ZTF20abmdgdn', 'AGN14'),
    ('ZTF18aaieyvr', 'AGN10'),
#     ('ZTF19abjprez', 'E3'),
#     ('ZTF19abgfgzy', 'AGN16'),
#     ('ZTF21aaygepu', 'E5'),
#     ('ZTF18absuhfp', 'E6'),
#     ('ZTF18adcbfig', 'E7'),
#     ('ZTF18aaowaij', 'E8'),
#     ('ZTF18acmziob', 'E9'),
]

assert np.all(np.array(agn_mask).astype(float) == np.array(agn_indicator).astype(float))

texts = []
for annotate_id in selected_AGN_IDs:
    # Skip AGNs <3 month since we remove them.
    if annotate_id[0] in test_objIds[agn_mask][~test_data_with_gte_3months_ids_mask[agn_mask]]:
        continue

    loca = np.where(test_objIds[agn_mask] == annotate_id[0])[0]
    coord_loca = test_outputs_condensed_condensed_transformed[agn_mask][loca, :].squeeze()

    # Skip the 'Unknown' class
    if milliquas_agn_labels.iloc[loca]['custom_type'].iloc[0] == 'Unknown':
        continue

    color = color_map_[milliquas_agn_labels.iloc[loca]['custom_type'].iloc[0]]

    texts.append(
        ax.text(
            coord_loca[0], coord_loca[1], annotate_id[1], c=color, fontsize=12
        )
    )

texts, arrows = adjust_text(
    texts, ax=ax, only_move={'points':'y', 'texts':'y'},
    arrowprops=dict(arrowstyle="-", color='gray', lw=0.4),
    ensure_inside_axes=False,
    expand=(4, 4),
    objects=[legend, ax.yaxis, ax.xaxis],
    force_explode=(0.5, 2.)
)

for i, (arrow, text) in enumerate(zip(arrows, texts)):
    if selected_AGN_IDs[i][1] == 'AGN2':
        posA, posB = arrow._posA_posB
        arrow.set_positions((4.7, 8.5), None)
        text.set_position((4.7-0.05, 8.5+0.05))
#     if selected_AGN_IDs[i][1] == 'AGN3':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((5.5, 9.3), None)
#         text.set_position((5.5-0.05, 9.3+0.05))
#     if selected_AGN_IDs[i][1] == 'B5':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((5.5, 8.3), None)
#         text.set_position((5.5+0.05, 8.3+0.05))
#     if selected_AGN_IDs[i][1] == 'AGN3':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((5.2, 6.5), None)
#         text.set_position((5.2+0.05, 6.5-0.05))
    if selected_AGN_IDs[i][1] == 'AGN3':
        posA, posB = arrow._posA_posB
        arrow.set_positions((6.2, 6.5), None)
        text.set_position((6.2+0.05, 6.5-0.05))
#     if selected_AGN_IDs[i][1] == 'AGN5':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((10.3, 5.4), None)
#         text.set_position((10.3+0.05, 5.4-0.05))
#     if selected_AGN_IDs[i][1] == 'AGN5':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((5.3, 8.7), None)
#         text.set_position((5.3+0.05, 8.7+0.05))
#     if selected_AGN_IDs[i][1] == 'B10':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((5.8, 6.9), None)
#         text.set_position((5.8+0.05, 6.9-0.05))
#     if selected_AGN_IDs[i][1] == 'B11':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((6.5, 6.5), None)
#         text.set_position((6.5+0.05, 6.5+0.05))
#     if selected_AGN_IDs[i][1] == 'AGN6':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((9.7, 5.2), None)
#         text.set_position((9.7, 5.2-0.05))
    if selected_AGN_IDs[i][1] == 'AGN4':
        posA, posB = arrow._posA_posB
        arrow.set_positions((9.3, 5.7), None)
        text.set_position((9.3+0.05, 5.7-0.05))
#     if selected_AGN_IDs[i][1] == 'AGN7':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((10.7, 5.7), None)
#         text.set_position((10.7+0.05, 5.7-0.05))
#     if selected_AGN_IDs[i][1] == 'C3':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((10, 5.1), None)
#         text.set_position((10+0.05, 5.1-0.05))
#     if selected_AGN_IDs[i][1] == 'C4':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((9.45, 5.4), None)
#         text.set_position((9.45, 5.4-0.05))
#     if selected_AGN_IDs[i][1] == 'C6':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((10.5, 5.7), None)
#         text.set_position((10.5+0.05, 5.7-0.05))
    if selected_AGN_IDs[i][1] == 'AGN5':
        posA, posB = arrow._posA_posB
        arrow.set_positions((9.7, 5.3), None)
        text.set_position((9.7-0.05, 5.3-0.05))
#     if selected_AGN_IDs[i][1] == 'C9':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((10.3, 5.2), None)
#         text.set_position((10.3+0.05, 5.2-0.05))
#     if selected_AGN_IDs[i][1] == 'C10':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((8.4, 6.35), None)
#         text.set_position((8.4-0.05, 6.35-0.05))
#     if selected_AGN_IDs[i][1] == 'C12':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((9.2, 5.3), None)
#         text.set_position((9.2-0.05, 5.3-0.05))
#     if selected_AGN_IDs[i][1] == 'D2':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((7.2, 11.7), None)
#         text.set_position((7.2-0.05, 11.7+0.05))
#     if selected_AGN_IDs[i][1] == 'AGN11':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((11.6, 5.5), None)
#         text.set_position((11.6+0.05, 5.5-0.05))
    if selected_AGN_IDs[i][1] == 'AGN6':
        posA, posB = arrow._posA_posB
        arrow.set_positions((9, 6), None)
        text.set_position((9+0.05, 6-0.05))
#     if selected_AGN_IDs[i][1] == 'D5':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((6, 10.5), None)
#         text.set_position((6-0.05, 10.5+0.05))
    if selected_AGN_IDs[i][1] == 'AGN7':
        posA, posB = arrow._posA_posB
        arrow.set_positions((10, 4.7), None)
        text.set_position((10-0.05, 4.7-0.05))
#     if selected_AGN_IDs[i][1] == 'AGN11':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((5.5, 9.8), None)
#         text.set_position((5.5-0.05, 9.8+0.05))
    if selected_AGN_IDs[i][1] == 'AGN8':
        posA, posB = arrow._posA_posB
        arrow.set_positions((6, 10.2), None)
        text.set_position((6-0.05, 10.2+0.05))
    if selected_AGN_IDs[i][1] == 'AGN9':
        posA, posB = arrow._posA_posB
        arrow.set_positions((5.8, 10.7), None)
        text.set_position((5.8-0.05, 10.7+0.05))
#     if selected_AGN_IDs[i][1] == 'AGN14':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((12.5, 12.8), None)
#         text.set_position((12.5+0.05, 12.8+0.05))
    if selected_AGN_IDs[i][1] == 'AGN10':
        posA, posB = arrow._posA_posB
        arrow.set_positions((14.3, 10), None)
        text.set_position((14.3+0.05, 10+0.05))
#     if selected_AGN_IDs[i][1] == 'AGN17':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((13.5, 11), None)
#         text.set_position((13.5+0.05, 11-0.05))
# #     if selected_AGN_IDs[i][1] == 'E3':
# #         posA, posB = arrow._posA_posB
# #         arrow.set_positions((13.5, 7.2), None)
# #         text.set_position((13.5+0.05, 7.2-0.05))
#     if selected_AGN_IDs[i][1] == 'AGN19':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((12.8, 11.8), None)
#         text.set_position((12.8+0.05, 11.8+0.05))
#     if selected_AGN_IDs[i][1] == 'E6':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((14.1, 9.5), None)
#         text.set_position((14.1+0.05, 9.5+0.05))
#     if selected_AGN_IDs[i][1] == 'E8':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((14.5, 8.5), None)
#         text.set_position((14.5+0.05, 8.5))
#     if selected_AGN_IDs[i][1] == 'E9':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((12, 6.3), None)
#         text.set_position((12+0.05, 6.3-0.05))
    arrow.set_mutation_scale(20)

save_fig('projection_classes_AGN_MILLIQUAS.pdf')

In [ ]:
# from sklearn.metrics.pairwise import cosine_similarity

# ###
# qso_x = test_outputs_condensed_condensed_transformed[agn_mask, 0][milliquas_agn_labels['custom_type'] == 'QSO'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'QSO']]
# qso_y = test_outputs_condensed_condensed_transformed[agn_mask, 1][milliquas_agn_labels['custom_type'] == 'QSO'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'QSO']]

# agn_x = test_outputs_condensed_condensed_transformed[agn_mask, 0][milliquas_agn_labels['custom_type'] == 'AGN'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'AGN']]
# agn_y = test_outputs_condensed_condensed_transformed[agn_mask, 1][milliquas_agn_labels['custom_type'] == 'AGN'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'AGN']]

# bllac_x = test_outputs_condensed_condensed_transformed[agn_mask, 0][milliquas_agn_labels['custom_type'] == 'BL Lac'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'BL Lac']]
# bllac_y = test_outputs_condensed_condensed_transformed[agn_mask, 1][milliquas_agn_labels['custom_type'] == 'BL Lac'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'BL Lac']]

# type2_x = test_outputs_condensed_condensed_transformed[agn_mask, 0][milliquas_agn_labels['custom_type'] == 'Type II'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'Type II']]
# type2_y = test_outputs_condensed_condensed_transformed[agn_mask, 1][milliquas_agn_labels['custom_type'] == 'Type II'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'Type II']]
# ###
# _qso_set = test_outputs_condensed_condensed[agn_mask][milliquas_agn_labels['custom_type'] == 'QSO'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'QSO']]
# qso_corr = agn_used_magpsf_corr[milliquas_agn_labels['custom_type'] == 'QSO'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'QSO']]

# _agn_set = test_outputs_condensed_condensed[agn_mask][milliquas_agn_labels['custom_type'] == 'AGN'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'AGN']]
# agn_corr = agn_used_magpsf_corr[milliquas_agn_labels['custom_type'] == 'AGN'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'AGN']]

# _bllac_set = test_outputs_condensed_condensed[agn_mask][milliquas_agn_labels['custom_type'] == 'BL Lac'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'BL Lac']]
# bllac_corr = agn_used_magpsf_corr[milliquas_agn_labels['custom_type'] == 'BL Lac'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'BL Lac']]

# _type2_set = test_outputs_condensed_condensed[agn_mask][milliquas_agn_labels['custom_type'] == 'Type II'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'Type II']]
# type2_corr = agn_used_magpsf_corr[milliquas_agn_labels['custom_type'] == 'Type II'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'Type II']]

# print(qso_corr.sum() + agn_corr.sum() + bllac_corr.sum() + type2_corr.sum())

# _sn_encodings = test_outputs_condensed_condensed[sn_mask]
# print(_sn_encodings.shape)
# _qso_encodings_contaminants = _qso_set[(qso_x < 6.5) & (qso_y < 7)]
# _bllac_encodings_contaminants = _bllac_set[(bllac_x < 6.5) & (bllac_y < 7)]
# _agn_encodings_contaminants = _agn_set[(agn_x < 6.5) & (agn_y < 7)]
# print(_qso_encodings_contaminants.shape, _bllac_encodings_contaminants.shape, _agn_encodings_contaminants.shape)
# _a_encodings_contaminants = np.vstack(
#     (_qso_encodings_contaminants, _bllac_encodings_contaminants, _agn_encodings_contaminants)
# )
# print(f'no. of contaminant AGNs: {_a_encodings_contaminants.shape}')
# ###
# _qso_encodings_noncontaminants = _qso_set[(qso_x >= 6.5) | (qso_y >= 7)]
# _bllac_encodings_noncontaminants = _bllac_set[(bllac_x >= 6.5) | (bllac_y >= 7)]
# _agn_encodings_noncontaminants = _agn_set[(agn_x >= 6.5) | (agn_y >= 7)]
# print(_qso_encodings_noncontaminants.shape, _bllac_encodings_noncontaminants.shape, _agn_encodings_noncontaminants.shape)
# _a_encodings_noncontaminants = np.vstack(
#     (_qso_encodings_noncontaminants, _bllac_encodings_noncontaminants, _agn_encodings_noncontaminants)
# )
# print(f'no. of non-contaminant AGNs: {_a_encodings_noncontaminants.shape}')

# print(len(_a_encodings_noncontaminants) + len(_a_encodings_contaminants), len(test_outputs_condensed_condensed_transformed[agn_mask][test_data_with_gte_3months_ids_mask[agn_mask]]))

# #####
# from sklearn.neighbors import NearestNeighbors

# k = 10
# print(test_outputs_condensed_condensed_transformed.shape)
# nbrs = NearestNeighbors(n_neighbors=k, metric='euclidean').fit(
#     test_outputs_condensed_condensed_transformed
# )
# distances, indices = nbrs.kneighbors(test_outputs_condensed_condensed_transformed)
# #####


# from sklearn.metrics.pairwise import cosine_similarity

# # (a) contaminant AGN vs non-contaminant AGN
# cos_sim_cont_vs_noncont = cosine_similarity(
#     _a_encodings_contaminants,
#     _a_encodings_noncontaminants
# )

# # (b) contaminant AGN vs SN
# cos_sim_cont_vs_sn = cosine_similarity(
#     _a_encodings_contaminants,
#     _sn_encodings
# )

# assert len(_a_encodings_contaminants) > 0
# assert len(_a_encodings_noncontaminants) > 0
# assert len(_sn_encodings) > 0

# assert not np.isnan(_a_encodings_contaminants).any()
# assert not np.isnan(_a_encodings_noncontaminants).any()
# assert not np.isnan(_sn_encodings).any()

# assert _a_encodings_contaminants.shape[1] == 546
# assert _a_encodings_noncontaminants.shape[1] == 546
# assert _sn_encodings.shape[1] == 546

# mean_sim_to_noncont = cos_sim_cont_vs_noncont.mean(axis=1)
# mean_sim_to_sn = cos_sim_cont_vs_sn.mean(axis=1)
# print(f'mean sim of the {len(_a_encodings_contaminants)} contaminant AGNs with non-cont AGNs: {mean_sim_to_noncont}')
# print(f'mean sim of the {len(_a_encodings_contaminants)} contaminant AGNs with SNe: {mean_sim_to_sn}')

# median_sim_to_noncont = np.median(cos_sim_cont_vs_noncont, axis=1)
# median_sim_to_sn = np.median(cos_sim_cont_vs_sn, axis=1)
# print(f'median sim of the {len(_a_encodings_contaminants)} contaminant AGNs with non-cont AGNs: {median_sim_to_noncont}, with corresponding MAD {stats.median_abs_deviation(cos_sim_cont_vs_noncont, axis=1)}')
# print(f'median sim of the {len(_a_encodings_contaminants)} contaminant AGNs with SNe: {median_sim_to_sn}, with corresponding MAD {stats.median_abs_deviation(cos_sim_cont_vs_sn, axis=1)}')

# from scipy import stats
# cos_sim_sn_vs_sn = cosine_similarity(
#     _sn_encodings,
#     _sn_encodings
# )
# median_median_sim_to_sn = np.median(np.median(cos_sim_sn_vs_sn, axis=1))
# print(f'summarized sim of all SNe with SNe: {median_median_sim_to_sn}, with MAD {np.median(stats.median_abs_deviation(cos_sim_sn_vs_sn, axis=1))}')

# # total = len(test_outputs_condensed_condensed_transformed[agn_mask][test_data_with_gte_3months_ids_mask[agn_mask]])
# # covered = (
# #     len(_qso_encodings_contaminants) + len(_qso_encodings_noncontaminants) +
# #     len(_bllac_encodings_contaminants) + len(_bllac_encodings_noncontaminants) +
# #     len(_agn_encodings_contaminants) + len(_agn_encodings_noncontaminants)
# # )

# # print("missing:", total - covered)

# # assert len(_a_encodings_noncontaminants) + len(_a_encodings_contaminants) == len(test_outputs_condensed_condensed_transformed[agn_mask][test_data_with_gte_3months_ids_mask[agn_mask]])

# ###
# # cosine_similarity(
# #     test_outputs_condensed_condensed[agn_mask][milliquas_agn_labels['custom_type'] != 'Unknown'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] != 'Unknown']],
# #     test_outputs_condensed_condensed[agn_mask][milliquas_agn_labels['custom_type'] != 'Unknown'][test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] != 'Unknown']]
# # )

In [ ]:
# import numpy as np
# from sklearn.neighbors import NearestNeighbors
# from sklearn.metrics.pairwise import cosine_similarity
# from scipy import stats

# # _a_encodings_contaminants : shape (Nc, d)
# # _a_encodings_noncontaminants : shape (Nn, d)
# # _sn_encodings : shape (Ns, d)
# # k : number of nearest neighbors
# # -----------------------------

# k = 10

# # Nearest neighbors in cosine space
# # fit on non-contaminant AGNs
# nbrs_noncont = NearestNeighbors(n_neighbors=k, metric='cosine').fit(_a_encodings_noncontaminants)
# dist_noncont, idx_noncont = nbrs_noncont.kneighbors(_a_encodings_contaminants)

# # fit on SNe
# nbrs_sn = NearestNeighbors(n_neighbors=k, metric='cosine').fit(_sn_encodings)
# dist_sn, idx_sn = nbrs_sn.kneighbors(_a_encodings_contaminants)


# A = _a_encodings_contaminants
# B_noncont = _a_encodings_noncontaminants
# B_sn = _sn_encodings

# from sklearn.metrics.pairwise import cosine_similarity

# # Using sklearn's cosine_similarity instead of manual dot-product
# cos_sim_knn_noncont = np.array([
#     cosine_similarity(A[i].reshape(1, -1), B_noncont[idx_noncont[i]]).flatten()
#     for i in range(len(A))
# ])

# cos_sim_knn_sn = np.array([
#     cosine_similarity(A[i].reshape(1, -1), B_sn[idx_sn[i]]).flatten()
#     for i in range(len(A))
# ])

# median_sim_to_noncont = np.median(cos_sim_knn_noncont, axis=1)
# median_sim_to_sn = np.median(cos_sim_knn_sn, axis=1)

# mad_sim_to_noncont = stats.median_abs_deviation(cos_sim_knn_noncont, axis=1)
# mad_sim_to_sn = stats.median_abs_deviation(cos_sim_knn_sn, axis=1)


# print(f"Contaminant AGNs vs kNN non-contaminant AGNs: median={median_sim_to_noncont}, MAD={mad_sim_to_noncont}")
# print(f"Contaminant AGNs vs kNN SNe: median={median_sim_to_sn}, MAD={mad_sim_to_sn}")

# cos_sim_sn_vs_sn = cosine_similarity(_sn_encodings)
# median_median_sim_sn = np.median(np.median(cos_sim_sn_vs_sn, axis=1))
# mad_median_sim_sn = np.median(stats.median_abs_deviation(cos_sim_sn_vs_sn, axis=1))
# print(f"All SNe vs SNe: median={median_median_sim_sn}, MAD={mad_median_sim_sn}")

In [ ]:
# umap_fit.graph_.todense()

In [ ]:
agn_corr.sum(), bllac_corr.sum(), type2_corr.sum(), qso_corr.sum()

In [ ]:
(~agn_corr).sum(), (~bllac_corr).sum(), (~type2_corr).sum(), (~qso_corr).sum()

In [ ]:
(~test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'Type II']).sum(),\
(~test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'AGN']).sum(),\
(~test_data_with_gte_3months_ids_mask[agn_mask][milliquas_agn_labels['custom_type'] == 'QSO']).sum()

One can visualize light curves at different locations in this projection space, if required.

In [ ]:
matplotlib.rcParams["font.size"] = "12"

In [ ]:
# from matplotlib.ticker import MaxNLocator, MultipleLocator

# decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
# check_indices = np.where(
#     (test_outputs_condensed_condensed_transformed[:, 0] > 4.6) &
#     (test_outputs_condensed_condensed_transformed[:, 0] < 5.2) &
#     (test_outputs_condensed_condensed_transformed[:, 1] > 8) &
#     (test_outputs_condensed_condensed_transformed[:, 1] < 8.6)
# #     (test_outputs_condensed_condensed_transformed[:, 1] < 1.1) &
# #     (test_outputs_condensed_condensed_transformed[:, 0] > 6.3) &
# #     (test_outputs_condensed_condensed_transformed[:, 0] < 6.7)
# #     (test_outputs_condensed_condensed_transformed[:, 1] < -0.5) &
# #     (test_outputs_condensed_condensed_transformed[:, 0] > 6) &
# #     (test_outputs_condensed_condensed_transformed[:, 0] < 6.5) &
# #     (test_outputs_condensed_condensed_transformed[:, 1] > -0.5) &
# #     (test_outputs_condensed_condensed_transformed[:, 1] < 0.3)
# #     (test_outputs_condensed_condensed_transformed[:, 1] > 1)
# )[0]

# print(len(check_indices))
# counter = 0
# for eidx in check_indices:
#     currObjId = test_objIds[eidx]
#     currObjId_index = np.where(total_objIds == currObjId)
#     currCommonFinkclass = total_common_finkclasses[currObjId_index].squeeze()
#     currCommonFinkclass = currCommonFinkclass[currCommonFinkclass != '0']
#     assert len(currCommonFinkclass) == 1
#     currCommonFinkclass = currCommonFinkclass[0]

#     if currCommonFinkclass != 'custom_sn':
#         continue
# #     if fine_classes_grouped[eidx] != 'SLSN':
# #         continue
# #     if fine_classes_grouped[eidx] != 'SN Ibc':
# #         continue
# #     if fine_classes_grouped[eidx] != 'Early SN Ia':
# #         continue
# #     if fine_classes_grouped[eidx] != 'SN Ia':
# #         continue
# #     if fine_classes_grouped[eidx] != 'SN II':
# #         continue

#     print(
#         currObjId,
#         test_outputs_condensed_condensed_transformed[np.where(test_objIds == currObjId), 0],
#         test_outputs_condensed_condensed_transformed[np.where(test_objIds == currObjId), 1],
#         fine_classes_grouped[eidx]
#     )

# #     print(scipy.spatial.distance.cosine(
# #         test_outputs_condensed_condensed[np.where(test_objIds == currObjId), :].squeeze(),
# #         test_outputs_condensed_condensed[np.where(test_objIds == 'ZTF22aafuind'), :].squeeze(),
# #     ))
# #     continue
# #     if fine_classes_grouped[eidx] == '(TNS) SN Ia':
# #         continue
# #     print(test_outputs_condensed_condensed_transformed[eidx, 0], currObjId)
# #     print(fine_classes[eidx])
# #     print(df_alerts_pandas[df_alerts_pandas['objectId'] == currObjId]['finkclass'].value_counts())

# #     if len(df_alerts_pandas[df_alerts_pandas['objectId'] == currObjId]) > 8:
# #         continue

# #     plot_lc(df_alerts_pandas, currObjId, title_suffix=fine_classes[eidx])

# #     print(test_outputs_condensed_condensed_transformed[eidx, 0], test_outputs_condensed_condensed_transformed[eidx, 1])
#     ax = quick_plot_decoded_lc(decoded_lcs_test, df_alerts_pandas, currObjId, fig_width=width*2/3, fig_height=height/4)
#     ax.tick_params(axis='both', which='minor', length=3)
#     ax.tick_params(axis='both', which='major', length=6)
#     counter += 1

#     if counter > 20:
#         break

In [ ]:
from matplotlib.ticker import MaxNLocator, MultipleLocator

decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
min_max_mags = np.load('min_max_mags.npy')

# selected_SN_IDs = [
#     # leftmost SN Ia
#     'ZTF22aaijrul',
#     'ZTF22abfyzir',
# #     'ZTF22abfvsbt',
# #     'ZTF22aavctvq',
# #     'ZTF22abhldat',
# #     'ZTF21acaegsr',
# #     'ZTF22abgteub',
# #     'ZTF21abjpphz',
#     'ZTF21abzithf',
# #     'ZTF21abkiefj',  # SN Ia variant
#     # middle SN Ia
#     'ZTF21abujejr',
# #     'ZTF22abgbffy',
# #     'ZTF22aafoqrd',
#     'ZTF21abnujlh',
#     'ZTF21abrgdug',
#     'ZTF22aafoqrd',
# #     'ZTF22aafwbey',  # SN Ia variant
#     # a bit on the right (UMAP_x bet 2 and 2.5)
# #     'ZTF22abyhvtk',
# #     'ZTF21abifxhe',
# #     'ZTF22aafuind',  # SN Ia variant
#     # outliers
# #     'ZTF22abyrovo',
#     'ZTF22aabwrov',
#     'ZTF19aaapnxn',  # extreme outlier at UMAP_x ~ 5

#     # leftmost SN II
#     'ZTF21abidrcn',
# #     'ZTF22abeyzcl',
#     'ZTF21abmwzxt',
#     # bottom SN II
#     'ZTF21acafqtj',
# #     'ZTF22aaayceh',
# #     'ZTF22aajjawt',
#     'ZTF22abhsxph',
#     'ZTF22aagvxjc',
#     # outliers SN II
#     'ZTF22aafxzfb',
#     'ZTF21abtdvpg',
#     'ZTF22aapkbkl',
#     'ZTF21abcjpnm',

#     # SN Ibc
#     'ZTF22aaahuly',
#     'ZTF22abieqzx',
#     'ZTF21abraccv',
#     'ZTF22aarwwyy',
#     'ZTF22aaafkvn',
#     'ZTF22aapubuy',
#     'ZTF22abgroeo',
#     'ZTF21abuvury',

#     # SLSN
#     'ZTF22aaftzcn',
#     'ZTF22aafumyr',
#     'ZTF20acbcfaa',
#     'ZTF22aaacvyy',
#     'ZTF21accwovq',
#     'ZTF22abcvfgs',
#     'ZTF20aacbwbm',

# #     # Early SN Ia
#     'ZTF21abqoodm'
#     'ZTF22ablepod',
#     'ZTF21abxdlyk',
# ]

for ii, (objId, text_label) in enumerate(selected_SN_IDs):
    objId_index = np.where(test_objIds == objId)[0][0]
    print(
        objId,
        test_outputs_condensed_condensed_transformed[objId_index, 0],
        test_outputs_condensed_condensed_transformed[objId_index, 1],
    )

    ax = quick_plot_decoded_lc(
        decoded_lcs_test, df_alerts_pandas, objId, min_max_mags=min_max_mags,
        fig_width=width*2/3, fig_height=height/4,
        show_legend=True if ii == 0 else False, legend_loc='upper center',
    )
    ax.tick_params(axis='both', which='minor', length=3)
    ax.tick_params(axis='both', which='major', length=6)

    if np.isnan(new_decline_rates_g[objId_index]) or np.isnan(new_decline_rates_r[objId_index]):
        dr = '-'
    else:
        dr = np.round(
            (np.array(new_decline_rates_g[objId_index]) + np.array(new_decline_rates_r[objId_index])) / 2,
            2
        )
    dur_text = (np.array(dur_g[objId_index]) + np.array(dur_r[objId_index])) / 2
#     time_min_mag_text = (np.array(time_min_mag_g[objId_index]) + np.array(time_min_mag_r[objId_index])) / 2
#     amplitudes_text = (np.array(amplitudes_g[objId_index]) + np.array(amplitudes_r[objId_index])) / 2

    color_at_peak1_text = color_at_peak1[objId_index]

    if ii == 0:
#         text_to_show = r'Duration (days), $t_{\mathrm{peak}} (r)$ (days), Decline rate (mag/day)\n{}, {}, {}'.format(dur_r[objId_index], time_min_mag_r[objId_index], decline_rate_r[objId_index])
#         text_to_show = f'Duration (days), $t_{{\\mathrm{{peak}}}}$ (days), Decline rate (mag/day) $(r)$\n{dur_r[objId_index]:.0f}, {time_min_mag_r[objId_index]:.0f}, {dr_r}'
#         text_to_show = f'{dur_text:.0f}, {time_min_mag_text:.0f}, {amplitudes_text:.1f}, {dr}'
        text_to_show = f'{dur_text:.0f}, {dr}, {color_at_peak1_text:.1f}'
    else:
        text_to_show = f'{dur_text:.0f}, {dr}, {color_at_peak1_text:.1f}'
    ax.text(
        0.95, 0.9, text_to_show,
        ha='right', va='top', transform=ax.transAxes, linespacing=1.5
    )
    ax.text(
        0.5, 0.05, text_label,
        ha='center', va='bottom', transform=ax.transAxes
    )
    # save_fig(f'light_curve_{objId}_{fine_classes_grouped[objId_index].strip()}.pdf')
    save_fig(f'light_curve_{objId}_SN_{text_label}.pdf')

In [ ]:
from matplotlib.ticker import MaxNLocator, MultipleLocator

##########################################################
decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
if correct_lcs_for_agns:
    decoded_lcs_test_corrected = np.load('evaluate_test_CORRECTED_decoded_lcs.npz')

    assert list(decoded_lcs_test.keys()) == list(decoded_lcs_test_corrected.keys()), "NPZ keys mismatch"

    # Build corrected_agn: start from original, replace AGN rows with corrected
    decoded_lcs_test_corrected_agn = {}
    for i, key in enumerate(decoded_lcs_test.keys()):
        if agn_indicator[i]:
            decoded_lcs_test_corrected_agn[key] = decoded_lcs_test_corrected[key].copy()
            # Verify AGN lc came from corrected
            np.testing.assert_array_equal(
                decoded_lcs_test_corrected_agn[key],
                decoded_lcs_test_corrected[key]
            )
        else:
            decoded_lcs_test_corrected_agn[key] = decoded_lcs_test[key].copy()
            # Verify non-AGN lc is unchanged
            np.testing.assert_array_equal(
                decoded_lcs_test_corrected_agn[key],
                decoded_lcs_test[key]
            )

    print(f"Replaced {agn_indicator.sum()} AGN rows out of {len(agn_indicator)} total")

use_corrected_agns = True
if use_corrected_agns:
    print('Using corrected AGN light curves...')
    assert np.all(
        np.load('evaluate_test_objIds_dataloader.npy') == test_objIds_corrected_agn
    )
    test_objIds = test_objIds_corrected_agn
    decoded_lcs_test = decoded_lcs_test_corrected_agn
    min_max_mags = np.load('min_max_mags.npy')
    min_max_mags_corrected = np.load('min_max_mags_ftransfer_ztf_2025_05_31_430518_CORRECTED.npy')
else:
    print('Using uncorrected AGN light curves...')
    test_objIds  = np.load('evaluate_test_objIds_dataloader.npy')
    decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
    min_max_mags = np.load('min_max_mags.npy')
##########################################################

df_alerts_corrected = pd.read_parquet(
    'alerts_processed_ftransfer_ztf_2025_05_31_430518_CORRECTED.parquet',
    columns=[
        'objectId', 'finkclass', 'tnsclass',
         'fid', 'magpsf_corr', 'sigmapsf_corr', 'jd'
    ]
)

for ii, (objId, text_label) in enumerate(selected_AGN_IDs):
    objId_index = np.where(test_objIds == objId)[0][0]
    print(
        objId,
        test_outputs_condensed_condensed_transformed[objId_index, 0],
        test_outputs_condensed_condensed_transformed[objId_index, 1],
    )
    loca = np.where(test_objIds[agn_mask] == objId)[0]
    whichclass = milliquas_agn_labels.iloc[loca]['custom_type'].iloc[0]
#     if whichclass == 'nan':
#         whichclass = 'Unknown'

    ax = quick_plot_decoded_lc(
        decoded_lcs_test, df_alerts_corrected, objId,
        min_max_mags=min_max_mags_corrected,
        fig_width=width*2/3, fig_height=height/4,
        show_legend=True if ii == 0 else False, legend_loc='upper center',
        whichclass=whichclass
    )
    ax.tick_params(axis='both', which='minor', length=3)
    ax.tick_params(axis='both', which='major', length=6)

    dur_text = (np.array(dur_g[objId_index]) + np.array(dur_r[objId_index])) / 2
    mean_color_text = mean_colors[objId_index]
    mean_variance_text = (
        (
            np.sqrt(np.array(excess_vars_g)) + \
            np.sqrt(np.array(excess_vars_r))
        ) / 2
    )[objId_index] * 100

    if ii == 0:
#         text_to_show = r'Duration (days), $t_{\mathrm{peak}} (r)$ (days), Decline rate (mag/day)\n{}, {}, {}'.format(dur_r[objId_index], time_min_mag_r[objId_index], decline_rate_r[objId_index])
#         text_to_show = f'Duration (days), $t_{{\\mathrm{{peak}}}}$ (days), Decline rate (mag/day) $(r)$\n{dur_r[objId_index]:.0f}, {time_min_mag_r[objId_index]:.0f}, {dr_r}'
        text_to_show = f'{dur_text:.0f}, {mean_color_text:.2f}, {mean_variance_text:.1f}'
    else:
        text_to_show = f'{dur_text:.0f}, {mean_color_text:.2f}, {mean_variance_text:.1f}'
    ax.text(
        0.95, 0.9, text_to_show,
        ha='right', va='top', transform=ax.transAxes, linespacing=1.5
    )
    ax.text(
        0.5, 0.05, text_label,
        ha='center', va='bottom', transform=ax.transAxes
    )
#     plt.show()
    # save_fig(f'light_curve_{objId}_{fine_classes_grouped[objId_index].strip()}.pdf')
    save_fig(f'light_curve_{objId}_AGN_{text_label}.pdf')

In [ ]:
decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
from matplotlib.ticker import MaxNLocator, MultipleLocator

# ax = quick_plot_decoded_lc('ZTF22aanmbrl', fig_width=width*2/3, fig_height=height/4)
# ax.set_xlim([300, 340])

In [ ]:
cc1 = np.where(test_objIds == 'ZTF21abrgdug')[0][0]
cc2 = np.where(test_objIds == 'ZTF22aaayceh')[0][0]
test_outputs_condensed_condensed_transformed[cc1, :], test_outputs_condensed_condensed_transformed[cc2, :]

In [ ]:
# qso_outlier = np.where(test_objIds == 'ZTF21aazltbp')[0][0]
# sn_close_to_qso_outlier = np.where(test_objIds == 'ZTF21abraccv')[0][0]
# another_qso = np.where(test_objIds == 'ZTF20abmdgdn')[0][0]
# b4_agn = np.where(test_objIds == 'ZTF19aakxiwm')[0][0]
# b6_agn = np.where(test_objIds == 'ZTF19abjascw')[0][0]
# b7_agn = np.where(test_objIds == 'ZTF18aakeosr')[0][0]
# b8_agn = np.where(test_objIds == 'ZTF17aacpvmc')[0][0]

# def cs(a,b):
#     cos_sim = np.dot(a, b) / (np.linalg.norm(a)*np.linalg.norm(b))
#     dot_product = np.dot(a, b)
#     norm_product = np.linalg.norm(a) * np.linalg.norm(b)
#     return cos_sim, dot_product, norm_product

# print(cs(
#     test_outputs_condensed_condensed[qso_outlier, :],
#     test_outputs_condensed_condensed[sn_close_to_qso_outlier, :]
# ), cs(
#     test_outputs_condensed_condensed[qso_outlier, :],
#     test_outputs_condensed_condensed[another_qso, :]
# ), cs(
#     test_outputs_condensed_condensed[b4_agn, :],
#     test_outputs_condensed_condensed[sn_close_to_qso_outlier, :]
# ), cs(
#     test_outputs_condensed_condensed[b6_agn, :],
#     test_outputs_condensed_condensed[sn_close_to_qso_outlier, :]
# ), cs(
#     test_outputs_condensed_condensed[b7_agn, :],
#     test_outputs_condensed_condensed[sn_close_to_qso_outlier, :]
# ), cs(
#     test_outputs_condensed_condensed[b8_agn, :],
#     test_outputs_condensed_condensed[sn_close_to_qso_outlier, :]
# ))
# print('+++')
# cs(
#     test_outputs_condensed_condensed[b4_agn, :],
#     test_outputs_condensed_condensed[another_qso, :]
# ), cs(
#     test_outputs_condensed_condensed[b6_agn, :],
#     test_outputs_condensed_condensed[another_qso, :]
# ), cs(
#     test_outputs_condensed_condensed[b7_agn, :],
#     test_outputs_condensed_condensed[another_qso, :]
# ), cs(
#     test_outputs_condensed_condensed[b8_agn, :],
#     test_outputs_condensed_condensed[another_qso, :]
# )

In [ ]:
decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')

check_indices = np.where(
    (test_outputs_condensed_condensed_transformed[:, 1] < 6)
#     (test_outputs_condensed_condensed_transformed[:, 0] < 5.5)
#     (test_outputs_condensed_condensed_transformed[:, 0] < 7.5) &
#     (test_outputs_condensed_condensed_transformed[:, 1] > 9)
#     (test_outputs_condensed_condensed_transformed[:, 0] < 6)
#     (test_outputs_condensed_condensed_transformed[:, 1] > 1)
)[0]
print(len(check_indices))
counter = 0
for eidx in check_indices:
    currObjId = test_objIds[eidx]
    currObjId_index = np.where(total_objIds == currObjId)
    currCommonFinkclass = total_common_finkclasses[currObjId_index].squeeze()
    currCommonFinkclass = currCommonFinkclass[currCommonFinkclass != '0']
    assert len(currCommonFinkclass) == 1
    currCommonFinkclass = currCommonFinkclass[0]

    if currCommonFinkclass != 'custom_agn':
        continue

    if fine_classes_grouped_finer[eidx] != 'Blazar':
        continue

#if fine_classes_grouped_finer[eidx] != 'AGN' and fine_classes_grouped_finer[eidx] != 'AGN':
#         continue
#     print(fine_classes[eidx])
#     print(df_alerts_pandas[df_alerts_pandas['objectId'] == currObjId]['finkclass'].value_counts())

    objId_index = np.where(test_objIds == currObjId)[0][0]

    print(
        objId_index, currObjId,
        test_outputs_condensed_condensed_transformed[objId_index, 0],
        test_outputs_condensed_condensed_transformed[objId_index, 1],
    )
    counter += 1

#     plot_lc(df_alerts_pandas, currObjId, title_suffix=fine_classes[eidx])

    quick_plot_decoded_lc(
        decoded_lcs_test, df_alerts, currObjId, min_max_mags,
        fig_width=width*2/3, fig_height=height/4
    )

    if counter >= 20:
        break

In [ ]:
test_outputs_condensed_condensed.shape

In [ ]:
len_one_band_outputs = test_outputs_condensed_condensed.shape[1] // 2
fig, ax = plt.subplots(1, 1, figsize=(20, 5))
for i in range(len(test_outputs_condensed_condensed)):
    ax.plot(test_outputs_condensed_condensed[i][:len_one_band_outputs])
    if i == 2:
        break
ax.set_ylim([-2, +2])
plt.show()

In [ ]:
# Plots the distribution of finkclasses across the entire dataset (train+val+test).
total_common_finkclasses_flatten = total_common_finkclasses.flatten()

fig, ax = plt.subplots(1, 1, figsize=(16, 5))
pd.Series(
    total_common_finkclasses_flatten[total_common_finkclasses_flatten != '0'].flatten()  # !=0 because those were added just for apdding.
).value_counts().plot(kind='bar', ax=ax, rot=0)
ax.set_yscale('log')
# ax.set_title('total_common_finkclasses')
ax.set_ylabel('Counts', fontsize=axeslabelsize)

ax.tick_params(axis='x', labelsize=ticklabelsize, width=2, length=10)
ax.tick_params(axis='y', labelsize=ticklabelsize, width=2, length=10)
ax.set_xticklabels(['AGN', 'SNe', 'Early SN Ia'])

In [ ]:
pd.Series(
    total_common_finkclasses_flatten[total_common_finkclasses_flatten != '0'].flatten()  # !=0 because those were added just for apdding.
).value_counts()

Coloring the projection space with light curve properties

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import iqr

# NOTE: num_datapoints_lcs is the no. of points summed over all filters.
num_datapoints_lcs = np.load('num_datapoints_lcs.npy')

import matplotlib.pyplot as plt
plt.hist(num_datapoints_lcs, histtype='step')
plt.yscale('log')
plt.xlabel('no. of datapoints', fontsize=axeslabelsize)
plt.ylabel('Counts', fontsize=axeslabelsize)
plt.axvline(x=np.mean(num_datapoints_lcs), linestyle='--', c='black', label='mean')
plt.axvline(x=np.median(num_datapoints_lcs), linestyle='--', c='red', label='median')
plt.tick_params(axis='x', labelsize=ticklabelsize, width=2, length=10)
plt.tick_params(axis='y', labelsize=ticklabelsize, width=2, length=10)
plt.legend(fontsize=ticklabelsize)
plt.show()

print(np.mean(num_datapoints_lcs), np.median(num_datapoints_lcs))

duration_lcs = np.load('duration_lcs.npy')
plt.hist(duration_lcs/24, histtype='step')
plt.yscale('log')
plt.xlabel('duration of light curve (days)', fontsize=axeslabelsize)
plt.ylabel('Counts', fontsize=axeslabelsize)
plt.axvline(x=np.mean(duration_lcs/24), linestyle='--', c='black', label='mean')
plt.axvline(x=np.median(duration_lcs/24), linestyle='--', c='red', label='median')
plt.tick_params(axis='x', labelsize=ticklabelsize, width=2, length=10)
plt.tick_params(axis='y', labelsize=ticklabelsize, width=2, length=10)
plt.legend(fontsize=ticklabelsize)
plt.show()

print(np.mean(duration_lcs), np.median(duration_lcs))


min_max_magdiffs = np.load('min_max_magdiffs.npy')

plt.hist(min_max_magdiffs, histtype='step')
plt.yscale('log')
plt.xlabel('(max - min) mag', fontsize=axeslabelsize)
plt.ylabel('Counts', fontsize=axeslabelsize)
plt.axvline(x=np.mean(min_max_magdiffs), linestyle='--', c='black', label='mean')
plt.axvline(x=np.median(min_max_magdiffs), linestyle='--', c='red', label='median')
plt.legend(fontsize=ticklabelsize)
plt.tick_params(axis='x', labelsize=ticklabelsize)
plt.tick_params(axis='y', labelsize=ticklabelsize)
plt.show()

print(np.mean(min_max_magdiffs), np.median(min_max_magdiffs), iqr(min_max_magdiffs), np.std(min_max_magdiffs))

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

import math
from scipy.stats import siegelslopes

# Compute properties
min_mags_g, min_mags_r, mean_mags_g, mean_mags_r, amplitudes_g, \
amplitudes_r, length_lc, time_min_mag_g, time_min_mag_r, \
decline_rate_g, decline_rate_r, decline_rate_g_numerator, decline_rate_r_numerator, \
decline_rate_g_modified, decline_rate_r_modified, \
dur, dur_g, dur_r, sigmas_mag_g, sigmas_mag_r, \
min_mags, mean_mags, amplitudes, time_min_mag, decline_rate, sigmas_mag, \
excess_vars_g, excess_vars_r, excess_vars, color_at_peak1, delta_t_peak, \
tmms_r, tmms_g, tmins_r, tmins_g, tmaxs_r, tmaxs_g, linear_slopes_g, linear_slopes_r,\
colors_at_peak_minus_color_at_faintest, color_at_faintest = \
[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], \
[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []

if use_corrected_agns:
    df_alerts_corrected = dd.read_parquet(
        'alerts_processed_ftransfer_ztf_2025_05_31_430518_CORRECTED.parquet',
        columns=[
            'objectId', 'finkclass', 'tnsclass',
            'fid', 'magpsf_corr', 'sigmapsf_corr', 'jd',
            'corrected', 'used_magpsf_corr'
        ]
    )
    df_alerts_corrected_pandas = df_alerts_corrected.compute()

uncorrected_agn_mask, corrected_agn_mask = [], []
for i, oi in enumerate(test_objIds):
    ## Checks for AGN. If AGN, use corrected dataframe of alerts ##
    if oi in milliquas_agn_labels['objectId'].values:
        test_currCommonFinkclass = milliquas_agn_labels[milliquas_agn_labels['objectId'] == oi]['custom_type']
        assert len(test_currCommonFinkclass) == 1
        test_currCommonFinkclass = test_currCommonFinkclass.iloc[0]
#         if test_currCommonFinkclass == 'nan':
#             test_currCommonFinkclass = 'Unknown'
    else:
        test_currObjId_index = np.where(test_objIds == oi)
        assert i == float(test_currObjId_index[0][0])
        test_currCommonFinkclass = fine_classes[test_currObjId_index[0][0]]

    uncorrected_agn_condition = (
        (test_currCommonFinkclass in milliquas_agn_labels['custom_type'].unique())
        and (use_corrected_agns)
        and not df_alerts_corrected_pandas.loc[df_alerts_corrected_pandas['objectId'] == oi, 'used_magpsf_corr'].any()
    )
    corrected_agn_condition = (
        (test_currCommonFinkclass in milliquas_agn_labels['custom_type'].unique())
        and (use_corrected_agns)
        and df_alerts_corrected_pandas.loc[df_alerts_corrected_pandas['objectId'] == oi, 'used_magpsf_corr'].all()
    )
    if use_corrected_agns:
        uncorrected_agn_mask.append(True) if uncorrected_agn_condition else uncorrected_agn_mask.append(False)
        corrected_agn_mask.append(True) if corrected_agn_condition else corrected_agn_mask.append(False)
    ###############################################################
    data = get_lc(
        df_alerts_corrected_pandas if corrected_agn_condition else df_alerts_pandas,
        oi, time_in_hrs=False,
        magpsf_column='magpsf_corr' if corrected_agn_condition else 'magpsf',
        sigmapsf_column='sigmapsf_corr' if corrected_agn_condition else 'sigmapsf'
    )
    obs = data[2]
    obs_mask = data[3]
    t = data[1]

    obs_mag = np.where(obs_mask, obs, np.nan)
    assert np.all(np.sum(~np.isnan(obs_mag), axis=1) >= 1)

    obs_mag_sorted_g_nonan = np.sort(obs_mag.T[0][~np.isnan(obs_mag.T[0])])
    obs_mag_sorted_r_nonan = np.sort(obs_mag.T[1][~np.isnan(obs_mag.T[1])])
    obs_mag_sorted_nonan = np.sort(obs_mag.T[~np.isnan(obs_mag.T)])

    ng = len(obs_mag_sorted_g_nonan)
    nr = len(obs_mag_sorted_r_nonan)
    n = len(obs_mag_sorted_nonan)

    amplitude_g = (
        np.median(obs_mag_sorted_g_nonan[int(-math.ceil(0.05 * ng)):]) - \
        np.median(obs_mag_sorted_g_nonan[0:int(math.ceil(0.05 * ng))])
    )
    amplitude_r = (
        np.median(obs_mag_sorted_r_nonan[int(-math.ceil(0.05 * nr)):]) - \
        np.median(obs_mag_sorted_r_nonan[0:int(math.ceil(0.05 * nr))])
    )
    amplitude = (
        np.median(obs_mag_sorted_nonan[int(-math.ceil(0.05 * n)):]) - \
        np.median(obs_mag_sorted_nonan[0:int(math.ceil(0.05 * n))])
    )

    min_mag_g = np.nanmin(obs_mag.T[0])
    min_mag_r = np.nanmin(obs_mag.T[1])
    min_mag = np.nanmin(obs_mag)
    min_mags_g.append(min_mag_g)
    min_mags_r.append(min_mag_r)
    min_mags.append(min_mag)

    mean_mag_g = np.nanmean(obs_mag.T[0])
    mean_mag_r = np.nanmean(obs_mag.T[1])
    mean_mag = np.nanmean(obs_mag)
    mean_mags_g.append(mean_mag_g)
    mean_mags_r.append(mean_mag_r)
    mean_mags.append(mean_mag)

    amplitudes_g.append(
#         np.nanmax(obs_mag[0]) - np.nanmin(obs_mag.T[0])
        amplitude_g
    )
    amplitudes_r.append(
#         np.nanmax(obs_mag[1]) - np.nanmin(obs_mag.T[1])
        amplitude_r
    )
    amplitudes.append(
#         np.nanmax(obs_mag) - np.nanmin(obs_mag)
        amplitude
    )

    length_lc.append(len(t))

    obs_ = np.where(obs != 0, obs, np.inf)
    _tmm_g = float(t[np.where(obs_.T[0] == obs_.T[0].min())[0]]) \
        if len(t[np.where(obs_.T[0] == obs_.T[0].min())[0]]) <= 1 else \
        float(t[np.where(obs_.T[0] == obs_.T[0].min())[0]][0])
    _tmm_r = float(t[np.where(obs_.T[1] == obs_.T[1].min())[0]]) \
        if len(t[np.where(obs_.T[1] == obs_.T[1].min())[0]]) <= 1 else \
        float(t[np.where(obs_.T[1] == obs_.T[1].min())[0]][0])
    t_magmin_index_g = np.where(obs_mag.T[0] == min_mag_g)[0]
    t_magmin_index_r = np.where(obs_mag.T[1] == min_mag_r)[0]
    if len(t_magmin_index_g) > 1:
        print(f'[warning] t_magmin_index_g had more than 1 entry for {oi}, selecting the first entry')
        t_magmin_index_g = t_magmin_index_g[0]
    if len(t_magmin_index_r) > 1:
        print(f'[warning] t_magmin_index_r had more than 1 entry for {oi}, selecting the first entry')
        t_magmin_index_r = t_magmin_index_r[0] 
    tmm_g = float(t[t_magmin_index_g])
    tmm_r = float(t[t_magmin_index_r])
    assert _tmm_g == tmm_g
    assert _tmm_r == tmm_r
    tmms_r.append(tmm_r)
    tmms_g.append(tmm_g)

    mask_g = obs_mask.T[0]
    mask_r = obs_mask.T[1]

    tmin_index_g = np.where(mask_g == 1)[0][0]
    tmin_index_r = np.where(mask_r == 1)[0][0]
    tmin_g = float(t[tmin_index_g])
    tmin_r = float(t[tmin_index_r])
    tmin = t.min()
    assert tmin == 0.

    tmins_r.append(tmin_r)
    tmins_g.append(tmin_g)

    time_min_mag_g.append(tmm_g - tmin_g)  # not tmm/24 because time_in_hrs=False above.
    time_min_mag_r.append(tmm_r - tmin_r)  # not tmm/24 because time_in_hrs=False above.
    t_magmin_index = np.where(obs_mag == min_mag)[0]
    tmm = float(t[t_magmin_index])
    time_min_mag.append(tmm - tmin)

    tmax_index_g = np.where(mask_g == 1)[0][-1]
    tmax_index_r = np.where(mask_r == 1)[0][-1]
    assert not np.isnan(obs_mag[tmax_index_g][0])
    assert not np.isnan(obs_mag[tmax_index_r][1])
    tmax_g = float(t[tmax_index_g])
    tmax_r = float(t[tmax_index_r])
    tmax = t.max()

    tmaxs_r.append(tmax_r)
    tmaxs_g.append(tmax_g)

    assert tmax_index_g >= t_magmin_index_g
    assert tmax_index_r >= t_magmin_index_r

    assert tmax_g == tmax or tmax_r == tmax

    decline_rate_g.append(
        (float(obs_mag.T[0][tmax_index_g] - min_mag_g)) / (tmax_g - tmm_g) \
#         2*amplitude_g/(tmax_g - tmm_g)
        # because decline rate makes sense only if its declining
        if tmax_g > tmm_g else np.nan
    )
    decline_rate_r.append(
        (float(obs_mag.T[1][tmax_index_r] - min_mag_r)) / (tmax_r - tmm_r) \
#         2*amplitude_r/(tmax_r - tmm_r)
        # because decline rate makes sense only if its declining
        if tmax_r > tmm_r else np.nan
    )
    decline_rate_g_modified.append(
        (float(obs_mag.T[0][tmax_index_g] - min_mag)) / (tmax_g - tmm) \
#         2*amplitude_g/(tmax_g - tmm_g)
        # because decline rate makes sense only if its declining
        if tmax_g > tmm_g else np.nan
    )
    decline_rate_r_modified.append(
        (float(obs_mag.T[1][tmax_index_r] - min_mag)) / (tmax_r - tmm) \
#         2*amplitude_r/(tmax_r - tmm_r)
        # because decline rate makes sense only if its declining
        if tmax_r > tmm_r else np.nan
    )
    decline_rate.append(
        (float(np.max(obs[np.where(t == tmax)[0]])) - min_mag) / (tmax - tmm) \
        # because decline rate makes sense only if its declining
        if tmax > tmm else np.nan
    )

    decline_rate_g_numerator.append(
        (float(obs_mag.T[0][tmax_index_g] - min_mag_g)) if tmax_g > tmm_g else np.nan
    )
    decline_rate_r_numerator.append(
        (float(obs_mag.T[1][tmax_index_r] - min_mag_r)) if tmax_r > tmm_r else np.nan
    )

    assert (tmin_index_g == 0) or (tmin_index_r == 0)

    assert t_magmin_index_g >= tmin_index_g
    assert t_magmin_index_r >= tmin_index_r

    assert not np.isnan(obs_mag[tmin_index_g][0])
    assert not np.isnan(obs_mag[tmin_index_r][1])

    dur.append(t[-1] - t[0])
    dur_g.append(tmax_g - tmin_g)
    dur_r.append(tmax_r - tmin_r)

    sigma_mag_g = np.nanstd(obs_mag.T[0])
    sigma_mag_r = np.nanstd(obs_mag.T[1])
    sigmas_mag_g.append(sigma_mag_g)
    sigmas_mag_r.append(sigma_mag_r)
    sigma_mag = np.nanstd(obs_mag)
    sigmas_mag.append(sigma_mag)

    excess_var_g = sigma_mag_g**2 / mean_mag_g**2
    excess_var_r = sigma_mag_r**2 / mean_mag_r**2
    excess_var = sigma_mag**2 / mean_mag**2
    excess_vars_g.append(excess_var_g)
    excess_vars_r.append(excess_var_r)
    excess_vars.append(excess_var)

    color_at_peak1.append(min_mag_g - min_mag_r)

    max_mag_g = np.nanmax(obs_mag.T[0])
    max_mag_r = np.nanmax(obs_mag.T[1])
    color_at_faintest.append(max_mag_g - max_mag_r)
    colors_at_peak_minus_color_at_faintest.append(
        (min_mag_g - min_mag_r) - (max_mag_g - max_mag_r)
    )

    delta_t_peak.append(tmm_g - tmm_r)

#     mask_g_for_siegelslopes = mask_g[tmin_index_g:tmax_index_g+1]
#     mask_r_for_siegelslopes = mask_r[tmin_index_r:tmax_index_r+1]

#     t_g_for_siegelslopes = t.squeeze()#[mask_g_for_siegelslopes]
#     t_r_for_siegelslopes = t.squeeze()#[mask_r_for_siegelslopes]

#     if (
#         np.all(np.isclose(t_g_for_siegelslopes, t_g_for_siegelslopes[0])) or
#         np.all(np.isclose(t_r_for_siegelslopes, t_r_for_siegelslopes[0]))
#     ):
#         linear_slopes_g.append(np.nan)
#         linear_slopes_r.append(np.nan)
#         continue

    valid_indices_g = ~np.isnan(obs_mag.T[0])
    valid_indices_r = ~np.isnan(obs_mag.T[1])

    # NOTE: scipy==1.12.0 is used, which takes y values first and then x values.
    linear_slope_g, _ = siegelslopes(
        obs_mag.T[0][valid_indices_g], t.squeeze()[valid_indices_g] #nan_policy='omit'
    )
    linear_slope_r, _ = siegelslopes(
        obs_mag.T[1][valid_indices_r], t.squeeze()[valid_indices_r], #nan_policy='omit'
    )
    linear_slopes_g.append(linear_slope_g)
    linear_slopes_r.append(linear_slope_r)

#     color_at_faintest_minus_color_at_peak = (
#         np.median(obs_mag_sorted_g_nonan[int(-math.ceil(0.05 * ng)):]) - \
#         np.median(obs_mag_sorted_r_nonan[int(-math.ceil(0.05 * nr)):])
#     ) - (
#         np.median(obs_mag_sorted_g_nonan[0:int(math.ceil(0.05 * ng))]) - \
#         np.median(obs_mag_sorted_r_nonan[0:int(math.ceil(0.05 * nr))])
#     )
#     colors_at_faintest_minus_color_at_peak.append(color_at_faintest_minus_color_at_peak)

print(len(uncorrected_agn_mask), sum(uncorrected_agn_mask))
print(len(corrected_agn_mask), sum(corrected_agn_mask))

In [ ]:
np.all(
    np.array(test_used_magpsf_corr_arr[agn_indicator]) == np.array(corrected_agn_mask)[agn_indicator]
)

In [ ]:
# mv = (
#     np.sqrt(np.array(excess_vars_g)[general_agn_or_not]) + \
#     np.sqrt(np.array(excess_vars_r)[general_agn_or_not])
# ) / 2
# mv.shape, test_objIds[agn_mask].shape

In [ ]:
# print(sum(type2_or_not), sum(type1_or_not))
# print(mv[type1_or_not].mean(), mv[type2_or_not].mean())
# print(np.median(mv[type1_or_not]), np.median(mv[type2_or_not]))

In [ ]:
matplotlib.rcParams["font.size"] = "12"

In [ ]:
# # SN
# fig, ax = plt.subplots(1, 2, figsize=(width, height))
# # scg = ax[0].scatter(
# #     test_outputs_condensed_condensed_transformed[sn_mask, 0],
# #     test_outputs_condensed_condensed_transformed[sn_mask, 1],
# #     c=np.array(excess_vars_g)[sn_mask]*1e5,
# #     s=150, alpha=0.3, norm=matplotlib.colors.LogNorm()
# # )
# scg = ax[0].hexbin(
#     test_outputs_condensed_condensed_transformed[sn_mask, 0],
#     test_outputs_condensed_condensed_transformed[sn_mask, 1],
#     C=np.array(excess_vars_g)[sn_mask]*1e5, #vmin=np.min(excess_vars_g)*1e5, vmax=np.max(excess_vars_g)*1e5,
# #     norm=matplotlib.colors.LogNorm(),
#     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
# )

# # scr = ax[1].scatter(
# #     test_outputs_condensed_condensed_transformed[sn_mask, 0],
# #     test_outputs_condensed_condensed_transformed[sn_mask, 1],
# #     c=np.array(excess_vars_r)[sn_mask]*1e5,
# #     s=150, alpha=0.3, norm=matplotlib.colors.LogNorm()
# # )
# scr = ax[1].hexbin(
#     test_outputs_condensed_condensed_transformed[sn_mask, 0],
#     test_outputs_condensed_condensed_transformed[sn_mask, 1],
#     C=np.array(excess_vars_r)[sn_mask]*1e5, #vmin=np.min(excess_vars_r)*1e5, vmax=np.max(excess_vars_r)*1e5,
# #     norm=matplotlib.colors.LogNorm(),
#     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
# )

# for a in ax:
#     a.set_xlabel('UMAP 1', fontsize=axeslabelsize)
#     a.set_ylabel('UMAP 2', fontsize=axeslabelsize)
#     a.tick_params(axis='x', labelsize=ticklabelsize)
#     a.tick_params(axis='y', labelsize=ticklabelsize)

# for i, sc in enumerate([scg, scr]):
#     cbar = plt.colorbar(sc)
#     cbar.set_label(
#         r'$\mathrm{ExcessVar}_g (\times10^5)$' if i == 0 else r'$\mathrm{ExcessVar}_r (\times10^5)$', rotation=270,
#         labelpad=30, fontsize=axeslabelsize
#     )
#     cbar.ax.tick_params(labelsize=ticklabelsize)

# # AGN
fig, ax = plt.subplots(1, 2, figsize=(width*2, height))
# scg = ax[0].scatter(
#     test_outputs_condensed_condensed_transformed[agn_mask, 0],
#     test_outputs_condensed_condensed_transformed[agn_mask, 1],
#     c=np.array(excess_vars_g)[agn_mask]*1e5,
#     s=150, alpha=0.3, norm=matplotlib.colors.LogNorm()
# )
scg = ax[0].hexbin(
    test_outputs_condensed_condensed_transformed[agn_mask, 0],
    test_outputs_condensed_condensed_transformed[agn_mask, 1],
    C=np.array(excess_vars_g)[agn_mask]*1e5, #vmin=np.min(excess_vars_g)*1e5, vmax=np.max(excess_vars_g)*1e5,
    norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)
# scr = ax[1].scatter(
#     test_outputs_condensed_condensed_transformed[agn_mask, 0],
#     test_outputs_condensed_condensed_transformed[agn_mask, 1],
#     c=np.array(excess_vars_r)[agn_mask]*1e5,
#     s=150, alpha=0.3, norm=matplotlib.colors.LogNorm()
# )
scr = ax[1].hexbin(
    test_outputs_condensed_condensed_transformed[agn_mask, 0],
    test_outputs_condensed_condensed_transformed[agn_mask, 1],
    C=np.array(excess_vars_r)[agn_mask]*1e5, #vmin=np.min(excess_vars_r)*1e5, vmax=np.max(excess_vars_r)*1e5,
    norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

for a in ax:
    a.set_xlabel('UMAP 1', fontsize=axeslabelsize)
    a.set_ylabel('UMAP 2', fontsize=axeslabelsize)
    a.tick_params(axis='x', labelsize=ticklabelsize)
    a.tick_params(axis='y', labelsize=ticklabelsize)

for i, sc in enumerate([scg, scr]):
    cbar = plt.colorbar(sc)
    cbar.set_label(
        r'$\mathrm{ExcessVar}_g (\times10^5)$' if i == 0 else r'$\mathrm{ExcessVar}_r (\times10^5)$', rotation=270,
        labelpad=30, fontsize=axeslabelsize
    )
    cbar.ax.tick_params(labelsize=ticklabelsize)


# ===============================================
# for i, obs_filter in enumerate(['g', 'r']):
#     color_array = np.array(excess_vars_g)*1e5 if i == 0 else np.array(excess_vars_r)*1e5

#     fig = plt.figure(figsize=(width/2, height/2))
#     ax = plt.gca()
#     sc = ax.hexbin(
#         test_outputs_condensed_condensed_transformed[:, 0],
#         test_outputs_condensed_condensed_transformed[:, 1],
#         C=color_array, #vmin=np.min(excess_vars_g)*1e5, vmax=np.max(excess_vars_g)*1e5,
#     #     norm=matplotlib.colors.LogNorm(),
#         mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
#     )

#     ax.tick_params(axis='x')#, labelsize=ticklabelsize)
#     ax.tick_params(axis='y')#, labelsize=ticklabelsize)

#     #if i == 0:
#     ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
#     ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

# #     ax.text(
# #         .93, .96, r'$g$' if i == 0 else r'$r$',
# #         ha='left', va='top', transform=ax.transAxes
# #     )

#     cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
#     cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)
#     cbar.ax.xaxis.set_ticks_position('top')
#     cbar.ax.xaxis.set_label_position('top')
#     cbar.set_label(
#         r'$\mathrm{ExcessVar}\,(g)\,(\times10^5)$' if i == 0 else r'$\mathrm{ExcessVar}\,(r)\,(\times10^5)$',
#         labelpad=10#, fontsize=axeslabelsize
#     )
#     cbar.ax.tick_params()#labelsize=ticklabelsize)

#     set_axes_style(ax)

#     save_fig(f'projection_general_excessvar_{obs_filter}.pdf')

In [ ]:
# fig, ax = plt.subplots(1, 2, figsize=(20, 8))
# scg = ax[0].scatter(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     c=min_mags_g, alpha=0.5
# )
# scr = ax[1].scatter(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     c=min_mags_r, alpha=0.5
# )
# for a in ax:
#     a.set_xlabel('UMAP 1', fontsize=axeslabelsize)
#     a.set_ylabel('UMAP 2', fontsize=axeslabelsize)
#     a.tick_params(axis='x', labelsize=ticklabelsize)
#     a.tick_params(axis='y', labelsize=ticklabelsize)

# for i, sc in enumerate([scg, scr]):
#     cbar = plt.colorbar(sc)
#     cbar.set_label(
#         r'$m_{g, \mathrm{peak}}$' if i == 0 else r'$m_{r, \mathrm{peak}}$', rotation=270,
#         labelpad=30, fontsize=axeslabelsize
#     )
#     cbar.ax.tick_params(labelsize=ticklabelsize)

# ==================================================================
fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][~np.array(uncorrected_agn_mask)],
    test_outputs_condensed_condensed_transformed[:, 1][~np.array(uncorrected_agn_mask)],
    C=np.array(min_mags)[~np.array(uncorrected_agn_mask)],
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

ax.set_xticks([4, 6, 8, 10, 12, 14])

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    r'$m_{\mathrm{peak}}\,(g, r)$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)

set_axes_style(ax)

save_fig(f'projection_general_peak_mag.pdf')

In [ ]:
# fig, ax = plt.subplots(1, 2, figsize=(width*2/2, height/2))
# # scg = ax[0].scatter(
# #     test_outputs_condensed_condensed_transformed[:, 0],
# #     test_outputs_condensed_condensed_transformed[:, 1],
# #     c=mean_mags_g, alpha=0.5
# # )
# # scr = ax[1].scatter(
# #     test_outputs_condensed_condensed_transformed[:, 0],
# #     test_outputs_condensed_condensed_transformed[:, 1],
# #     c=mean_mags_r, alpha=0.5
# # )

# scg = ax[0].hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     C=np.array(mean_mags_g), #norm=matplotlib.colors.LogNorm(),
#     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
# )
# scr = ax[1].hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     C=np.array(mean_mags_r), #norm=matplotlib.colors.LogNorm(),
#     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
# )

# for a in ax:
#     a.tick_params(axis='x')#, labelsize=ticklabelsize)
#     a.tick_params(axis='y')#, labelsize=ticklabelsize)

# ax[0].set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
# ax[0].set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

# for i, sc in enumerate([scg, scr]):
#     cbar = plt.colorbar(sc)
#     cbar.set_label(
#         r'$\mu_g(m)$' if i == 0 else r'$\mu_r(m)$', rotation=270,
#         labelpad=15, #fontsize=axeslabelsize
#     )
#     cbar.ax.tick_params()#labelsize=ticklabelsize)
# save_fig('projection_general_mean_mag.pdf')


# =======================================================
color_array = (np.array(mean_mags_g) + np.array(mean_mags_r)) / 2

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][~np.array(uncorrected_agn_mask)],
    test_outputs_condensed_condensed_transformed[:, 1][~np.array(uncorrected_agn_mask)],
    C=color_array[~np.array(uncorrected_agn_mask)],
#     norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

#if i == 0:
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

ax.set_xticks([4, 6, 8, 10, 12, 14])

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
ticks = [18, 18.5, 19, 19.5, 20]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=ticks)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.ax.set_xticklabels(ticks)
cbar.set_label(
    r'$\mu\,\,(g, r)$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)

set_axes_style(ax)

save_fig(f'projection_general_mean_mag.pdf')

In [ ]:
fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0],
    test_outputs_condensed_condensed_transformed[:, 1],
    C=np.array(amplitudes),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    r'$m_{\mathrm{max}} - m_{\mathrm{peak}}\,(g, r)$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)

set_axes_style(ax)

save_fig(f'projection_general_peak_minus_max_mag.pdf')



color_array = (np.array(amplitudes_g) + np.array(amplitudes_r)) / 2

vmin = np.quantile(color_array, 0.02)
vmax = np.quantile(color_array, 0.98)

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][~np.array(uncorrected_agn_mask)],
    test_outputs_condensed_condensed_transformed[:, 1][~np.array(uncorrected_agn_mask)],
    C=color_array[~np.array(uncorrected_agn_mask)], vmin=vmin, vmax=vmax,
#     norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

#if i == 0:
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

ax.set_xticks([4, 6, 8, 10, 12, 14])

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=[0.2, 0.6, 1., 1.4, 1.8])
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    'Amplitude '+ r'$(g, r)$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)

set_axes_style(ax)

save_fig(f'projection_general_amplitude.pdf')

In [ ]:
# fig, ax = plt.subplots(1, 2, figsize=(22, 7))
# # scg = ax[0].scatter(
# #     test_outputs_condensed_condensed_transformed[:, 0],
# #     test_outputs_condensed_condensed_transformed[:, 1],
# #     c=amplitudes_g, alpha=0.5
# # )
# scg = ax[0].hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     C=np.array(amplitudes_g),
#     mincnt=1, edgecolor='grey', cmap='OrRd', gridsize=20
# )

# # scr = ax[1].scatter(
# #     test_outputs_condensed_condensed_transformed[:, 0],
# #     test_outputs_condensed_condensed_transformed[:, 1],
# #     c=amplitudes_r, alpha=0.5
# # )
# scr = ax[1].hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     C=np.array(amplitudes_r),
#     mincnt=1, edgecolor='grey', cmap='OrRd', gridsize=20
# )

# for a in ax:
#     a.set_xlabel('UMAP 1', fontsize=axeslabelsize)
#     a.set_ylabel('UMAP 2', fontsize=axeslabelsize)
#     a.tick_params(axis='x', labelsize=ticklabelsize)
#     a.tick_params(axis='y', labelsize=ticklabelsize)

# for i, sc in enumerate([scg, scr]):
#     cbar = plt.colorbar(sc)
#     cbar.set_label(
# #         r'$m_{g, \mathrm{max}} - m_{g, \mathrm{peak}}$' if i == 0 else r'$m_{r, \mathrm{max}} - m_{r, \mathrm{peak}}$', rotation=270,
#         r'$A (g)$' if i == 0 else r'$A (r)$', rotation=270,
#         labelpad=30, fontsize=axeslabelsize
#     )
#     cbar.ax.tick_params(labelsize=ticklabelsize)

In [ ]:
# fig = plt.figure(figsize=(width/2, height/2))
# ax = plt.gca()
# sc = ax.hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0][sn_mask][np.array(dur_r)[sn_mask] < 50],
#     test_outputs_condensed_condensed_transformed[:, 1][sn_mask][np.array(dur_r)[sn_mask] < 50],
#     C=(np.array(length_lc)/2)[sn_mask][np.array(dur_r)[sn_mask] < 50],
#     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20,
#     vmin=0, vmax=10
# )

# ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
# ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
# ax.tick_params(axis='x')#, labelsize=ticklabelsize)
# ax.tick_params(axis='y')#, labelsize=ticklabelsize)

# cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
# cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)
# cbar.ax.xaxis.set_ticks_position('top')
# cbar.ax.xaxis.set_label_position('top')
# cbar.set_label(
#     r'$\langle L \rangle$',
#     labelpad=10#, fontsize=axeslabelsize
# )
# cbar.ax.tick_params()#labelsize=ticklabelsize)

# set_axes_style(ax)

In [ ]:
fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][~np.array(uncorrected_agn_mask)],
    test_outputs_condensed_condensed_transformed[:, 1][~np.array(uncorrected_agn_mask)],
    C=(np.array(length_lc)/2)[~np.array(uncorrected_agn_mask)],
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

ax.set_xticks([4, 6, 8, 10, 12, 14])

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    r'$\langle L \rangle$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)

set_axes_style(ax)

save_fig(f'projection_general_average_number_of_datapoints.pdf')

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 8))
sc = ax.scatter(
    test_outputs_condensed_condensed_transformed[:, 0],
    test_outputs_condensed_condensed_transformed[:, 1],
    c=(np.array(length_lc)/2)/np.array(dur).squeeze(), alpha=0.5,
    norm=matplotlib.colors.LogNorm()
)
cbar = plt.colorbar(sc)
ax.set_xlabel('UMAP 1', fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2', fontsize=axeslabelsize)
ax.tick_params(axis='x', labelsize=ticklabelsize)
ax.tick_params(axis='y', labelsize=ticklabelsize)
cbar.ax.set_ylabel(
    r'$\frac{\langle L \rangle}{Duration}$', rotation=270,
    labelpad=30, fontsize=axeslabelsize
)
cbar.ax.tick_params(labelsize=ticklabelsize)
set_axes_style(ax)

In [ ]:
# fig, ax = plt.subplots(1, 2, figsize=(width*2/2, height/2))
# # scg = ax[0].scatter(
# #     test_outputs_condensed_condensed_transformed[:, 0],
# #     test_outputs_condensed_condensed_transformed[:, 1],
# #     c=time_min_mag_g, alpha=0.5
# # )
# scg = ax[0].hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     C=np.array(time_min_mag_g), #norm=matplotlib.colors.LogNorm(),
#     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
# )
# # scr = ax[1].scatter(
# #     test_outputs_condensed_condensed_transformed[:, 0],
# #     test_outputs_condensed_condensed_transformed[:, 1],
# #     c=time_min_mag_r, alpha=0.5
# # )
# scr = ax[1].hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     C=np.array(time_min_mag_r), #norm=matplotlib.colors.LogNorm(),
#     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
# )
# for a in ax:
#     a.tick_params(axis='x')#, labelsize=ticklabelsize-9)
#     a.tick_params(axis='y')#, labelsize=ticklabelsize-9)

# ax[0].set_xlabel('UMAP 1',)# fontsize=axeslabelsize-9)
# ax[0].set_ylabel('UMAP 2',)# fontsize=axeslabelsize-9)

# for i, sc in enumerate([scg, scr]):
#     cbar = plt.colorbar(sc)
#     cbar.set_label(
#         r'$t_{\mathrm{peak}} (g)$ (days)' if i == 0 else r'$t_{\mathrm{peak}} (r)$ (days)', rotation=270,
#         labelpad=15, #fontsize=axeslabelsize-9
#     )
#     cbar.ax.tick_params()#labelsize=ticklabelsize-9)

# save_fig('projection_general_time_of_peak_mag.pdf')

# # fig, ax = plt.subplots(1, 1, figsize=(15, 8))
# # sc = ax.scatter(
# #     test_outputs_condensed_condensed_transformed[:, 0],
# #     test_outputs_condensed_condensed_transformed[:, 1],
# #     c=time_min_mag, alpha=0.5
# # )
# # cbar = plt.colorbar(sc)
# # ax.set_xlabel('UMAP 1', fontsize=axeslabelsize)
# # ax.set_ylabel('UMAP 2', fontsize=axeslabelsize)
# # ax.tick_params(axis='x', labelsize=ticklabelsize)
# # ax.tick_params(axis='y', labelsize=ticklabelsize)
# # cbar.ax.set_ylabel(
# #     r'$t_{m_{\mathrm{peak}}}$ (days)', rotation=270,
# #     labelpad=30, fontsize=axeslabelsize
# # )
# # cbar.ax.tick_params(labelsize=ticklabelsize)
# save_fig('projection_time_of_peak_mag.pdf')

# ======================================================
color_array = (np.array(time_min_mag_g) + np.array(time_min_mag_r)) / 2

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][~np.array(uncorrected_agn_mask)],
    test_outputs_condensed_condensed_transformed[:, 1][~np.array(uncorrected_agn_mask)],
    C=color_array[~np.array(uncorrected_agn_mask)],
#     norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

#if i == 0:
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

ax.set_xticks([4, 6, 8, 10, 12, 14])

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=[100, 200, 300, 400])
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    r'$t_{\mathrm{peak}}\,\,(g, r)$ (days)',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)

set_axes_style(ax)

save_fig(f'projection_general_time_of_peak_mag.pdf')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(width*2, height))
# scg = ax[0].scatter(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     c=time_min_mag_g, alpha=0.5
# )
assert np.all(
    np.array(time_min_mag_g)/np.array(dur_g) <= 1
) and np.all(
    np.array(time_min_mag_g)/np.array(dur_g) >= 0
)
assert np.all(
    np.array(time_min_mag_r)/np.array(dur_r) <= 1
) and np.all(
    np.array(time_min_mag_r)/np.array(dur_r) >= 0
)

scg = ax[0].hexbin(
    test_outputs_condensed_condensed_transformed[:, 0],
    test_outputs_condensed_condensed_transformed[:, 1],
    C=np.array(time_min_mag_g)/np.array(dur_g), norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)
# scr = ax[1].scatter(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     c=time_min_mag_r, alpha=0.5
# )
scr = ax[1].hexbin(
    test_outputs_condensed_condensed_transformed[:, 0],
    test_outputs_condensed_condensed_transformed[:, 1],
    C=np.array(time_min_mag_r)/np.array(dur_r), norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)
for a in ax:
    a.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    a.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
    a.tick_params(axis='x')#, labelsize=ticklabelsize)
    a.tick_params(axis='y')#, labelsize=ticklabelsize)

for i, sc in enumerate([scg, scr]):
    cbar = plt.colorbar(sc)
    cbar.set_label(
        r'$t_{m_{g, \mathrm{peak}}}  / Duration$ (days)' if i == 0 else r'$t_{m_{r, \mathrm{peak}}} / Duration$ (days)', rotation=270,
        labelpad=35, #fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)

for a in ax:
    set_axes_style(a)

save_fig('projection_time_of_peak_mag_by_duration.pdf')

# fig, ax = plt.subplots(1, 1, figsize=(15, 8))
# sc = ax.scatter(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     c=time_min_mag, alpha=0.5
# )
# cbar = plt.colorbar(sc)
# ax.set_xlabel('UMAP 1', fontsize=axeslabelsize)
# ax.set_ylabel('UMAP 2', fontsize=axeslabelsize)
# ax.tick_params(axis='x', labelsize=ticklabelsize)
# ax.tick_params(axis='y', labelsize=ticklabelsize)
# cbar.ax.set_ylabel(
#     r'$t_{m_{\mathrm{peak}}}$ (days)', rotation=270,
#     labelpad=30, fontsize=axeslabelsize
# )
# cbar.ax.tick_params(labelsize=ticklabelsize)

# save_fig('projection_time_of_peak_mag_by_duration.pdf')

In [ ]:
# # fig, ax = plt.subplots(1, 2, figsize=(width*2/2, height/2))
# # # sc = ax.scatter(
# # #     test_outputs_condensed_condensed_transformed[:, 0],
# # #     test_outputs_condensed_condensed_transformed[:, 1],
# # #     c=dur, alpha=0.5
# # # )
# # sc = ax.hexbin(
# #     test_outputs_condensed_condensed_transformed[:, 0],
# #     test_outputs_condensed_condensed_transformed[:, 1],
# #     C=np.array(dur), #norm=matplotlib.colors.LogNorm(),
# #     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
# # )
# # cbar = plt.colorbar(sc)
# # ax.set_xlabel('UMAP 1', fontsize=axeslabelsize)
# # ax.set_ylabel('UMAP 2', fontsize=axeslabelsize)
# # ax.tick_params(axis='x', labelsize=ticklabelsize)
# # ax.tick_params(axis='y', labelsize=ticklabelsize)
# # cbar.ax.set_ylabel(
# #     'Duration (days)', rotation=270,
# #     labelpad=30, fontsize=axeslabelsize
# # )
# # cbar.ax.tick_params(labelsize=ticklabelsize)
# # save_fig('projection_general_duration.pdf')

# # ============================================
# fig, ax = plt.subplots(1, 2, figsize=(width*2/2, height/2))
# # scg = ax[0].scatter(
# #     test_outputs_condensed_condensed_transformed[:, 0],
# #     test_outputs_condensed_condensed_transformed[:, 1],
# #     c=time_min_mag_g, alpha=0.5
# # )
# scg = ax[0].hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     C=np.array(dur_g), #norm=matplotlib.colors.LogNorm(),
#     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
# )
# # scr = ax[1].scatter(
# #     test_outputs_condensed_condensed_transformed[:, 0],
# #     test_outputs_condensed_condensed_transformed[:, 1],
# #     c=time_min_mag_r, alpha=0.5
# # )
# scr = ax[1].hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     C=np.array(dur_r), #norm=matplotlib.colors.LogNorm(),
#     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
# )
# for a in ax:
#     a.tick_params(axis='x')#, labelsize=ticklabelsize-9)
#     a.tick_params(axis='y')#, labelsize=ticklabelsize-9)

# ax[0].set_xlabel('UMAP 1',)# fontsize=axeslabelsize-9)
# ax[0].set_ylabel('UMAP 2',)# fontsize=axeslabelsize-9)

# for i, sc in enumerate([scg, scr]):
#     cbar = plt.colorbar(sc)
#     cbar.set_label(
#         'Duration '+ r'$(g)$ '+ '(days)' if i == 0 else 'Duration '+ r'$(r)$ '+ '(days)', rotation=270,
#         labelpad=15, #fontsize=axeslabelsize-9
#     )
#     cbar.ax.tick_params()#labelsize=ticklabelsize-9)

# save_fig('projection_general_duration.pdf')


# ==========================================
color_array = (np.array(dur_g) + np.array(dur_r)) / 2

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][~np.array(uncorrected_agn_mask)],
    test_outputs_condensed_condensed_transformed[:, 1][~np.array(uncorrected_agn_mask)],
    C=color_array[~np.array(uncorrected_agn_mask)],
#     norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

#if i == 0:
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

ax.set_xticks([4, 6, 8, 10, 12, 14])

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=[0, 100, 200, 300, 400])
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    'Duration '+ r'$(g, r)$ '+ '(days)',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)

set_axes_style(ax)

save_fig(f'projection_general_duration.pdf')

In [ ]:
#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
fig, ax = plt.subplots(figsize=(width/2, height/2))
# sc = ax.hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0][~np.array(uncorrected_agn_mask)],
#     test_outputs_condensed_condensed_transformed[:, 1][~np.array(uncorrected_agn_mask)],
#     C=color_array[~np.array(uncorrected_agn_mask)],
# #     norm=matplotlib.colors.LogNorm(),
#     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
# )

sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[agn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[agn_mask, 1],
    fill=True, cmap=sns.light_palette(orangish_color, as_cmap=True), ax=ax,
    levels=15, thresh=0.01, alpha=0.4,
)
sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap=sns.light_palette(bluish_color, as_cmap=True), ax=ax,
    levels=15, thresh=0.01, alpha=0.4,
)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=orangish_color, edgecolor='none', alpha=0.4, label='AGN'),
    Patch(facecolor=bluish_color, edgecolor='none', alpha=0.4, label='SN'),
]
ax.legend(handles=legend_elements, loc='upper left', frameon=False)

ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

#if i == 0:
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

ax.set_xticks([4, 6, 8, 10, 12, 14])

set_axes_style(ax)

save_fig(f'projection_general_SN_AGN_separation.pdf')

In [ ]:
# fig, ax = plt.subplots(1, 2, figsize=(width*2/2, height/2))
# # scg = ax[0].scatter(
# #     test_outputs_condensed_condensed_transformed[:, 0],
# #     test_outputs_condensed_condensed_transformed[:, 1],
# #     c=sigmas_mag_g, alpha=0.5
# # )
# scg = ax[0].hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     C=np.array(sigmas_mag_g), #norm=matplotlib.colors.LogNorm(),
#     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
# )
# # scr = ax[1].scatter(
# #     test_outputs_condensed_condensed_transformed[:, 0],
# #     test_outputs_condensed_condensed_transformed[:, 1],
# #     c=sigmas_mag_r, alpha=0.5
# # )
# scr = ax[1].hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     C=np.array(sigmas_mag_r), #norm=matplotlib.colors.LogNorm(),
#     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
# )
# for a in ax:
#     a.tick_params(axis='x')#, labelsize=ticklabelsize)
#     a.tick_params(axis='y')#, labelsize=ticklabelsize)

# ax[0].set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
# ax[0].set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

# for i, sc in enumerate([scg, scr]):
#     cbar = plt.colorbar(sc)
#     cbar.set_label(
#         r'$\sigma_g(m)$' if i == 0 else r'$\sigma_r(m)$', rotation=270,
#         labelpad=15,# fontsize=axeslabelsize
#     )
#     cbar.ax.tick_params()#labelsize=ticklabelsize)

# save_fig('projection_general_sigma_mag.pdf')

# ========================================================
for i, obs_filter in enumerate(['g', 'r']):
    color_array = np.array(sigmas_mag_g) if i == 0 else np.array(sigmas_mag_r)

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
    fig = plt.figure(figsize=(width/2, height/2))
    ax = plt.gca()
    sc = ax.hexbin(
        test_outputs_condensed_condensed_transformed[:, 0],
        test_outputs_condensed_condensed_transformed[:, 1],
        C=color_array,
    #     norm=matplotlib.colors.LogNorm(),
        mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
    )

    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    #if i == 0:
    ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

    cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
    cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=[0.1, 0.3, 0.5])
    cbar.ax.xaxis.set_ticks_position('top')
    cbar.ax.xaxis.set_label_position('top')
    cbar.set_label(
        r'$\sigma\,(g)$' if i == 0 else r'$\sigma\,(r)$',
        labelpad=10#, fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)

    set_axes_style(ax)

    save_fig(f'projection_general_sigma_mag_{obs_filter}.pdf')

In [ ]:
# fig, ax = plt.subplots(1, 1, figsize=(11, 7))
# # scg = ax.scatter(
# #     test_outputs_condensed_condensed_transformed[:, 0],
# #     test_outputs_condensed_condensed_transformed[:, 1],
# #     c=np.array(color_at_peak1), alpha=0.5
# # )
# sc = ax.hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     C=np.array(color_at_peak1),
#     mincnt=3, edgecolor='grey', cmap='RdYlBu_r', gridsize=20
# )

# ax.set_xlabel('UMAP 1', fontsize=axeslabelsize)
# ax.set_ylabel('UMAP 2', fontsize=axeslabelsize)
# ax.tick_params(axis='x', labelsize=ticklabelsize)
# ax.tick_params(axis='y', labelsize=ticklabelsize)

# cbar = plt.colorbar(sc)
# cbar.set_label(
#     r'$(g - r)_{\mathrm{peak}}$', rotation=270,
#     labelpad=30, fontsize=axeslabelsize
# )
# cbar.ax.tick_params(labelsize=ticklabelsize)

# ==============================================================
import matplotlib.colors as mcolors

# fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
# sc = ax.scatter(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     c=color_at_peak1, alpha=0.5, cmap='RdYlBu_r'
# )

# Get color range centered around zero
norm = mcolors.TwoSlopeNorm(
    vmin=np.quantile(color_at_peak1, 0.02), vcenter=0, vmax=np.quantile(color_at_peak1, 0.98)
)

sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][~np.array(uncorrected_agn_mask)],
    test_outputs_condensed_condensed_transformed[:, 1][~np.array(uncorrected_agn_mask)],
    C=np.array(color_at_peak1)[~np.array(uncorrected_agn_mask)], norm=norm,
    mincnt=3, edgecolor='grey', cmap='RdYlBu_r', gridsize=20
)
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

ax.set_xticks([4, 6, 8, 10, 12, 14])

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
if use_corrected_agns:
    ticks = [-0.15, -0.05, 0, 0.3, 0.6]
else:
    ticks = [-0.3, -0.15, 0, 0.3, 0.6]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=ticks)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.ax.set_xticklabels(ticks)
cbar.set_label(
    r'$(g - r)_{\mathrm{peak}}$',
    labelpad=10,# fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
cbar.ax.tick_params()#labelsize=ticklabelsize)

set_axes_style(ax)

save_fig(f'projection_general_color_at_peak.pdf')

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(11, 7))
# scg = ax.scatter(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     c=np.array(color_at_peak1), alpha=0.5
# )
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0],
    test_outputs_condensed_condensed_transformed[:, 1],
    C=np.array(delta_t_peak),
    mincnt=3, edgecolor='grey', cmap='RdYlBu_r', gridsize=20
)

ax.set_xlabel('UMAP 1', fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2', fontsize=axeslabelsize)
ax.tick_params(axis='x', labelsize=ticklabelsize)
ax.tick_params(axis='y', labelsize=ticklabelsize)

ax.set_xticks([4, 6, 8, 10, 12, 14])

cbar = plt.colorbar(sc)
cbar.set_label(
    r'$\Delta t_{\mathrm{peak}}$', rotation=270,
    labelpad=30, fontsize=axeslabelsize
)
cbar.ax.tick_params(labelsize=ticklabelsize)
set_axes_style(ax)

In [ ]:
# decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
# decoded_lcs_test_keys = list(decoded_lcs_test.keys())
mean_colors = []
median_colors = []
std_colors = []
color_at_peak = []
color_mag_corrs = []


## Added code block for correction (same as used in the interpolation section below)
decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
if correct_lcs_for_agns:
    decoded_lcs_test_corrected = np.load('evaluate_test_CORRECTED_decoded_lcs.npz')

    assert list(decoded_lcs_test.keys()) == list(decoded_lcs_test_corrected.keys()), "NPZ keys mismatch"

    # Build corrected_agn: start from original, replace AGN rows with corrected
    decoded_lcs_test_corrected_agn = {}
    for i, key in enumerate(decoded_lcs_test.keys()):
        if agn_indicator[i]:
            decoded_lcs_test_corrected_agn[key] = decoded_lcs_test_corrected[key].copy()
            # Verify AGN lc came from corrected
            np.testing.assert_array_equal(
                decoded_lcs_test_corrected_agn[key],
                decoded_lcs_test_corrected[key]
            )
        else:
            decoded_lcs_test_corrected_agn[key] = decoded_lcs_test[key].copy()
            # Verify non-AGN lc is unchanged
            np.testing.assert_array_equal(
                decoded_lcs_test_corrected_agn[key],
                decoded_lcs_test[key]
            )

    print(f"Replaced {agn_indicator.sum()} AGN rows out of {len(agn_indicator)} total")

decoded_lcs_test_keys = list(decoded_lcs_test.keys())
print(decoded_lcs_test.f.arr_2.shape)

use_corrected_agns = True
if use_corrected_agns:
    print('Using corrected AGN light curves...')
    assert np.all(
        np.load('evaluate_test_objIds_dataloader.npy') == test_objIds_corrected_agn
    )
    test_objIds = test_objIds_corrected_agn
    decoded_lcs_test = decoded_lcs_test_corrected_agn
    min_max_mags = np.load('min_max_mags.npy')
    min_max_mags_corrected = np.load('min_max_mags_ftransfer_ztf_2025_05_31_430518_CORRECTED.npy')
else:
    print('Using uncorrected AGN light curves...')
    test_objIds  = np.load('evaluate_test_objIds_dataloader.npy')
    decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
    min_max_mags = np.load('min_max_mags.npy')
####################################################################


for k, key in enumerate(decoded_lcs_test_keys):
    decoded_lc_test = decoded_lcs_test[key]
    decoded_lc_test_pred = decoded_lc_test[0]
    observed_mask = decoded_lc_test[2]

    pred_x, observed_data, observed_mask, observed_times = decoded_lcs_test[key]

    objId = test_objIds[k]

    ## Added code to check corrected for AGN
    if objId in milliquas_agn_labels['objectId'].values:
        test_currCommonFinkclass = milliquas_agn_labels[milliquas_agn_labels['objectId'] == objId]['custom_type']
        assert len(test_currCommonFinkclass) == 1
        test_currCommonFinkclass = test_currCommonFinkclass.iloc[0]
#         if test_currCommonFinkclass == 'nan':
#             test_currCommonFinkclass = 'Unknown'
    else:
        test_currObjId_index = np.where(test_objIds == objId)
        test_currCommonFinkclass = fine_classes[test_currObjId_index[0][0]]
    corrected_agn_condition = (
        (test_currCommonFinkclass in milliquas_agn_labels['custom_type'].unique())
        and (use_corrected_agns)
        and df_alerts_corrected_pandas.loc[df_alerts_corrected_pandas['objectId'] == objId, 'used_magpsf_corr'].all()
    )
    if corrected_agn_condition:
        print(f'Using corrected min/max mags')
    else:
        print('Using uncorrected min/max mags')
    use_min_max_mags = min_max_mags_corrected if corrected_agn_condition else min_max_mags
    ########################################

    row = use_min_max_mags[np.where(use_min_max_mags[:, 0] == objId)].squeeze()

    # Pass a mask of all ones for prediction because for
    # interpolation all points can be considered observed.
    decoded_lc_test_pred = unnormalize_mag(
        decoded_lc_test_pred, np.ones_like(observed_mask), float(row[1]), float(row[2])
    )
    
    mean_color = (decoded_lc_test_pred[:, 0] - decoded_lc_test_pred[:, 1]).mean()
    median_color = np.median(
        (decoded_lc_test_pred[:, 0] - decoded_lc_test_pred[:, 1])
    )
    std_color = (decoded_lc_test_pred[:, 0] - decoded_lc_test_pred[:, 1]).std()
    mean_colors.append(mean_color)
    std_colors.append(std_color)
    median_colors.append(median_color)
    color_at_peak.append(
        decoded_lc_test_pred[:, 0].min() - decoded_lc_test_pred[:, 1].min()
    )

#     # color-magnitude corr to check bluer-when-brighter (mainly for AGNs)\
    indices = np.argsort(-decoded_lc_test_pred[:, 0])
    color_mag_corr, _ = scipy.stats.spearmanr(
        -decoded_lc_test_pred[:, 0][indices],
        (decoded_lc_test_pred[:, 0] - decoded_lc_test_pred[:, 1])[indices],
        #nan_policy='omit'
    )
    color_mag_corrs.append(color_mag_corr)

In [ ]:
plt.hist(color_mag_corrs)

In [ ]:
# for oi in test_objIds[np.array(color_mag_corrs) > 0.75]:
#     ax = quick_plot_decoded_lc(oi, fig_width=width*2/3, fig_height=height/4)
#     plt.show()

In [ ]:
# fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
# sc = ax.scatter(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     c=mean_colors, alpha=0.5, cmap='RdYlBu_r'
# )

import matplotlib.colors as mcolors

norm = mcolors.TwoSlopeNorm(
    vmin=np.quantile(mean_colors, 0.02), vcenter=0, vmax=np.quantile(mean_colors, 0.98)
)

sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0],
    test_outputs_condensed_condensed_transformed[:, 1],
    C=np.array(mean_colors), norm=norm,
    mincnt=3, edgecolor='grey', cmap='RdYlBu_r', gridsize=20
)
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

ax.set_xticks([4, 6, 8, 10, 12, 14])

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
ticks = [-0.4, -0.2, 0, 0.35, 0.7]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=ticks)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.ax.set_xticklabels(ticks)
cbar.set_label(
    r'$\mu_{\mathrm{g - r}}$',
    labelpad=10,# fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
cbar.ax.tick_params()#labelsize=ticklabelsize)

set_axes_style(ax)

save_fig(f'projection_general_mean_color.pdf')

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 8))
sc = ax.scatter(
    test_outputs_condensed_condensed_transformed[:, 0],
    test_outputs_condensed_condensed_transformed[:, 1],
    c=median_colors, alpha=0.5, cmap='RdYlBu_r'
)
cbar = plt.colorbar(sc)
ax.set_xlabel('UMAP 1', fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2', fontsize=axeslabelsize)
ax.tick_params(axis='x', labelsize=ticklabelsize)
ax.tick_params(axis='y', labelsize=ticklabelsize)
cbar.ax.set_ylabel(
    r'$\mathrm{median}_{\mathrm{g - r}}$ (mag)', rotation=270,
    labelpad=30, fontsize=axeslabelsize
)
cbar.ax.tick_params(labelsize=ticklabelsize)
set_axes_style(ax)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 8))
# sc = ax.scatter(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     c=std_colors, alpha=0.5
# )
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0],
    test_outputs_condensed_condensed_transformed[:, 1],
    C=np.array(std_colors),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=25
)
cbar = plt.colorbar(sc)
ax.set_xlabel('UMAP 1', fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2', fontsize=axeslabelsize)
ax.tick_params(axis='x', labelsize=ticklabelsize)
ax.tick_params(axis='y', labelsize=ticklabelsize)

ax.set_xticks([4, 6, 8, 10, 12, 14])

cbar.ax.set_ylabel(
    r'$\sigma_{\mathrm{g - r}}$ (mag)', rotation=270,
    labelpad=30, fontsize=axeslabelsize
)
cbar.ax.tick_params(labelsize=ticklabelsize)
print(f'Average of sigma_color = {np.mean(std_colors):.3f}')
set_axes_style(ax)

In [ ]:
# plt.hist(std_colors); plt.show()
# plt.hist(np.array(std_colors)[sn_mask]); plt.show()
# plt.hist(np.array(std_colors)[agn_mask]); plt.show()

Mean and median color are similar which means for any light curve the distribution of magnitudes is not heavily skewed towards the left or the right. The standard deviation of color is low for all light curves which means the variability around the mean is low (less than a few-tenths of a magnitude). Low standard deviation of AGNs is expected because their variability is stochastic and color mostly stays the same (mostly constant $g - r$). Also, standard deviation may not capture sudden flares in AGN light curves, and so their $\sigma$ being similar to SNe light curves is expected. Astrophysically, SNe show color evolution so it was expected they have a high $\sigma$, but that's not the case here.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 8))
sc = ax.scatter(
    test_outputs_condensed_condensed_transformed[:, 0],
    test_outputs_condensed_condensed_transformed[:, 1],
    c=np.array(std_colors)/np.squeeze(np.array(dur)), alpha=0.5,
)
cbar = plt.colorbar(sc)
ax.set_xlabel('UMAP 1', fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2', fontsize=axeslabelsize)
ax.tick_params(axis='x', labelsize=ticklabelsize)
ax.tick_params(axis='y', labelsize=ticklabelsize)
cbar.ax.set_ylabel(
    r'$\sigma_{\mathrm{g - r}} / Duration$ (mag/days)', rotation=270,
    labelpad=30, fontsize=axeslabelsize
)
cbar.ax.tick_params(labelsize=ticklabelsize)
set_axes_style(ax)

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

blazar_or_not = []
general_agn_or_not = []
qso_or_not = []
type1_or_not = []
type2_or_not = []
seyfert1_or_not = []

# _raw_finkclasses = df_alerts_raw[['objectId', 'finkclass']].compute()

for i, oi in enumerate(test_objIds):
    # NOTE: For collecting the classes, we don't necessarily need the corrected dataframe.
    _coarse_finkclass = df_alerts_pandas[df_alerts_pandas['objectId'] == oi]['finkclass'].mode().iloc[0]
    if _coarse_finkclass != 'custom_agn':
        general_agn_or_not.append(False)
#         type1_or_not.append(False)
#         type2_or_not.append(False)
#         blazar_or_not.append(False)
#         qso_or_not.append(False)
#         seyfert1_or_not.append(False)
        continue
    general_agn_or_not.append(True)
    ######### If using SIMBAD #########
#     type1_or_not.append(fine_classes_grouped_finer[i] == 'Type I')
#     type2_or_not.append(fine_classes_grouped_finer[i] == 'Type II')
#     blazar_or_not.append(fine_classes_grouped_finer[i] == 'Blazar')
#     qso_or_not.append(
# #         (fine_classes_grouped_finer[i] == 'Type I') &
#         (fine_classes[i] == 'QSO')
#     )
#     seyfert1_or_not.append(
#         (fine_classes_grouped_finer[i] == 'Type I') &
#         (fine_classes[i] == 'Seyfert_1')
#     )
# #     seyfert_or_not.append(fine_classes_grouped_finer[i] == 'Seyfert')

    ###################################

type1_or_not = milliquas_agn_labels['custom_type'] == 'AGN'
qso_or_not = milliquas_agn_labels['custom_type'] == 'QSO'
type2_or_not = milliquas_agn_labels['custom_type'] == 'Type II'
blazar_or_not = milliquas_agn_labels['custom_type'] == 'BL Lac'
seyfert1_or_not = milliquas_agn_labels['custom_type'] == 'AGN'

blazar_or_not = np.array(blazar_or_not)
qso_or_not = np.array(qso_or_not)
type1_or_not = np.array(type1_or_not)
type2_or_not = np.array(type2_or_not)
seyfert1_or_not = np.array(seyfert1_or_not)
# seyfert_or_not = np.array(seyfert_or_not)
general_agn_or_not = np.array(general_agn_or_not)
assert np.all(np.array(general_agn_or_not) == agn_mask)
assert np.all(type1_or_not == seyfert1_or_not)

In [ ]:
# plt.hist(np.array(min_mags)[general_sn_or_not], histtype='step', color='black')
# plt.hist(np.array(min_mags)[general_agn_or_not], histtype='step', color='red')

# print(np.median(np.array(min_mags)[general_sn_or_not]))
# print(np.median(np.array(min_mags)[general_agn_or_not]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(width*2/2, height/2))

color_type1 = 100 * (
    np.sqrt(np.array(excess_vars_g)[general_agn_or_not][type1_or_not]) + \
    np.sqrt(np.array(excess_vars_r)[general_agn_or_not][type1_or_not])
) / 2
color_type2 = 100 * (
    np.sqrt(np.array(excess_vars_g)[general_agn_or_not][type2_or_not]) + \
    np.sqrt(np.array(excess_vars_r)[general_agn_or_not][type2_or_not])
) / 2

# color_type1 = np.sqrt(np.array(excess_vars_g)[general_agn_or_not])[type1_or_not]
# color_type2 = np.sqrt(np.array(excess_vars_g)[general_agn_or_not])[type2_or_not]
color_type1_stats = np.percentile(color_type1, [16, 50, 84])
color_type2_stats = np.percentile(color_type2, [16, 50, 84])

sc0 = axes[0].scatter(
    (np.array(mean_mags_g) + np.array(mean_mags_r))[general_agn_or_not][type1_or_not] / 2,
    color_type1
)
sc1 = axes[1].scatter(
    (np.array(mean_mags_g) + np.array(mean_mags_r))[general_agn_or_not][type2_or_not] / 2,
    color_type2
)

plt.show()


plt.hist(((np.array(mean_mags_g) + np.array(mean_mags_r))/2)[general_agn_or_not][type1_or_not], color='black')
plt.hist(((np.array(mean_mags_g) + np.array(mean_mags_r))/2)[general_agn_or_not][type2_or_not], color='red')
plt.show()

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

xticks = [4, 6, 8, 10, 12, 14]
yticks = [6, 8, 10, 12]
xlim = (2.5, 15)
ylim = (4.5, 13)

fig, axes = plt.subplots(1, 2, figsize=(width*2/2, height/2))

#####################################################################
type_titles = ['Type I', 'Type II']
mincnt1, mincnt2 = 3, 1
subtype_mask1 = type1_or_not
subtype_mask2 = type2_or_not
color_type1 = np.array(mean_colors)[general_agn_or_not][subtype_mask1]
color_type2 = np.array(mean_colors)[general_agn_or_not][subtype_mask2]

color_type1_stats = np.percentile(color_type1, [16, 50, 84])
color_type2_stats = np.percentile(color_type2, [16, 50, 84])
#####################################################################

import matplotlib.colors as mcolors
vmin0, vmax0 = np.min(
    [np.quantile(color_type1, 0.02), np.quantile(color_type2, 0.02)]
), np.max(
    [np.quantile(color_type1, 0.98), np.quantile(color_type2, 0.98)]
)
vmin1, vmax1 = vmin0, vmax0

norm0 = mcolors.TwoSlopeNorm(
    vmin=vmin0, vcenter=0, vmax=vmax1
)
norm1 = mcolors.TwoSlopeNorm(
    vmin=vmin0, vcenter=0, vmax=vmax1
)

sc0 = axes[0].hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_agn_or_not][subtype_mask1],
    test_outputs_condensed_condensed_transformed[:, 1][general_agn_or_not][subtype_mask1],
    C=color_type1, norm=norm0,
    mincnt=mincnt1, edgecolor='grey', cmap='RdYlBu_r', gridsize=20
)
sc1 = axes[1].hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_agn_or_not][subtype_mask2],
    test_outputs_condensed_condensed_transformed[:, 1][general_agn_or_not][subtype_mask2],
    C=color_type2, norm=norm1,
    mincnt=mincnt2, edgecolor='grey', cmap='RdYlBu_r', gridsize=20
)

assert len(general_agn_or_not) > sum(general_agn_or_not)

for ii, ax in enumerate(axes):
    ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('top', size='5%', pad=0.05)

    cbar = fig.colorbar(sc0 if ii == 0 else sc1, orientation='horizontal', cax=cax)
    cbar.ax.xaxis.set_ticks_position('top')
    cbar.ax.xaxis.set_label_position('top')
    # cbar.ax.set_xticklabels(ticks)
    cbar.set_label(
        r'$\mu_{\mathrm{g - r}}$',
        labelpad=10,# fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    set_axes_style(ax)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    ax.text(
        0.05, 0.9, type_titles[ii],
        ha='left', va='top', transform=ax.transAxes, linespacing=1.5
    )
    stats_text = f'{color_type1_stats[0]:.2f}, {color_type1_stats[1]:.2f}, {color_type1_stats[2]:.2f}'\
    if ii == 0 else f'{color_type2_stats[0]:.2f}, {color_type2_stats[1]:.2f}, {color_type2_stats[2]:.2f}'
    ax.text(
        0.96, 0.03, stats_text,
        ha='right', va='bottom', transform=ax.transAxes, linespacing=1.5
    )

plt.show()

# =====

fig, axes = plt.subplots(1, 2, figsize=(width*2/2, height/2))

color_type1 = 100 * (
    np.sqrt(np.array(excess_vars_g)[general_agn_or_not][subtype_mask1]) + \
    np.sqrt(np.array(excess_vars_r)[general_agn_or_not][subtype_mask1])
) / 2
color_type2 = 100 * (
    np.sqrt(np.array(excess_vars_g)[general_agn_or_not][subtype_mask2]) + \
    np.sqrt(np.array(excess_vars_r)[general_agn_or_not][subtype_mask2])
) / 2

# color_type1 = np.sqrt(np.array(excess_vars_g)[general_agn_or_not])[subtype_mask1]
# color_type2 = np.sqrt(np.array(excess_vars_g)[general_agn_or_not])[subtype_mask2]
color_type1_stats = np.percentile(color_type1, [16, 50, 84])
color_type2_stats = np.percentile(color_type2[color_type2<np.max(color_type2)], [16, 50, 84])

vmin0, vmax0 = np.min(
    [np.quantile(color_type1, 0.02), np.quantile(color_type2, 0.02)]
), np.max(
    [np.quantile(color_type1, 0.98), np.quantile(color_type2, 0.98)]
)
vmin1, vmax1 = vmin0, vmax0
# vmin0, vmax0 = np.quantile(color_type1, 0.02), np.quantile(color_type1, 0.98)
# vmin1, vmax1 = np.quantile(color_type2, 0.02), np.quantile(color_type2, 0.98)

sc0 = axes[0].hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_agn_or_not][subtype_mask1],
    test_outputs_condensed_condensed_transformed[:, 1][general_agn_or_not][subtype_mask1],
    C=color_type1, vmin=vmin0, vmax=vmax0,
    mincnt=mincnt1, edgecolor='grey', cmap='OrRd', gridsize=20
)
sc1 = axes[1].hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_agn_or_not][subtype_mask2],
    test_outputs_condensed_condensed_transformed[:, 1][general_agn_or_not][subtype_mask2],
    C=color_type2, vmin=vmin1, vmax=vmax1,
    mincnt=mincnt2, edgecolor='grey', cmap='OrRd', gridsize=20
)

assert len(general_agn_or_not) > sum(general_agn_or_not)

for ii, ax in enumerate(axes):
    ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('top', size='5%', pad=0.05)

    cbar = fig.colorbar(sc0 if ii == 0 else sc1, orientation='horizontal', cax=cax)
    cbar.ax.xaxis.set_ticks_position('top')
    cbar.ax.xaxis.set_label_position('top')
    # cbar.ax.set_xticklabels(ticks)
    cbar.set_label(
        r'$\sigma / \mu\,\,(g, r)\,\,(\times 100)$',
        labelpad=10,# fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    set_axes_style(ax)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    ax.text(
        0.05, 0.9, type_titles[ii],
        ha='left', va='top', transform=ax.transAxes, linespacing=1.5
    )
    stats_text = f'{color_type1_stats[0]:.2f}, {color_type1_stats[1]:.2f}, {color_type1_stats[2]:.2f}'\
    if ii == 0 else f'{color_type2_stats[0]:.2f}, {color_type2_stats[1]:.2f}, {color_type2_stats[2]:.2f}'
    ax.text(
        0.96, 0.03, stats_text,
        ha='right', va='bottom', transform=ax.transAxes, linespacing=1.5
    )

plt.show()

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

xticks = [4, 6, 8, 10, 12, 14]
yticks = [6, 8, 10, 12]
xlim = (2.5, 15)
ylim = (4.5, 13)

fig, axes = plt.subplots(1, 2, figsize=(width*2/2, height/2))

#####################################################################
type_titles = ['Blazar', 'QSO']
mincnt1, mincnt2 = 1, 3
subtype_mask1 = blazar_or_not
subtype_mask2 = qso_or_not
color_type1 = np.array(mean_colors)[general_agn_or_not][subtype_mask1]
color_type2 = np.array(mean_colors)[general_agn_or_not][subtype_mask2]

color_type1_stats = np.percentile(color_type1, [16, 50, 84])
color_type2_stats = np.percentile(color_type2, [16, 50, 84])
#####################################################################

import matplotlib.colors as mcolors
vmin0, vmax0 = np.min(
    [np.quantile(color_type1, 0.02), np.quantile(color_type2, 0.02)]
), np.max(
    [np.quantile(color_type1, 0.98), np.quantile(color_type2, 0.98)]
)
vmin1, vmax1 = vmin0, vmax0

norm0 = mcolors.TwoSlopeNorm(
    vmin=vmin0, vcenter=0, vmax=vmax1
)
norm1 = mcolors.TwoSlopeNorm(
    vmin=vmin0, vcenter=0, vmax=vmax1
)

sc0 = axes[0].hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_agn_or_not][subtype_mask1],
    test_outputs_condensed_condensed_transformed[:, 1][general_agn_or_not][subtype_mask1],
    C=color_type1, norm=norm0,
    mincnt=mincnt1, edgecolor='grey', cmap='RdYlBu_r', gridsize=15
)
sc1 = axes[1].hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_agn_or_not][subtype_mask2],
    test_outputs_condensed_condensed_transformed[:, 1][general_agn_or_not][subtype_mask2],
    C=color_type2, norm=norm1,
    mincnt=mincnt2, edgecolor='grey', cmap='RdYlBu_r', gridsize=15
)

assert len(general_agn_or_not) > sum(general_agn_or_not)

for ii, ax in enumerate(axes):
    ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('top', size='5%', pad=0.05)

    cbar = fig.colorbar(sc0 if ii == 0 else sc1, orientation='horizontal', cax=cax)
    cbar.ax.xaxis.set_ticks_position('top')
    cbar.ax.xaxis.set_label_position('top')
    # cbar.ax.set_xticklabels(ticks)
    cbar.set_label(
        r'$\mu_{\mathrm{g - r}}$',
        labelpad=10,# fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    set_axes_style(ax)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    ax.text(
        0.05, 0.9, type_titles[ii],
        ha='left', va='top', transform=ax.transAxes, linespacing=1.5
    )
    stats_text = f'{color_type1_stats[0]:.2f}, {color_type1_stats[1]:.2f}, {color_type1_stats[2]:.2f}'\
    if ii == 0 else f'{color_type2_stats[0]:.2f}, {color_type2_stats[1]:.2f}, {color_type2_stats[2]:.2f}'
    ax.text(
        0.96, 0.03, stats_text,
        ha='right', va='bottom', transform=ax.transAxes, linespacing=1.5
    )

plt.show()

# =====

fig, axes = plt.subplots(1, 2, figsize=(width*2/2, height/2))

color_type1 = 100 * (
    np.sqrt(np.array(excess_vars_g)[general_agn_or_not][subtype_mask1]) + \
    np.sqrt(np.array(excess_vars_r)[general_agn_or_not][subtype_mask1])
) / 2
color_type2 = 100 * (
    np.sqrt(np.array(excess_vars_g)[general_agn_or_not][subtype_mask2]) + \
    np.sqrt(np.array(excess_vars_r)[general_agn_or_not][subtype_mask2])
) / 2
color_type1_stats = np.percentile(color_type1, [16, 50, 84])
color_type2_stats = np.percentile(color_type2, [16, 50, 84])

vmin0, vmax0 = np.min(
    [np.quantile(color_type1, 0.02), np.quantile(color_type2, 0.02)]
), np.max(
    [np.quantile(color_type1, 0.98), np.quantile(color_type2, 0.98)]
)
vmin1, vmax1 = vmin0, vmax0
# vmin0, vmax0 = np.quantile(color_type1, 0.02), np.quantile(color_type1, 0.98)
# vmin1, vmax1 = np.quantile(color_type2, 0.02), np.quantile(color_type2, 0.98)

sc0 = axes[0].hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_agn_or_not][subtype_mask1],
    test_outputs_condensed_condensed_transformed[:, 1][general_agn_or_not][subtype_mask1],
    C=color_type1, vmin=vmin0, vmax=vmax0,
    mincnt=mincnt1, edgecolor='grey', cmap='OrRd', gridsize=15
)
sc1 = axes[1].hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_agn_or_not][subtype_mask2],
    test_outputs_condensed_condensed_transformed[:, 1][general_agn_or_not][subtype_mask2],
    C=color_type2, vmin=vmin1, vmax=vmax1,
    mincnt=mincnt2, edgecolor='grey', cmap='OrRd', gridsize=15
)

assert len(general_agn_or_not) > sum(general_agn_or_not)

for ii, ax in enumerate(axes):
    ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('top', size='5%', pad=0.05)

    cbar = fig.colorbar(sc0 if ii == 0 else sc1, orientation='horizontal', cax=cax)
    cbar.ax.xaxis.set_ticks_position('top')
    cbar.ax.xaxis.set_label_position('top')
    # cbar.ax.set_xticklabels(ticks)
    cbar.set_label(
        r'$\sigma / \mu\,\,(g, r)\,\,(\times 100)$',
        labelpad=10,# fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    set_axes_style(ax)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    ax.text(
        0.05, 0.9, type_titles[ii],
        ha='left', va='top', transform=ax.transAxes, linespacing=1.5
    )
    stats_text = f'{color_type1_stats[0]:.2f}, {color_type1_stats[1]:.2f}, {color_type1_stats[2]:.2f}'\
    if ii == 0 else f'{color_type2_stats[0]:.2f}, {color_type2_stats[1]:.2f}, {color_type2_stats[2]:.2f}'
    ax.text(
        0.96, 0.03, stats_text,
        ha='right', va='bottom', transform=ax.transAxes, linespacing=1.5
    )

plt.show()

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

xticks = [4, 6, 8, 10, 12, 14]
yticks = [6, 8, 10, 12]
xlim = (2.5, 15)
ylim = (4.5, 13)

fig, axes = plt.subplots(1, 2, figsize=(width*2/2, height/2))

#####################################################################
type_titles = ['Blazar', 'Seyfert 1']
mincnt1, mincnt2 = 1, 1
subtype_mask1 = blazar_or_not
subtype_mask2 = seyfert1_or_not
color_type1 = np.array(mean_colors)[general_agn_or_not][subtype_mask1]
color_type2 = np.array(mean_colors)[general_agn_or_not][subtype_mask2]

color_type1_stats = np.percentile(color_type1, [16, 50, 84])
color_type2_stats = np.percentile(color_type2, [16, 50, 84])
#####################################################################

import matplotlib.colors as mcolors
vmin0, vmax0 = np.min(
    [np.quantile(color_type1, 0.02), np.quantile(color_type2, 0.02)]
), np.max(
    [np.quantile(color_type1, 0.98), np.quantile(color_type2, 0.98)]
)
vmin1, vmax1 = vmin0, vmax0

norm0 = mcolors.TwoSlopeNorm(
    vmin=vmin0, vcenter=0, vmax=vmax1
)
norm1 = mcolors.TwoSlopeNorm(
    vmin=vmin0, vcenter=0, vmax=vmax1
)

sc0 = axes[0].hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_agn_or_not][subtype_mask1],
    test_outputs_condensed_condensed_transformed[:, 1][general_agn_or_not][subtype_mask1],
    C=color_type1, norm=norm0,
    mincnt=mincnt1, edgecolor='grey', cmap='RdYlBu_r', gridsize=15
)
sc1 = axes[1].hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_agn_or_not][subtype_mask2],
    test_outputs_condensed_condensed_transformed[:, 1][general_agn_or_not][subtype_mask2],
    C=color_type2, norm=norm1,
    mincnt=mincnt2, edgecolor='grey', cmap='RdYlBu_r', gridsize=15
)

assert len(general_agn_or_not) > sum(general_agn_or_not)

for ii, ax in enumerate(axes):
    ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('top', size='5%', pad=0.05)

    cbar = fig.colorbar(sc0 if ii == 0 else sc1, orientation='horizontal', cax=cax)
    cbar.ax.xaxis.set_ticks_position('top')
    cbar.ax.xaxis.set_label_position('top')
    # cbar.ax.set_xticklabels(ticks)
    cbar.set_label(
        r'$\mu_{\mathrm{g - r}}$',
        labelpad=10,# fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    set_axes_style(ax)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    ax.text(
        0.05, 0.9, type_titles[ii],
        ha='left', va='top', transform=ax.transAxes, linespacing=1.5
    )
    stats_text = f'{color_type1_stats[0]:.2f}, {color_type1_stats[1]:.2f}, {color_type1_stats[2]:.2f}'\
    if ii == 0 else f'{color_type2_stats[0]:.2f}, {color_type2_stats[1]:.2f}, {color_type2_stats[2]:.2f}'
    ax.text(
        0.96, 0.03, stats_text,
        ha='right', va='bottom', transform=ax.transAxes, linespacing=1.5
    )

plt.show()

# =====

fig, axes = plt.subplots(1, 2, figsize=(width*2/2, height/2))

color_type1 = 100 * (
    np.sqrt(np.array(excess_vars_g)[general_agn_or_not][subtype_mask1]) + \
    np.sqrt(np.array(excess_vars_r)[general_agn_or_not][subtype_mask1])
) / 2
color_type2 = 100 * (
    np.sqrt(np.array(excess_vars_g)[general_agn_or_not][subtype_mask2]) + \
    np.sqrt(np.array(excess_vars_r)[general_agn_or_not][subtype_mask2])
) / 2
color_type1_stats = np.percentile(color_type1, [16, 50, 84])
color_type2_stats = np.percentile(color_type2, [16, 50, 84])

vmin0, vmax0 = np.min(
    [np.quantile(color_type1, 0.02), np.quantile(color_type2, 0.02)]
), np.max(
    [np.quantile(color_type1, 0.98), np.quantile(color_type2, 0.98)]
)
vmin1, vmax1 = vmin0, vmax0
# vmin0, vmax0 = np.quantile(color_type1, 0.02), np.quantile(color_type1, 0.98)
# vmin1, vmax1 = np.quantile(color_type2, 0.02), np.quantile(color_type2, 0.98)

sc0 = axes[0].hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_agn_or_not][subtype_mask1],
    test_outputs_condensed_condensed_transformed[:, 1][general_agn_or_not][subtype_mask1],
    C=color_type1, vmin=vmin0, vmax=vmax0,
    mincnt=mincnt1, edgecolor='grey', cmap='OrRd', gridsize=15
)
sc1 = axes[1].hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_agn_or_not][subtype_mask2],
    test_outputs_condensed_condensed_transformed[:, 1][general_agn_or_not][subtype_mask2],
    C=color_type2, vmin=vmin1, vmax=vmax1,
    mincnt=mincnt2, edgecolor='grey', cmap='OrRd', gridsize=15
)

assert len(general_agn_or_not) > sum(general_agn_or_not)

for ii, ax in enumerate(axes):
    ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('top', size='5%', pad=0.05)

    cbar = fig.colorbar(sc0 if ii == 0 else sc1, orientation='horizontal', cax=cax)
    cbar.ax.xaxis.set_ticks_position('top')
    cbar.ax.xaxis.set_label_position('top')
    # cbar.ax.set_xticklabels(ticks)
    cbar.set_label(
        r'$\sigma / \mu\,\,(g, r)\,\,(\times 100)$',
        labelpad=10,# fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    set_axes_style(ax)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    ax.text(
        0.05, 0.9, type_titles[ii],
        ha='left', va='top', transform=ax.transAxes, linespacing=1.5
    )
    stats_text = f'{color_type1_stats[0]:.2f}, {color_type1_stats[1]:.2f}, {color_type1_stats[2]:.2f}'\
    if ii == 0 else f'{color_type2_stats[0]:.2f}, {color_type2_stats[1]:.2f}, {color_type2_stats[2]:.2f}'
    ax.text(
        0.96, 0.03, stats_text,
        ha='right', va='bottom', transform=ax.transAxes, linespacing=1.5
    )

plt.show()

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

xticks = [4, 6, 8, 10, 12, 14]
yticks = [6, 8, 10, 12]
xlim = (2.5, 15)
ylim = (4.5, 13)

fig, axes = plt.subplots(1, 2, figsize=(width*2/2, height/2))

#####################################################################
type_titles = ['QSO', 'Seyfert 1']
mincnt1, mincnt2 = 3, 1
subtype_mask1 = qso_or_not
subtype_mask2 = seyfert1_or_not
color_type1 = np.array(mean_colors)[general_agn_or_not][subtype_mask1]
color_type2 = np.array(mean_colors)[general_agn_or_not][subtype_mask2]

color_type1_stats = np.percentile(color_type1, [16, 50, 84])
color_type2_stats = np.percentile(color_type2, [16, 50, 84])
#####################################################################

import matplotlib.colors as mcolors
vmin0, vmax0 = np.min(
    [np.quantile(color_type1, 0.02), np.quantile(color_type2, 0.02)]
), np.max(
    [np.quantile(color_type1, 0.98), np.quantile(color_type2, 0.98)]
)
vmin1, vmax1 = vmin0, vmax0

norm0 = mcolors.TwoSlopeNorm(
    vmin=vmin0, vcenter=0, vmax=vmax1
)
norm1 = mcolors.TwoSlopeNorm(
    vmin=vmin0, vcenter=0, vmax=vmax1
)

sc0 = axes[0].hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_agn_or_not][subtype_mask1],
    test_outputs_condensed_condensed_transformed[:, 1][general_agn_or_not][subtype_mask1],
    C=color_type1, norm=norm0,
    mincnt=mincnt1, edgecolor='grey', cmap='RdYlBu_r', gridsize=15
)
sc1 = axes[1].hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_agn_or_not][subtype_mask2],
    test_outputs_condensed_condensed_transformed[:, 1][general_agn_or_not][subtype_mask2],
    C=color_type2, norm=norm1,
    mincnt=mincnt2, edgecolor='grey', cmap='RdYlBu_r', gridsize=15
)

assert len(general_agn_or_not) > sum(general_agn_or_not)

for ii, ax in enumerate(axes):
    ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('top', size='5%', pad=0.05)

    cbar = fig.colorbar(sc0 if ii == 0 else sc1, orientation='horizontal', cax=cax)
    cbar.ax.xaxis.set_ticks_position('top')
    cbar.ax.xaxis.set_label_position('top')
    # cbar.ax.set_xticklabels(ticks)
    cbar.set_label(
        r'$\mu_{\mathrm{g - r}}$',
        labelpad=10,# fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    set_axes_style(ax)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    ax.text(
        0.05, 0.9, type_titles[ii],
        ha='left', va='top', transform=ax.transAxes, linespacing=1.5
    )
    stats_text = f'{color_type1_stats[0]:.2f}, {color_type1_stats[1]:.2f}, {color_type1_stats[2]:.2f}'\
    if ii == 0 else f'{color_type2_stats[0]:.2f}, {color_type2_stats[1]:.2f}, {color_type2_stats[2]:.2f}'
    ax.text(
        0.96, 0.03, stats_text,
        ha='right', va='bottom', transform=ax.transAxes, linespacing=1.5
    )

plt.show()

# =====

fig, axes = plt.subplots(1, 2, figsize=(width*2/2, height/2))

color_type1 = 100 * (
    np.sqrt(np.array(excess_vars_g)[general_agn_or_not][subtype_mask1]) + \
    np.sqrt(np.array(excess_vars_r)[general_agn_or_not][subtype_mask1])
) / 2
color_type2 = 100 * (
    np.sqrt(np.array(excess_vars_g)[general_agn_or_not][subtype_mask2]) + \
    np.sqrt(np.array(excess_vars_r)[general_agn_or_not][subtype_mask2])
) / 2
color_type1_stats = np.percentile(color_type1, [16, 50, 84])
color_type2_stats = np.percentile(color_type2, [16, 50, 84])

vmin0, vmax0 = np.min(
    [np.quantile(color_type1, 0.02), np.quantile(color_type2, 0.02)]
), np.max(
    [np.quantile(color_type1, 0.98), np.quantile(color_type2, 0.98)]
)
vmin1, vmax1 = vmin0, vmax0
# vmin0, vmax0 = np.quantile(color_type1, 0.02), np.quantile(color_type1, 0.98)
# vmin1, vmax1 = np.quantile(color_type2, 0.02), np.quantile(color_type2, 0.98)

sc0 = axes[0].hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_agn_or_not][subtype_mask1],
    test_outputs_condensed_condensed_transformed[:, 1][general_agn_or_not][subtype_mask1],
    C=color_type1, vmin=vmin0, vmax=vmax0,
    mincnt=mincnt1, edgecolor='grey', cmap='OrRd', gridsize=15
)
sc1 = axes[1].hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_agn_or_not][subtype_mask2],
    test_outputs_condensed_condensed_transformed[:, 1][general_agn_or_not][subtype_mask2],
    C=color_type2, vmin=vmin1, vmax=vmax1,
    mincnt=mincnt2, edgecolor='grey', cmap='OrRd', gridsize=15
)

assert len(general_agn_or_not) > sum(general_agn_or_not)

for ii, ax in enumerate(axes):
    ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('top', size='5%', pad=0.05)

    cbar = fig.colorbar(sc0 if ii == 0 else sc1, orientation='horizontal', cax=cax)
    cbar.ax.xaxis.set_ticks_position('top')
    cbar.ax.xaxis.set_label_position('top')
    # cbar.ax.set_xticklabels(ticks)
    cbar.set_label(
        r'$\sigma / \mu\,\,(g, r)\,\,(\times 100)$',
        labelpad=10,# fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    set_axes_style(ax)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    ax.text(
        0.05, 0.9, type_titles[ii],
        ha='left', va='top', transform=ax.transAxes, linespacing=1.5
    )
    stats_text = f'{color_type1_stats[0]:.2f}, {color_type1_stats[1]:.2f}, {color_type1_stats[2]:.2f}'\
    if ii == 0 else f'{color_type2_stats[0]:.2f}, {color_type2_stats[1]:.2f}, {color_type2_stats[2]:.2f}'
    ax.text(
        0.96, 0.03, stats_text,
        ha='right', va='bottom', transform=ax.transAxes, linespacing=1.5
    )

plt.show()

In [ ]:
color_type11 = 100 * (
    np.sqrt(np.array(excess_vars_g)[general_agn_or_not][qso_or_not]) + \
    np.sqrt(np.array(excess_vars_r)[general_agn_or_not][qso_or_not])
) / 2
color_type21 = 100 * (
    np.sqrt(np.array(excess_vars_g)[general_agn_or_not][seyfert1_or_not]) + \
    np.sqrt(np.array(excess_vars_r)[general_agn_or_not][seyfert1_or_not])
) / 2
color_type31 = 100 * (
    np.sqrt(np.array(excess_vars_g)[general_agn_or_not][blazar_or_not]) + \
    np.sqrt(np.array(excess_vars_r)[general_agn_or_not][blazar_or_not])
) / 2

color_type12 = np.array(mean_colors)[general_agn_or_not][qso_or_not]
color_type22 = np.array(mean_colors)[general_agn_or_not][seyfert1_or_not]
color_type32 = np.array(mean_colors)[general_agn_or_not][blazar_or_not]

from matplotlib import lines
colors = [sn_color_map['SN Ia'], rgb_values2[4], rgb_values[-1]]
sns.kdeplot(
    x=color_type11, y=color_type12, fill=False, alpha=0.5, color=colors[0],
    linewidths=2.5
)
sns.kdeplot(
    x=color_type21, y=color_type22, fill=False, alpha=0.5, color=colors[1],
    linewidths=2.5
)
sns.kdeplot(
    x=color_type31, y=color_type32, fill=False, alpha=0.5, color=colors[2],
    linewidths=2.5
)
handles = [lines.Line2D([0], [0], color=color) for color in colors]
labels = ['QSO', 'Seyfert 1', 'Blazar']
plt.legend(handles=handles, labels=labels)

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

sn1a_or_not = []
general_sn_or_not = []
sn1bc_or_not = []
slsn_or_not = []
sn2_or_not = []
early_sn_1a = []

for i, oi in enumerate(test_objIds):
    _coarse_finkclass = df_alerts_pandas[df_alerts_pandas['objectId'] == oi]['finkclass'].mode().iloc[0]
    if _coarse_finkclass == 'custom_agn':
        general_sn_or_not.append(False)
        continue
    assert _coarse_finkclass == 'custom_sn' or _coarse_finkclass == 'Early SN Ia candidate'
    general_sn_or_not.append(True)
    sn1a_or_not.append(fine_classes_grouped_finer[i] == 'SN Ia')
    sn1bc_or_not.append(fine_classes_grouped_finer[i] == 'SN Ibc')
    slsn_or_not.append(fine_classes_grouped_finer[i] == 'SLSN')
    sn2_or_not.append(fine_classes_grouped_finer[i] == 'SN II')
    early_sn_1a.append(fine_classes_grouped_finer[i] == 'Early SN Ia')

sn1a_or_not = np.array(sn1a_or_not)
sn1bc_or_not = np.array(sn1bc_or_not)
slsn_or_not = np.array(slsn_or_not)
sn2_or_not = np.array(sn2_or_not)
early_sn_1a = np.array(early_sn_1a)

assert np.all(np.array(general_sn_or_not) == np.array(sn_mask))

In [ ]:
general_mask = general_sn_or_not
matplotlib.rcParams["font.size"] = "12"

xticks = [4, 6, 8, 10]#, 12, 14]
yticks = [4, 6, 8, 10]
# xlim = (2.5, 15)
# ylim = (4.5, 13)
xlim = (3.5, 9)
ylim = (4.5, 9.5)

fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask],
    C=np.array(min_mags_g)[general_mask],
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    r'$m_{\mathrm{peak}}\,(g, r)$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
plt.show()


fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask],
    C=np.array(min_mags_r)[general_mask],
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    r'$m_{\mathrm{peak}}\,(g, r)$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
plt.show()


fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask],
    C=np.array(amplitudes_g)[general_mask],
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    'Amplitude ' + r'$(g)$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
plt.show()


fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()

color_array = (np.array(amplitudes_g)[general_mask] + np.array(amplitudes_r)[general_mask]) / 2
# sns.kdeplot(
#     x=test_outputs_condensed_condensed_transformed[agn_mask, 0],
#     y=test_outputs_condensed_condensed_transformed[agn_mask, 1],
#     fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
# )

vmin = np.quantile(color_array, 0.03)
vmax = np.quantile(color_array, 0.97)

sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask],
    C=color_array, vmin=vmin, vmax=vmax,
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=15
)

ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=[0.1, 0.5, 1, 1.5, 2])
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    'Amplitude ' + r'$(g, r)$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('amplitude_SN.pdf')
plt.show()




# for i, obs_filter in enumerate(['g', 'r']):
color_array = (np.array(time_min_mag_g)[general_mask] + np.array(time_min_mag_r)[general_mask]) / 2

#     vmin = np.quantile(color_array, 0.05)
#     vmax = np.quantile(color_array, 0.95)

# vmin = np.quantile(color_array, 0.03)
# vmax = np.quantile(color_array, 0.97)
vmin, vmax = 0, 15

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
# sns.kdeplot(
#     x=test_outputs_condensed_condensed_transformed[agn_mask, 0],
#     y=test_outputs_condensed_condensed_transformed[agn_mask, 1],
#     fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
# )
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask],
    C=color_array, vmin=vmin, vmax=vmax,
#         norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=15
)

ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

#if i == 0:
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=[2, 10, 18])
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    r'$t_{\mathrm{peak}}\,(g, r)$ (days)',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('time_of_peak_SN.pdf')
plt.show()


# color_array = np.abs(np.array(time_min_mag_g)[general_mask] - np.array(time_min_mag_r)[general_mask])

# #     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
# fig = plt.figure(figsize=(width/2, height/2))
# ax = plt.gca()
# sc = ax.hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0][general_mask],
#     test_outputs_condensed_condensed_transformed[:, 1][general_mask],
#     C=color_array,
#     norm=matplotlib.colors.LogNorm(),
#     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
# )

# ax.tick_params(axis='x')#, labelsize=ticklabelsize)
# ax.tick_params(axis='y')#, labelsize=ticklabelsize)

# #if i == 0:
# ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
# ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

# #     ax.text(
# #         .93, .96, r'$g$' if i == 0 else r'$r$',
# #         ha='left', va='top', transform=ax.transAxes
# #     )

# cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
# #     ticks = [0, 100, 200, 300, 400]
# cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, ticks=ticks)
# cbar.ax.xaxis.set_ticks_position('top')
# cbar.ax.xaxis.set_label_position('top')
# cbar.set_label(
#     r'$abs(t_{\mathrm{peak}}\,(g) - t_{\mathrm{peak}}\,(r))$ (days)',
#     labelpad=10#, fontsize=axeslabelsize
# )
# cbar.ax.tick_params()#labelsize=ticklabelsize)
# set_axes_style(ax)
# ax.set_xticks(xticks)
# ax.set_yticks(yticks)
# ax.set_xlim(xlim)
# ax.set_ylim(ylim)
# plt.show()




# for i, obs_filter in enumerate(['g', 'r']):
color_array = (np.array(dur_g)[general_mask] + np.array(dur_r)[general_mask]) / 2

#     vmin = np.quantile(color_array, 0.05)
#     vmax = np.quantile(color_array, 0.95)
# vmin = np.quantile(color_array, 0.03)
# vmax = np.quantile(color_array, 0.97)
vmin, vmax = 0, 100

print(color_array.min(), vmin, color_array.max(), vmax)

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
# sns.kdeplot(
#     x=test_outputs_condensed_condensed_transformed[agn_mask, 0],
#     y=test_outputs_condensed_condensed_transformed[agn_mask, 1],
#     fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
# )
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask],
    C=color_array, vmin=vmin, vmax=vmax,
#         norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=15
)

ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

#if i == 0:
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
ticks = [10, 50, 90]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=ticks)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    'Duration '+ r'$(g, r)$ '+ '(days)',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('duration_SN.pdf')
plt.show()



# fig, ax = plt.subplots(1, 2, figsize=(width*2/2, height/2))
# # scg = ax[0].scatter(
# #     test_outputs_condensed_condensed_transformed[:, 0],
# #     test_outputs_condensed_condensed_transformed[:, 1],
# #     c=sigmas_mag_g, alpha=0.5
# # )
# scg = ax[0].hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     C=np.array(sigmas_mag_g), #norm=matplotlib.colors.LogNorm(),
#     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
# )
# # scr = ax[1].scatter(
# #     test_outputs_condensed_condensed_transformed[:, 0],
# #     test_outputs_condensed_condensed_transformed[:, 1],
# #     c=sigmas_mag_r, alpha=0.5
# # )
# scr = ax[1].hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     C=np.array(sigmas_mag_r), #norm=matplotlib.colors.LogNorm(),
#     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
# )
# for a in ax:
#     a.tick_params(axis='x')#, labelsize=ticklabelsize)
#     a.tick_params(axis='y')#, labelsize=ticklabelsize)

# ax[0].set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
# ax[0].set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

# for i, sc in enumerate([scg, scr]):
#     cbar = plt.colorbar(sc)
#     cbar.set_label(
#         r'$\sigma_g(m)$' if i == 0 else r'$\sigma_r(m)$', rotation=270,
#         labelpad=15,# fontsize=axeslabelsize
#     )
#     cbar.ax.tick_params()#labelsize=ticklabelsize)

# save_fig('projection_general_sigma_mag.pdf')

# ========================================================
for i, obs_filter in enumerate(['g', 'r']):
    color_array = np.array(sigmas_mag_g)[general_mask] if i == 0 else np.array(sigmas_mag_r)[general_mask]

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
    fig = plt.figure(figsize=(width/2, height/2))
    ax = plt.gca()
    sc = ax.hexbin(
        test_outputs_condensed_condensed_transformed[:, 0][general_mask],
        test_outputs_condensed_condensed_transformed[:, 1][general_mask],
        C=color_array,
    #     norm=matplotlib.colors.LogNorm(),
        mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=15
    )

    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    #if i == 0:
    ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

    cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
    cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=[0.1, 0.3, 0.5])
    cbar.ax.xaxis.set_ticks_position('top')
    cbar.ax.xaxis.set_label_position('top')
    cbar.set_label(
        r'$\sigma\,(g)$' if i == 0 else r'$\sigma\,(r)$',
        labelpad=10#, fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)

    set_axes_style(ax)

#     save_fig(f'projection_general_sigma_mag_{obs_filter}_SN.pdf')


fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()

# sns.kdeplot(
#     x=test_outputs_condensed_condensed_transformed[agn_mask, 0],
#     y=test_outputs_condensed_condensed_transformed[agn_mask, 1],
#     fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
# )

import matplotlib.colors as mcolors
# Get color range centered around zero
color_at_peak1_general_mask = np.array(color_at_peak1)[general_mask]
# vmin = np.percentile(color_at_peak1_general_mask, 0.01)
# vmax = np.percentile(color_at_peak1_general_mask, 0.99)
norm = mcolors.TwoSlopeNorm(
    vmin=np.quantile(color_at_peak1_general_mask, 0.02), vcenter=0,
    vmax=np.quantile(color_at_peak1_general_mask, 0.98)
)

sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask],
    C=color_at_peak1_general_mask, norm=norm,
    mincnt=3, edgecolor='grey', cmap='RdYlBu_r', gridsize=15
)
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
# ticks = [-0.3, -0.15, 0, 0.3, 0.6]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=[-0.2, 0, 0.7])
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
# cbar.ax.set_xticklabels(ticks)
cbar.set_label(
    r'$(g - r)_{\mathrm{peak}}$',
    labelpad=10,# fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('color_at_peak_SN.pdf')
plt.show()




fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()

mean_colors_general_mask = np.array(mean_colors)[general_mask]

# vmin = np.percentile(mean_colors_general_mask, 0.01)
# vmax = np.percentile(mean_colors_general_mask, 0.99)
norm = mcolors.TwoSlopeNorm(
    vmin=-0.2, vcenter=0, vmax=0.2
)

sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask],
    C=mean_colors_general_mask, norm=norm,
    mincnt=3, edgecolor='grey', cmap='RdYlBu_r', gridsize=20
)
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
# ticks = [-0.7, -0.35, 0, 0.35, 0.7]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, ticks=ticks)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
# cbar.ax.set_xticklabels(ticks)
cbar.set_label(
    r'$\mu_{\mathrm{g - r}}$',
    labelpad=10,# fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
plt.show()



fig = plt.figure(figsize=(width/3, height/3))
ax = plt.gca()

sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask],
    C=np.array(std_colors)[general_mask],
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=15
)
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
ticks = [0.01, 0.05, 0.1]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=ticks)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
# cbar.ax.set_xticklabels(ticks)
cbar.set_label(
    r'$\sigma_{\mathrm{g - r}}$',
    labelpad=10,# fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('sigma_color_SN.pdf')

plt.show()



for i, obs_filter in enumerate(['g', 'r']):
    color_array = np.array(excess_vars_g)[general_mask]*1e5 if i == 0 else np.array(excess_vars_r)[general_mask]*1e5

    fig = plt.figure(figsize=(width/2, height/2))
    ax = plt.gca()
    sc = ax.hexbin(
        test_outputs_condensed_condensed_transformed[:, 0][general_mask],
        test_outputs_condensed_condensed_transformed[:, 1][general_mask],
        C=color_array, #vmin=np.min(excess_vars_g)*1e5, vmax=np.max(excess_vars_g)*1e5,
    #     norm=matplotlib.colors.LogNorm(),
        mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
    )

    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    #if i == 0:
    ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

    cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
    cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)
    cbar.ax.xaxis.set_ticks_position('top')
    cbar.ax.xaxis.set_label_position('top')
    cbar.set_label(
        r'$\mathrm{ExcessVar}\,(g)\,(\times10^5)$' if i == 0 else r'$\mathrm{ExcessVar}\,(r)\,(\times10^5)$',
        labelpad=10#, fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    set_axes_style(ax)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

#### NOTE: LINEAR DECLINE RATE only makes sense for SN light curves. ####
for i, obs_filter in enumerate(['g', 'r']):
    color_array = (np.array(dur_g) - np.array(time_min_mag_g))[general_mask] if i == 0 else (np.array(dur_r) - np.array(time_min_mag_r))[general_mask]

    print(f'avg duration-tpeak = {color_array.mean()} in {obs_filter} filter')

    assert np.all(
        color_array >= 0
    )

#     fig = plt.figure(figsize=(width/2, height/2))
#     ax = plt.gca()
#     ax.hist(color_array, bins=[0, 0.2, 0.4, 0.6, 0.8, 1])
#     plt.show()

#     vmin = np.quantile(color_array, 0.05)
#     vmax = np.quantile(color_array, 0.95)

    vmin, vmax = 0, 100

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
    fig = plt.figure(figsize=(width/2, height/2))
    ax = plt.gca()
    sc = ax.hexbin(
        test_outputs_condensed_condensed_transformed[:, 0][general_mask],
        test_outputs_condensed_condensed_transformed[:, 1][general_mask],
        C=color_array, vmin=vmin, vmax=vmax,
#         norm=matplotlib.colors.LogNorm(),
        mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
    )

    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    #if i == 0:
    ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

    cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
#     ticks = [2, 7, 12, 17]
    cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, ticks=ticks)
    cbar.ax.xaxis.set_ticks_position('top')
    cbar.ax.xaxis.set_label_position('top')
    cbar.set_label(
        r'$Duration (g) - t_{\mathrm{peak}}\,(g)$' if i == 0 else r'$Duration (r) - t_{\mathrm{peak}}\,(r)$',
        labelpad=10#, fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    set_axes_style(ax)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    plt.show()

for i, obs_filter in enumerate(['g', 'r']):
    color_array = (np.array(decline_rate_g_numerator))[general_mask] if i == 0 else (np.array(decline_rate_r_numerator))[general_mask]

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
    fig = plt.figure(figsize=(width/2, height/2))
    ax = plt.gca()
    sc = ax.hexbin(
        test_outputs_condensed_condensed_transformed[:, 0][general_mask],
        test_outputs_condensed_condensed_transformed[:, 1][general_mask],
        C=color_array,
#         norm=matplotlib.colors.LogNorm(),
        mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
    )

    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    #if i == 0:
    ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

    cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
#     ticks = [2, 7, 12, 17]
    cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, ticks=ticks)
    cbar.ax.xaxis.set_ticks_position('top')
    cbar.ax.xaxis.set_label_position('top')
    cbar.set_label(
        'Decline rate (numerator) (g)' if i == 0 else 'Decline rate numerator (r)',
        labelpad=10#, fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    set_axes_style(ax)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    plt.show()

for i, obs_filter in enumerate(['g', 'r']):
    color_array = np.array(decline_rate_g)[general_mask] if i == 0 else np.array(decline_rate_r)[general_mask]

    color_array_removed_nans = color_array[~np.isnan(color_array)]

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
    fig = plt.figure(figsize=(width/3, height/3))
    ax = plt.gca()
    sc = ax.hexbin(
        test_outputs_condensed_condensed_transformed[:, 0][general_mask][~np.isnan(color_array)],
        test_outputs_condensed_condensed_transformed[:, 1][general_mask][~np.isnan(color_array)],
        C=color_array_removed_nans,
    #     norm=matplotlib.colors.LogNorm(),
        mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
    )

    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    #if i == 0:
    ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

    cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
#     ticks = [0, 0.05, 0.1, 0.15, 0.2, 0.25]
    cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, ticks=ticks)
    cbar.ax.xaxis.set_ticks_position('top')
    cbar.ax.xaxis.set_label_position('top')
    cbar.set_label(
        'Decline rate ' + r'$(g)$ ' + r'$(\mathrm{mag}/\mathrm{days})$' if i == 0 else 'Decline rate ' + r'$(r)$ ' + r'$(\mathrm{mag}/\mathrm{days})$',
        labelpad=10#, fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    set_axes_style(ax)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    if i == 1:
        save_fig('linear_decline_rate_r_SN.pdf')
    plt.show()

decline_rate_denominator = np.array(decline_rate_r_numerator)[general_mask] / np.array(decline_rate_r)[general_mask]
decline_rate_denominator = np.nan_to_num(decline_rate_denominator, nan=0.)

assert np.allclose(
    decline_rate_denominator,
    (np.array(dur_r) - np.array(time_min_mag_r))[general_mask]
)




# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# def custom_reduce_C_function(C):
#     return np.std(C) / np.mean(C)

# Standard deviation of values
# for i, obs_filter in enumerate(['g', 'r']):
color_array = (np.array(dur_g)[general_mask] + np.array(dur_r)[general_mask]) / 2

#     vmin = np.quantile(color_array, 0.05)
#     vmax = np.quantile(color_array, 0.95)
# vmin = np.quantile(color_array, 0.03)
# vmax = np.quantile(color_array, 0.97)
vmin, vmax = np.quantile(color_array, 0.02), np.quantile(color_array, 0.98)

print(color_array.min(), vmin, color_array.max(), vmax)

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
# sns.kdeplot(
#     x=test_outputs_condensed_condensed_transformed[agn_mask, 0],
#     y=test_outputs_condensed_condensed_transformed[agn_mask, 1],
#     fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
# )
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask],
    C=color_array, reduce_C_function=np.std, #vmin=vmin, vmax=vmax,
#         norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=15
)

ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

#if i == 0:
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
# ticks = [10, 50, 90]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, ticks=ticks)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    r'$\sigma_{\mathrm{Duration}\,\,(g, r)}$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('duration_std_SN.pdf')
plt.show()




fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()

# sns.kdeplot(
#     x=test_outputs_condensed_condensed_transformed[agn_mask, 0],
#     y=test_outputs_condensed_condensed_transformed[agn_mask, 1],
#     fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
# )

import matplotlib.colors as mcolors
# Get color range centered around zero
color_at_peak1_general_mask = np.array(color_at_peak1)[general_mask]
# vmin = np.percentile(color_at_peak1_general_mask, 0.01)
# vmax = np.percentile(color_at_peak1_general_mask, 0.99)
#norm = mcolors.TwoSlopeNorm(
#    vmin=np.quantile(color_at_peak1_general_mask, 0.02), vcenter=0,
#    vmax=np.quantile(color_at_peak1_general_mask, 0.98)
#)

sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask],
    C=color_at_peak1_general_mask, reduce_C_function=np.std,
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=15
)
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
# ticks = [-0.3, -0.15, 0, 0.3, 0.6]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, ticks=[-0.2, 0, 0.7])
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
# cbar.ax.set_xticklabels(ticks)
cbar.set_label(
    r'$\sigma_{(g - r)_{\mathrm{peak}}}$',
    labelpad=10,# fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('color_at_peak_std_SN.pdf')
plt.show()



color_array = (np.array(time_min_mag_g)[general_mask] + np.array(time_min_mag_r)[general_mask]) / 2
fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
# sns.kdeplot(
#     x=test_outputs_condensed_condensed_transformed[agn_mask, 0],
#     y=test_outputs_condensed_condensed_transformed[agn_mask, 1],
#     fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
# )
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask],
    C=color_array, #vmin=vmin, vmax=vmax,
    reduce_C_function=np.std,
#         norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=15
)

ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

#if i == 0:
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, ticks=[2, 10, 18])
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    r'$\sigma_{t_{\mathrm{peak}}\,\,(g, r)}$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('time_of_peak_std_SN.pdf')
plt.show()





fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()

color_array = (np.array(amplitudes_g)[general_mask] + np.array(amplitudes_r)[general_mask]) / 2
# sns.kdeplot(
#     x=test_outputs_condensed_condensed_transformed[agn_mask, 0],
#     y=test_outputs_condensed_condensed_transformed[agn_mask, 1],
#     fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
# )

#vmin = np.quantile(color_array, 0.03)
#vmax = np.quantile(color_array, 0.97)

sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask],
    C=color_array, #vmin=vmin, vmax=vmax,
    reduce_C_function=np.std,
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=15
)

ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, ticks=[0.1, 0.5, 1, 1.5, 2])
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    r'$\sigma_{\mathrm{Amplitude}\,\,(g, r)}$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('amplitude_std_SN.pdf')
plt.show()

In [ ]:
general_mask = general_agn_or_not
matplotlib.rcParams["font.size"] = "12"

xticks = [4, 6, 8, 10, 12, 14]
yticks = [5, 7, 9, 11, 13]
xlim = (4, 14)
ylim = (5, 13.5)

fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask],
    C=np.array(min_mags_g)[general_mask],
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    r'$m_{\mathrm{peak}}\,(g, r)$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
plt.show()


fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask],
    C=np.array(min_mags_r)[general_mask],
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    r'$m_{\mathrm{peak}}\,(g, r)$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
plt.show()


fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask],
    C=np.array(amplitudes_g)[general_mask],
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    'Amplitude ' + r'$(g)$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
plt.show()


fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()

color_array = (np.array(amplitudes_g)[general_mask][test_data_with_gte_3months_ids_mask[general_mask]] + np.array(amplitudes_r)[general_mask][test_data_with_gte_3months_ids_mask[general_mask]]) / 2
sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
)

vmin = np.quantile(color_array, 0.03)
vmax = np.quantile(color_array, 0.97)

sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    C=color_array, vmin=vmin, vmax=vmax,
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=[0.1, 0.5, 1, 1.5, 2])
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    'Amplitude ' + r'$(g, r)$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('amplitude_AGN.pdf')
plt.show()




# for i, obs_filter in enumerate(['g', 'r']):
color_array = (np.array(time_min_mag_g)[general_mask][test_data_with_gte_3months_ids_mask[general_mask]] + np.array(time_min_mag_r)[general_mask][test_data_with_gte_3months_ids_mask[general_mask]]) / 2

#     vmin = np.quantile(color_array, 0.05)
#     vmax = np.quantile(color_array, 0.95)

# vmin = np.quantile(color_array, 0.03)
# vmax = np.quantile(color_array, 0.97)
vmin, vmax = 0, 15

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
)
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    C=color_array, vmin=vmin, vmax=vmax,
#         norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

#if i == 0:
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=[2, 10, 18])
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    r'$t_{\mathrm{peak}}\,(g, r)$ (days)',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('time_of_peak_AGN.pdf')
plt.show()


# color_array = np.abs(np.array(time_min_mag_g)[general_mask] - np.array(time_min_mag_r)[general_mask])

# #     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
# fig = plt.figure(figsize=(width/2, height/2))
# ax = plt.gca()
# sc = ax.hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0][general_mask],
#     test_outputs_condensed_condensed_transformed[:, 1][general_mask],
#     C=color_array,
#     norm=matplotlib.colors.LogNorm(),
#     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
# )

# ax.tick_params(axis='x')#, labelsize=ticklabelsize)
# ax.tick_params(axis='y')#, labelsize=ticklabelsize)

# #if i == 0:
# ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
# ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

# #     ax.text(
# #         .93, .96, r'$g$' if i == 0 else r'$r$',
# #         ha='left', va='top', transform=ax.transAxes
# #     )

# cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
# #     ticks = [0, 100, 200, 300, 400]
# cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, ticks=ticks)
# cbar.ax.xaxis.set_ticks_position('top')
# cbar.ax.xaxis.set_label_position('top')
# cbar.set_label(
#     r'$abs(t_{\mathrm{peak}}\,(g) - t_{\mathrm{peak}}\,(r))$ (days)',
#     labelpad=10#, fontsize=axeslabelsize
# )
# cbar.ax.tick_params()#labelsize=ticklabelsize)
# set_axes_style(ax)
# ax.set_xticks(xticks)
# ax.set_yticks(yticks)
# ax.set_xlim(xlim)
# ax.set_ylim(ylim)
# plt.show()




# for i, obs_filter in enumerate(['g', 'r']):
color_array = (
    np.array(dur_g)[general_mask][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]]\
    + np.array(dur_r)[general_mask][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]]
) / 2

#     vmin = np.quantile(color_array, 0.05)
#     vmax = np.quantile(color_array, 0.95)
# vmin = np.quantile(color_array, 0.03)
# vmax = np.quantile(color_array, 0.97)
vmin, vmax = np.quantile(color_array, 0.02), np.quantile(color_array, 0.98)

print(color_array.min(), vmin, color_array.max(), vmax)

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
)
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]],
    C=color_array, vmin=vmin, vmax=vmax,
#         norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

#if i == 0:
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
# ticks = [10, 50, 90]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, ticks=ticks)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    'Duration '+ r'$(g, r)$ '+ '(days)',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('duration_AGN.pdf')
plt.show()



# fig, ax = plt.subplots(1, 2, figsize=(width*2/2, height/2))
# # scg = ax[0].scatter(
# #     test_outputs_condensed_condensed_transformed[:, 0],
# #     test_outputs_condensed_condensed_transformed[:, 1],
# #     c=sigmas_mag_g, alpha=0.5
# # )
# scg = ax[0].hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     C=np.array(sigmas_mag_g), #norm=matplotlib.colors.LogNorm(),
#     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
# )
# # scr = ax[1].scatter(
# #     test_outputs_condensed_condensed_transformed[:, 0],
# #     test_outputs_condensed_condensed_transformed[:, 1],
# #     c=sigmas_mag_r, alpha=0.5
# # )
# scr = ax[1].hexbin(
#     test_outputs_condensed_condensed_transformed[:, 0],
#     test_outputs_condensed_condensed_transformed[:, 1],
#     C=np.array(sigmas_mag_r), #norm=matplotlib.colors.LogNorm(),
#     mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
# )
# for a in ax:
#     a.tick_params(axis='x')#, labelsize=ticklabelsize)
#     a.tick_params(axis='y')#, labelsize=ticklabelsize)

# ax[0].set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
# ax[0].set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

# for i, sc in enumerate([scg, scr]):
#     cbar = plt.colorbar(sc)
#     cbar.set_label(
#         r'$\sigma_g(m)$' if i == 0 else r'$\sigma_r(m)$', rotation=270,
#         labelpad=15,# fontsize=axeslabelsize
#     )
#     cbar.ax.tick_params()#labelsize=ticklabelsize)

# save_fig('projection_general_sigma_mag.pdf')

# ========================================================
for i, obs_filter in enumerate(['g', 'r']):
    color_array = np.array(sigmas_mag_g)[general_mask] if i == 0 else np.array(sigmas_mag_r)[general_mask]

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
    fig = plt.figure(figsize=(width/2, height/2))
    ax = plt.gca()
    sc = ax.hexbin(
        test_outputs_condensed_condensed_transformed[:, 0][general_mask],
        test_outputs_condensed_condensed_transformed[:, 1][general_mask],
        C=color_array,
    #     norm=matplotlib.colors.LogNorm(),
        mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
    )

    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    #if i == 0:
    ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

    cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
    cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=[0.1, 0.3, 0.5])
    cbar.ax.xaxis.set_ticks_position('top')
    cbar.ax.xaxis.set_label_position('top')
    cbar.set_label(
        r'$\sigma\,(g)$' if i == 0 else r'$\sigma\,(r)$',
        labelpad=10#, fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)

    set_axes_style(ax)

#     save_fig(f'projection_general_sigma_mag_{obs_filter}_AGN.pdf')


fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()

sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
)

import matplotlib.colors as mcolors
# Get color range centered around zero
color_at_peak1_general_mask = np.array(color_at_peak1)[general_mask][test_data_with_gte_3months_ids_mask[general_mask]]
# vmin = np.percentile(color_at_peak1_general_mask, 0.01)
# vmax = np.percentile(color_at_peak1_general_mask, 0.99)
norm = mcolors.TwoSlopeNorm(
    vmin=np.quantile(color_at_peak1_general_mask, 0.02), vcenter=0,
    vmax=np.quantile(color_at_peak1_general_mask, 0.98)
)

assert color_at_peak1_general_mask.shape == test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]].shape

sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    C=color_at_peak1_general_mask, norm=norm,
    mincnt=3, edgecolor='grey', cmap='RdYlBu_r', gridsize=20
)
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
# ticks = [-0.3, -0.15, 0, 0.3, 0.6]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=[-0.2, 0, 0.7])
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
# cbar.ax.set_xticklabels(ticks)
cbar.set_label(
    r'$(g - r)_{\mathrm{peak}}$',
    labelpad=10,# fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('color_at_peak_AGN.pdf')
plt.show()


fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()

sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
)

import matplotlib.colors as mcolors
# Get color range centered around zero
color_at_faintest_general_mask = np.array(color_at_faintest)[general_mask][test_data_with_gte_3months_ids_mask[general_mask]]
norm = mcolors.TwoSlopeNorm(
    vmin=np.quantile(color_at_faintest_general_mask, 0.02), vcenter=0,
    vmax=np.quantile(color_at_faintest_general_mask, 0.98)
)

assert color_at_faintest_general_mask.shape == test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]].shape

sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    C=color_at_faintest_general_mask, norm=norm,
    mincnt=3, edgecolor='grey', cmap='RdYlBu_r', gridsize=20
)
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
# ticks = [-0.3, -0.15, 0, 0.3, 0.6]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=[-0.2, 0, 0.7])
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
# cbar.ax.set_xticklabels(ticks)
cbar.set_label(
    r'$(g - r)_{\mathrm{faint}}$',
    labelpad=10,# fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('color_at_faintest_AGN.pdf')
plt.show()


fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()

sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
)

import matplotlib.colors as mcolors
# Get color range centered around zero
colors_at_peak_minus_color_at_faintest_general_mask = np.array(colors_at_peak_minus_color_at_faintest)[general_mask][test_data_with_gte_3months_ids_mask[general_mask]]
norm = mcolors.TwoSlopeNorm(
    vmin=np.quantile(colors_at_peak_minus_color_at_faintest_general_mask, 0.02), vcenter=0,
    vmax=np.quantile(colors_at_peak_minus_color_at_faintest_general_mask, 0.98)
)

sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    C=colors_at_peak_minus_color_at_faintest_general_mask, norm=norm,
    mincnt=3, edgecolor='grey', cmap='RdYlBu_r', gridsize=20
)
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
# ticks = [-0.3, -0.15, 0, 0.3, 0.6]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=[-1, -0.5, 0, 0.2, 0.4])
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
# cbar.ax.set_xticklabels(ticks)
cbar.set_label(
    r'$(g - r)_{\mathrm{peak}} - (g - r)_{\mathrm{faint}}$',
    labelpad=10,# fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('colors_at_peak_minus_color_at_faintest_AGN.pdf')
plt.show()



fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()

sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
)

import matplotlib.colors as mcolors
# Get color range centered around zero
color_at_peak1_general_mask = np.array(color_at_peak1)[general_mask][test_data_with_gte_3months_ids_mask[general_mask]]
# vmin = np.percentile(color_at_peak1_general_mask, 0.01)
# vmax = np.percentile(color_at_peak1_general_mask, 0.99)
norm = mcolors.TwoSlopeNorm(
    vmin=np.quantile(color_at_peak1_general_mask, 0.02), vcenter=0,
    vmax=np.quantile(color_at_peak1_general_mask, 0.98)
)

assert color_at_peak1_general_mask.shape == test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]].shape

sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    C=color_at_peak1_general_mask, reduce_C_function=np.std,
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
# ticks = [-0.3, -0.15, 0, 0.3, 0.6]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
# cbar.ax.set_xticklabels(ticks)
cbar.set_label(
    r'$\sigma_{(g - r)_{\mathrm{peak}}}$',
    labelpad=10,# fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('color_at_peak_std_AGN.pdf')
plt.show()


fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()

sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
)

import matplotlib.colors as mcolors
# Get color range centered around zero
color_at_faintest_general_mask = np.array(color_at_faintest)[general_mask][test_data_with_gte_3months_ids_mask[general_mask]]


assert color_at_faintest_general_mask.shape == test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]].shape

sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    C=color_at_faintest_general_mask, reduce_C_function=np.std,
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
# ticks = [-0.3, -0.15, 0, 0.3, 0.6]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
# cbar.ax.set_xticklabels(ticks)
cbar.set_label(
    r'$\sigma_{(g - r)_{\mathrm{faint}}}$',
    labelpad=10,# fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('color_at_faintest_std_AGN.pdf')
plt.show()


fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()

sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
)

import matplotlib.colors as mcolors
# Get color range centered around zero
colors_at_peak_minus_color_at_faintest_general_mask = np.array(colors_at_peak_minus_color_at_faintest)[general_mask][test_data_with_gte_3months_ids_mask[general_mask]]

sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    C=colors_at_peak_minus_color_at_faintest_general_mask, reduce_C_function=np.std,
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
# ticks = [-0.3, -0.15, 0, 0.3, 0.6]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
# cbar.ax.set_xticklabels(ticks)
cbar.set_label(
    r'$\sigma_{(g - r)_{\mathrm{peak}} - (g - r)_{\mathrm{faint}}}$',
    labelpad=10,# fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('colors_at_peak_minus_color_at_faintest_std_AGN.pdf')
plt.show()


fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()

mean_colors_general_mask = np.array(mean_colors)[general_mask][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]]

# vmin = np.percentile(mean_colors_general_mask, 0.01)
# vmax = np.percentile(mean_colors_general_mask, 0.99)
norm = mcolors.TwoSlopeNorm(
    vmin=np.quantile(mean_colors_general_mask, 0.02), vcenter=0,
    vmax=np.quantile(mean_colors_general_mask, 0.98)
)

sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
)
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]],
    C=mean_colors_general_mask, norm=norm,
    mincnt=3, edgecolor='grey', cmap='RdYlBu_r', gridsize=20
)
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
ticks = [-0.4, -0.2, 0, 0.35, 0.7]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=ticks)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.ax.set_xticklabels(ticks)
cbar.set_label(
    r'$\mu_{\mathrm{g - r}}$',
    labelpad=10,# fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('mean_color_AGN.pdf')
plt.show()



fig = plt.figure(figsize=(width/3, height/3))
ax = plt.gca()

sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    C=np.array(std_colors)[general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
ticks = [0.01, 0.05, 0.1]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax, ticks=ticks)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
# cbar.ax.set_xticklabels(ticks)
cbar.set_label(
    r'$\sigma_{\mathrm{g - r}}$',
    labelpad=10,# fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('sigma_color_AGN.pdf')

plt.show()



# for i, obs_filter in enumerate(['g', 'r']):
#     color_array = np.array(excess_vars_g)[general_mask]*1e5 if i == 0 else np.array(excess_vars_r)[general_mask]*1e5
color_array = (
    np.sqrt(np.array(excess_vars_g)[general_agn_or_not][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]]) + \
    np.sqrt(np.array(excess_vars_r)[general_agn_or_not][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]])
) / 2

vmin, vmax = np.quantile(color_array*100, 0.02), np.quantile(color_array*100, 0.98)

fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
)
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]],
    C=color_array*100, vmin=vmin, vmax=vmax, #vmin=np.min(excess_vars_g)*1e5, vmax=np.max(excess_vars_g)*1e5,
    #     norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

#if i == 0:
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

# import matplotlib.ticker as ticker
# def fmt(x, pos):
#     a, b = '{:.10f}'.format(x).split('e')
#     b = int(b)
#     return r'${} \times 10^{{{}}}$'.format(a, b)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, format=ticker.FuncFormatter(fmt))
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    r'$\sigma / \mu\,\,(g, r)\,\,(\times 100)$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('mean_variance_AGN.pdf')

color_array = (
    np.array(linear_slopes_g)[general_agn_or_not] + \
    np.array(linear_slopes_r)[general_agn_or_not]
) / 2

vmin, vmax = np.quantile(color_array, 0.02), np.quantile(color_array, 0.98)

fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
)
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    C=color_array, vmin=vmin, vmax=vmax,
    #     norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='RdYlBu_r', gridsize=20
)

ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

#if i == 0:
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

# import matplotlib.ticker as ticker
# def fmt(x, pos):
#     a, b = '{:.10f}'.format(x).split('e')
#     b = int(b)
#     return r'${} \times 10^{{{}}}$'.format(a, b)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, format=ticker.FuncFormatter(fmt))
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    'Linear trend slope',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('linear_slope_AGN.pdf')



color_array = color_mag_corrs

vmin, vmax = np.quantile(color_array, 0.02), np.quantile(color_array, 0.98)

fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
)
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask][test_data_with_gte_3months_ids_mask[general_mask]],
    C=color_array, vmin=vmin, vmax=vmax,
    #     norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='RdYlBu_r', gridsize=20
)

ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

#if i == 0:
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

# import matplotlib.ticker as ticker
# def fmt(x, pos):
#     a, b = '{:.10f}'.format(x).split('e')
#     b = int(b)
#     return r'${} \times 10^{{{}}}$'.format(a, b)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, format=ticker.FuncFormatter(fmt))
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    'color_mag_corrs',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('color_mag_corrs_AGN.pdf')


#### NOTE: LINEAR DECLINE RATE only makes sense for SN light curves. ####
for i, obs_filter in enumerate(['g', 'r']):
    color_array = (np.array(dur_g) - np.array(time_min_mag_g))[general_mask] if i == 0 else (np.array(dur_r) - np.array(time_min_mag_r))[general_mask]

    print(f'avg duration-tpeak = {color_array.mean()} in {obs_filter} filter')

    assert np.all(
        color_array >= 0
    )

#     fig = plt.figure(figsize=(width/2, height/2))
#     ax = plt.gca()
#     ax.hist(color_array, bins=[0, 0.2, 0.4, 0.6, 0.8, 1])
#     plt.show()

#     vmin = np.quantile(color_array, 0.05)
#     vmax = np.quantile(color_array, 0.95)

    vmin, vmax = 0, 100

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
    fig = plt.figure(figsize=(width/2, height/2))
    ax = plt.gca()
    sc = ax.hexbin(
        test_outputs_condensed_condensed_transformed[:, 0][general_mask],
        test_outputs_condensed_condensed_transformed[:, 1][general_mask],
        C=color_array, vmin=vmin, vmax=vmax,
#         norm=matplotlib.colors.LogNorm(),
        mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
    )

    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    #if i == 0:
    ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

    cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
#     ticks = [2, 7, 12, 17]
    cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, ticks=ticks)
    cbar.ax.xaxis.set_ticks_position('top')
    cbar.ax.xaxis.set_label_position('top')
    cbar.set_label(
        r'$Duration (g) - t_{\mathrm{peak}}\,(g)$' if i == 0 else r'$Duration (r) - t_{\mathrm{peak}}\,(r)$',
        labelpad=10#, fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    set_axes_style(ax)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    plt.show()

for i, obs_filter in enumerate(['g', 'r']):
    color_array = (np.array(decline_rate_g_numerator))[general_mask] if i == 0 else (np.array(decline_rate_r_numerator))[general_mask]

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
    fig = plt.figure(figsize=(width/2, height/2))
    ax = plt.gca()
    sc = ax.hexbin(
        test_outputs_condensed_condensed_transformed[:, 0][general_mask],
        test_outputs_condensed_condensed_transformed[:, 1][general_mask],
        C=color_array,
#         norm=matplotlib.colors.LogNorm(),
        mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
    )

    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    #if i == 0:
    ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

    cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
#     ticks = [2, 7, 12, 17]
    cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, ticks=ticks)
    cbar.ax.xaxis.set_ticks_position('top')
    cbar.ax.xaxis.set_label_position('top')
    cbar.set_label(
        'Decline rate (numerator) (g)' if i == 0 else 'Decline rate numerator (r)',
        labelpad=10#, fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    set_axes_style(ax)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    plt.show()

for i, obs_filter in enumerate(['g', 'r']):
    color_array = np.array(decline_rate_g)[general_mask] if i == 0 else np.array(decline_rate_r)[general_mask]

    color_array_removed_nans = color_array[~np.isnan(color_array)]

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
    fig = plt.figure(figsize=(width/3, height/3))
    ax = plt.gca()
    sc = ax.hexbin(
        test_outputs_condensed_condensed_transformed[:, 0][general_mask][~np.isnan(color_array)],
        test_outputs_condensed_condensed_transformed[:, 1][general_mask][~np.isnan(color_array)],
        C=color_array_removed_nans,
    #     norm=matplotlib.colors.LogNorm(),
        mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
    )

    ax.tick_params(axis='x')#, labelsize=ticklabelsize)
    ax.tick_params(axis='y')#, labelsize=ticklabelsize)

    #if i == 0:
    ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
    ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

    cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
#     ticks = [0, 0.05, 0.1, 0.15, 0.2, 0.25]
    cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, ticks=ticks)
    cbar.ax.xaxis.set_ticks_position('top')
    cbar.ax.xaxis.set_label_position('top')
    cbar.set_label(
        'Decline rate ' + r'$(g)$ ' + r'$(\mathrm{mag}/\mathrm{days})$' if i == 0 else 'Decline rate ' + r'$(r)$ ' + r'$(\mathrm{mag}/\mathrm{days})$',
        labelpad=10#, fontsize=axeslabelsize
    )
    cbar.ax.tick_params()#labelsize=ticklabelsize)
    set_axes_style(ax)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    if i == 1:
        save_fig('linear_decline_rate_r_AGN.pdf')
    plt.show()

decline_rate_denominator = np.array(decline_rate_r_numerator)[general_mask] / np.array(decline_rate_r)[general_mask]
decline_rate_denominator = np.nan_to_num(decline_rate_denominator, nan=0.)

assert np.allclose(
    decline_rate_denominator,
    (np.array(dur_r) - np.array(time_min_mag_r))[general_mask]
)


# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# def custom_reduce_C_function(C):
#     return np.std(C) / np.mean(C)

# Standard deviation of values
# for i, obs_filter in enumerate(['g', 'r']):
color_array = (np.array(dur_g)[general_mask][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]] + np.array(dur_r)[general_mask][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]]) / 2

#     vmin = np.quantile(color_array, 0.05)
#     vmax = np.quantile(color_array, 0.95)
# vmin = np.quantile(color_array, 0.03)
# vmax = np.quantile(color_array, 0.97)
vmin, vmax = np.quantile(color_array, 0.02), np.quantile(color_array, 0.98)

print(color_array.min(), vmin, color_array.max(), vmax)

#     fig, ax = plt.subplots(1, 1, figsize=(width/2, height/2))
fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
)
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]],
    C=color_array, reduce_C_function=np.std, #vmin=vmin, vmax=vmax,
#         norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

#if i == 0:
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
# ticks = [10, 50, 90]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, ticks=ticks)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    r'$\sigma_{\mathrm{Duration}\,\,(g, r)}$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('duration_std_AGN.pdf')
plt.show()



fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()

# vmin = np.percentile(mean_colors_general_mask, 0.01)
# vmax = np.percentile(mean_colors_general_mask, 0.99)
# norm = mcolors.TwoSlopeNorm(
#     vmin=np.quantile(mean_colors_std_general_mask, 0.02), vcenter=0,
#     vmax=np.quantile(mean_colors_std_general_mask, 0.98)
# )

sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
)

sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]],
    C=mean_colors_general_mask, reduce_C_function=np.std,
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
# ticks = [0.05, 0.1, 0.15]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, ticks=ticks)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
# cbar.ax.set_xticklabels(ticks)
cbar.set_label(
    r'$\sigma_{\mu_{\mathrm{g - r}}}$',
    labelpad=10,# fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('mean_color_std_AGN.pdf')
plt.show()




# for i, obs_filter in enumerate(['g', 'r']):
#     color_array = np.array(excess_vars_g)[general_mask]*1e5 if i == 0 else np.array(excess_vars_r)[general_mask]*1e5
color_array = (
    np.sqrt(np.array(excess_vars_g)[general_agn_or_not][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]]) + \
    np.sqrt(np.array(excess_vars_r)[general_agn_or_not][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]])
) / 2

vmin, vmax = np.quantile(color_array*100, 0.02), np.quantile(color_array*100, 0.98)

fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()
sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[sn_mask, 0],
    y=test_outputs_condensed_condensed_transformed[sn_mask, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
)
sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask][test_data_with_gte_3months_ids_mask[general_mask]][agn_used_magpsf_corr[test_data_with_gte_3months_ids_mask[general_mask]]],
    C=color_array*100, reduce_C_function=np.std, #vmin=vmin, vmax=vmax, #vmin=np.min(excess_vars_g)*1e5, vmax=np.max(excess_vars_g)*1e5,
    #     norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=20
)

ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

#if i == 0:
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

#     ax.text(
#         .93, .96, r'$g$' if i == 0 else r'$r$',
#         ha='left', va='top', transform=ax.transAxes
#     )

# import matplotlib.ticker as ticker
# def fmt(x, pos):
#     a, b = '{:.10f}'.format(x).split('e')
#     b = int(b)
#     return r'${} \times 10^{{{}}}$'.format(a, b)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, format=ticker.FuncFormatter(fmt))
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    r'$\sigma_{\sigma / \mu\,\,(g, r)\,\,(\times 100)}$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig('mean_variance_std_AGN.pdf')

In [ ]:
from scipy.interpolate import interp1d

decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
decoded_lcs_test_keys = list(decoded_lcs_test.keys())

test_objIds  = np.load('evaluate_test_objIds_dataloader.npy')
test_outputs_condensed = np.load('evaluate_test_outputs_condensed.npy')
min_max_mags = np.load('min_max_mags.npy') 

xticks = [4, 6, 8, 10]#, 12, 14]
yticks = [4, 6, 8, 10]
# xlim = (2.5, 15)
# ylim = (4.5, 13)
xlim = (3.5, 9)
ylim = (4.5, 9.5)

new_decline_rates_g, new_decline_rates_r = [], []
#################################################
# NOTE: As this is only meant to be used for SN,
# we don't use the correction anywhere.
#################################################
for k, key in enumerate(decoded_lcs_test_keys):
    pred_x, observed_data, observed_mask, observed_times = decoded_lcs_test[key]

    objId = test_objIds[k]
    row = min_max_mags[np.where(min_max_mags[:, 0] == objId)].squeeze()

    assert np.all(observed_times[:, 0] == observed_times[:, 1])  # Because we simply repeated the dimension in the evaluation_unsupervised.py code.
    observed_times = observed_times[:, 0]
    # Set to NaN to ensure non-essential time values for this lc are not shown.
    observed_times[observed_times == 0.0] = np.nan
    # Recall that we use make_first_time_zero=True in prepare_data (see get_lc), so the first time value will be zero
    # But we don't want that first time value to be set to NaN. So we undo that below.
    observed_times[0] = 0.0
    observed_times = unnormalize_time(observed_times, min_time, max_time)
    observed_times = observed_times/24

    observed_data = unnormalize_mag(
        observed_data, observed_mask, float(row[1]), float(row[2])
    )
    # Pass a mask of all ones for prediction because for
    # interpolation all points can be considered observed.
    pred_x = unnormalize_mag(
        pred_x, np.ones_like(observed_mask), float(row[1]), float(row[2])
    )

    tmm_g, tmax_g, min_mag_g, tmin_g = tmms_g[k], tmaxs_g[k], min_mags_g[k], tmins_g[k]
    index_g = 0
    tmm_r, tmax_r, min_mag_r, tmin_r = tmms_r[k], tmaxs_r[k], min_mags_r[k], tmins_r[k]
    index_r = 1

    data = get_lc(df_alerts_pandas, objId, time_in_hrs=False)
    obs = data[2]
    obs_mask = data[3]
    t = data[1]
    obs_mag = np.where(obs_mask, obs, np.nan)

    if (tmax_g - tmm_g < 15) or (tmax_r - tmm_r < 15):
        new_decline_rates_g.append(np.nan)
        new_decline_rates_r.append(np.nan)
        continue

    observed_times_ = np.nan_to_num(observed_times, nan=np.nanmax(observed_times))
    assert len(observed_times_) == len(pred_x.T[index_g])
    assert len(observed_times_) == len(pred_x.T[index_r])

    z1g = interp1d(
        observed_times_,
        pred_x.T[index_g],
        kind='linear'
    )
    pred_x_15g = z1g(tmm_g + 15)
    z1r = interp1d(
        observed_times_,
        pred_x.T[index_r],
        kind='linear'
    )
    pred_x_15r = z1r(tmm_r + 15)

    # just checking for g band. same checks cannbe done for r band
    if not objId in milliquas_agn_labels['objectId'].values:
        assert t[np.where(obs_mag.T[index_g] == min_mag_g)[0]] == tmm_g
        assert np.isclose(
            observed_times[np.where(obs_mag.T[index_g] == min_mag_g)[0][0]],
            tmm_g
        )
        assert np.isclose(
            np.nanmin(observed_data.T[index_g]), min_mag_g
        )

    new_decline_rate_g = (pred_x_15g - min_mag_g)
    new_decline_rate_r = (pred_x_15r - min_mag_r)
    new_decline_rates_g.append(new_decline_rate_g)
    new_decline_rates_r.append(new_decline_rate_r)


general_mask = general_sn_or_not
color_array_g = np.array(new_decline_rates_g)[general_mask]
color_array_removed_nans_g = color_array_g[~np.isnan(color_array_g)]
color_array_r = np.array(new_decline_rates_r)[general_mask]
color_array_removed_nans_r = color_array_r[~np.isnan(color_array_r)]

fig = plt.figure(figsize=(width/2, height/2))
ax = plt.gca()

# sns.kdeplot(
#     x=test_outputs_condensed_condensed_transformed[agn_mask, 0],
#     y=test_outputs_condensed_condensed_transformed[agn_mask, 1],
#     fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2, ax=ax
# )

sc = ax.hexbin(
    test_outputs_condensed_condensed_transformed[:, 0][general_mask][~np.isnan(color_array_g)],
    test_outputs_condensed_condensed_transformed[:, 1][general_mask][~np.isnan(color_array_g)],
    C=(color_array_removed_nans_g + color_array_removed_nans_r)/2,
#     norm=matplotlib.colors.LogNorm(),
    mincnt=3, edgecolor='grey', cmap='OrRd', gridsize=15
)

ax.tick_params(axis='x')#, labelsize=ticklabelsize)
ax.tick_params(axis='y')#, labelsize=ticklabelsize)

#if i == 0:
ax.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
ax.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)

cb_ax = fig.add_axes([0.12, 0.9, 0.78, 0.04])
#     ticks = [0, 0.05, 0.1, 0.15, 0.2, 0.25]
cbar = fig.colorbar(sc, orientation='horizontal', cax=cb_ax)#, ticks=ticks)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.set_label(
    r'$\Delta m_{15}\,\,(g, r)$',
    labelpad=10#, fontsize=axeslabelsize
)
cbar.ax.tick_params()#labelsize=ticklabelsize)
set_axes_style(ax)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
save_fig(f'linear_decline_rate_gr_SN_15days.pdf')
plt.show()

In [ ]:
import matplotlib.colors as mcolors

fig, ax = plt.subplots(1, 2, figsize=(20, 6))

sn1a_mean_colors = np.array(mean_colors)[general_sn_or_not][sn1a_or_not]
sn1bc_mean_colors = np.array(mean_colors)[general_sn_or_not][(~sn1a_or_not) & (sn1bc_or_not)]

norm0 = mcolors.TwoSlopeNorm(
    vmin=sn1a_mean_colors.min(), vcenter=0, vmax=sn1a_mean_colors.max()
)
norm1 = mcolors.TwoSlopeNorm(
    vmin=sn1bc_mean_colors.min(), vcenter=0, vmax=sn1bc_mean_colors.max()
)
plot0 = ax[0].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][sn1a_or_not],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][sn1a_or_not],
    c=sn1a_mean_colors, s=250, alpha=0.5, cmap='RdYlBu_r', norm=norm0
)
plot1 = ax[1].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][(~sn1a_or_not) & (sn1bc_or_not)],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][(~sn1a_or_not) & (sn1bc_or_not)],
    c=sn1bc_mean_colors, s=250, alpha=0.5, cmap='RdYlBu_r', norm=norm1
)

ax0_xlim = ax[0].get_xlim()
ax0_ylim = ax[0].get_ylim()
ax[1].set_xlim(ax0_xlim)
ax[1].set_ylim(ax0_ylim)

assert len(general_sn_or_not) > sum(general_sn_or_not)

divider0 = make_axes_locatable(ax[0])
cax0 = divider0.append_axes('right', size='5%', pad=0.05)
cbar0 = fig.colorbar(plot0, cax=cax0, orientation='vertical')
divider1 = make_axes_locatable(ax[1])
cax1 = divider1.append_axes('right', size='5%', pad=0.05)
cbar1 = fig.colorbar(plot1, cax=cax1, orientation='vertical')

# ax[0].set_xlim(ax[1].get_xlim())
# ax[0].set_ylim(ax[1].get_ylim())

for a in ax:
    a.set_xlabel('UMAP 1', fontsize=axeslabelsize)
    a.set_ylabel('UMAP 2', fontsize=axeslabelsize)
    a.tick_params(axis='x', labelsize=ticklabelsize)
    a.tick_params(axis='y', labelsize=ticklabelsize)

cbar0.ax.set_ylabel(
    r'$\mu_{\mathrm{g - r}}$', rotation=270,
    labelpad=30, fontsize=axeslabelsize
)
cbar0.ax.tick_params(labelsize=ticklabelsize)
cbar1.ax.set_ylabel(
    r'$\mu_{\mathrm{g - r}}$', rotation=270,
    labelpad=30, fontsize=axeslabelsize
)
cbar1.ax.tick_params(labelsize=ticklabelsize)

ax[0].set_title('(TNS) SN Ia', fontsize=titlesize)
ax[1].set_title('(TNS) SN Ibc', fontsize=titlesize)
plt.show()

# ==============

sn1a_colors_at_peak = np.array(color_at_peak1)[general_sn_or_not][sn1a_or_not]
sn1bc_colors_at_peak = np.array(color_at_peak1)[general_sn_or_not][(~sn1a_or_not) & (sn1bc_or_not)]
norm0 = mcolors.TwoSlopeNorm(
    vmin=sn1a_colors_at_peak.min(), vcenter=0, vmax=sn1a_colors_at_peak.max()
)
norm1 = mcolors.TwoSlopeNorm(
    vmin=sn1bc_colors_at_peak.min(), vcenter=0, vmax=sn1bc_colors_at_peak.max()
)

fig, ax = plt.subplots(1, 2, figsize=(20, 6))

plot0 = ax[0].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][sn1a_or_not],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][sn1a_or_not],
    c=sn1a_colors_at_peak, s=250, alpha=0.5, cmap='RdYlBu_r', norm=norm0
)
plot1 = ax[1].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][(~sn1a_or_not) & (sn1bc_or_not)],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][(~sn1a_or_not) & (sn1bc_or_not)],
    c=sn1bc_colors_at_peak, s=250, alpha=0.5, cmap='RdYlBu_r', norm=norm1
)

ax0_xlim = ax[0].get_xlim()
ax0_ylim = ax[0].get_ylim()
ax[1].set_xlim(ax0_xlim)
ax[1].set_ylim(ax0_ylim)

assert len(general_sn_or_not) > sum(general_sn_or_not)

divider0 = make_axes_locatable(ax[0])
cax0 = divider0.append_axes('right', size='5%', pad=0.05)
cbar0 = fig.colorbar(plot0, cax=cax0, orientation='vertical')
divider1 = make_axes_locatable(ax[1])
cax1 = divider1.append_axes('right', size='5%', pad=0.05)
cbar1 = fig.colorbar(plot1, cax=cax1, orientation='vertical')

# ax[0].set_xlim(ax[1].get_xlim())
# ax[0].set_ylim(ax[1].get_ylim())

for a in ax:
    a.set_xlabel('UMAP 1', fontsize=axeslabelsize)
    a.set_ylabel('UMAP 2', fontsize=axeslabelsize)
    a.tick_params(axis='x', labelsize=ticklabelsize)
    a.tick_params(axis='y', labelsize=ticklabelsize)

cbar0.ax.set_ylabel(
    r'$(g - r)_{\mathrm{peak}}$', rotation=270,
    labelpad=30, fontsize=axeslabelsize
)
cbar0.ax.tick_params(labelsize=ticklabelsize)
cbar1.ax.set_ylabel(
    r'$(g - r)_{\mathrm{peak}}$', rotation=270,
    labelpad=30, fontsize=axeslabelsize
)
cbar1.ax.tick_params(labelsize=ticklabelsize)

ax[0].set_title('(TNS) SN Ia', fontsize=titlesize)
ax[1].set_title('(TNS) SN Ibc', fontsize=titlesize)
plt.show()

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

fig, ax = plt.subplots(1, 2, figsize=(20, 6))
plot0 = ax[0].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][sn2_or_not],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][sn2_or_not],
    c=np.array(mean_colors)[general_sn_or_not][sn2_or_not], alpha=0.5, cmap='RdYlBu_r'
)
plot1 = ax[1].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][(~sn2_or_not) & (slsn_or_not)],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][(~sn2_or_not) & (slsn_or_not)],
    c=np.array(mean_colors)[general_sn_or_not][(~sn2_or_not) & (slsn_or_not)], alpha=0.5, cmap='RdYlBu_r'
)

assert len(general_sn_or_not) > sum(general_sn_or_not)

divider0 = make_axes_locatable(ax[0])
cax0 = divider0.append_axes('right', size='5%', pad=0.05)
cbar0 = fig.colorbar(plot0, cax=cax0, orientation='vertical')
divider1 = make_axes_locatable(ax[1])
cax1 = divider1.append_axes('right', size='5%', pad=0.05)
cbar1 = fig.colorbar(plot1, cax=cax1, orientation='vertical')

# ax[0].set_xlim(ax[1].get_xlim())
# ax[0].set_ylim(ax[1].get_ylim())

for a in ax:
    a.set_xlabel('UMAP 1', fontsize=axeslabelsize)
    a.set_ylabel('UMAP 2', fontsize=axeslabelsize)
    a.tick_params(axis='x', labelsize=ticklabelsize)
    a.tick_params(axis='y', labelsize=ticklabelsize)

cbar0.ax.set_ylabel(
    r'$\mu_{\mathrm{g - r}}$ (mag)', rotation=270,
    labelpad=30, fontsize=axeslabelsize
)
cbar0.ax.tick_params(labelsize=ticklabelsize)
cbar1.ax.set_ylabel(
    r'$\mu_{\mathrm{g - r}}$ (mag)', rotation=270,
    labelpad=30, fontsize=axeslabelsize
)
cbar1.ax.tick_params(labelsize=ticklabelsize)

ax[0].set_title('(TNS) SN II', fontsize=titlesize)
ax[1].set_title('(TNS) SLSN', fontsize=titlesize)

# del _raw_classes

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

fig, ax = plt.subplots(1, 2, figsize=(22, 7))

# Scatter plots with color mapping
decline_rate_g_sn2 = np.array(decline_rate_g)[general_sn_or_not][sn2_or_not]
decline_rate_r_sn2 = np.array(decline_rate_r)[general_sn_or_not][sn2_or_not]
decline_rate_g_sn1a = np.array(decline_rate_g)[general_sn_or_not][(~sn2_or_not) & (sn1a_or_not)]
decline_rate_r_sn1a = np.array(decline_rate_r)[general_sn_or_not][(~sn2_or_not) & (sn1a_or_not)]

scg = ax[0].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][(~sn2_or_not) & (sn1a_or_not)],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][(~sn2_or_not) & (sn1a_or_not)],
    s=200, vmin=0, vmax=0.1,
    c=decline_rate_r_sn1a, alpha=0.4,
)
scg = ax[0].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][sn2_or_not],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][sn2_or_not],
    s=200, vmin=0, vmax=0.1, marker='^',
    c=decline_rate_g_sn2, alpha=1,
)

scr = ax[1].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][(~sn2_or_not) & (sn1a_or_not)],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][(~sn2_or_not) & (sn1a_or_not)],
    s=200, vmin=0, vmax=0.1,
    c=decline_rate_r_sn1a, alpha=0.4,
)
scr = ax[1].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][sn2_or_not],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][sn2_or_not],
    s=200, vmin=0, vmax=0.1, marker='^',
    c=decline_rate_r_sn2, alpha=1,
)

handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='darkgray', label='SN Ia', markersize=20) \
]
handles += [
    Line2D([0], [0], marker='^', color='w', markerfacecolor='darkgray', label='SN II', markersize=20)
]

ax[0].legend(handles=handles, loc='upper right', fontsize=ticklabelsize, frameon=True)


# Axis formatting
for a in ax:
    a.set_xlabel('UMAP 1', fontsize=axeslabelsize)
    a.set_ylabel('UMAP 2', fontsize=axeslabelsize)
    a.tick_params(axis='x', labelsize=ticklabelsize)
    a.tick_params(axis='y', labelsize=ticklabelsize)

# Colorbars with histograms
for i, (sc) in enumerate([scg, scr]):
    cbar = plt.colorbar(sc, location='top', ax=ax[i], pad=0.1, aspect=30)
    cbar.ax.set_xticks([])
#     cbar.set_label(
#         'Decline rate (g) (mag/days)' if i == 0 else 'Decline rate (r) (mag/days)',
#         labelpad=35, fontsize=axeslabelsize
#     )
#     cbar.ax.tick_params(labelsize=ticklabelsize)

    # Add histogram on top of the colorbar
    hist_ax = cbar.ax.inset_axes([0, 2.5, 1, 10])
    if i == 0:
        hist_ax.hist(decline_rate_g_sn2, bins=np.linspace(0, 0.1, 10), color='gray', edgecolor='gray', histtype='step', linewidth=3, label='SN II')
        hist_ax.hist(decline_rate_g_sn1a, bins=np.linspace(0, 0.1, 10), color='gray', edgecolor='gray', histtype='step', linestyle='dashed', linewidth=3, label='SN Ia')
    else:
        hist_ax.hist(decline_rate_r_sn2, bins=np.linspace(0, 0.1, 10), color='gray', edgecolor='gray', histtype='step', linewidth=3, label='SN II')
        hist_ax.hist(decline_rate_r_sn1a, bins=np.linspace(0, 0.1, 10), color='gray', edgecolor='gray', histtype='step', linestyle='dashed', linewidth=3, label='SN Ia')
    hist_ax.set_xlim([0, 0.1])
    hist_ax.legend(fontsize=ticklabelsize)
    hist_ax.set_ylabel('Counts', fontsize=axeslabelsize)
    hist_ax.set_xlabel('Decline rate ' + r'$(g)\,(\mathrm{mag}\,\mathrm{day}^{-1})$' if i == 0 else 'Decline rate ' + r'$(r)\,(\mathrm{mag}\,\mathrm{day}^{-1})$', fontsize=axeslabelsize)
    hist_ax.tick_params(axis='x', labelsize=ticklabelsize)
    hist_ax.tick_params(axis='y', labelsize=ticklabelsize)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

fig, ax = plt.subplots(1, 2, figsize=(22, 7))

# Scatter plots with color mapping
decline_rate_g_slsn = np.array(decline_rate_g)[general_sn_or_not][slsn_or_not]
decline_rate_r_slsn = np.array(decline_rate_r)[general_sn_or_not][slsn_or_not]
decline_rate_g_sn1a = np.array(decline_rate_g)[general_sn_or_not][(~slsn_or_not) & (sn1a_or_not)]
decline_rate_r_sn1a = np.array(decline_rate_r)[general_sn_or_not][(~slsn_or_not) & (sn1a_or_not)]

scg = ax[0].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][(~slsn_or_not) & (sn1a_or_not)],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][(~slsn_or_not) & (sn1a_or_not)],
    s=200, vmin=0, vmax=0.1,
    c=decline_rate_r_sn1a, alpha=0.4,
)
scg = ax[0].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][slsn_or_not],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][slsn_or_not],
    s=200, vmin=0, vmax=0.1, marker='^',
    c=decline_rate_g_slsn, alpha=1,
)

scr = ax[1].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][(~slsn_or_not) & (sn1a_or_not)],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][(~slsn_or_not) & (sn1a_or_not)],
    s=200, vmin=0, vmax=0.1,
    c=decline_rate_r_sn1a, alpha=0.4,
)
scr = ax[1].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][slsn_or_not],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][slsn_or_not],
    s=200, vmin=0, vmax=0.1, marker='^',
    c=decline_rate_r_slsn, alpha=1,
)

handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='darkgray', label='SN Ia', markersize=20) \
]
handles += [
    Line2D([0], [0], marker='^', color='w', markerfacecolor='darkgray', label='SLSN', markersize=20)
]

ax[0].legend(handles=handles, loc='upper right', fontsize=ticklabelsize, frameon=True)

# Axis formatting
for a in ax:
    a.set_xlabel('UMAP 1', fontsize=axeslabelsize)
    a.set_ylabel('UMAP 2', fontsize=axeslabelsize)
    a.tick_params(axis='x', labelsize=ticklabelsize)
    a.tick_params(axis='y', labelsize=ticklabelsize)

# Colorbars with histograms
for i, (sc) in enumerate([scg, scr]):
    cbar = plt.colorbar(sc, location='top', ax=ax[i], pad=0.1, aspect=30)
    cbar.ax.set_xticks([])
#     cbar.set_label(
#         'Decline rate (g) (mag/days)' if i == 0 else 'Decline rate (r) (mag/days)',
#         labelpad=35, fontsize=axeslabelsize
#     )
#     cbar.ax.tick_params(labelsize=ticklabelsize)

    # Add histogram on top of the colorbar
    hist_ax = cbar.ax.inset_axes([0, 2.5, 1, 10])
    if i == 0:
        hist_ax.hist(decline_rate_g_slsn, bins=np.linspace(0, 0.1, 10), color='gray', edgecolor='gray', histtype='step', linewidth=3, label='SLSN')
        hist_ax.hist(decline_rate_g_sn1a, bins=np.linspace(0, 0.1, 10), color='gray', edgecolor='gray', histtype='step', linestyle='dashed', linewidth=3, label='SN Ia')
    else:
        hist_ax.hist(decline_rate_r_slsn, bins=np.linspace(0, 0.1, 10), color='gray', edgecolor='gray', histtype='step', linewidth=3, label='SLSN')
        hist_ax.hist(decline_rate_r_sn1a, bins=np.linspace(0, 0.1, 10), color='gray', edgecolor='gray', histtype='step', linestyle='dashed', linewidth=3, label='SN Ia')
    hist_ax.set_xlim([0, 0.1])
    hist_ax.legend(fontsize=ticklabelsize)
    hist_ax.set_ylabel('Counts', fontsize=axeslabelsize)
    hist_ax.set_xlabel('Decline rate ' + r'$(g)\,(\mathrm{mag}\,\mathrm{day}^{-1})$' if i == 0 else 'Decline rate ' + r'$(r)\,(\mathrm{mag}\,\mathrm{day}^{-1})$', fontsize=axeslabelsize)
    hist_ax.tick_params(axis='x', labelsize=ticklabelsize)
    hist_ax.tick_params(axis='y', labelsize=ticklabelsize)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import numpy as np

fig, ax = plt.subplots(3, 4, figsize=(width*7/2, height*3/2))

# Scatter plots with color mapping
decline_rate_r_sn1a = np.array(decline_rate_r)[general_sn_or_not][sn1a_or_not]
decline_rate_r_sn1bc = np.array(decline_rate_r)[general_sn_or_not][sn1bc_or_not]
decline_rate_r_sn2 = np.array(decline_rate_r)[general_sn_or_not][sn2_or_not]
decline_rate_r_slsn = np.array(decline_rate_r)[general_sn_or_not][slsn_or_not]

sn1a_colors_at_peak = np.array(color_at_peak1)[general_sn_or_not][sn1a_or_not]
sn1bc_colors_at_peak = np.array(color_at_peak1)[general_sn_or_not][sn1bc_or_not]
sn2_colors_at_peak = np.array(color_at_peak1)[general_sn_or_not][sn2_or_not]
slsn_colors_at_peak = np.array(color_at_peak1)[general_sn_or_not][slsn_or_not]

sn1a_tpeak_duration = (np.array(time_min_mag_r)/np.array(dur_r))[general_sn_or_not][sn1a_or_not]
sn1bc_tpeak_duration = (np.array(time_min_mag_r)/np.array(dur_r))[general_sn_or_not][sn1bc_or_not]
sn2_tpeak_duration = (np.array(time_min_mag_r)/np.array(dur_r))[general_sn_or_not][sn2_or_not]
slsn_tpeak_duration = (np.array(time_min_mag_r)/np.array(dur_r))[general_sn_or_not][slsn_or_not]

norm0 = mcolors.TwoSlopeNorm(
    vmin=sn1a_colors_at_peak.min(), vcenter=0, vmax=sn1a_colors_at_peak.max()
)
norm1 = mcolors.TwoSlopeNorm(
    vmin=sn2_colors_at_peak.min(), vcenter=0, vmax=sn2_colors_at_peak.max()
)
norm2 = mcolors.TwoSlopeNorm(
    vmin=sn1bc_colors_at_peak.min(), vcenter=0, vmax=sn1bc_colors_at_peak.max()
)
norm3 = mcolors.TwoSlopeNorm(
    vmin=slsn_colors_at_peak.min(), vcenter=0, vmax=slsn_colors_at_peak.max()
)

ax[0, 0].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][sn1a_or_not],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][sn1a_or_not],
    s=200,
    c=sn1a_tpeak_duration, alpha=0.4,
)
ax[0, 0].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][sn2_or_not],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][sn2_or_not],
    s=200, marker='D',
    c=sn2_tpeak_duration, alpha=0.4,
)
handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='darkgray', label='SN Ia', markersize=20) \
]
handles += [
    Line2D([0], [0], marker='D', color='w', markerfacecolor='darkgray', label='SN II', markersize=20)
]
ax[0, 0].legend(handles=handles)

ax[0, 1].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][sn1a_or_not],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][sn1a_or_not],
    s=180,
    c=sn1a_tpeak_duration, alpha=0.4,
)
ax[0, 1].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][slsn_or_not],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][slsn_or_not],
    s=200, marker='^',
    c=slsn_tpeak_duration, alpha=0.4,
)
handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='darkgray', label='SN Ia', markersize=20) \
]
handles += [
    Line2D([0], [0], marker='^', color='w', markerfacecolor='darkgray', label='SLSN', markersize=20)
]
ax[0, 1].legend(handles=handles)
ax[0, 2].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][sn1a_or_not],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][sn1a_or_not],
    s=180,
    c=sn1a_tpeak_duration, alpha=0.4,
)
ax[0, 2].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][sn1bc_or_not],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][sn1bc_or_not],
    s=200, marker='s',
    c=sn1bc_tpeak_duration, alpha=0.4,
)
handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='darkgray', label='SN Ia', markersize=20) \
]
handles += [
    Line2D([0], [0], marker='s', color='w', markerfacecolor='darkgray', label='SN Ibc', markersize=20)
]
ax[0, 2].legend(handles=handles)
ax[0, 3].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][sn2_or_not],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][sn2_or_not],
    s=200, marker='D',
    c=sn2_tpeak_duration, alpha=0.4,
)
scr = ax[0, 3].scatter(
    test_outputs_condensed_condensed_transformed[:, 0][general_sn_or_not][slsn_or_not],
    test_outputs_condensed_condensed_transformed[:, 1][general_sn_or_not][slsn_or_not],
    s=200, marker='^',
    c=slsn_tpeak_duration, alpha=0.4,
)
handles = [
    Line2D([0], [0], marker='D', color='w', markerfacecolor='darkgray', label='SN II', markersize=20) \
]
handles += [
    Line2D([0], [0], marker='^', color='w', markerfacecolor='darkgray', label='SLSN', markersize=20)
]
ax[0, 3].legend(handles=handles)

ax[0, 0].sharex(ax[0, 1])
# ax[0, 0].sharex(ax[0, 2])
# ax[0, 0].sharex(ax[0, 3])
ax[0, 0].sharey(ax[0, 1])
# ax[0, 0].sharey(ax[0, 2])
# ax[0, 0].sharey(ax[0, 3])

divider = make_axes_locatable(ax[0, 3])
cax = divider.append_axes('right', size='5%', pad=0.05)
fig.colorbar(scr, cax=cax, orientation='vertical');

# # Axis formatting
# for a in ax:
#     for b in a:
#         b.set_xlabel('UMAP 1')#, fontsize=axeslabelsize)
#         b.set_ylabel('UMAP 2')#, fontsize=axeslabelsize)
#         b.tick_params(axis='x')#, labelsize=ticklabelsize)
#         b.tick_params(axis='y')#, labelsize=ticklabelsize)

# # Colorbars with histograms
# for i, (sc) in enumerate([scg, scr]):
#     cbar = plt.colorbar(sc, location='top', ax=ax[i], pad=0.1, aspect=30)
#     cbar.ax.set_xticks([])
# #     cbar.set_label(
# #         'Decline rate (g) (mag/days)' if i == 0 else 'Decline rate (r) (mag/days)',
# #         labelpad=35, fontsize=axeslabelsize
# #     )
# #     cbar.ax.tick_params(labelsize=ticklabelsize)

#     # Add histogram on top of the colorbar
#     hist_ax = cbar.ax.inset_axes([0, 2.5, 1, 10])
#     if i == 0:
#         hist_ax.hist(decline_rate_g_sn2, bins=np.linspace(0, 0.1, 10), color='gray', edgecolor='gray', histtype='step', linewidth=3, label='SN II')
#         hist_ax.hist(decline_rate_g_sn1a, bins=np.linspace(0, 0.1, 10), color='gray', edgecolor='gray', histtype='step', linestyle='dashed', linewidth=3, label='SN Ia')
#     else:
#         hist_ax.hist(decline_rate_r_sn2, bins=np.linspace(0, 0.1, 10), color='gray', edgecolor='gray', histtype='step', linewidth=3, label='SN II')
#         hist_ax.hist(decline_rate_r_sn1a, bins=np.linspace(0, 0.1, 10), color='gray', edgecolor='gray', histtype='step', linestyle='dashed', linewidth=3, label='SN Ia')
#     hist_ax.set_xlim([0, 0.1])
#     hist_ax.legend(fontsize=ticklabelsize)
#     hist_ax.set_ylabel('Counts', fontsize=axeslabelsize)
#     hist_ax.set_xlabel('Decline rate ' + r'$(g)\,(\mathrm{mag}\,\mathrm{day}^{-1})$' if i == 0 else 'Decline rate ' + r'$(r)\,(\mathrm{mag}\,\mathrm{day}^{-1})$', fontsize=axeslabelsize)
#     hist_ax.tick_params(axis='x', labelsize=ticklabelsize)
#     hist_ax.tick_params(axis='y', labelsize=ticklabelsize)

# Similarity search using encoded representations

In [ ]:
show_decoded_lcs_along_with_sim_search = True

In [ ]:
test_outputs_condensed.shape

In [ ]:
if show_decoded_lcs_along_with_sim_search:
    decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
    decoded_lcs_val = np.load('evaluate_val_decoded_lcs.npz')
    decoded_lcs_train = np.load('evaluate_train_decoded_lcs.npz')
    dlte = list(decoded_lcs_test.keys())
    dlv = list(decoded_lcs_val.keys())
    dltr = list(decoded_lcs_train.keys())
    dltr_v_te = np.concatenate([dltr, dlv, dlte])

In [ ]:
# Redefine plot_lc. This is ONLY done because we want to look at the query and the closest lc side by side.
# So here we pass an axes as input (which means fig needs to be created beforehand).
# Some motivation from here: https://stackoverflow.com/a/66371682

def plot_lc_ax_visualize(
    df_alerts, name, title_suffix='', fid_column='fid', magpsf_column='magpsf',
    jd_column='jd', sigmapsf_column='sigmapsf', finkclass_column='finkclass',
    objectId_column='objectId', ax=None, annotate_finkclass=False
):
    """Plots photometry for the given name (objectId) from the alerts dataframe.

    Parameters
    ----------
    name: str
        objectID
    """
    if ax is None:
        ax = plt.gca()

    # Accumulate all alerts for the provided objectId
    pdf = df_alerts[df_alerts[objectId_column] == name].sort_values(
        by=jd_column
    )

    if len(pdf) == 0:
        raise ValueError(f'No alerts exist for objectId = {name}!')

    # Colors to plot
    colordic = {1: 'C0', 2: 'C1'}

    # Labels of ZTF filters
    filtdic = {1: 'g', 2: 'r'}

    for filt in np.unique(pdf[fid_column]):
        # select data from one filter at a time
        maskFilt = pdf[fid_column] == filt

        times = pdf[maskFilt][jd_column] - pdf[maskFilt][jd_column].min()

        ax.errorbar(
            times,
            pdf[maskFilt][magpsf_column],
            pdf[maskFilt][sigmapsf_column],
            ls = '', marker='o', color=colordic[filt], label=filtdic[filt]
        )

        ax.errorbar(
            times,
            pdf[maskFilt][magpsf_column],
            pdf[maskFilt][sigmapsf_column],
            ls='', marker='^', color=colordic[filt]
        )

        if (finkclass_column is not None) and (annotate_finkclass):
            _offset_x = (times.max() - times.min()) / 200
            _offset_y = (pdf[maskFilt][magpsf_column].max() - pdf[maskFilt][magpsf_column].min()) / 100
            for x, y, string in zip(times, pdf[maskFilt][magpsf_column], pdf[maskFilt][finkclass_column]):
                ax.text(
                    x+_offset_x, y+_offset_y, string,
                    color=colordic[filt]
                )

    ax.invert_yaxis()
    ax.legend(fontsize=ticklabelsize)
    assert len(pdf[objectId_column].unique()) == 1
    ax.set_title(f'{pdf[objectId_column].unique()[0]}'+': '+title_suffix, fontsize=titlesize)
    ax.set_xlabel('time (days)', fontsize=axeslabelsize)
    ax.set_ylabel('mag', fontsize=axeslabelsize)

    ax.tick_params(axis='x', labelsize=ticklabelsize, width=2, length=10)
    ax.tick_params(axis='y', labelsize=ticklabelsize, width=2, length=10)
    return ax

See [this link](https://aws.plainenglish.io/unlocking-faiss-for-efficient-search-vector-indexing-and-ann-with-serverless-architecture-5b2b59ead20f) for `faiss` usage, for example.

In [ ]:
def pad_and_concat(array_list, pad_value=0.0):
    # Find maximum length across dim=2
    # this should be the `max_num_ref_points_dataset` dimension
    max_len = max(arr.shape[2] for arr in array_list)

    # Pad each tensor along dim=2
    padded = []
    for arr in array_list:
        pad_width = [(0, 0)] * arr.ndim
        pad_width[2] = (0, max_len - arr.shape[2])  # pad dim=1 at the end
        padded_arr = np.pad(arr, pad_width, mode='constant', constant_values=pad_value)
        padded.append(padded_arr)
    
    return np.concatenate(padded, axis=0)

In [ ]:
train_objIds_dataloader = np.load('evaluate_train_objIds_dataloader.npy')
val_objIds_dataloader = np.load('evaluate_val_objIds_dataloader.npy')
test_objIds_dataloader = np.load('evaluate_test_objIds_dataloader.npy')

# NOTE: The order (train first, then val, and then test),
# must match the order used in `outputs` in the below cell.
objIds_dataloader = np.concatenate([train_objIds_dataloader, val_objIds_dataloader, test_objIds_dataloader])

In [ ]:
## Requires `pip install faiss-cpu` ##

import numpy as np
import pandas as pd 
import faiss

train_outputs_condensed = np.load('evaluate_train_outputs_condensed.npy')
val_outputs_condensed = np.load('evaluate_val_outputs_condensed.npy')
test_outputs_condensed = np.load('evaluate_test_outputs_condensed.npy')

# Each light curve will have different number of query points due to our approach of
# using custom query for each light curve. Thus, we set the values to nan in
# `evaluate_unsupervised.py`. Here we replace them to zero to make similarity search work.
test_outputs_condensed = np.nan_to_num(test_outputs_condensed, nan=0.)
val_outputs_condensed = np.nan_to_num(val_outputs_condensed, nan=0.)
train_outputs_condensed = np.nan_to_num(train_outputs_condensed, nan=0.)

num_test_examples = len(test_outputs_condensed)
outputs = [train_outputs_condensed, val_outputs_condensed, test_outputs_condensed]
outputs = pad_and_concat(outputs, pad_value=0.)
print(outputs.shape, train_outputs_condensed.shape, val_outputs_condensed.shape, test_outputs_condensed.shape)

assert np.all(val_objIds_dataloader == val_objIds)
assert np.all(test_objIds_dataloader == test_objIds)
assert np.all(train_objIds_dataloader == train_objIds)

# Average across the `num_sample` dimension. Has no effect if num_sample=1.
outputs = outputs.mean(1)
outputs = outputs.reshape(outputs.shape[0], -1)
print(outputs.shape)

# processed_outputs = []
# for o in outputs:
#     # `o` is of shape:
#     # (num_examples_dataset, num_sample, max_num_ref_points_dataset, number_of_filters)

#     # Average across the `num_sample` dimension. Has no effect if num_sample=1.
#     o = o.mean(1)
#     # Flatten across the `number_of_filters` dimension.
#     # Below idea of flattening came from:
#     # https://ai.stackexchange.com/questions/37233/similarities-between-2d-vectors-to-flatten-or-to-not
#     o = o.reshape(o.shape[0], o.shape[1], -1)
#     processed_outputs.append(o)

# No. of max reference points in the combined dataset
vector_dimension = outputs.shape[1]
                                 
faiss.normalize_L2(outputs)
# Note that `IndexFlatIP` in itself only performs the inner dot product.
# This is cosine similarity only because we normalize the vectors in the previous line.
# Cosine sim = inner product on normalized vectors.
index = faiss.IndexFlatIP(vector_dimension)
index.add(outputs)

In [ ]:
outputs.shape

In [ ]:
train_min_max_times = np.load('train_min_max_times.npy')
min_time, max_time = train_min_max_times[0], train_min_max_times[1]

In [ ]:
import time

k = 5  # How many similar examples to return.
t = time.time()
# NOTE: No need to normalize query here because `outputs`
# was already normalized when creating the index above.
similarities, similarities_ids = index.search(outputs, k)
print(f'time required (ms): {((time.time() - t)*1e3):.1f}, for {len(similarities)} vectors')

similarities = np.clip(similarities, 0, 1)

results = pd.DataFrame(
    np.hstack(
        (similarities, similarities_ids)
    )
)
sim_columns = [
        f'sim_{i}' for i in range(similarities.shape[1])
    ]
sim_id_columns = [
        f'sim_id_{i}' for i in range(similarities_ids.shape[1])
    ]
results.columns = sim_columns + sim_id_columns
for col in sim_id_columns:
    results[col] = results[col].astype('int') 
results.shape

In [ ]:
results.head()

In [ ]:
# **NOTE**: THIS ASSUMES TEST EXAMPLES ARE AT THE END.
# THIS IS DECIDED BASED ON THE FOLLOWING DEFINITION IN AN ABOVE CELL:
# `outputs = [train_outputs_condensed, val_outputs_condensed, test_outputs_condensed]`
only_test_outputs = outputs[-num_test_examples:]
assert len(only_test_outputs) == num_test_examples
# loop over test examples since those are queries
for c, idx in enumerate(range(len(only_test_outputs))):
    query_currObjId = test_objIds_dataloader[idx]
    query_currObjId_index = np.where(total_objIds == query_currObjId)
    query_currCommonFinkclass = total_common_finkclasses[query_currObjId_index].squeeze()
    query_currCommonFinkclass = query_currCommonFinkclass[query_currCommonFinkclass != '0']
    assert len(query_currCommonFinkclass) == 1

    fig, ax = plt.subplots(1, k, figsize=(30, 3))
    # Plot query
    assert test_objIds_dataloader[idx] == objIds_dataloader[-num_test_examples + idx]
    plot_lc_ax_visualize(
        df_alerts_pandas, test_objIds_dataloader[idx],
        ax=ax[0], title_suffix=query_currCommonFinkclass[0]
    )

    # Extract the similarity search result for the current test example
    row = results.iloc[-num_test_examples + idx]
    # Ignore the first entry since it will be same as query with sim=1.0
    in_test_or_not = []
    in_val_or_not = []
    currCommonFinkclasses = []
    for i_ in range(1, k):
        currObjId = objIds_dataloader[int(row[f'sim_id_{i_}'])]
        if currObjId in train_objIds_dataloader:
            in_test_or_not.append(0)
            in_val_or_not.append(0)
        elif currObjId in val_objIds_dataloader:
            in_test_or_not.append(0)
            in_val_or_not.append(1)
        elif currObjId in test_objIds_dataloader:
            in_test_or_not.append(1)
            in_val_or_not.append(0)

        currObjId_index = np.where(total_objIds == currObjId)
        # NOTE: The calculation of `currCommonFinkclass` has one caveat
        # which is that if there are multiple modes (two classes equally common)
        # then we would inadvertently only select the first mode
        # (which is based on lexicographical ordering of the finkclass names).
        # But this is not very common and probably is okay.
        currCommonFinkclass = total_common_finkclasses[currObjId_index].squeeze()
        currCommonFinkclass = currCommonFinkclass[currCommonFinkclass != '0']

        assert len(currCommonFinkclass) == 1
        currCommonFinkclasses.append(currCommonFinkclass[0])
        assert df_alerts_pandas[df_alerts_pandas['objectId'] == currObjId]['finkclass'].mode().tolist()[0] == currCommonFinkclass[0] 

        plot_lc_ax_visualize(
            df_alerts_pandas, objIds_dataloader[int(row[f'sim_id_{i_}'])], ax=ax[i_],
            title_suffix=currCommonFinkclass[0]+f'\nsim = {row[f"sim_{i_}"]:.3f}'
        )  # this is the closest to the query in the database.

    if show_decoded_lcs_along_with_sim_search:
        # Also, we pass the same min_max_mags even if we use local normalization
        # for the mags because in plot_decoded_lc, we select the min/max for
        # that light curve, so it takes care of that.

        # Plot query decoded lc
        _ = plot_decoded_lc(
            decoded_lcs_test, key=dlte[idx], df_alerts=df_alerts, objId=query_currObjId,
            unnormalize_lcs=True, min_time=min_time, max_time=max_time,
            min_max_mags=min_max_mags, prob_plot=False, also_show_sigmapsf_and_rmse=False,
            whichclass=query_currCommonFinkclass[0]
        )
        plt.show()

        # Only show decoded lc for the closest (note, we ignore
        # sim_id_0 because it will be the same as query)
        # `in_test_or_not[0]` gives whether the closest example
        # (excluding 0th element) is in test set or not
        # It's `in_test_or_not[0]` and not `in_test_or_not[1]`
        # to get the closest element to the query because
        # `in_test_or_not` is populated from index 1:
        # see `for i_ in range(1, k)`. Same for `in_val_or_not`.
        if in_test_or_not[0]:
            use_decoded_lcs = decoded_lcs_test
        elif in_val_or_not[0]:
            use_decoded_lcs = decoded_lcs_val
        else:
            use_decoded_lcs = decoded_lcs_train

        _ = plot_decoded_lc(
            use_decoded_lcs, key=dltr_v_te[int(row['sim_id_1'])],
            df_alerts=df_alerts,
            objId=objIds_dataloader[int(row[f'sim_id_1'])], unnormalize_lcs=True,
            min_time=min_time, max_time=max_time, min_max_mags=min_max_mags,
            prob_plot=False, also_show_sigmapsf_and_rmse=False, whichclass=currCommonFinkclasses[0]
        )
        plt.show()
        plt.close()

    if c == 3:
        break

# Diagnostic plot

In [ ]:
matplotlib.rcParams["font.size"] = "12"

In [ ]:

length_lc_all, dur_g_all, dur_r_all = [], [], []

for i, oi in enumerate(total_objIds):
    dataA = get_lc(df_alerts_pandas, oi, time_in_hrs=False)
    obsA = dataA[2]
    obs_maskA = dataA[3]
    tA = dataA[1]

    obs_magA = np.where(obs_maskA, obsA, np.nan)
    assert np.all(np.sum(~np.isnan(obs_magA), axis=1) >= 1)

    min_magA_g = np.nanmin(obs_magA.T[0])
    min_magA_r = np.nanmin(obs_magA.T[1])

    length_lc_all.append(len(tA))

    obsA_ = np.where(obsA != 0, obsA, np.inf)

    # Hacky adjustment for a special case.
    try:
        _tmmA_g = float(tA[np.where(obsA_.T[0] == obsA_.T[0].min())[0]].squeeze())
    except TypeError:
        print('Type error in _tmmA_g calculation')
        _tmmA_g = float(tA[np.where(obsA_.T[0] == obsA_.T[0].min())[0]].squeeze()[0])
    _tmmA_r = float(tA[np.where(obsA_.T[1] == obsA_.T[1].min())[0]].squeeze())
    t_magmin_indexA_g = np.where(obs_magA.T[0] == min_magA_g)[0]
    t_magmin_indexA_r = np.where(obs_magA.T[1] == min_magA_r)[0]
    try:
        tmmA_g = float(tA[t_magmin_indexA_g].squeeze())
    except TypeError:
        tmmA_g = float(tA[t_magmin_indexA_g].squeeze()[0])
    tmmA_r = float(tA[t_magmin_indexA_r].squeeze())
    assert _tmmA_g == tmmA_g
    assert _tmmA_r == tmmA_r

    maskA_g = obs_maskA.T[0]
    maskA_r = obs_maskA.T[1]

    tmin_indexA_g = np.where(maskA_g == 1)[0][0]
    tmin_indexA_r = np.where(maskA_r == 1)[0][0]
    tminA_g = float(tA[tmin_indexA_g])
    tminA_r = float(tA[tmin_indexA_r])
    tminA = tA.min()
    assert tminA == 0.

#     time_min_mag_g.append(tmm_g - tmin_g)  # not tmm/24 because time_in_hrs=False above.
#     time_min_mag_r.append(tmm_r - tmin_r)  # not tmm/24 because time_in_hrs=False above.
#     t_magmin_index = np.where(obs_magA == min_magA)[0]
#     tmm = float(tA[t_magminA_index])

    tmax_indexA_g = np.where(maskA_g == 1)[0][-1]
    tmax_indexA_r = np.where(maskA_r == 1)[0][-1]
    assert not np.isnan(obs_magA[tmax_indexA_g][0])
    assert not np.isnan(obs_magA[tmax_indexA_r][1])
    tmaxA_g = float(tA[tmax_indexA_g])
    tmaxA_r = float(tA[tmax_indexA_r])
    tmaxA = tA.max()

    try:
        assert tmax_indexA_g >= t_magmin_indexA_g
    except ValueError:
        assert np.all(tmax_indexA_g >= t_magmin_indexA_g)
    assert tmax_indexA_r >= t_magmin_indexA_r

    assert tmaxA_g == tmaxA or tmaxA_r == tmaxA
    assert (tmin_indexA_g == 0) or (tmin_indexA_r == 0)
    try:
        assert t_magmin_indexA_g >= tmin_indexA_g
    except ValueError:
        assert np.all(t_magmin_indexA_g >= tmin_indexA_g)
    assert t_magmin_indexA_r >= tmin_indexA_r
    assert not np.isnan(obs_magA[tmin_indexA_g][0])
    assert not np.isnan(obs_magA[tmin_indexA_r][1])

#     dur.append(t[-1] - t[0])
    dur_g_all.append(tmaxA_g - tminA_g)
    dur_r_all.append(tmaxA_r - tminA_r)

In [ ]:
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator

num_datapoints_lcs = np.load('num_datapoints_lcs.npy')
classes = []
for tcf in total_common_finkclasses:
    if tcf == 'custom_sn':
        classes.append(0)
    elif tcf == 'custom_agn':
        classes.append(1)
    elif tcf == 'Early SN Ia candidate':
        classes.append(2)
classes = np.array(classes)

# classes = np.array([0 if tcf == 'custom_sn' else 1 for tcf in total_common_finkclasses])
# duration_lcs = np.load('duration_lcs.npy')

duration_lcs = (np.array(dur_g_all) + np.array(dur_r_all)) / 2

assert np.all(
    num_datapoints_lcs/2 == np.array(length_lc_all)/2
)

g = sns.JointGrid(height=height/2)

x1 = num_datapoints_lcs[classes==1]/2
y1 = (duration_lcs)[classes==1]
x2 = num_datapoints_lcs[classes==0]/2
y2 = (duration_lcs)[classes==0]
x3 = num_datapoints_lcs[classes==2]/2
y3 = (duration_lcs)[classes==2]

print(x1.max(), x2.max(), x3.max())
xmax = max(x1.max(), x2.max(), x3.max())
ymax = max(y1.max(), y2.max(), y3.max())

g.ax_joint.hexbin(
    x1, y1, mincnt=3, edgecolor='grey',
    gridsize=20, cmap="Blues", alpha=0.5,
    label='AGN', xscale='log'
)

g.ax_joint.hexbin(
    x2, y2, mincnt=3, edgecolor='grey',
    gridsize=(15, 12), cmap="Reds", alpha=0.5,
    label='SN', xscale='log'
)
g.ax_joint.scatter(
    x3, y3, c="gray", alpha=0.5,
    label='Early SN Ia', marker='x'
)

cmap1 = plt.cm.Blues
cmap2 = plt.cm.Reds
cmap3 = plt.cm.gray
blue_mid = cmap1(0.5)
red_mid  = cmap2(0.5)
gray_mid = cmap3(0.5)

# Add marginal histograms
sns.histplot(
    x=x1, ax=g.ax_marg_x, color=blue_mid, kde=True, alpha=0.6,
    log_scale=True, bins=np.linspace(np.log10(3), np.log10(xmax), num=10)
)
sns.histplot(
    x=x2, ax=g.ax_marg_x, color=red_mid, kde=True, alpha=0.6,
    log_scale=True, bins=np.linspace(np.log10(3), np.log10(xmax), num=10)
)
sns.histplot(
    x=x3, ax=g.ax_marg_x, color=gray_mid, kde=False, alpha=0.6,
    log_scale=True, bins=np.linspace(np.log10(3), np.log10(xmax), num=10)
)

sns.histplot(y=y1, ax=g.ax_marg_y, color=blue_mid, kde=True, alpha=0.6, binwidth=30)#, log_scale=True)
sns.histplot(y=y2, ax=g.ax_marg_y, color=red_mid, kde=True, alpha=0.6, binwidth=30)#, log_scale=True)
sns.histplot(y=y3, ax=g.ax_marg_y, color=gray_mid, kde=False, alpha=0.6, binwidth=30)#, log_scale=True)

g.ax_joint.set_xlabel(r'$\langle L \rangle$')#, fontsize=axeslabelsize)
g.ax_joint.set_ylabel('Duration (days)')#, fontsize=axeslabelsize)

g.ax_joint.tick_params(axis='x')#, labelsize=ticklabelsize)
g.ax_joint.tick_params(axis='y')#, labelsize=ticklabelsize)

proxy1 = Patch(facecolor=blue_mid, edgecolor='grey', alpha=0.75, label='AGN')
proxy2 = Patch(facecolor=red_mid, edgecolor='grey', alpha=0.75, label='SN')
proxy3 = Patch(facecolor=gray_mid, edgecolor='grey', alpha=0.75, label='Early SN Ia')

g.ax_joint.legend(
    handles=[proxy1, proxy2, proxy3], loc='lower right',
    #fontsize=ticklabelsize-8
#     bbox_to_anchor=(1.2, 0.5)
)

g.ax_marg_x.tick_params(labelleft=True)#, labelsize=ticklabelsize-15)
# g.ax_marg_x.set_ylabel('Counts', fontsize=axeslabelsize-15)
g.ax_marg_y.tick_params(labeltop=True, rotation=270)#, labelsize=ticklabelsize-15)
# g.ax_marg_y.set_ylabel('Counts', fontsize=axeslabelsize-15)
g.ax_marg_y.grid(True, axis='x', ls=':')
g.ax_marg_y.xaxis.set_major_locator(MaxNLocator(2))
g.ax_marg_x.grid(True, axis='y', ls=':')
g.ax_marg_x.yaxis.set_major_locator(MaxNLocator(3))

set_axes_style(g.ax_joint)
# set_axes_style(g.ax_marg_x)
# set_axes_style(g.ax_marg_y)

save_fig('data_diagnostic_plot.pdf')

plt.show()

In [ ]:
y1.max(), y2.max(), y3.max()

In [ ]:
num_datapoints_lcs/2, np.array(length_lc_all)

# Interpolation

Instead of aiming for an interpolation that perfectly matches the observed points, our aim is to capture more general evolution (but still capturing local structure).

In [ ]:
matplotlib.rcParams["font.size"] = "18"

In [ ]:
decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
if correct_lcs_for_agns:
    decoded_lcs_test_corrected = np.load('evaluate_test_CORRECTED_decoded_lcs.npz')

    assert list(decoded_lcs_test.keys()) == list(decoded_lcs_test_corrected.keys()), "NPZ keys mismatch"

    # Build corrected_agn: start from original, replace AGN rows with corrected
    decoded_lcs_test_corrected_agn = {}
    for i, key in enumerate(decoded_lcs_test.keys()):
        if agn_indicator[i]:
            decoded_lcs_test_corrected_agn[key] = decoded_lcs_test_corrected[key].copy()
            # Verify AGN lc came from corrected
            np.testing.assert_array_equal(
                decoded_lcs_test_corrected_agn[key],
                decoded_lcs_test_corrected[key]
            )
        else:
            decoded_lcs_test_corrected_agn[key] = decoded_lcs_test[key].copy()
            # Verify non-AGN lc is unchanged
            np.testing.assert_array_equal(
                decoded_lcs_test_corrected_agn[key],
                decoded_lcs_test[key]
            )

    print(f"Replaced {agn_indicator.sum()} AGN rows out of {len(agn_indicator)} total")

dlte = list(decoded_lcs_test.keys())
print(decoded_lcs_test.f.arr_2.shape)

use_corrected_agns = True
if use_corrected_agns:
    print('Using corrected AGN light curves...')
    assert np.all(
        np.load('evaluate_test_objIds_dataloader.npy') == test_objIds_corrected_agn
    )
    test_objIds = test_objIds_corrected_agn
    decoded_lcs_test = decoded_lcs_test_corrected_agn
    min_max_mags = np.load('min_max_mags.npy')
    min_max_mags_corrected = np.load('min_max_mags_ftransfer_ztf_2025_05_31_430518_CORRECTED.npy')
else:
    print('Using uncorrected AGN light curves...')
    test_objIds  = np.load('evaluate_test_objIds_dataloader.npy')
    decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
    min_max_mags = np.load('min_max_mags.npy')

In [ ]:
duration_lcs = np.load('duration_lcs.npy')

In [ ]:
# quick_plot_decoded_lc('ZTF18aaedrvf', fig_width=width*2/3, fig_height=height/4)

In [ ]:
if use_corrected_agns:
    test_objIds_dataloader = np.load('evaluate_test_objIds_dataloader.npy')
    test_objIds_dataloader_corrected = np.load('evaluate_test_CORRECTED_objIds_dataloader.npy')
    assert np.all(test_objIds_dataloader == test_objIds_dataloader_corrected)

In [ ]:
assert np.all(test_objIds_dataloader == test_objIds)

In [ ]:
def get_duration_from_normalized_lc_with_key(lc, min_time, max_time):
    lc_unnormalized = unnormalize_time(lc, min_time, max_time)
    return lc_unnormalized.max() - lc_unnormalized.min()

def get_counters_and_keys_from_mode(decoded_lcs_test, mode='random'):
    if mode == 'low_N_high_D':
        counters_and_keys = [
            # observed_mask is same as `decoded_lcs_test[key][2]`
            # observed_time is same as `decoded_lcs_test[key][3]`
            (counter, key) for counter, key in enumerate(decoded_lcs_test.keys()) \
            if (np.all(decoded_lcs_test[key][2].sum(0) < min_datapoints_each_filter)) \
            and (
                get_duration_from_normalized_lc_with_key(
                    decoded_lcs_test[key][3], min_time, max_time
                ) >= np.median(duration_lcs)
            )
        ]
    elif mode == 'low_N_low_D':
        counters_and_keys = [
            # observed_mask is same as `decoded_lcs_test[key][2]`
            # observed_time is same as `decoded_lcs_test[key][3]`
            (counter, key) for counter, key in enumerate(decoded_lcs_test.keys()) \
            if (np.all(decoded_lcs_test[key][2].sum(0) < min_datapoints_each_filter)) \
            and (
                get_duration_from_normalized_lc_with_key(
                    decoded_lcs_test[key][3], min_time, max_time
                ) < np.median(duration_lcs)
            )
        ]
    elif mode == 'high_N_high_D':
        counters_and_keys = [
            # observed_mask is same as `decoded_lcs_test[key][2]`
            # observed_time is same as `decoded_lcs_test[key][3]`
            (counter, key) for counter, key in enumerate(decoded_lcs_test.keys()) \
            if (np.all(decoded_lcs_test[key][2].sum(0) >= min_datapoints_each_filter)) \
            and (
                get_duration_from_normalized_lc_with_key(
                    decoded_lcs_test[key][3], min_time, max_time
                ) >= np.median(duration_lcs)
            )
        ]
    elif mode == 'high_N_low_D':
        counters_and_keys = [
            # observed_mask is same as `decoded_lcs_test[key][2]`
            # observed_time is same as `decoded_lcs_test[key][3]`
            (counter, key) for counter, key in enumerate(decoded_lcs_test.keys()) \
            if (np.all(decoded_lcs_test[key][2].sum(0) >= min_datapoints_each_filter)) \
            and (
                get_duration_from_normalized_lc_with_key(
                    decoded_lcs_test[key][3], min_time, max_time
                ) < np.median(duration_lcs)
            )    
        ]
    elif mode == 'random':
        counters_and_keys = [
            # observed_mask is same as `decoded_lcs_test[key][2]`
            # observed_time is same as `decoded_lcs_test[key][3]`
            (counter, key) for counter, key in enumerate(decoded_lcs_test.keys())
        ]

    return counters_and_keys

In [ ]:
if use_corrected_agns:
    df_alerts_corrected = dd.read_parquet(
        'alerts_processed_ftransfer_ztf_2025_05_31_430518_CORRECTED.parquet',
        columns=[
            'objectId', 'finkclass', 'tnsclass',
            'fid', 'magpsf_corr', 'sigmapsf_corr', 'jd', 'corrected',
            'used_magpsf_corr'
        ]
    )

In [ ]:
min_datapoints_each_filter = 10
num_to_show = 15

modes = [
#     'random',
#     'low_N_high_D',
#     'low_N_low_D',
    'high_N_high_D',
#     'high_N_low_D'
]
for mode in modes:
    counters_and_keys = get_counters_and_keys_from_mode(decoded_lcs_test, mode)
    show_counter = 0
    print(f'Total {len(counters_and_keys)} examples with mode = "{mode}"')

    for counter, key in counters_and_keys:
        test_currObjId = test_objIds_dataloader[counter]

        if test_currObjId in milliquas_agn_labels['objectId'].values:
            test_currCommonFinkclass = milliquas_agn_labels[milliquas_agn_labels['objectId'] == test_currObjId]['custom_type']
            assert len(test_currCommonFinkclass) == 1
            test_currCommonFinkclass = test_currCommonFinkclass.iloc[0]
#             if test_currCommonFinkclass == 'nan':
#                 test_currCommonFinkclass = 'Unknown'
        else:
            test_currObjId_index = np.where(test_objIds == test_currObjId)
            test_currCommonFinkclass = fine_classes[test_currObjId_index[0][0]]

        ############################ Subselection of specific subtypes ############################
#         if test_currCommonFinkclass != '(TNS) SN Ia':
#             continue
        ###########################################################################################

#         _coarse_finkclass = df_alerts_pandas[df_alerts_pandas['objectId'] == test_currObjId]['finkclass'].mode().iloc[0]
#         if _coarse_finkclass == 'custom_agn':
#             continue

        # Also, we pass the same min_max_mags even if we use local normalization
        # for the mags because in plot_decoded_lc, we select the min/max for
        # that light curve, so it takes care of that.

        # Plot test decoded lc
        corrected_agn_condition = (
            (test_currCommonFinkclass in milliquas_agn_labels['custom_type'].unique())
            and (use_corrected_agns)
            and df_alerts_corrected_pandas.loc[df_alerts_corrected_pandas['objectId'] == test_currObjId, 'used_magpsf_corr'].all()
        )
        print(corrected_agn_condition)
        _ = plot_decoded_lc(
            decoded_lcs_test, key=key, df_alerts=df_alerts_corrected if corrected_agn_condition else df_alerts, objId=test_currObjId,
            unnormalize_lcs=True, min_time=min_time, max_time=max_time,
            min_max_mags=min_max_mags_corrected if corrected_agn_condition else min_max_mags, prob_plot=False,
            whichclass=test_currCommonFinkclass,
            also_show_sigmapsf_and_rmse=True, show_separate_bands=False,
            show_only_rmse=True,
            fig_width=1.5*width*2/3, fig_height=1.5*height/4,
            show_legend=True if custom_counter == 0 else False, legend_loc='best',
            show_corrected_mags=True if corrected_agn_condition else False
        )
        plt.show()
        plt.close()

        if show_counter == num_to_show:
            break

        show_counter += 1

In [ ]:
modes = [
    'random',
]
for mode in modes:
    counters_and_keys = get_counters_and_keys_from_mode(decoded_lcs_test, mode)
    show_counter = 0
    print(f'Total {len(counters_and_keys)} examples with mode = "{mode}"')

    for counter, key in counters_and_keys:
        test_currObjId = test_objIds_dataloader[counter]

        if test_currObjId in milliquas_agn_labels['objectId'].values:
            test_currCommonFinkclass = milliquas_agn_labels[milliquas_agn_labels['objectId'] == test_currObjId]['custom_type']
            assert len(test_currCommonFinkclass) == 1
            test_currCommonFinkclass = test_currCommonFinkclass.iloc[0]
#             if test_currCommonFinkclass == 'nan':
#                 test_currCommonFinkclass = 'Unknown'
        else:
            test_currObjId_index = np.where(test_objIds == test_currObjId)
            test_currCommonFinkclass = fine_classes[test_currObjId_index[0][0]]

#         _coarse_finkclass = df_alerts_pandas[df_alerts_pandas['objectId'] == test_currObjId]['finkclass'].mode().iloc[0]
#         if _coarse_finkclass == 'custom_agn':
#             continue

        # Also, we pass the same min_max_mags even if we use local normalization
        # for the mags because in plot_decoded_lc, we select the min/max for
        # that light curve, so it takes care of that.

        # Plot test decoded lc
        corrected_agn_condition = (
            (test_currCommonFinkclass in milliquas_agn_labels['custom_type'].unique())
            and (use_corrected_agns)
            and df_alerts_corrected_pandas.loc[df_alerts_corrected_pandas['objectId'] == test_currObjId, 'used_magpsf_corr'].all()
        )
        print(test_currObjId, key, corrected_agn_condition)

In [ ]:
## For paper
decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
if correct_lcs_for_agns:
    decoded_lcs_test_corrected = np.load('evaluate_test_CORRECTED_decoded_lcs.npz')

    assert list(decoded_lcs_test.keys()) == list(decoded_lcs_test_corrected.keys()), "NPZ keys mismatch"

    # Build corrected_agn: start from original, replace AGN rows with corrected
    decoded_lcs_test_corrected_agn = {}
    for i, key in enumerate(decoded_lcs_test.keys()):
        if agn_indicator[i]:
            decoded_lcs_test_corrected_agn[key] = decoded_lcs_test_corrected[key].copy()
            # Verify AGN lc came from corrected
            np.testing.assert_array_equal(
                decoded_lcs_test_corrected_agn[key],
                decoded_lcs_test_corrected[key]
            )
        else:
            decoded_lcs_test_corrected_agn[key] = decoded_lcs_test[key].copy()
            # Verify non-AGN lc is unchanged
            np.testing.assert_array_equal(
                decoded_lcs_test_corrected_agn[key],
                decoded_lcs_test[key]
            )

    print(f"Replaced {agn_indicator.sum()} AGN rows out of {len(agn_indicator)} total")

dlte = list(decoded_lcs_test.keys())
print(decoded_lcs_test.f.arr_2.shape)

use_corrected_agns = True
if use_corrected_agns:
    print('Using corrected AGN light curves...')
    assert np.all(
        np.load('evaluate_test_objIds_dataloader.npy') == test_objIds_corrected_agn
    )
    test_objIds = test_objIds_corrected_agn
    decoded_lcs_test = decoded_lcs_test_corrected_agn
    min_max_mags = np.load('min_max_mags.npy')
    min_max_mags_corrected = np.load('min_max_mags_ftransfer_ztf_2025_05_31_430518_CORRECTED.npy')
else:
    print('Using uncorrected AGN light curves...')
    test_objIds  = np.load('evaluate_test_objIds_dataloader.npy')
    decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
    min_max_mags = np.load('min_max_mags.npy')
###

min_datapoints_each_filter = 10
custom_counter = 0

interpolation_section_examples_good = [
    # AGN
    'ZTF18aceytnk',
    'ZTF19adaiusp',
    'ZTF18acdyndq',
    'ZTF19aasszjp',
    'ZTF19abggdjc',
    'ZTF19acxyrwu',#*
    'ZTF18abkuccd',#*

    # bl lacs:
    'ZTF18admwqvn',
    'ZTF18aabyvml',#*
    'ZTF17aabwhei',#**
    'ZTF19agwxlv',
#     'ZTF19aakjfaw',#**

    # SN
    'ZTF22abjagqj',
    'ZTF21abghumo',
    'ZTF22abjvyib',
    'ZTF22aabbfre'

#     'ZT21abvfacx', 'ZTF22aaacxkp',
# #     'ZTF22aanesux',
#     'ZTF18aasszwr', 'ZTF17aadptpa',
# #     'ZTF19acvrkas',
#     'ZTF18aaupmln',
# #     'ZTF18acvgzbp',
# #     'ZTF20aapadcm',
# #     'ZTF22aaohpmz',
# #     'ZTF22aafupdg',
#     'ZTF18aaevepq',
#     'ZTF19aasboss'
]

interpolation_section_examples_bad = [
#     'ZTF18adllqvx',
    'ZTF22aakaygk',
#     'ZTF18aavnfwe',
#     'ZTF21aacugue',
    'ZTF22aanmbrl',
    'ZTF21abrgbex',
    'ZTF22aaahuly',
    'ZTF22aaijrul',

    # AGN
    'ZTF19abgfgzy',  # flare at ~300d not captured.
    'ZTF18admwqvn',
    'ZTF20acdcick',
##########################     'ZTF18aagteoy',  # shows that large RMSE/residuals for general AGNs can be dominated by flare/flare-like signatures
]

modes = [
    'random',
]
for mode in modes:
    counters_and_keys = get_counters_and_keys_from_mode(decoded_lcs_test, mode)
    show_counter = 0
    print(f'Total {len(counters_and_keys)} examples with mode = "{mode}"')

    for counter, key in counters_and_keys:
        test_currObjId = test_objIds_dataloader[counter]

        if test_currObjId not in interpolation_section_examples_bad:
            continue

        if test_currObjId in milliquas_agn_labels['objectId'].values:
            test_currCommonFinkclass = milliquas_agn_labels[milliquas_agn_labels['objectId'] == test_currObjId]['custom_type']
            assert len(test_currCommonFinkclass) == 1
            test_currCommonFinkclass = test_currCommonFinkclass.iloc[0]
#             if test_currCommonFinkclass == 'nan':
#                 test_currCommonFinkclass = 'Unknown'
        else:
            test_currObjId_index = np.where(test_objIds == test_currObjId)
            test_currCommonFinkclass = fine_classes[test_currObjId_index[0][0]]

#         _coarse_finkclass = df_alerts_pandas[df_alerts_pandas['objectId'] == test_currObjId]['finkclass'].mode().iloc[0]
#         if _coarse_finkclass == 'custom_agn':
#             continue

        # Also, we pass the same min_max_mags even if we use local normalization
        # for the mags because in plot_decoded_lc, we select the min/max for
        # that light curve, so it takes care of that.

        # Plot test decoded lc
        corrected_agn_condition = (
            (test_currCommonFinkclass in milliquas_agn_labels['custom_type'].unique())
            and (use_corrected_agns)
            and df_alerts_corrected_pandas.loc[df_alerts_corrected_pandas['objectId'] == test_currObjId, 'used_magpsf_corr'].all()
        )
        print(test_currObjId, key, corrected_agn_condition)

        ax = plot_decoded_lc(
            decoded_lcs_test, key=key, df_alerts=df_alerts_corrected if corrected_agn_condition else df_alerts, objId=test_currObjId,
            unnormalize_lcs=True, min_time=min_time, max_time=max_time,
            min_max_mags=min_max_mags_corrected if corrected_agn_condition else min_max_mags, prob_plot=False,
            whichclass=test_currCommonFinkclass,
            also_show_sigmapsf_and_rmse=True, show_separate_bands=False,
            show_only_rmse=True,
            fig_width=1.5*width*2/3, fig_height=1.5*height/4,
            show_legend=True if custom_counter == 0 else False, legend_loc='best',
            show_corrected_mags=True if corrected_agn_condition else False
        )
        ax.tick_params(axis='both', which='minor', length=3)
        ax.tick_params(axis='both', which='major', length=6)

        custom_counter += 1
#         plt.close()
        save_fig(f'light_curve_{test_currObjId}_interpolation_section.pdf')

In [ ]:
n_unmatched = ((~df_alerts_corrected['corrected']) & (df_alerts_corrected['finkclass'] == 'custom_agn')).sum().compute()
print(f"Unmatched AGN alert rows: {n_unmatched} / {len(df_alerts_corrected[df_alerts_corrected['finkclass'] == 'custom_agn'])}")

In [ ]:
# orig = pd.read_parquet('alerts_processed_ftransfer_ztf_2025_05_31_430518.parquet', columns=['objectId', 'candid', 'fid', 'jd', 'finkclass'])
# orig[orig['objectId'] == 'ZTF22abnoofs'][['objectId', 'candid', 'fid', 'jd', 'finkclass']]

In [ ]:
# # Uncomment below code to show big residual examples:


# mode = 'random'
# counters_and_keys = get_counters_and_keys_from_mode(decoded_lcs_test, mode)
# show_counter = 0
# print(f'Total {len(counters_and_keys)} examples with mode = "{mode}"')

# pdfs_by_objId = {obj: g for obj, g in df_alerts_pandas.groupby('objectId')}

# residuals_all, sigmapsf_all = [], []
# for counter, key in counters_and_keys:
#     test_currObjId = test_objIds_dataloader[counter]

#     if test_currObjId in milliquas_agn_labels['objectId'].values:
#         test_currCommonFinkclass = milliquas_agn_labels[milliquas_agn_labels['objectId'] == test_currObjId]['custom_type']
#         assert len(test_currCommonFinkclass) == 1
#         test_currCommonFinkclass = test_currCommonFinkclass.iloc[0]
#         if test_currCommonFinkclass == 'nan':
#             test_currCommonFinkclass = 'Unknown'
#     else:
#         test_currObjId_index = np.where(test_objIds == test_currObjId)
#         test_currCommonFinkclass = fine_classes[test_currObjId_index[0][0]]
    
#     if test_currCommonFinkclass != 'AGN':
#         continue

#     residuals, sigmapsf_list = get_residuals_from_decoded_lc(
#         pdfs_by_objId,
#         decoded_lcs_test, key=key, objId=test_currObjId,
#         unnormalize_lcs=True, min_time=min_time, max_time=max_time,
#         min_max_mags=min_max_mags,
#         whichclass=test_currCommonFinkclass,
#     )
#     if np.abs(np.any(residuals)) > 0.2:
#         print(test_currObjId)

#         # Plot test decoded lc
#         ax = plot_decoded_lc(
#             decoded_lcs_test, key=key, objId=test_currObjId,
#             unnormalize_lcs=True, min_time=min_time, max_time=max_time,
#             min_max_mags=min_max_mags, prob_plot=False,
#             whichclass=test_currCommonFinkclass,
#             also_show_sigmapsf_and_rmse=True, show_separate_bands=False,
#             show_only_rmse=True,
#             fig_width=1.5*width*2/3, fig_height=1.5*height/4,
#             show_legend=True if custom_counter == 0 else False, legend_loc='best'
#         )
#         ax.tick_params(axis='both', which='minor', length=3)
#         ax.tick_params(axis='both', which='major', length=6)

#         custom_counter += 1

#         plt.show()

In [ ]:
mode = 'random'
counters_and_keys = get_counters_and_keys_from_mode(decoded_lcs_test, mode)
show_counter = 0
print(f'Total {len(counters_and_keys)} examples with mode = "{mode}"')

###
decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
if correct_lcs_for_agns:
    decoded_lcs_test_corrected = np.load('evaluate_test_CORRECTED_decoded_lcs.npz')

    assert list(decoded_lcs_test.keys()) == list(decoded_lcs_test_corrected.keys()), "NPZ keys mismatch"

    # Build corrected_agn: start from original, replace AGN rows with corrected
    decoded_lcs_test_corrected_agn = {}
    for i, key in enumerate(decoded_lcs_test.keys()):
        if agn_indicator[i]:
            decoded_lcs_test_corrected_agn[key] = decoded_lcs_test_corrected[key].copy()
            # Verify AGN lc came from corrected
            np.testing.assert_array_equal(
                decoded_lcs_test_corrected_agn[key],
                decoded_lcs_test_corrected[key]
            )
        else:
            decoded_lcs_test_corrected_agn[key] = decoded_lcs_test[key].copy()
            # Verify non-AGN lc is unchanged
            np.testing.assert_array_equal(
                decoded_lcs_test_corrected_agn[key],
                decoded_lcs_test[key]
            )

    print(f"Replaced {agn_indicator.sum()} AGN rows out of {len(agn_indicator)} total")

dlte = list(decoded_lcs_test.keys())
print(decoded_lcs_test.f.arr_2.shape)

use_corrected_agns = True
if use_corrected_agns:
    print('Using corrected AGN light curves...')
    assert np.all(
        np.load('evaluate_test_objIds_dataloader.npy') == test_objIds_corrected_agn
    )
    test_objIds = test_objIds_corrected_agn
    decoded_lcs_test = decoded_lcs_test_corrected_agn
    min_max_mags = np.load('min_max_mags.npy')
    min_max_mags_corrected = np.load('min_max_mags_ftransfer_ztf_2025_05_31_430518_CORRECTED.npy')
else:
    print('Using uncorrected AGN light curves...')
    test_objIds  = np.load('evaluate_test_objIds_dataloader.npy')
    decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
    min_max_mags = np.load('min_max_mags.npy')
###

df_alerts_corrected = pd.read_parquet(
    'alerts_processed_ftransfer_ztf_2025_05_31_430518_CORRECTED.parquet',
    columns=[
        'objectId', 'finkclass', 'tnsclass',
         'fid', 'magpsf_corr', 'sigmapsf_corr', 'jd', 'used_magpsf_corr'
    ]
)

residuals_all, sigmapsf_all = [], []
for counter, key in counters_and_keys:
    test_currObjId = test_objIds_dataloader[counter]

    if test_currObjId in milliquas_agn_labels['objectId'].values:
        test_currCommonFinkclass = milliquas_agn_labels[milliquas_agn_labels['objectId'] == test_currObjId]['custom_type']
        assert len(test_currCommonFinkclass) == 1
        test_currCommonFinkclass = test_currCommonFinkclass.iloc[0]
#             if test_currCommonFinkclass == 'nan':
#                 test_currCommonFinkclass = 'Unknown'
    else:
        test_currObjId_index = np.where(test_objIds == test_currObjId)
        test_currCommonFinkclass = fine_classes[test_currObjId_index[0][0]]

    corrected_agn_condition = (
        (test_currCommonFinkclass in milliquas_agn_labels['custom_type'].unique())
        and (use_corrected_agns)
        and df_alerts_corrected.loc[df_alerts_corrected['objectId'] == test_currObjId, 'used_magpsf_corr'].all()
    )
    uncorrected_agn_condition = (
        (test_currCommonFinkclass in milliquas_agn_labels['custom_type'].unique())
        and (use_corrected_agns)
        and not df_alerts_corrected.loc[df_alerts_corrected['objectId'] == test_currObjId, 'used_magpsf_corr'].any()
    )
    if uncorrected_agn_condition:
        continue
    if corrected_agn_condition:
        df_alerts_to_use = df_alerts_corrected
    else:
        df_alerts_to_use = df_alerts

    pdfs_by_objId = {test_currObjId: df_alerts_to_use.groupby('objectId').get_group(test_currObjId)}

    residuals, sigmapsf_list = get_residuals_from_decoded_lc(
        pdfs_by_objId,
        decoded_lcs_test, key=key, objId=test_currObjId,
        unnormalize_lcs=True, min_time=min_time, max_time=max_time,
        min_max_mags=min_max_mags_corrected if corrected_agn_condition else min_max_mags,
        whichclass=test_currCommonFinkclass, sigmapsf_column='sigmapsf_corr' if corrected_agn_condition else 'sigmapsf',
        magpsf_column='magpsf_corr' if corrected_agn_condition else 'magpsf'
    )
    residuals_all.append(residuals)
    sigmapsf_all.append(sigmapsf_list)

residuals_all = np.concatenate(residuals_all, axis=0)
sigmapsf_all = np.concatenate(sigmapsf_all, axis=0)

print(residuals_all.shape, sigmapsf_all.shape)
assert len(residuals_all) == len(sigmapsf_all)

In [ ]:
print(residuals_all.shape)

# Below from https://stackoverflow.com/a/11886564
def is_outlier(points, thresh=3.5):
    """
    Returns a boolean array with True if points are outliers and False 
    otherwise.

    Parameters:
    -----------
        points : An numobservations by numdimensions array of observations
        thresh : The modified z-score to use as a threshold. Observations with
            a modified z-score (based on the median absolute deviation) greater
            than this value will be classified as outliers.

    Returns:
    --------
        mask : A numobservations-length boolean array.

    References:
    ----------
        Boris Iglewicz and David Hoaglin (1993), "Volume 16: How to Detect and
        Handle Outliers", The ASQC Basic References in Quality Control:
        Statistical Techniques, Edward F. Mykytka, Ph.D., Editor. 
    """
    if len(points.shape) == 1:
        points = points[:,None]
    median = np.median(points, axis=0)
    diff = np.sum((points - median)**2, axis=-1)
    diff = np.sqrt(diff)
    med_abs_deviation = np.median(diff)

    modified_z_score = 0.6745 * diff / med_abs_deviation

    return modified_z_score > thresh

print(residuals_all[~is_outlier(residuals_all)].shape)
plt.hist(residuals_all[~is_outlier(residuals_all)])

from scipy.stats import norm

mean, std = norm.fit(residuals_all[~is_outlier(residuals_all)])
# print(mean, std)
x = np.linspace(mean - 3.5*std, mean + 3.5*std, 100)
plt.plot(x, 0.5e4*norm.pdf(x, mean, std), c='black')  # scaled by some random constant
plt.show()

print(np.mean(residuals_all[~is_outlier(residuals_all)]))
print(np.std(residuals_all[~is_outlier(residuals_all)]))
print(np.mean(residuals_all))
print(np.std(residuals_all))

plt.hexbin(
    sigmapsf_all[~is_outlier(residuals_all)],
    residuals_all[~is_outlier(residuals_all)],
    gridsize=30
)
plt.xlabel(r'$\sigma_{\mathrm{psf}}$')
plt.ylabel('Residual')
plt.show()

In [ ]:
# print(np.round(np.mean(residuals_all), 3))
# print(np.round(np.std(residuals_all), 3))

In [ ]:
# plt.hexbin(
#     sigmapsf_all[~is_outlier(residuals_all)][(residuals_all[~is_outlier(residuals_all)] >= sigmapsf_all[~is_outlier(residuals_all)])],
#     residuals_all[~is_outlier(residuals_all)][(residuals_all[~is_outlier(residuals_all)] >= sigmapsf_all[~is_outlier(residuals_all)])],
#     gridsize=30
# )
# plt.xlabel(r'$\sigma_{\mathrm{psf}}$')
# plt.ylabel('Residual')
# plt.show()

import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# z = residuals_all[~is_outlier(residuals_all)] / sigmapsf_all[~is_outlier(residuals_all)]
z = residuals_all / sigmapsf_all

print("mean(z) =", z.mean())
print("std(z)  =", z.std(ddof=1))
chi2_red = np.sum(z**2) / (len(z) - 1)   # use p=1 if unknown
print("reduced chi2 =", chi2_red)

for k in (1.0, 2.0):
    print(f"frac |z|<{k}:", np.mean(np.abs(z) < k))

# KS test vs normal
ks_stat, ks_p = stats.kstest(z, 'norm', args=(0, z.std(ddof=1)))
print("KS p-value:", ks_p)

# Plots
plt.figure(); plt.hist(z, bins=60, density=True); plt.title('z histogram')
plt.figure(); stats.probplot(z, dist="norm", plot=plt); plt.title('QQ plot')
plt.show()


## Interpolation OOD data

In [ ]:
matplotlib.rcParams["font.size"] = "18"

decoded_lcs_test = np.load('evaluate_OOD_test_decoded_lcs.npz')
dlte = list(decoded_lcs_test.keys())
print(decoded_lcs_test.f.arr_0.shape)
OOD_test_objIds_dataloader = np.load('evaluate_OOD_test_objIds_dataloader.npy')

min_datapoints_each_filter = 10
num_to_show = 15
custom_counter = 0

train_val_test_min_max_times = np.load('train_val_test_min_max_times.npy')
train_min_time, train_max_time = train_val_test_min_max_times[0], train_val_test_min_max_times[1]

# NOTE: Unlike the other sections, here OOD_df_alerts already contains
# corrected magnitudes wherever needed and uncorrected magnitudes
# otherwise. So in the subsequent code, simply passing OOD_df_alerts
# works irrespective of whether corrected or not.
# Similarly for OOD_min_max_mags, no change to handle corrected/uncorrected
# required in subsequent code.
OOD_min_max_mags = np.load('OOD_min_max_mags.npy')
OOD_df_alerts = pd.read_json('OOD_df_alerts.json')

assert (OOD_df_alerts[(OOD_df_alerts['v:classification'] == 'RRLyr') | (OOD_df_alerts['v:classification'] == 'LPV*')]['corrected'] == 0.).sum() == 0
assert (OOD_df_alerts[(OOD_df_alerts['v:classification'] == 'RRLyr') | (OOD_df_alerts['v:classification'] == 'LPV*')]['magpsf_corr'] == 100.).sum() == 0

modes = [
    'random',
]

######################################################################
# Below two variables taken from `OOD_evaluate_unsupervised_prepare_data.py`.
object_ids = {
    'rrlyrae': ['ZTF17aaaebgd', 'ZTF18abadkhl', 'ZTF18adalmbo', 'ZTF18abcosdx', 'ZTF18abklama'],
    'lpv':     ['ZTF18abryodl', 'ZTF18aabjjhq', 'ZTF18aaaaogd'],
    'tde':     ['ZTF20aahmtso', 'ZTF25aclckzl', 'ZTF20accxwrk', 'ZTF22aadesap'],  # ZTF25aclckzl: https://www.wis-tns.org/astronotes/astronote/2026-7
}
rrlyrae_lpv_ids = set(object_ids['rrlyrae'] + object_ids['lpv'])
######################################################################
label_letters = {
    oid: chr(ord('A') + i)
    for i, oid in enumerate(oid for ids in object_ids.values() for oid in ids)
}
print(label_letters)

for mode in modes:
    # `get_counters_and_keys_from_mode` was defined in the interpolation section before.
    counters_and_keys = get_counters_and_keys_from_mode(decoded_lcs_test, mode)
    show_counter = 0
    print(f'Total {len(counters_and_keys)} examples with mode = "{mode}"')

    for counter, key in counters_and_keys:
        test_currObjId = OOD_test_objIds_dataloader[counter]

        # These are custom examples. I'm setting it to None to make the code work easily.
        # But given the small no. of examples generally here, individual fink classes may be easily
        # accessible through manual search on Fink, for example.
        test_currCommonFinkclass = ''

        # Also, we pass a single min_max_mag array even if we use local normalization
        # for the mags because in plot_decoded_lc, we select the min/max for
        # that light curve, so it takes care of that.

        # Plot test decoded lc
        print(test_currObjId in rrlyrae_lpv_ids)
        ax = plot_decoded_lc(
            decoded_lcs_test, key=key, df_alerts=OOD_df_alerts, objId=test_currObjId,
            unnormalize_lcs=True, min_time=train_min_time, max_time=train_max_time,
            min_max_mags=OOD_min_max_mags, prob_plot=False,
            whichclass=test_currCommonFinkclass + f' ({label_letters[test_currObjId]})',
            also_show_sigmapsf_and_rmse=True, show_separate_bands=False,
            show_only_rmse=True,
            fig_width=1.5*width*2/3, fig_height=1.5*height/4,
            show_legend=True if custom_counter == 0 else False, legend_loc='best',
            fid_column='fid', objectId_column='objectId',
            show_corrected_mags=True if test_currObjId in rrlyrae_lpv_ids else False
        )

        ax.tick_params(axis='both', which='minor', length=3)
        ax.tick_params(axis='both', which='major', length=6)

        custom_counter += 1

#         plt.show()
#         plt.close()
        save_fig(f'light_curve_{test_currObjId}_interpolation_OOD.pdf')

In [ ]:
OOD_min_max_mags

In [ ]:
OOD_objIds = np.load('evaluate_OOD_test_objIds_dataloader.npy')
OOD_objIds

In [ ]:
OOD_test_outputs_condensed = np.load('evaluate_OOD_test_outputs_condensed.npy')
OOD_test_outputs_condensed_condensed = OOD_test_outputs_condensed.mean(1)

# Flatten across latent_dim dimension.
OOD_test_outputs_condensed_condensed = OOD_test_outputs_condensed_condensed.transpose(0, 2, 1).reshape(OOD_test_outputs_condensed_condensed.shape[0], -1)

OOD_test_outputs_condensed_condensed = np.nan_to_num(OOD_test_outputs_condensed_condensed, nan=0.)

# print(f'shape before padding: {OOD_test_outputs_condensed_condensed.shape}')

# def pad_to_length(x, target_len, axis=1):
#     pad_width = [(0, 0)] * x.ndim
#     pad_width[axis] = (0, max(0, target_len - x.shape[axis]))
#     return np.pad(x, pad_width, mode='constant')

assert test_outputs_condensed_condensed.shape[1] == 546

# print(OOD_test_outputs_condensed_condensed.shape)
# OOD_test_outputs_condensed_condensed = pad_to_length(
#     OOD_test_outputs_condensed_condensed, 546
# )
# print(f'shape after padding: {OOD_test_outputs_condensed_condensed.shape}')

# NOTE: Truncating the encodings to match the shape is not the best choice
# because the truncated encoding is in general not the same as the encoding
# obtained on the truncated time interval. But since we truncated the OOD light
# curves to match the maximum duration seen in training, the difference between
# the encoded vectors and the truncated vectors to match the training duration
# will not be huge. Indeed, the difference is 552-546, which is small, and so
# truncation is okay. If this difference is large, simply truncating the encodings
# may not be the right choice.
OOD_test_outputs_condensed_condensed_transformed = umap_fit.transform(
    # TEMPORARY HACK: indexing over the first 546 elements is necessary since umap_fit
    # was fitted on SN+AGN data that had that many entries (check: test_outputs_condensed_condensed.shape[1]).
    # Due to much longer light curves for the OOD data here, the dimensions was ~2k, which gave an error due to mismatch.
    OOD_test_outputs_condensed_condensed[:, :546]
)

In [ ]:
matplotlib.rcParams["font.size"] = 12

import seaborn as sns
fig = plt.figure(figsize=(width/2, height/3))
ax = plt.gca()

sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[:, 0],
    y=test_outputs_condensed_condensed_transformed[:, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2
)

# ax.scatter(
#     test_outputs_condensed_condensed_transformed[:, 0][agn_indicator==1], test_outputs_condensed_condensed_transformed[:, 1][agn_indicator==1],
#     c=colors[2], label='AGN', s=150, alpha=0.3
# )
# ax.scatter(
#     test_outputs_condensed_condensed_transformed[:, 0][sn_indicator==1], test_outputs_condensed_condensed_transformed[:, 1][sn_indicator==1],
#     c=colors[0], label='SN', s=150, alpha=0.3
# )
ax.scatter(
    OOD_test_outputs_condensed_condensed_transformed[:, 0],
    OOD_test_outputs_condensed_condensed_transformed[:, 1],
    c='black', s=20, alpha=1., marker='x'
)

texts = []
for i in range(len(OOD_test_outputs_condensed_condensed_transformed)):
    texts.append(ax.text(
        OOD_test_outputs_condensed_condensed_transformed[i, 0],
        OOD_test_outputs_condensed_condensed_transformed[i, 1],
        label_letters[OOD_objIds[i]],
        fontsize=13
    ))

adjust_text(
    texts, ax=ax,
    x=OOD_test_outputs_condensed_condensed_transformed[:, 0],
    y=OOD_test_outputs_condensed_condensed_transformed[:, 1],
    arrowprops=dict(arrowstyle='-', color=orangish_color, lw=1.),
    expand=(2.2, 2.2)
)
# for i, (arrow, text) in enumerate(zip(arrows, texts)):
#     if label_letters[OOD_objIds[i]] == 'A':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((9, 8), None)
#         text.set_position((9-0.05, 8+0.05))
#     if label_letters[OOD_objIds[i]] == 'C':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((9, 5.5), None)
#         text.set_position((9-0.05, 5.5-0.05))
#     if label_letters[OOD_objIds[i]] == 'E':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((10.2, 13), None)
#         text.set_position((10.2, 13+0.05))
#     if label_letters[OOD_objIds[i]] == 'F':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((11.5, 5.5), None)
#         text.set_position((11.5, 5.5-0.05))
#     if label_letters[OOD_objIds[i]] == 'I':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((9, 13), None)
#         text.set_position((9, 13+0.05))
#     if label_letters[OOD_objIds[i]] == 'K':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((12, 12), None)
#         text.set_position((12, 12-0.05))
#     if label_letters[OOD_objIds[i]] == 'L':
#         posA, posB = arrow._posA_posB
#         arrow.set_positions((7.6, 9), None)
#         text.set_position((7.6, 9-0.05))
#     arrow.set_mutation_scale(20)

"""
# for i in range(len(OOD_test_outputs_condensed_condensed_transformed)):
# #     ax.text(
# #         OOD_test_outputs_condensed_condensed_transformed[i, 0],
# #         OOD_test_outputs_condensed_condensed_transformed[i, 1],
# #         label_letters[i],
# #         fontsize=14
# #     )
#     ax.annotate(
#         label_letters[OOD_objIds[i]], xy=(
#             OOD_test_outputs_condensed_condensed_transformed[i, 0],
#             OOD_test_outputs_condensed_condensed_transformed[i, 1]
#         ), xytext=(
#             OOD_test_outputs_condensed_condensed_transformed[i, 0]+0.1,
#             OOD_test_outputs_condensed_condensed_transformed[i, 1]+0.1
#         ),
#         fontsize=13
#     )
"""

xticks = [4, 6, 8, 10, 12, 14]
yticks = [4, 6, 8, 10, 12, 14]
ax.set_xticks(xticks)
ax.set_yticks(yticks)

ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
set_axes_style(ax)

save_fig('projection_classes_OOD.pdf')

## Graph-based classification (unused approach)

In [ ]:
# Prepare data.
OOD_test_outputs_condensed = np.load('evaluate_test_ftransfer_ztf_2026_02_01_378646_outputs_condensed.npy')
# Since OOD_test_outputs_condensed is of shape:
# (num_test_examples, num_samples_VAE, max_num_ref_points_test_set, number_of_filters),
# we are averaging over the num_sample dimension.
OOD_test_outputs_condensed_condensed = OOD_test_outputs_condensed.mean(1)

# Flatten across latent_dim dimension.
OOD_test_outputs_condensed_condensed = OOD_test_outputs_condensed_condensed.transpose(0, 2, 1).reshape(OOD_test_outputs_condensed_condensed.shape[0], -1)

OOD_test_outputs_condensed_condensed = np.nan_to_num(OOD_test_outputs_condensed_condensed, nan=0.)

def pad_to_length(x, target_len, axis=1):
    pad_width = [(0, 0)] * x.ndim
    pad_width[axis] = (0, max(0, target_len - x.shape[axis]))
    return np.pad(x, pad_width, mode='constant')

assert test_outputs_condensed_condensed.shape[1] == 546
print(OOD_test_outputs_condensed_condensed.shape)
OOD_test_outputs_condensed_condensed = pad_to_length(
    OOD_test_outputs_condensed_condensed, 546
)
print(OOD_test_outputs_condensed_condensed.shape)

OOD_test_outputs_condensed_condensed_transformed = umap_fit.transform(
    # TEMPORARY HACK: indexing over the first 546 elements is necessary since umap_fit
    # was fitted on SN+AGN data that had that many entries (check: test_outputs_condensed_condensed.shape[1]).
    # Due to much longer light curves for the OOD data here, the dimensions was ~2k, which gave an error due to mismatch.
    OOD_test_outputs_condensed_condensed[:, :546]
)

In [ ]:
matplotlib.rcParams["font.size"] = 12

import seaborn as sns
fig = plt.figure(figsize=(width/2, height/3))
ax = plt.gca()

sns.kdeplot(
    x=test_outputs_condensed_condensed_transformed[:, 0],
    y=test_outputs_condensed_condensed_transformed[:, 1],
    fill=True, cmap='gray_r', levels=15, thresh=0, alpha=0.2
)

# ax.scatter(
#     test_outputs_condensed_condensed_transformed[:, 0][agn_indicator==1], test_outputs_condensed_condensed_transformed[:, 1][agn_indicator==1],
#     c=colors[2], label='AGN', s=150, alpha=0.3
# )
# ax.scatter(
#     test_outputs_condensed_condensed_transformed[:, 0][sn_indicator==1], test_outputs_condensed_condensed_transformed[:, 1][sn_indicator==1],
#     c=colors[0], label='SN', s=150, alpha=0.3
# )
ax.scatter(
    OOD_test_outputs_condensed_condensed_transformed[:, 0], OOD_test_outputs_condensed_condensed_transformed[:, 1],
    c='black', s=20, alpha=0.5, marker='x'
)

# for i in range(len(OOD_test_outputs_condensed_condensed_transformed)):
# #     ax.text(
# #         OOD_test_outputs_condensed_condensed_transformed[i, 0],
# #         OOD_test_outputs_condensed_condensed_transformed[i, 1],
# #         label_letters[i],
# #         fontsize=14
# #     )
#     ax.annotate(
#         label_letters[i], xy=(
#             OOD_test_outputs_condensed_condensed_transformed[i, 0],
#             OOD_test_outputs_condensed_condensed_transformed[i, 1]
#         ), xytext=(
#             OOD_test_outputs_condensed_condensed_transformed[i, 0]+0.1,
#             OOD_test_outputs_condensed_condensed_transformed[i, 1]+0.1
#         ),
#         fontsize=13
#     )

ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
set_axes_style(ax)

save_fig('projection_classes_OOD_GraphTest.pdf')

In [ ]:
df_alerts_graph_test = dd.read_parquet(
    'alerts_processed_ftransfer_ztf_2026_02_01_378646.parquet',
    columns=[
            'objectId', 'finkclass', 'tnsclass',
             'fid', 'magpsf', 'sigmapsf', 'jd'
        ]
)
df_alerts_graph_test_pandas = df_alerts_graph_test.compute()
df_alerts_graph_test_pandas.columns

In [ ]:
objectIds = np.load('evaluate_test_ftransfer_ztf_2026_02_01_378646_objIds_dataloader.npy')
latent_encodings = np.load('evaluate_test_ftransfer_ztf_2026_02_01_378646_outputs_condensed.npy').mean(1)

# For finding classes, do smth similar to what was done for milliquas labels
df_alerts_graph_test = dd.read_parquet(
    'alerts_processed_ftransfer_ztf_2026_02_01_378646.parquet',
    columns=[
            'objectId', 'finkclass', 'tnsclass',
             'fid', 'magpsf', 'sigmapsf', 'jd'
        ]
)
df_alerts_graph_test_pandas = df_alerts_graph_test.compute()

def get_mode(series):
    mode = series.mode()
    if mode.size > 1:
        print('Warning: more than one mode present. Selecting first mode, which may be random.')
    return mode[0]
#     if mode.size == 1:
#         return mode[0]
#     return "x"

df_alerts_graph_test_pandas['custom_type'] = df_alerts_graph_test_pandas.groupby('objectId')['finkclass'].transform(get_mode)

df_alerts_graph_test_pandas = df_alerts_graph_test_pandas.sort_values(
    by='objectId',
    key=lambda s: pd.Categorical(s, categories=objectIds, ordered=True)
)

assert np.all(
    df_alerts_graph_test_pandas['objectId'].unique().tolist() == objectIds
)
object_classes = df_alerts_graph_test_pandas.groupby('objectId')['custom_type'].first()
classes = object_classes.reindex(objectIds).tolist()

print(len(objectIds), len(latent_encodings), len(classes))
assert len(objectIds) == len(latent_encodings) == len(classes)

np.savez(
    "latents_with_metadata.npz",
    latent_encodings=latent_encodings,
    objectIds=objectIds,
    classes=classes
)

In [ ]:
data = np.load("latents_with_metadata.npz")

latents = data["latent_encodings"]
objectIds = data["objectIds"]
labels = data["classes"]
latents[12], objectIds[12], labels[12]

In [ ]:
data["latent_encodings"][0].shape, data["latent_encodings"][3].shape, data["latent_encodings"][5].shape

# Interpretation of encoder attention scores

In [ ]:
from evaluate_unsupervised import (
    dim, latent_dim, rec_hidden, gen_hidden, sample_tp,
    embed_time, learn_emb, enc_num_heads, dec_num_heads, device,
    model_file_path, DATA_COMBINED_PATH,
    DATA_IDS_PATH, ref_resolution_days
)
from models import enc_mtan_rnn, dec_mtan_rnn

In [ ]:
import torch
rec = enc_mtan_rnn(
    dim, latent_dim, rec_hidden,
    embed_time=embed_time, learn_emb=learn_emb,
    num_heads=enc_num_heads, device=device
).to(device)


dec = dec_mtan_rnn(
    dim, latent_dim, gen_hidden,
    embed_time=embed_time, learn_emb=learn_emb,
    num_heads=dec_num_heads, device=device
).to(device)

model_file = torch.load(model_file_path, map_location=torch.device('cpu'))
rec.load_state_dict(model_file['rec_state_dict'])
rec.eval()

dec.load_state_dict(model_file['dec_state_dict'])
dec.eval()

rec, dec

In [ ]:
rec.att.linears[-1].weight[:, 0].shape, rec.att.linears[-1].weight[:, 1].shape

In [ ]:
U_g = rec.att.linears[-1].weight[:, 0]
U_r = rec.att.linears[-1].weight[:, 1]
plt.hist(U_g.detach().numpy(), label='g')
plt.hist(U_r.detach().numpy(), label='r')
plt.legend()

See [these lines in the original mTAN code](https://github.com/reml-lab/mTAN/blob/7a3d536ee742f1cacb4a6d3478ac78a228d995ff/src/models.py#L109). We first calculate the key and query, followed by mTAN to calculate the attention, which is then passed to a GRU and then a hidden linear layer module.

In [ ]:
import torch
# import numpy as np
# from torch.utils.data import DataLoader
# from prepare_data import MyDataSet

# data_combined = torch.load(DATA_COMBINED_PATH)
# data_Ids = np.load(DATA_IDS_PATH)
# dataset = MyDataSet(data_combined, data_Ids)
# test_loader = DataLoader(dataset, batch_size=1, num_workers=1, shuffle=False)

test_loader = torch.load('test_dataloader.pth')
decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
test_objIds_dataloader = np.load('evaluate_test_objIds_dataloader.npy')
OOD_test_dataloader = torch.load('OOD_test_dataloader.pth')
OOD_test_objIds = np.load('OOD_test_objIds.npy')

In [ ]:
matplotlib.rcParams["font.size"] = 14

##########################################################
decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
if correct_lcs_for_agns:
    decoded_lcs_test_corrected = np.load('evaluate_test_CORRECTED_decoded_lcs.npz')

    assert list(decoded_lcs_test.keys()) == list(decoded_lcs_test_corrected.keys()), "NPZ keys mismatch"

    # Build corrected_agn: start from original, replace AGN rows with corrected
    decoded_lcs_test_corrected_agn = {}
    for i, key in enumerate(decoded_lcs_test.keys()):
        if agn_indicator[i]:
            decoded_lcs_test_corrected_agn[key] = decoded_lcs_test_corrected[key].copy()
            # Verify AGN lc came from corrected
            np.testing.assert_array_equal(
                decoded_lcs_test_corrected_agn[key],
                decoded_lcs_test_corrected[key]
            )
        else:
            decoded_lcs_test_corrected_agn[key] = decoded_lcs_test[key].copy()
            # Verify non-AGN lc is unchanged
            np.testing.assert_array_equal(
                decoded_lcs_test_corrected_agn[key],
                decoded_lcs_test[key]
            )

    print(f"Replaced {agn_indicator.sum()} AGN rows out of {len(agn_indicator)} total")

use_corrected_agns = True
if use_corrected_agns:
    print('Using corrected AGN light curves...')
    assert np.all(
        np.load('evaluate_test_objIds_dataloader.npy') == test_objIds_corrected_agn
    )
    test_objIds = test_objIds_corrected_agn
    decoded_lcs_test = decoded_lcs_test_corrected_agn
    min_max_mags = np.load('min_max_mags.npy')
    min_max_mags_corrected = np.load('min_max_mags_ftransfer_ztf_2025_05_31_430518_CORRECTED.npy')
else:
    print('Using uncorrected AGN light curves...')
    test_objIds  = np.load('evaluate_test_objIds_dataloader.npy')
    decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
    min_max_mags = np.load('min_max_mags.npy')
##########################################################

import utils
from mpl_toolkits.axes_grid1 import make_axes_locatable

from torch_intermediate_layer_getter import IntermediateLayerGetter as MidGetter
return_layers = {
    'att': 'att'
#     'hiddens_to_z0': 'hiddens_to_z0',
}
mid_getter = MidGetter(rec, return_layers=return_layers, keep_output=True)
mid_getter2 = MidGetter(dec, return_layers=return_layers, keep_output=True)

from matplotlib.ticker import FormatStrFormatter

def _normalize_line_plot(y, a=0., b=1.):
    norm = (y - y.min()) / (y.max() - y.min())
    scaled = a + norm * (b - a)
    return scaled

train_min_max_times = np.load('train_min_max_times.npy')
min_time, max_time = train_min_max_times[0], train_min_max_times[1]

#################################### NOTE #####################################
# The below code is partly a copy-paste of the code in evaluate_unsupervised.py
# NOTE: The below code for visualization assumes a batch size of 1.
# Otherwise it will give an error or not work as expected.
###############################################################################

train_val_test_min_max_times_filename = 'train_val_test_min_max_times.npy'
train_val_test_min_max_times = np.load(train_val_test_min_max_times_filename)
train_max_time = train_val_test_min_max_times[1]
delta_t = (ref_resolution_days * 24) / train_max_time

objectId_column, fid_column = 'objectId', 'fid'

df_alerts_corrected = pd.read_parquet(
    'alerts_processed_ftransfer_ztf_2025_05_31_430518_CORRECTED.parquet',
    columns=[
        'objectId', 'finkclass', 'tnsclass',
         'fid', 'magpsf_corr', 'sigmapsf_corr', 'jd'
    ]
)

with torch.no_grad():
    for c, batch in enumerate(test_loader):
        objId = batch[1][0]

        if objId != 'ZTF22abjvyib' and objId != 'ZTF19abgfgzy' and objId != 'ZTF22aanmbrl' and objId != 'ZTF19aagwxlv':  # ZTF19aagwxlv
            continue

        print(objId)
        
        ## Added code to check corrected for AGN
        if objId in milliquas_agn_labels['objectId'].values:
            test_currCommonFinkclass = milliquas_agn_labels[milliquas_agn_labels['objectId'] == objId]['custom_type']
            assert len(test_currCommonFinkclass) == 1
            test_currCommonFinkclass = test_currCommonFinkclass.iloc[0]
        #         if test_currCommonFinkclass == 'nan':
        #             test_currCommonFinkclass = 'Unknown'
        else:
            test_currObjId_index = np.where(test_objIds == objId)
            test_currCommonFinkclass = fine_classes[test_currObjId_index[0][0]]
        corrected_agn_condition = (
            (test_currCommonFinkclass in milliquas_agn_labels['custom_type'].unique())
            and (use_corrected_agns)
            and df_alerts_corrected_pandas.loc[df_alerts_corrected_pandas['objectId'] == objId, 'used_magpsf_corr'].all()
        )
        if corrected_agn_condition:
            print(f'Using corrected min/max mags')
        else:
            print('Using uncorrected min/max mags')
        use_min_max_mags = min_max_mags_corrected if corrected_agn_condition else min_max_mags
        print(objId, corrected_agn_condition)
        ########################################

#         if objId != 'ZTF22aafoqrd' and objId != 'ZTF18abtrubp':
#             continue

        key = list(decoded_lcs_test.keys())[int(np.where(test_objIds == objId)[0][0])]

        test_batch = batch[0]
        test_batch = test_batch.to(device)
        observed_data, observed_mask, observed_tp = (
            test_batch[:, :, :dim],
            test_batch[:, :, dim: 2 * dim],
            test_batch[:, :, -1],
        )
        if sample_tp and sample_tp < 1:
            subsampled_data, subsampled_tp, subsampled_mask = subsample_timepoints(
                observed_data.clone(), observed_tp.clone(), observed_mask.clone(), sample_tp)
        else:
            subsampled_data, subsampled_tp, subsampled_mask = \
                observed_data, observed_tp, observed_mask

        if sample_tp == 1.:
            assert torch.all(observed_tp == subsampled_tp)

        # Thq query times
        query = utils.generate_query_matrix(subsampled_tp, delta_t, padding_value=-999)

        mid_outputs, model_output = mid_getter(
            torch.cat(
                (subsampled_data, subsampled_mask), 2
            ),
            subsampled_tp, query, return_att=True
        )
        # Shape of mid_outputs['att'][1] for the encoder is (B, h, Tq, Tk, 2*num_filters)
#         print(mid_outputs['att'][0].shape, mid_outputs['att'][1].shape)
#         print(subsampled_data.shape, subsampled_mask.shape, subsampled_tp.shape)

        qz0_mean, qz0_logvar = (
            model_output[:, :, : latent_dim],
            model_output[:, :, latent_dim:],
        )
        num_sample = 1
        epsilon = torch.randn(
            num_sample, qz0_mean.shape[0], qz0_mean.shape[1], qz0_mean.shape[2]
        ).to(device)
        z0 = epsilon * torch.exp(0.5 * qz0_logvar) + qz0_mean
        z0 = z0.view(-1, qz0_mean.shape[1], qz0_mean.shape[2])
        batch, seqlen = observed_tp.size()
        time_steps = (
            observed_tp[None, :, :].repeat(num_sample, 1, 1).view(-1, seqlen)
        )
        pred_x = dec(z0, time_steps, query)
        dmid_outputs, dmodel_output = mid_getter2(
            z0, time_steps, query
        )
#         print(dmid_outputs['att'].shape, mid_outputs['att'][1].shape)
        # Average over the batch dimension.
        dattn_map = dmid_outputs['att'].mean(0)
        # After the above operation, dattn_map is a 2d matrix: 347 x 2*gen_n_hidden.
        dattn_map = dattn_map[:, :gen_hidden]

        # Convert to numpy
        subsampled_data, subsampled_tp, subsampled_mask = subsampled_data.numpy(), subsampled_tp.numpy(), subsampled_mask.numpy()
        observed_data, observed_tp, observed_mask = observed_data.numpy(), observed_tp.numpy(), observed_mask.numpy()
        query = query.numpy()

        # The query must be evenly spaced by design
        assert np.allclose(
            np.diff(query.squeeze()), np.diff(query.squeeze())[0]
        )

        df_alerts_to_use = df_alerts_corrected if corrected_agn_condition else df_alerts
        df_subset = df_alerts_to_use[df_alerts_to_use[objectId_column] == objId]
        if not corrected_agn_condition:
            df_subset = df_subset.compute()

        num_filters = 2
        for i in range(num_filters):
            maskFilt = df_subset[fid_column] == (i+1)
            assert len(df_subset[maskFilt]) == observed_mask[:, :, i].sum()

        # Calculate attention map for plotting
        # First, average over the batch and head dimension.
        attn_map = mid_outputs['att'][1].mean(0).mean(0)
        # `attn_map` contains 2*num_filters entries in the last dimension
        # where first `num_filters` dimensions correspond to the mean and
        # the latter `num_filters` dimensions correspond to standard deviation
        attn_map = attn_map[:, :, :num_filters]
        print(f'attn_map shape = {attn_map.shape}')

        test_currObjId = test_objIds_dataloader[c]
        test_currObjId_index = np.where(total_objIds == test_currObjId)
        test_currCommonFinkclass = total_common_finkclasses[test_currObjId_index].squeeze()
        test_currCommonFinkclass = test_currCommonFinkclass[test_currCommonFinkclass != '0']
        assert len(test_currCommonFinkclass) == 1

        observed_tp[observed_tp == 0.0] = np.nan
        observed_tp[:, 0] = 0.0
        observed_tp = observed_tp.squeeze()
        tmax = np.nanmax(observed_tp)
        arg_tmax = np.nanargmax(observed_tp)
        assert observed_tp[arg_tmax] == tmax
        observed_data[observed_mask == 0] = np.nan
        row = use_min_max_mags[np.where(use_min_max_mags[:, 0] == objId)].squeeze()
        obs_to_show = unnormalize_mag(
            observed_data, observed_mask, float(row[1]), float(row[2])
        )

        print(observed_tp.shape, obs_to_show.shape, observed_mask.shape, query.shape)

        observed_tp_unnormalized = unnormalize_time(
            observed_tp[observed_tp <= tmax].squeeze(), min_time, max_time
        ) / 24
        query_tp_unnormalized = unnormalize_time(
            query.squeeze(), min_time, max_time
        ) / 24
        assert np.allclose(
            np.diff(query_tp_unnormalized.squeeze()),
            np.diff(query_tp_unnormalized.squeeze())[0],
            rtol=1e-3
        )

        # Store xlim for matching across plots
        xlim_maxs = []

        X, Y = np.meshgrid(
            observed_tp_unnormalized,
            query_tp_unnormalized
        )

#         fig, ax = plt.subplots(1, num_filters, figsize=(width*2, height/2))
        fig, ax = plt.subplots(2, 1, figsize=(width*0.9, height/1.5), sharex=True, sharey=True)  
#         plt.subplots_adjust(hspace=0.09)
        # NOTE: It is true that the attention map is calculated for each
        # filter separately. It's not calculated for each `latent_dim` separately.
        # So the visualization for g and r bands separately is as expected
        # and not a coincidence due to internally using `latent_dim` which is equal
        # to the no. of filters in our case.

        mesh_vmin, mesh_vmax = attn_map[:, :arg_tmax+1].min(), attn_map[:, :arg_tmax+1].max()
        print(mesh_vmin, mesh_vmax)

        for ii in range(num_filters):
            # arg_tmax_ii = np.nanargmax(
            #    observed_tp[observed_mask.squeeze()[:, ii].astype(bool)]
            # )
            # Only select the relevant region of attention map
            # i.e., only uptil the max time of observation for this light curve
            attn_map_to_show = attn_map[:, :arg_tmax+1, ii]
            print(f'attn_map_to_show shape, min, max = {attn_map_to_show.shape}, {attn_map_to_show.min()}, {attn_map_to_show.max()}')

#             ax[ii].set(adjustable='box', aspect='equal')

            pc = ax[ii].pcolormesh(
                X, Y, attn_map_to_show,  # attn_map[:, :arg_tmax+1, 0]+attn_map[:, :arg_tmax+1, 1]
                shading='nearest', cmap='OrRd',
                vmin=mesh_vmin, vmax=mesh_vmax
            )
#             pc = ax[ii].scatter(
#                 X, Y, attn_map_to_show
#             )
            obs_to_show_ii = obs_to_show.squeeze()[:, ii]
            obs_to_show_ii = obs_to_show_ii[~np.isnan(obs_to_show_ii)]

            observed_tp_unnormalized_ii = unnormalize_time(
                observed_tp[observed_mask.squeeze()[:, ii].astype(bool)],
                min_time, max_time
            ) / 24
            assert set(observed_tp_unnormalized_ii).issubset(
                set(observed_tp_unnormalized)
            )

            # Show light curve on top
#             ax[ii].plot(
#                 observed_tp_unnormalized_ii,
#                 _normalize_line_plot(
#                     obs_to_show_ii,
#                     a=np.quantile(Y, 0.25), b=np.quantile(Y, 0.75)
#                 ),
#                 '-o', label=r'$m_{\mathrm{obs}}$', markersize=15,
#                 linewidth=2.5, c='#0072B2',
#             )
#             ax[ii].scatter(
#                 observed_tp_unnormalized_ii,
#                 _normalize_line_plot(
#                     obs_to_show_ii,
#                     a=np.quantile(Y, 0.25), b=np.quantile(Y, 0.75)
#                 ),
#                 label=r'$m_{\mathrm{obs}}$', s=15, c='#0072B2',
#             )
            # Show column averaged attention map
#             normalize_attn_map_to_show = _normalize_line_plot(
#                 attn_map_to_show.mean(axis=0),
#                 a=np.quantile(Y, 0.25), b=np.quantile(Y, 0.75)
#             )
# #             from mpl_toolkits.axes_grid1.inset_locator import inset_axes
# #             axins = inset_axes(
# #                 ax[ii], width="100%", height="100%", loc='upper left',
# #                    bbox_to_anchor=(0.5,1-0.3,.3,.3), bbox_transform=ax[ii].transAxes
# #             )
#             ax[ii].plot(
#                 observed_tp_unnormalized,
#                 normalize_attn_map_to_show, '-o',
#                 linewidth=2.5, c='darkgray', alpha=0.9
#             )
#             ax[ii].axhline(
#                 y=normalize_attn_map_to_show.min(),
#                 linestyle='--', c='darkgray', alpha=0.8
#             )

            # Show y=x line
            ax[ii].plot(
                observed_tp_unnormalized_ii, observed_tp_unnormalized_ii,
                linestyle='--', c='gray'
            )

            ymin, ymax = ax[ii].get_ylim()
            ax[ii].vlines(
                observed_tp_unnormalized_ii,
                ymin=ymin, ymax=ymin+0.05 * (ymax - ymin),
                color='black'
            )

            band_label = r' ($g$) ' if ii == 0 else r' ($r$) '
            ax[ii].set_xlabel('Observed time' + band_label + '(days)')#, fontsize=axeslabelsize)
            ax[ii].set_ylabel('Query time (days)')#, fontsize=axeslabelsize)
            ax[ii].tick_params(axis='x')#, labelsize=ticklabelsize)
            ax[ii].tick_params(axis='y')#, labelsize=ticklabelsize)
#             if ii == 0:
#                 ax[ii].text(0.03, 0.9, r'$g$', transform=ax[ii].transAxes)#, fontsize=titlesize)
#             elif ii == 1:
#                 ax[ii].text(0.03, 0.9, r'$r$', transform=ax[ii].transAxes)#, fontsize=titlesize)

#             ax[ii].set_xlim(left=0)
            xlim_maxs.append(observed_tp_unnormalized_ii.max())

            # Plot query points as horizontal lines to give an idea
            # of the no. of query points and spacing between adjacent
            # query points.
#             for y in query_tp_unnormalized:
#                 ax[ii].axhline(y=y, linestyle='dashed', alpha=0.2, c='gray')

#         from mpl_toolkits.axes_grid1 import make_axes_locatable
#         divider = make_axes_locatable(ax[1])
#         cax = divider.append_axes("right", size="5%", pad=0.05)
#         cbar = fig.colorbar(pc, ax=cax)
#         cbar.set_label(
#             'attention weight', rotation=270,
#             labelpad=10,
#         )
#         cbar.ax.tick_params()#labelsize=ticklabelsize)

        test_currObjId_index = np.where(test_objIds == test_currObjId)
        key = list(decoded_lcs_test.keys())[test_currObjId_index[0][0]]
        if test_currObjId in milliquas_agn_labels['objectId'].values:
            test_currCommonFinkclass = milliquas_agn_labels[milliquas_agn_labels['objectId'] == test_currObjId]['custom_type']
            assert len(test_currCommonFinkclass) == 1
            test_currCommonFinkclass = test_currCommonFinkclass.iloc[0]
#             if test_currCommonFinkclass == 'nan':
#                 test_currCommonFinkclass = 'Unknown'
        else:
            test_currObjId_index = np.where(test_objIds == test_currObjId)
            test_currCommonFinkclass = fine_classes[test_currObjId_index[0][0]]
#         ax[0].set_title(objId + f' - {test_currCommonFinkclass}')

        if objId == 'ZTF19abgfgzy' or objId == 'ZTF19aagwxlv':
            ax[1].axvline(x=310, linestyle='dotted')
            ax[1].axvline(x=340, linestyle='dotted')

        save_fig(f'attn_map_{objId}.pdf')
        plt.show()

        axx = plot_decoded_lc(
            decoded_lcs_test, key=key, df_alerts=df_alerts_corrected if corrected_agn_condition else df_alerts, objId=test_currObjId,
            unnormalize_lcs=True, min_time=min_time, max_time=max_time,
            min_max_mags=use_min_max_mags, prob_plot=False, show_nothing=True,
#             whichclass=test_currCommonFinkclass,
            also_show_sigmapsf_and_rmse=False, show_only_rmse=True,
            show_separate_bands=False, fig_width=width*0.9, fig_height=height/3,
            show_legend=False, legend_loc='best', #show_separate_bands_wspace=0.05,
            show_corrected_mags=True if corrected_agn_condition else False,
        )
        if objId == 'ZTF19abgfgzy' or objId == 'ZTF19aagwxlv':
            axx.axvline(x=310, linestyle='dotted')
            axx.axvline(x=340, linestyle='dotted')

        # Match x-axis limits
#         axx.set_xlim(left=0, right=max(xlim_maxs))
        axx.set_title(objId + f' - {test_currCommonFinkclass}')
        save_fig(f'lc_attn_map_{objId}.pdf')

#         # Show decoder attention map
#         fig, ax = plt.subplots()
#         ax.imshow(dattn_map[:arg_tmax+1, :])
#         ax.set_xlabel('No. of decoder hidden dimensions')
#         ax.set_ylabel('Observation location')
#         plt.show()

In [ ]:
# test_currObjId_index = np.where(test_objIds == 'ZTF22aaseqhf')
# key = list(decoded_lcs_test.keys())[test_currObjId_index[0][0]]
# ax = plot_decoded_lc(
#     decoded_lcs_test, key=key, df_alerts=df_alerts, objId='ZTF22aaseqhf',
#     unnormalize_lcs=True, min_time=min_time, max_time=max_time,
#     min_max_mags=min_max_mags, prob_plot=False,
#     whichclass='check',
#     also_show_sigmapsf_and_rmse=False, show_only_rmse=True,
#     show_separate_bands=False, fig_width=width, fig_height=height, show_legend=True,
#     legend_loc='best'
# )

## Interpretation (for OOD data)

In [ ]:
import utils

from torch_intermediate_layer_getter import IntermediateLayerGetter as MidGetter
return_layers = {
    'att': 'att'
#     'hiddens_to_z0': 'hiddens_to_z0',
}
mid_getter = MidGetter(rec, return_layers=return_layers, keep_output=True)
mid_getter2 = MidGetter(dec, return_layers=return_layers, keep_output=True)

from matplotlib.ticker import FormatStrFormatter

def _normalize_line_plot(y, a=0., b=1.):
    norm = (y - y.min()) / (y.max() - y.min())
    scaled = a + norm * (b - a)
    return scaled

train_min_max_times = np.load('train_min_max_times.npy')
min_time, max_time = train_min_max_times[0], train_min_max_times[1]

#################################### NOTE #####################################
# The below code is partly a copy-paste of the code in evaluate_unsupervised.py
# NOTE: The below code for visualization assumes a batch size of 1.
# Otherwise it will give an error or not work as expected.
###############################################################################

train_val_test_min_max_times_filename = 'train_val_test_min_max_times.npy'
train_val_test_min_max_times = np.load(train_val_test_min_max_times_filename)
train_max_time = train_val_test_min_max_times[1]
delta_t = (ref_resolution_days * 24) / train_max_time

objectId_column, fid_column = 'i:objectId', 'i:fid'

with torch.no_grad():
    for c, batch in enumerate(OOD_test_dataloader):
        objId = batch[1][0]
        print(objId)

        key = list(decoded_lcs_test.keys())[int(np.where(OOD_test_objIds == objId)[0])]

        test_batch = batch[0]
        test_batch = test_batch.to(device)
        observed_data, observed_mask, observed_tp = (
            test_batch[:, :, :dim],
            test_batch[:, :, dim: 2 * dim],
            test_batch[:, :, -1],
        )
        if sample_tp and sample_tp < 1:
            subsampled_data, subsampled_tp, subsampled_mask = subsample_timepoints(
                observed_data.clone(), observed_tp.clone(), observed_mask.clone(), sample_tp)
        else:
            subsampled_data, subsampled_tp, subsampled_mask = \
                observed_data, observed_tp, observed_mask

        if sample_tp == 1.:
            assert torch.all(observed_tp == subsampled_tp)

        # Thq query times
        query = utils.generate_query_matrix(subsampled_tp, delta_t, padding_value=-999)

        mid_outputs, model_output = mid_getter(
            torch.cat(
                (subsampled_data, subsampled_mask), 2
            ),
            subsampled_tp, query, return_att=True
        )
        # Shape of mid_outputs['att'][1] for the encoder is (B, h, Tq, Tk, 2*num_filters)
#         print(mid_outputs['att'][0].shape, mid_outputs['att'][1].shape)
#         print(subsampled_data.shape, subsampled_mask.shape, subsampled_tp.shape)
        
        qz0_mean, qz0_logvar = (
            model_output[:, :, : latent_dim],
            model_output[:, :, latent_dim:],
        )
        num_sample = 1
        epsilon = torch.randn(
            num_sample, qz0_mean.shape[0], qz0_mean.shape[1], qz0_mean.shape[2]
        ).to(device)
        z0 = epsilon * torch.exp(0.5 * qz0_logvar) + qz0_mean
        z0 = z0.view(-1, qz0_mean.shape[1], qz0_mean.shape[2])
        batch, seqlen = observed_tp.size()
        time_steps = (
            observed_tp[None, :, :].repeat(num_sample, 1, 1).view(-1, seqlen)
        )
        pred_x = dec(z0, time_steps, query)
        dmid_outputs, dmodel_output = mid_getter2(
            z0, time_steps, query
        )
#         print(dmid_outputs['att'].shape, mid_outputs['att'][1].shape)
        # Average over the batch dimension.
        dattn_map = dmid_outputs['att'].mean(0)
        # After the above operation, dattn_map is a 2d matrix: 347 x 2*gen_n_hidden.
        dattn_map = dattn_map[:, :gen_hidden]

        # Convert to numpy
        subsampled_data, subsampled_tp, subsampled_mask = subsampled_data.numpy(), subsampled_tp.numpy(), subsampled_mask.numpy()
        observed_data, observed_tp, observed_mask = observed_data.numpy(), observed_tp.numpy(), observed_mask.numpy()
        query = query.numpy()

        # The query must be evenly spaced by design
#         assert np.allclose(
#             np.diff(query.squeeze()), np.diff(query.squeeze())[0]
#         )

        df_subset = OOD_df_alerts[OOD_df_alerts[objectId_column] == objId]

        num_filters = 2
        for i in range(num_filters):
            maskFilt = df_subset[fid_column] == (i+1)
            assert len(df_subset[maskFilt]) == observed_mask[:, :, i].sum()

        # Calculate attention map for plotting
        # First, average over the batch and head dimension.
        attn_map = mid_outputs['att'][1].mean(0).mean(0)
        # `attn_map` contains 2*num_filters entries in the last dimension
        # where first `num_filters` dimensions correspond to the mean and
        # the latter `num_filters` dimensions correspond to standard deviation
        attn_map = attn_map[:, :, :num_filters]

        test_currObjId = test_objIds_dataloader[c]
        test_currObjId_index = np.where(total_objIds == test_currObjId)
        test_currCommonFinkclass = total_common_finkclasses[test_currObjId_index].squeeze()
        test_currCommonFinkclass = test_currCommonFinkclass[test_currCommonFinkclass != '0']
        assert len(test_currCommonFinkclass) == 1

        observed_tp[observed_tp == 0.0] = np.nan
        observed_tp[:, 0] = 0.0
        observed_tp = observed_tp.squeeze()
        tmax = np.nanmax(observed_tp)
        arg_tmax = np.nanargmax(observed_tp)
        assert observed_tp[arg_tmax] == tmax
        observed_data[observed_mask == 0] = np.nan
        row = OOD_min_max_mags[np.where(OOD_min_max_mags[:, 0] == objId)].squeeze()
        obs_to_show = unnormalize_mag(
            observed_data, observed_mask, float(row[1]), float(row[2])
        )

        print(observed_tp.shape, obs_to_show.shape, observed_mask.shape, query.shape)

        observed_tp_unnormalized = unnormalize_time(
            observed_tp[observed_tp <= tmax].squeeze(), min_time, max_time
        ) / 24
        query_tp_unnormalized = unnormalize_time(
            query.squeeze(), min_time, max_time
        ) / 24
        assert np.allclose(
            np.diff(query_tp_unnormalized.squeeze()),
            np.diff(query_tp_unnormalized.squeeze())[0],
            rtol=1e-3
        )

        X, Y = np.meshgrid(
            observed_tp_unnormalized,
            query_tp_unnormalized
        )

        fig, ax = plt.subplots(1, num_filters, figsize=(width*2, height/3))
        # NOTE: It is true that the attention map is calculated for each
        # filter separately. It's not calculated for each `latent_dim` separately.
        # So the visualization for g and r bands separately is as expected
        # and not a coincidence due to internally using `latent_dim` which is equal
        # to the no. of filters in our case.
        for ii in range(num_filters):
            # arg_tmax_ii = np.nanargmax(
            #    observed_tp[observed_mask.squeeze()[:, ii].astype(bool)]
            # )
            # Only select the relevant region of attention map
            # i.e., only uptil the max time of observation for this light curve
            attn_map_to_show = attn_map[:, :arg_tmax+1, ii]
            pc = ax[ii].pcolormesh(
                X, Y, attn_map_to_show,
                shading='nearest', cmap='OrRd'
            )
            obs_to_show_ii = obs_to_show.squeeze()[:, ii]
            obs_to_show_ii = obs_to_show_ii[~np.isnan(obs_to_show_ii)]

            observed_tp_unnormalized_ii = unnormalize_time(
                observed_tp[observed_mask.squeeze()[:, ii].astype(bool)],
                min_time, max_time
            ) / 24
            assert set(observed_tp_unnormalized_ii).issubset(
                set(observed_tp_unnormalized)
            )

            # Show light curve on top
#             ax[ii].plot(
#                 observed_tp_unnormalized_ii,
#                 _normalize_line_plot(
#                     obs_to_show_ii,
#                     a=np.quantile(Y, 0.25), b=np.quantile(Y, 0.75)
#                 ),
#                 '-o', label=r'$m_{\mathrm{obs}}$', markersize=15,
#                 linewidth=2.5, c='#0072B2',
#             )
            # Show column averaged attention map
            normalize_attn_map_to_show = _normalize_line_plot(
                    attn_map_to_show.mean(axis=0),
                    a=np.quantile(Y, 0.25), b=np.quantile(Y, 0.75)
                )
            ax[ii].plot(
                observed_tp_unnormalized,
                normalize_attn_map_to_show, '-o',
                linewidth=2.5, c='darkgray', alpha=0.9
            )
            ax[ii].axhline(
                y=normalize_attn_map_to_show.min(),
                linestyle='--', c='darkgray', alpha=0.8
            )

            ax[ii].set_xlabel('Observed time (days)')#, fontsize=axeslabelsize)
            ax[ii].set_ylabel('Query time (days)')#, fontsize=axeslabelsize)
            ax[ii].tick_params(axis='x')#, labelsize=ticklabelsize)
            ax[ii].tick_params(axis='y')#, labelsize=ticklabelsize)
            if ii == 0:
                ax[ii].set_title(r'$g$')#, fontsize=titlesize)
            elif ii == 1:
                ax[ii].set_title(r'$r$')#, fontsize=titlesize)

            # Plot query points as horizontal lines to give an idea
            # of the no. of query points and spacing between adjacent
            # query points.
#             for y in query_tp_unnormalized:
#                 ax[ii].axhline(y=y, linestyle='dashed', alpha=0.2, c='gray')

        cbar = fig.colorbar(pc, ax=ax.ravel().tolist())
        cbar.set_label(
            'attention score', rotation=270,
            labelpad=5,# fontsize=axeslabelsize
        )
        cbar.ax.tick_params()#labelsize=ticklabelsize)

        plt.show()

#         # Show decoder attention map
#         fig, ax = plt.subplots()
#         ax.imshow(dattn_map[:arg_tmax+1, :])
#         ax.set_xlabel('No. of decoder hidden dimensions')
#         ax.set_ylabel('Observation location')
#         plt.show()

        if c == 7:
            break

## GP regression

In [ ]:
# from functools import partial
# import numpy as np
# import jax
# import jax.numpy as jnp
# import matplotlib.pyplot as plt

# import tinygp
# from tinygp import GaussianProcess, kernels

# jax.config.update("jax_enable_x64", True)

# # --- multiband wrapper (very small) ---
# @tinygp.helpers.dataclass
# class Multiband(tinygp.kernels.quasisep.Wrapper):
#     amplitudes: jnp.ndarray  # shape (n_bands,)

#     def coord_to_sortable(self, X):
#         # X is a pytree like (t_array, band_array) and we sort by time
#         return X[0]

#     def observation_model(self, X):
#         # X[1] are integer band ids (0..n_bands-1)
#         return self.amplitudes[X[1]] * self.kernel.observation_model(X[0])

# # custom data
# def stack_multiband_data(data):
#     t, band, y, yerr = [], [], [], []
#     for b, (tb, yb, yerrb) in data.items():
#         t.append(tb)
#         band.append(np.full_like(tb, b, dtype=int))
#         y.append(yb)
#         yerr.append(yerrb)
#     return (
#         jnp.array(np.concatenate(t)),
#         jnp.array(np.concatenate(band), dtype=jnp.int32),
#         jnp.array(np.concatenate(y)),
#         jnp.array(np.concatenate(yerr)),
#     )

# ### Data here ###
# ID = 'ZTF18abtrubp'
# df_id = df_alerts_pandas[df_alerts_pandas['objectId'] == ID]
# minjd = df_id['jd'].min()
# t_band0 = df_id[df_id['fid'] == 1]['jd']-minjd
# t_band1 = df_id[df_id['fid'] == 2]['jd']-minjd
# mag_band0 = df_id[df_id['fid'] == 1]['magpsf']
# mag_band1 = df_id[df_id['fid'] == 2]['magpsf']
# magerr_band0 = np.zeros_like(mag_band0)
# magerr_band1 = np.zeros_like(mag_band1)
# #################

# data = {
#     0: (t_band0, mag_band0, magerr_band0),
#     1: (t_band1, mag_band1, magerr_band1),
# }

# t, band_id, y_obs, yerr = stack_multiband_data(data)
# order = jnp.argsort(t)
# t, band_id, y_obs, yerr = t[order], band_id[order], y_obs[order], yerr[order]
# X = (t, band_id)
# #print(X)

# # --- kernel: latent Matern52 in time, per-band amplitudes ---
# amplitudes = jnp.array([1.0, 1.0])              # scale each band's latent process
# base = kernels.quasisep.Matern52(scale=1.5)     # 1D time kernel
# kernel = Multiband(kernel=base, amplitudes=amplitudes)

# # --- build GP (pass observational variances via diag) ---
# gp = GaussianProcess(kernel, X, diag=(yerr ** 2)+1e-10)

# # --- generate a toy observed y from the GP prior + extra noise (for demo) ---
# key = jax.random.PRNGKey(0)
# y_true = gp.sample(key)                          # prior sample at X (joint across bands)
# # add independent measurement noise (consistent with yerr)
# key2 = jax.random.PRNGKey(1)
# y_obs = y_true + yerr * jax.random.normal(key2, shape=(len(t),))

# # --- prediction grid ---
# t_pred = np.arange(t.min() - 0.5, t.max() + 0.5, 2)  # 2 days
# t_pred_j = jnp.array(np.concatenate([t_pred, t_pred]))
# band_pred = jnp.concatenate([np.zeros_like(t_pred, dtype=int), np.ones_like(t_pred, dtype=int)])
# X_pred = (t_pred_j, jnp.array(band_pred, dtype=jnp.int32))

# # --- condition and get posterior mean/std at X_pred ---
# _, cond_gp = gp.condition(y_obs, X_test=X_pred)   # returns (logp, conditioned GP at X_pred)
# mu = cond_gp.mean                                  # shape (len(X_pred),)
# std = jnp.sqrt(cond_gp.variance)

# # --- plot (offset bands vertically for clarity) ---
# plt.figure(figsize=(9,4))
# for b in [0, 1]:
#     mask_obs = np.array(band_id == b)
#     mask_pred = np.array(band_pred == b)
#     offset = -2.5 * b
#     plt.errorbar(t[mask_obs], np.array(y_obs)[mask_obs] + offset, yerr=np.array(yerr)[mask_obs],
#                  fmt=".", ms=10, label=f"band {b+1} obs")
#     plt.plot(t_pred, np.array(mu)[mask_pred] + offset, label=f"band {b+1} mean")
#     plt.fill_between(t_pred,
#                      np.array(mu)[mask_pred] - 2*np.array(std)[mask_pred] + offset,
#                      np.array(mu)[mask_pred] + 2*np.array(std)[mask_pred] + offset,
#                      alpha=0.25)
# plt.xlabel("time (days)")
# plt.ylabel("magnitude (offset per band)")
# plt.legend(fontsize=9)
# plt.tight_layout()
# plt.show()

In [ ]:
# import time
# import numpy as np
# import matplotlib.pyplot as plt
# from sklearn.gaussian_process import GaussianProcessRegressor
# from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel

# def run_gp(ID, plot=True):
#     df_id = df_alerts_pandas[df_alerts_pandas['objectId'] == ID]
#     minjd = df_id['jd'].min()
#     t_band0 = (df_id[df_id['fid'] == 1]['jd']-minjd).to_numpy()
#     t_band1 = (df_id[df_id['fid'] == 2]['jd']-minjd).to_numpy()
#     mag_band0 = (df_id[df_id['fid'] == 1]['magpsf']).to_numpy()
#     mag_band1 = (df_id[df_id['fid'] == 2]['magpsf']).to_numpy()
#     magerr_band0 = np.zeros_like(mag_band0)
#     magerr_band1 = np.zeros_like(mag_band1)

#     # Prediction grid
#     min_t, max_t = min(t_band0.min(), t_band1.min()), max(t_band0.max(), t_band1.max())
#     # t_pred = np.arange(min_t - 0.5, max_t + 0.5, 2)
#     t_pred = np.sort(np.concatenate((t_band0, t_band1)))

#     # Independent GP per band
#     mu_pred, std_pred, exec_time = [], [], []

#     kernel = ConstantKernel() + Matern(length_scale=2, nu=3/2) + WhiteKernel(noise_level=1)

#     for t_b, y_b, yerr_b in [(t_band0, mag_band0, magerr_band0),
#                              (t_band1, mag_band1, magerr_band1)]:

#         X = t_b.reshape(-1,1)
#     #     kernel = 1.0*Matern(length_scale=1.5, nu=2.5) + WhiteKernel(noise_level=0.01)
        
#         # Suppressing the convergence warning as per https://news.ycombinator.com/item?id=26294156
#         gp = GaussianProcessRegressor(kernel=kernel, alpha=yerr_b**2, normalize_y=True)
#         start_time = time.time()
#         gp.fit(X, y_b)
#         mu, std = gp.predict(t_pred.reshape(-1,1), return_std=True)
#         end_time = time.time()
#         wall_clock_time = end_time - start_time
#         exec_time.append(wall_clock_time)
#         mu_pred.append(mu)
#         std_pred.append(std)

#     if plot:
#         fig = plt.figure(figsize=(10,4))
#         ax = plt.gca()
#         for i, (t_b, y_b) in enumerate([(t_band0, mag_band0),(t_band1, mag_band1)]):
#             offset = -2.5*i
#             ax.errorbar(t_b, y_b+offset, yerr=(magerr_band0 if i==0 else magerr_band1),
#                          fmt='.', ms=5, label=f'band {i+1} obs')
#             ax.plot(t_pred, mu_pred[i]+offset, label=f'band {i+1} mean')
#             ax.fill_between(t_pred, mu_pred[i]-2*std_pred[i]+offset, mu_pred[i]+2*std_pred[i]+offset, alpha=0.25)

#         ax.invert_yaxis()
#         ax.set_xlabel("time (days)")
#         ax.set_ylabel("magnitude (offset per band)")
#         ax.legend(fontsize=9)
#         plt.tight_layout()
#         plt.show()

#     return mu_pred, std_pred, np.sum(exec_time)

# IDs = ['ZTF19aagwxlv', 'ZTF18abtrubp', 'ZTF21abidrcn']
# for ID in IDs:
#     _, _, exec_time = run_gp(ID, plot=True)
#     print(ID, exec_time)

In [ ]:
from functools import partial
from typing import Dict, List, Union

import george
import numpy as np
import pandas as pd
import scipy.optimize as op
from astropy.table import Table, vstack

import time

matplotlib.rcParams["font.size"] = "16"

# From https://learn.astropy.org/tutorials/1_models-quick-fit.html
def calc_reduced_chi_square(fit, x, y, yerr, N, n_free=0.):
    """
    fit (array) values for the fit
    x,y,yerr (arrays) data
    N total number of points
    n_free number of parameters we are fitting
    """
    return 1.0 / (N - n_free) * sum(((fit - y) / yerr) ** 2)

def calc_rmse(fit, obs):
    rmse = np.sqrt(
        mean_squared_error(obs, fit, np.ones_like(obs))
    )
    return rmse

pb_wavelengths = {
    1: 4802.0,
    2: 6231.0,
}
def fit_2d_gp(
    obj_data: pd.DataFrame,
    return_kernel: bool = False,
    pb_wavelengths: Dict = pb_wavelengths,
    magpsf_col='magpsf', sigmapsf_col='sigmapsf',
    **kwargs,
):
    """Fit a 2D Gaussian process.

    If required, predict the GP at evenly spaced points along a light curve.

    Parameters
    ----------
    obj_data : pd.DataFrame
        Time, flux and flux error of the data (specific filter of an object).
    return_kernel : bool, default = False
        Whether to return the used kernel.
    pb_wavelengths: dict
        Mapping of the passband wavelengths for each filter used.
    kwargs : dict
        Additional keyword arguments that are ignored at the moment. We allow
        additional keyword arguments so that the various functions that
        call this one can be called with the same arguments.

    Returns
    -------
    kernel: george.gp.GP.kernel, optional
        The kernel used to fit the GP.
    gp_predict : functools.partial of george.gp.GP
        The GP instance that was used to fit the object.

    Examples
    --------
    >>> gp_wavelengths = np.vectorize(pb_wavelengths.get)(filters)
    >>> inverse_pb_wavelengths = {v: k for k, v in pb_wavelengths.items()}
    >>> gp_predict = fit_2d_gp(df, pb_wavelengths=pb_wavelengths)
    ...
    """
    guess_length_scale = 20.0  # a parameter of the Matern32Kernel

    obj_times = obj_data.jd.astype(float)
    obj_flux = obj_data[magpsf_col].astype(float)
    obj_flux_error = obj_data[sigmapsf_col].astype(float)
    obj_wavelengths = obj_data["fid"].map(pb_wavelengths)

    def neg_log_like(p):  # Objective function: negative log-likelihood
        gp.set_parameter_vector(p)
        loglike = gp.log_likelihood(obj_flux, quiet=True)
        return -loglike if np.isfinite(loglike) else 1e25

    def grad_neg_log_like(p):  # Gradient of the objective function.
        gp.set_parameter_vector(p)
        return -gp.grad_log_likelihood(obj_flux, quiet=True)

    # Use the highest signal-to-noise observation to estimate the scale. We
    # include an error floor so that in the case of very high
    # signal-to-noise observations we pick the maximum flux value.
    signal_to_noises = np.abs(obj_flux) / np.sqrt(
        obj_flux_error**2 + (1e-2 * np.max(obj_flux)) ** 2
    )
    scale = np.abs(obj_flux[signal_to_noises.idxmax()])

    kernel = (0.5 * scale) ** 2 * george.kernels.Matern32Kernel(
        [guess_length_scale**2, 6000**2], ndim=2
    )
    kernel.freeze_parameter("k2:metric:log_M_1_1")

    gp = george.GP(kernel)
    default_gp_param = gp.get_parameter_vector()
    x_data = np.vstack([obj_times, obj_wavelengths]).T
    gp.compute(x_data, obj_flux_error)

    bounds = [(0, np.log(1000**2))]
    bounds = [(default_gp_param[0] - 10, default_gp_param[0] + 10)] + bounds
    results = op.minimize(
        neg_log_like,
        gp.get_parameter_vector(),
        jac=grad_neg_log_like,
        method="L-BFGS-B",
        bounds=bounds,
        tol=1e-6,
    )

    if results.success:
        gp.set_parameter_vector(results.x)
    else:
        # Fit failed. Print out a warning, and use the initial guesses for fit
        # parameters.
        obj = obj_data["object_id"][0]
        print("GP fit failed for {}! Using guessed GP parameters.".format(obj))
        gp.set_parameter_vector(default_gp_param)

    gp_predict = partial(gp.predict, obj_flux)

    if return_kernel:
        return kernel, gp_predict
    return gp_predict


def predict_2d_gp(gp_predict, gp_times, gp_wavelengths, use_corr=False):
    """Outputs the predictions of a Gaussian Process.

    Parameters
    ----------
    gp_predict : functools.partial of george.gp.GP
        The GP instance that was used to fit the object.
    gp_times : numpy.ndarray
        Times to evaluate the Gaussian Process at.
    gp_wavelengths : numpy.ndarray
        Wavelengths to evaluate the Gaussian Process at.

    Returns
    -------
    obj_gps : pandas.core.frame.DataFrame, optional
        Time, flux and flux error of the fitted Gaussian Process.

    Examples
    --------
    >>> gp_predict = fit_2d_gp(df, pb_wavelengths=pb_wavelengths)
    >>> number_gp = timesteps
    >>> gp_times = np.linspace(min(df["mjd"]), max(df["mjd"]), number_gp)
    >>> obj_gps = predict_2d_gp(gp_predict, gp_times, gp_wavelengths)
    >>> obj_gps["filter"] = obj_gps["filter"].map(inverse_pb_wavelengths)
    ...
    """
    unique_wavelengths = np.unique(gp_wavelengths)
    number_gp = len(gp_times)
    obj_gps = []
    for wavelength in unique_wavelengths:
        gp_wavelengths = np.ones(number_gp) * wavelength
        pred_x_data = np.vstack([gp_times, gp_wavelengths]).T
        pb_pred, pb_pred_var = gp_predict(pred_x_data, return_var=True)
        # stack the GP results in a array momentarily
        obj_gp_pb_array = np.column_stack((gp_times, pb_pred, np.sqrt(pb_pred_var)))
        _cols = ["jd", "magpsf_corr", "sigmapsf_corr", "fid"] if use_corr else ["jd", "magpsf", "sigmapsf", "fid"]
        obj_gp_pb = Table(
            [
                obj_gp_pb_array[:, 0],
                obj_gp_pb_array[:, 1],
                obj_gp_pb_array[:, 2],
                [wavelength] * number_gp,
            ],
            names=_cols,
        )
        if len(obj_gps) == 0:  # initialize the table for 1st passband
            obj_gps = obj_gp_pb
        else:  # add more entries to the table
            obj_gps = vstack((obj_gps, obj_gp_pb))

    obj_gps = obj_gps.to_pandas()
    return obj_gps

objIds_to_test = ['ZTF19aagwxlv', 'ZTF18abtrubp', 'ZTF22abahblc']#, 'ZTF22aafxzfb', 'ZTF22aafumyr', 'ZTF20acbcfaa']

df_alerts = dd.read_parquet(
    'alerts_processed_ftransfer_ztf_2025_05_31_430518.parquet',
    columns=[
            'objectId', 'finkclass', 'tnsclass',
             'fid', 'magpsf', 'sigmapsf', 'jd'
        ]
)
df_alerts_corrected = pd.read_parquet(
    'alerts_processed_ftransfer_ztf_2025_05_31_430518_CORRECTED.parquet',
    columns=[
        'objectId', 'finkclass', 'tnsclass',
         'fid', 'magpsf_corr', 'sigmapsf_corr', 'jd', 'used_magpsf_corr',
    ]
)
df_alerts_pandas = df_alerts.compute()
total_time = []

train_min_max_times = np.load('train_min_max_times.npy')
min_time, max_time = train_min_max_times[0], train_min_max_times[1]
min_max_mags = np.load('min_max_mags.npy')

test_objIds_dataloader = np.load('evaluate_test_objIds_dataloader.npy')
# test_objIds = np.load('evaluate_test_objIds_dataloader.npy')
###
decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
if correct_lcs_for_agns:
    decoded_lcs_test_corrected = np.load('evaluate_test_CORRECTED_decoded_lcs.npz')

    assert list(decoded_lcs_test.keys()) == list(decoded_lcs_test_corrected.keys()), "NPZ keys mismatch"

    # Build corrected_agn: start from original, replace AGN rows with corrected
    decoded_lcs_test_corrected_agn = {}
    for i, key in enumerate(decoded_lcs_test.keys()):
        if agn_indicator[i]:
            decoded_lcs_test_corrected_agn[key] = decoded_lcs_test_corrected[key].copy()
            # Verify AGN lc came from corrected
            np.testing.assert_array_equal(
                decoded_lcs_test_corrected_agn[key],
                decoded_lcs_test_corrected[key]
            )
        else:
            decoded_lcs_test_corrected_agn[key] = decoded_lcs_test[key].copy()
            # Verify non-AGN lc is unchanged
            np.testing.assert_array_equal(
                decoded_lcs_test_corrected_agn[key],
                decoded_lcs_test[key]
            )

    print(f"Replaced {agn_indicator.sum()} AGN rows out of {len(agn_indicator)} total")

dlte = list(decoded_lcs_test.keys())
print(decoded_lcs_test.f.arr_2.shape)

use_corrected_agns = True
if use_corrected_agns:
    print('Using corrected AGN light curves...')
    assert np.all(
        np.load('evaluate_test_objIds_dataloader.npy') == test_objIds_corrected_agn
    )
    test_objIds = test_objIds_corrected_agn
    decoded_lcs_test = decoded_lcs_test_corrected_agn
    min_max_mags = np.load('min_max_mags.npy')
    min_max_mags_corrected = np.load('min_max_mags_ftransfer_ztf_2025_05_31_430518_CORRECTED.npy')
else:
    print('Using uncorrected AGN light curves...')
    test_objIds  = np.load('evaluate_test_objIds_dataloader.npy')
    decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
    min_max_mags = np.load('min_max_mags.npy')
###
assert np.all(test_objIds_dataloader == np.load('evaluate_test_objIds_dataloader.npy'))

# If True, evaluates at evenly spaced by 2 days, else only observed time points.
uniform_grid_gp_eval = False
for test_currObjId in objIds_to_test:
    if test_currObjId in milliquas_agn_labels['objectId'].values:
        test_currCommonFinkclass = milliquas_agn_labels[milliquas_agn_labels['objectId'] == test_currObjId]['custom_type']
        assert len(test_currCommonFinkclass) == 1
        test_currCommonFinkclass = test_currCommonFinkclass.iloc[0]
#         if test_currCommonFinkclass == 'nan':
#             test_currCommonFinkclass = 'Unknown'
    else:
        test_currObjId_index = np.where(test_objIds == test_currObjId)
        test_currCommonFinkclass = fine_classes[test_currObjId_index[0][0]]

    corrected_agn_condition = (
        (test_currCommonFinkclass in milliquas_agn_labels['custom_type'].unique())
        and (use_corrected_agns)
        and df_alerts_corrected.loc[df_alerts_corrected['objectId'] == test_currObjId, 'used_magpsf_corr'].all()
    )
    uncorrected_agn_condition = (
        (test_currCommonFinkclass in milliquas_agn_labels['custom_type'].unique())
        and (use_corrected_agns)
        and not df_alerts_corrected.loc[df_alerts_corrected['objectId'] == test_currObjId, 'used_magpsf_corr'].any()
    )
    print(test_currObjId, key, corrected_agn_condition, uncorrected_agn_condition)

    magpsf_col = 'magpsf_corr' if corrected_agn_condition else 'magpsf'
    sigmapsf_col = 'sigmapsf_corr' if corrected_agn_condition else 'sigmapsf'
    print(magpsf_col, sigmapsf_col)

#     if uncorrected_agn_condition:
#         continue
    if corrected_agn_condition:
        df_alerts_to_use = df_alerts_corrected
    else:
        df_alerts_to_use = df_alerts_pandas

    df = df_alerts_to_use[df_alerts_to_use['objectId'] == test_currObjId]

    start = time.time()
    gp_predict = fit_2d_gp(df, pb_wavelengths=pb_wavelengths, magpsf_col=magpsf_col, sigmapsf_col=sigmapsf_col)
    end = time.time()

    if uniform_grid_gp_eval:
        gp_times = np.arange(min(df["jd"]), max(df["jd"]), 2)  # 2 days
    else:
        gp_times = df["jd"]

    filters = df["fid"]
    filters = list(np.unique(filters))
    gp_wavelengths = np.vectorize(pb_wavelengths.get)(filters)
    obj_gps = predict_2d_gp(
        gp_predict, gp_times, gp_wavelengths, use_corr=corrected_agn_condition
    )

    #end = time.time()

    inverse_pb_wavelengths = {v: k for k, v in pb_wavelengths.items()}
    obj_gps["fid"] = obj_gps["fid"].map(inverse_pb_wavelengths)
    #for _ in range(5):

    #lc_times.append(t)
    #exec_time = np.median(lc_times)
    print(end-start)
    total_time.append(end - start)

    # Plot test decoded lc
    indx = np.where(test_objIds_dataloader == test_currObjId)[0][0]
    key = list(decoded_lcs_test.keys())[indx]

    # Calculate reduced chi-sq
    pred_x, observed_data, observed_mask, observed_times = decoded_lcs_test[key]

    # 1. PREPARE GROUND TRUTH
    # Recall: the actual dataframe values correspond to the observations
    _y = df[magpsf_col].values
    _yerr = df[sigmapsf_col].values
    N = len(_y)

    if not uniform_grid_gp_eval:
        assert len(gp_times) == N

    # 2. GP PREDICTION AT OBSERVED POINTS
    # obj_gps has 62 rows (31 for fid 1, 31 for fid 2). 
    # We merge with the original df to pick the correct 31 rows.
    alignment_df = pd.merge(
        df[['jd', 'fid']],
        obj_gps[['jd', 'fid', magpsf_col]].drop_duplicates(subset=['jd', 'fid']),
        on=['jd', 'fid'],
        how='left'
    )
#     print("=====")
#     print(df[sigmapsf_col])
#     print("=====")

    _fit_gp = alignment_df[magpsf_col].values
#     _fit_gp = obj_gps['magpsf']

    # --- 3. MTAN PREDICTION AT OBSERVED POINTS ---
    # Convert mask to boolean for indexing
    mask_bool = observed_mask.astype(bool)

    # Select only the predicted values (pred_x) where a real observation exists (mask is True)
    # This flattens the array to match the 1D structure of _y
    # Pass a mask of all ones for prediction because for
    # interpolation all points can be considered observed.
    use_min_max_mags = min_max_mags_corrected if corrected_agn_condition else min_max_mags
    row = use_min_max_mags[np.where(use_min_max_mags[:,0] == test_currObjId)].squeeze()
    pred_x = unnormalize_mag(
        pred_x, np.ones_like(observed_mask), float(row[1]), float(row[2])
    )
    _fit_mtan = pred_x[mask_bool]

    # Optional: Verify shapes align
    print(len(_fit_gp), len(_fit_mtan), N, observed_mask.sum())
    if len(alignment_df) == len(alignment_df.drop_duplicates(['jd', 'fid'])):
        assert len(_fit_gp) == N
        assert len(_fit_mtan) == N

    reduced_chisq_gp = calc_reduced_chi_square(
        _fit_gp, df['jd'], _y, _yerr, N, n_free=0.
    )
    reduced_chisq_mtan = calc_reduced_chi_square(
        _fit_mtan, df['jd'], _y, _yerr, N, n_free=0.
    )

    rmse_gp = calc_rmse(_fit_gp, _y)
    rmse_mtan = calc_rmse(_fit_mtan, _y)

    print(reduced_chisq_gp, reduced_chisq_mtan, rmse_gp, rmse_mtan)

    ax = plot_decoded_lc(
        decoded_lcs_test, key=key, df_alerts=df_alerts_corrected if corrected_agn_condition else df_alerts,
        objId=test_currObjId,
        unnormalize_lcs=True, min_time=min_time, max_time=max_time,
        min_max_mags=use_min_max_mags, prob_plot=False,
        whichclass=test_currCommonFinkclass,
        also_show_sigmapsf_and_rmse=True, show_separate_bands=False,
        show_only_rmse=True,
        fig_width=1.5*width*2/3, fig_height=1.5*height/3,
        show_legend=False, legend_loc='best', show_nothing=True,
        show_corrected_mags=True if corrected_agn_condition else False,
    )
    ax.tick_params(axis='both', which='minor', length=3)
    ax.tick_params(axis='both', which='major', length=6)

    # Plot GP
    minjd = df['jd'].min()
#     ax.scatter(df[df['fid'] == 1]['jd']-minjd, df[df['fid'] == 1]['magpsf'])

#     if not uniform_grid_gp_eval:
    # Colors mapping to match your existing style
    colors = {1: bluish_color, 2: orangish_color}
    for fid, group in obj_gps.groupby('fid'):
        group = group.sort_values(by='jd')
        ax.plot(group['jd'] - minjd, group[magpsf_col], linestyle='--', c=colors.get(fid, 'black'))
#     else:
#         ax.plot(gp_times-minjd, obj_gps[obj_gps['fid'] == 1]['magpsf'], linestyle='--', c=bluish_color)
#     #     ax.scatter(df[df['fid'] == 2]['jd']-minjd, df[df['fid'] == 2]['magpsf'])
#         ax.plot(gp_times-minjd, obj_gps[obj_gps['fid'] == 2]['magpsf'], linestyle='--', c=orangish_color)

#     ax.set_ylim([obj_gps['magpsf'].min(), obj_gps['magpsf'].max()])
#     ax.invert_yaxis()

    ax.text(
        0.06, 0.9, rf'$\mathrm{{RMSE}}_{{\mathrm{{GP}}}} = {rmse_gp:.2f}$',
        transform=ax.transAxes,
    )
    ax.text(
        0.06, 0.8, rf'$\mathrm{{RMSE}}_{{\mathrm{{mTAND}}}} = {rmse_mtan:.2f}$',
        transform=ax.transAxes,
    )
#     plt.show()

    save_fig(f'mtan_vs_gp_{test_currObjId}.pdf')

print(np.sum(total_time))

In [ ]:
rmse_gps, rmse_mtans = [], []
fail_counter = 0

for test_currObjId in test_objIds:
    if test_currObjId in milliquas_agn_labels['objectId'].values:
        test_currCommonFinkclass = milliquas_agn_labels[milliquas_agn_labels['objectId'] == test_currObjId]['custom_type']
        assert len(test_currCommonFinkclass) == 1
        test_currCommonFinkclass = test_currCommonFinkclass.iloc[0]
#         if test_currCommonFinkclass == 'nan':
#             test_currCommonFinkclass = 'Unknown'
    else:
        test_currObjId_index = np.where(test_objIds == test_currObjId)
        test_currCommonFinkclass = fine_classes[test_currObjId_index[0][0]]

    corrected_agn_condition = (
        (test_currCommonFinkclass in milliquas_agn_labels['custom_type'].unique())
        and (use_corrected_agns)
        and df_alerts_corrected.loc[df_alerts_corrected['objectId'] == test_currObjId, 'used_magpsf_corr'].all()
    )
    uncorrected_agn_condition = (
        (test_currCommonFinkclass in milliquas_agn_labels['custom_type'].unique())
        and (use_corrected_agns)
        and not df_alerts_corrected.loc[df_alerts_corrected['objectId'] == test_currObjId, 'used_magpsf_corr'].any()
    )

    magpsf_col = 'magpsf_corr' if corrected_agn_condition else 'magpsf'
    sigmapsf_col = 'sigmapsf_corr' if corrected_agn_condition else 'sigmapsf'

#     if uncorrected_agn_condition:
#         continue
    if corrected_agn_condition:
        df_alerts_to_use = df_alerts_corrected
    else:
        df_alerts_to_use = df_alerts_pandas

    df = df_alerts_to_use[df_alerts_to_use['objectId'] == test_currObjId]

    start = time.time()
    gp_predict = fit_2d_gp(df, pb_wavelengths=pb_wavelengths, magpsf_col=magpsf_col, sigmapsf_col=sigmapsf_col)
    end = time.time()

    if uniform_grid_gp_eval:
        gp_times = np.arange(min(df["jd"]), max(df["jd"]), 2)  # 2 days
    else:
        gp_times = df["jd"]

    filters = df["fid"]
    filters = list(np.unique(filters))
    gp_wavelengths = np.vectorize(pb_wavelengths.get)(filters)
    
    
    obj_gps = predict_2d_gp(
        gp_predict, gp_times, gp_wavelengths, use_corr=corrected_agn_condition
    )

    #end = time.time()

    inverse_pb_wavelengths = {v: k for k, v in pb_wavelengths.items()}
    obj_gps["fid"] = obj_gps["fid"].map(inverse_pb_wavelengths)
    #for _ in range(5):

    #lc_times.append(t)
    #exec_time = np.median(lc_times)
    total_time.append(end - start)

    # Plot test decoded lc
    indx = np.where(test_objIds_dataloader == test_currObjId)[0][0]
    key = list(decoded_lcs_test.keys())[indx]

    # Calculate reduced chi-sq
    pred_x, observed_data, observed_mask, observed_times = decoded_lcs_test[key]

    # 1. PREPARE GROUND TRUTH
    # Recall: the actual dataframe values correspond to the observations
    _y = df[magpsf_col].values
    _yerr = df[sigmapsf_col].values
    N = len(_y)

    if not uniform_grid_gp_eval:
        assert len(gp_times) == N

    # 2. GP PREDICTION AT OBSERVED POINTS
    # obj_gps has 62 rows (31 for fid 1, 31 for fid 2). 
    # We merge with the original df to pick the correct 31 rows.
    alignment_df = pd.merge(
        df[['jd', 'fid']],
        obj_gps[['jd', 'fid', magpsf_col]].drop_duplicates(subset=['jd', 'fid']),
        on=['jd', 'fid'],
        how='left',
    )
#     print("=====")
#     print(df[sigmapsf_col])
#     print("=====")

    _fit_gp = alignment_df[magpsf_col].values
#     _fit_gp = obj_gps['magpsf']

    # --- 3. MTAN PREDICTION AT OBSERVED POINTS ---
    # Convert mask to boolean for indexing
    mask_bool = observed_mask.astype(bool)

    # Select only the predicted values (pred_x) where a real observation exists (mask is True)
    # This flattens the array to match the 1D structure of _y
    # Pass a mask of all ones for prediction because for
    # interpolation all points can be considered observed.
    use_min_max_mags = min_max_mags_corrected if corrected_agn_condition else min_max_mags
    row = use_min_max_mags[np.where(use_min_max_mags[:,0] == test_currObjId)].squeeze()
    pred_x = unnormalize_mag(
        pred_x, np.ones_like(observed_mask), float(row[1]), float(row[2])
    )
    _fit_mtan = pred_x[mask_bool]

    # Optional: Verify shapes align
#     print(len(_fit_gp), len(_fit_mtan), N, observed_mask.sum())
    if len(alignment_df) == len(alignment_df.drop_duplicates(['jd', 'fid'])):
        assert len(_fit_gp) == N
        assert len(_fit_mtan) == N

#     reduced_chisq_gp = calc_reduced_chi_square(
#         _fit_gp, df['jd'], _y, _yerr, N, n_free=0.
#     )
#     reduced_chisq_mtan = calc_reduced_chi_square(
#         _fit_mtan, df['jd'], _y, _yerr, N, n_free=0.
#     )

    try:
        rmse_gp = calc_rmse(_fit_gp, _y)
        rmse_mtan = calc_rmse(_fit_mtan, _y)
    except ValueError:  # NOTE: Hacky fix for a minor no. of cases hopefully.
        fail_counter += 1
        continue

    rmse_gps.append(rmse_gp)
    rmse_mtans.append(rmse_mtan)

print(f'{fail_counter} light curves had failed RMSE calculation because of repeated values for (jd, fid) causing shape mismatch..')

In [ ]:
matplotlib.rcParams["font.size"] = "14"
fig = plt.figure(figsize=(1.5*width/2, height/2))
ax = plt.gca()
ax.hist(rmse_gps, bins=10, label='GP', color=orangish_color, alpha=0.5)
ax.hist(rmse_mtans, bins=10, label='mTAND', color=bluish_color, alpha=0.5)
ax.legend()
ax.set_xlabel('RMSE')
ax.set_ylabel('Counts')
ax.set_yscale('log')

In [ ]:
np.round(np.median(rmse_gps), 2), np.round(np.median(rmse_mtans), 2), \
np.round(scipy.stats.median_abs_deviation(rmse_gps), 2), np.round(scipy.stats.median_abs_deviation(rmse_mtans), 2)

In [ ]:
mtan_exec_times = np.load('mtan_exec_time_vs_nobs_1.npy').T
gp_exec_times = np.load('gp_exec_time_vs_nobs.npy')
mtan_exec_times.shape, gp_exec_times.shape

In [ ]:
from operator import itemgetter
mtan_exec_times = sorted(mtan_exec_times, key=itemgetter(0))
gp_exec_times = sorted(gp_exec_times, key=itemgetter(0))
assert np.all([m[0] for m in mtan_exec_times] == [g[0] for g in gp_exec_times])

In [ ]:
import numpy as np
np.seterr(all='warn') # default behavior (warnings, not errors)

matplotlib.rcParams["font.size"] = "14"
fig = plt.figure(figsize=(1.5*width/2, height/2))
ax = plt.gca()

X_mtan = np.array([int(float(m[2])) for m in mtan_exec_times])
Y_mtan = np.array([float(m[1]) for m in mtan_exec_times])
X_gp = np.array([int(m[2]) for m in gp_exec_times])
Y_gp = np.array([float(m[1]) for m in gp_exec_times])
assert np.all(X_mtan == X_gp)
ax.scatter(
    X_mtan, Y_mtan,
    color=bluish_color, alpha=0.1, label='mTAND encoder (GPU)',
    s=20, edgecolors='none'
)
ax.scatter(
    X_gp, Y_gp,
    color=orangish_color, alpha=0.1, marker='^', label='GP fit',
    s=20, edgecolors='none'
)
ax.set_xlabel(r'$2\langle L \rangle$')
ax.set_ylabel('Execution time (sec)')
ax.grid(True)#, which='both', alpha=0.3)
leg = ax.legend(frameon=False)
set_axes_style(ax)
ax.set_xscale('log')

for lh in leg.legend_handles:
    lh.set_alpha(1)
for handle in leg.legend_handles:
    handle.set_alpha(1)
    handle.set_sizes([81]) # Equivalent to markersize=9 (9^2 = 81)

#######
# Source - https://stackoverflow.com/a/23221786
# Posted by Hooked
# Retrieved 2026-04-29, License - CC BY-SA 3.0

total_bins = 50

bins = np.linspace(X_mtan.min(), X_mtan.max(), total_bins)
delta = bins[1]-bins[0]
idx  = np.digitize(X_mtan,bins)
running_median = [np.median(Y_mtan[idx==k]) for k in range(total_bins)]
ax.plot(bins-delta/2,running_median,lw=2,alpha=1, c=bluish_color)

assert np.all(X_mtan == X_gp
             )
delta = bins[1]-bins[0]
idx  = np.digitize(X_gp,bins)
running_median = [np.median(Y_gp[idx==k]) for k in range(total_bins)]
ax.plot(bins-delta/2,running_median,lw=2,alpha=1, c=orangish_color)
#######
# import seaborn as sns
# sns.regplot(x=X_mtan, y=Y_mtan, lowess=True)
# sns.regplot(x=X_gp, y=Y_gp, lowess=True)

ax.set_ylim(top=0.1)
save_fig('mtan_vs_gp_per_light_curve.pdf')

plt.show()


# Duration on x-axis
# Reorder the durations (that correspond to test_objIds) to match
# the ordering used in mtan_exec_times.
durations = (np.array(dur_g) + np.array(dur_r)) / 2
id_to_duration = dict(zip(test_objIds, durations))
mtan_ids = [row[0] for row in mtan_exec_times]
durations_reordered = [id_to_duration[obj_id] for obj_id in mtan_ids]
missing = [obj_id for obj_id in mtan_ids if obj_id not in id_to_duration]
assert len(missing) == 0, f"Missing IDs: {missing}"

# Now plot
fig = plt.figure(figsize=(1.5*width/2, height/2))
ax = plt.gca()

Y_mtan = np.array([float(m[1]) for m in mtan_exec_times])
Y_gp = np.array([float(m[1]) for m in gp_exec_times])
ax.scatter(
    durations_reordered, Y_mtan,
    color=bluish_color, alpha=0.1, label='mTAND encoder (GPU)',
    s=20, edgecolors='none'
)
ax.scatter(
    durations_reordered, Y_gp,
    color=orangish_color, alpha=0.1, marker='^', label='GP fit',
    s=20, edgecolors='none'
)
ax.set_xlabel('Duration (days)')
ax.set_ylabel('Execution time (sec)')
ax.grid(True)#, which='both', alpha=0.3)
leg = ax.legend(frameon=False)
for lh in leg.legend_handles:
    lh.set_alpha(1)
for handle in leg.legend_handles:
    handle.set_alpha(1)
    handle.set_sizes([81]) # Equivalent to markersize=9 (9^2 = 81)
set_axes_style(ax)

#######
# Source - https://stackoverflow.com/a/23221786
# Posted by Hooked
# Retrieved 2026-04-29, License - CC BY-SA 3.0

total_bins = 50

bins = np.linspace(np.array(durations_reordered).min(), np.array(durations_reordered).max(), total_bins)
delta = bins[1]-bins[0]
idx  = np.digitize(np.array(durations_reordered),bins)
running_median = [np.median(Y_mtan[idx==k]) for k in range(total_bins)]
ax.plot(bins-delta/2,running_median,lw=2,alpha=1, c=bluish_color)

# delta = bins[1]-bins[0]
# idx  = np.digitize(np.array(durations_reordered),bins)
running_median = [np.median(Y_gp[idx==k]) for k in range(total_bins)]
ax.plot(bins-delta/2,running_median,lw=2,alpha=1, c=orangish_color)
#######

ax.set_ylim(top=0.1)
save_fig('mtan_vs_gp_per_light_curve_duration.pdf')
plt.show()

In [ ]:
plt.hist(Y_mtan)
Y_mtan.mean(), np.median(Y_mtan)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data Preparation
X = np.array([int(float(m[2])) for m in mtan_exec_times])
dur = np.array(durations_reordered)
exec_mtan = np.array([float(m[1]) for m in mtan_exec_times])
exec_gp = np.array([float(m[1]) for m in gp_exec_times])
assert np.all(Y_mtan == exec_mtan)
assert np.all(Y_gp == exec_gp)

# Scaling sizes for visibility
size_mtan = 2000 * exec_mtan
size_gp = 2000 * exec_gp

# Create Figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(width, height/2), sharey=True)

# --- Subplot 1: mTAND ---
ax1.scatter(
    X, dur,
    s=size_mtan,
    color='#3182bd', alpha=0.05, label='mTAND encoder',
    facecolors='none', edgecolors='#3182bd', linewidths=1.5
)
ax1.set_title('mTAND Encoder')
ax1.set_xlabel(r'$2\langle L \rangle$')
ax1.set_ylabel('Duration (days)')
ax1.set_xlim([0, 100])
ax1.grid(True, linestyle='--', alpha=0.6)

# --- Subplot 2: Gaussian Process ---
ax2.scatter(
    X, dur,
    s=size_gp,
    color='#e6550d', alpha=0.05, marker='^', label='GP fit',
    facecolors='none', edgecolors='#e6550d', linewidths=1.5
)
ax2.set_title('GP Fit')
ax2.set_xlabel(r'$2\langle L \rangle$')
ax2.set_xlim([0, 100])
ax2.grid(True, linestyle='--', alpha=0.6)

# Apply styling and fix legend markers
for ax in [ax1, ax2]:
    set_axes_style(ax)
    leg = ax.legend(frameon=False, loc='upper left')
    # Fixed marker size logic for PathCollections
    for handle in leg.legend_handles:
        handle.set_alpha(1)
        if hasattr(handle, 'set_sizes'):
            handle.set_sizes([81]) 
    
    ax.set_xlim([0, 80])

plt.tight_layout()
plt.show()

In [ ]:
len(exec_mtan[X < 80]), len(exec_mtan[(X < 80) & (dur < 120)]), \
len(exec_gp[X < 80]), len(exec_gp[(X < 80) & (dur < 120)])

In [ ]:
np.round(np.median(exec_mtan[X < 80]), 3), np.round(np.median(exec_gp[X < 80]), 3), \
np.round(scipy.stats.median_abs_deviation(exec_mtan[X < 80]), 3), np.round(scipy.stats.median_abs_deviation(exec_gp[X < 80]), 3)

In [ ]:
np.round(np.median(exec_mtan[(X < 80) & (dur < 120)]), 3), np.round(np.median(exec_gp[(X < 80) & (dur < 120)]), 3), \
np.round(scipy.stats.median_abs_deviation(exec_mtan[(X < 80) & (dur < 120)]), 3), np.round(scipy.stats.median_abs_deviation(exec_gp[(X < 80) & (dur < 120)]), 3)

In [ ]:
np.round(np.median(exec_mtan[X >= 80]), 3), np.round(np.median(exec_gp[X >= 80]), 3)

In [ ]:
np.median(exec_mtan[(X < 80) & (dur < 200)]), np.median(exec_gp[(X < 80) & (dur < 200)])

In [ ]:
plt.scatter(
    exec_mtan[(X < 80) & (dur < 180)],
    exec_gp[(X < 80) & (dur < 180)]
)
plt.xlabel('mTAND (<80 obs and dur <6months)')
plt.ylabel('GP (<80 obs and dur <6months)')
plt.plot(exec_mtan[(X < 80) & (dur < 180)], exec_mtan[(X < 80) & (dur < 180)])

In [ ]:
np.sum([float(m[1]) for m in mtan_exec_times]), \
np.sum([float(m[1]) for m in gp_exec_times])

In [ ]:
np.mean([int(m[2]) for m in mtan_exec_times]), \
np.mean([int(m[2]) for m in gp_exec_times]),
np.median([int(m[2]) for m in mtan_exec_times]), \
np.median([int(m[2]) for m in gp_exec_times])

## mTAN schema example light curve save

In [ ]:
def quick_quick_plot_decoded_lc(
    objectId, fig_width=15, fig_height=12,
    show_legend=False, legend_loc='best',
    whichclass=None, show_only_rmse=False,
    show_pred=True,
):
    """
    Plot interpolated light curve using objectId

    Requires `fine_classes` variable to be defined.
    See one of the above cells.

    If whichclass is None, will use `fine_classes`
    which must be globally defined.
    """
    test_currObjId_index = np.where(test_objIds == objectId)
    key = list(decoded_lcs_test.keys())[test_currObjId_index[0][0]]
    ax = plot_decoded_lc(
        decoded_lcs_test, key=key, df_alerts=df_alerts, objId=objectId,
        unnormalize_lcs=True, min_time=min_time, max_time=max_time,
        min_max_mags=min_max_mags, prob_plot=False,
        whichclass=fine_classes[test_currObjId_index[0][0]] if whichclass is None else whichclass,
        also_show_sigmapsf_and_rmse=False, show_only_rmse=show_only_rmse,
        show_separate_bands=False, fig_width=fig_width, fig_height=fig_height, show_legend=show_legend,
        legend_loc=legend_loc, show_pred=show_pred,
    )

    return ax

In [ ]:
# plt.rcParams['figure.dpi'] = 300

# from matplotlib.ticker import MaxNLocator, MultipleLocator

# decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')

# ids = [
#     'ZTF21abmwzxt',
#     'ZTF18abuhbtj',
#     'ZTF18abnnchv',
#     'ZTF18abjzard'
# ]

# for ii, objId in enumerate(ids):
#     objId_index = np.where(test_objIds == objId)[0][0]
# #     print(
# #         objId,
# #         test_outputs_condensed_condensed_transformed[objId_index, 0],
# #         test_outputs_condensed_condensed_transformed[objId_index, 1],
# #     )

#     ax = quick_quick_plot_decoded_lc(
#         decoded_lcs_test, df_alerts, objId, min_max_mags, fig_width=width*2/3, fig_height=height/4,
#         show_legend=True if ii == 0 else False, legend_loc='best',
#         whichclass='', show_pred=False
#     )
#     ax.tick_params(axis='both', which='minor', length=3)
#     ax.tick_params(axis='both', which='major', length=6)
    
#     ax.spines[['right', 'top']].set_visible(False)
#     ax.get_xaxis().tick_bottom()
#     ax.get_yaxis().tick_left()
#     ax.set_title(None)
    
#     ax.set_xlim([0, 365+365/2])

#     # save_fig(f'light_curve_{objId}_{fine_classes_grouped[objId_index].strip()}.pdf')
# #     save_fig(f'light_curve_{objId}_SN_{text_label}.pdf')

## Corrected vs uncorrected AGN light curves (visual inspection)

In [ ]:
df_alerts_corrected = dd.read_parquet(
    'alerts_processed_ftransfer_ztf_2025_05_31_430518_CORRECTED.parquet',
    columns=[
        'objectId', 'finkclass', 'tnsclass',
         'fid', 'magpsf_corr', 'sigmapsf_corr', 'jd'
    ]
)

In [ ]:
test_objIds_dataloader = np.load('evaluate_test_objIds_dataloader.npy')
test_objIds_dataloader_corrected = np.load('evaluate_test_CORRECTED_objIds_dataloader.npy')
assert np.all(test_objIds_dataloader == test_objIds_dataloader_corrected)

In [ ]:
min_datapoints_each_filter = 10
num_to_show = 15

_decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
_decoded_lcs_test_corrected = np.load('evaluate_test_CORRECTED_decoded_lcs.npz')
_min_max_mags_corrected = np.load('min_max_mags_ftransfer_ztf_2025_05_31_430518_CORRECTED.npy')

modes = [
    'random',
]
for mode in modes:
    counters_and_keys = get_counters_and_keys_from_mode(_decoded_lcs_test, mode)
    show_counter = 0
    print(f'Total {len(counters_and_keys)} examples with mode = "{mode}"')

    for counter, key in counters_and_keys:
        test_currObjId = test_objIds_dataloader[counter]

        if test_currObjId in milliquas_agn_labels['objectId'].values:
            test_currCommonFinkclass = milliquas_agn_labels[milliquas_agn_labels['objectId'] == test_currObjId]['custom_type']
            assert len(test_currCommonFinkclass) == 1
            test_currCommonFinkclass = test_currCommonFinkclass.iloc[0]
#             if test_currCommonFinkclass == 'nan':
#                 test_currCommonFinkclass = 'Unknown'
        else:
            test_currObjId_index = np.where(test_objIds == test_currObjId)
            test_currCommonFinkclass = fine_classes[test_currObjId_index[0][0]]

        ############################ Subselection of non-SN subtypes ############################
        if 'SN' in test_currCommonFinkclass:  # this is a hacky way, not robust, but okay for purposes here.
            continue
        ###########################################################################################

        # Plot uncorrected (original)
        ax = plot_decoded_lc(
            _decoded_lcs_test, key=key, df_alerts=df_alerts, objId=test_currObjId,
            unnormalize_lcs=True, min_time=min_time, max_time=max_time,
            min_max_mags=min_max_mags, prob_plot=False,
            whichclass=test_currCommonFinkclass,
            also_show_sigmapsf_and_rmse=True, show_separate_bands=False,
            show_only_rmse=True,
            fig_width=width*2/2, fig_height=height/2,
            show_legend=True if counter == 0 else False, legend_loc='best',
            show_corrected_mags=False
        )
        ax.set_title(f'Uncorrected | {test_currObjId} | {test_currCommonFinkclass} | AGN: {agn_indicator[counter]}')

        # Plot corrected
        ax = plot_decoded_lc(
            _decoded_lcs_test_corrected, key=key, df_alerts=df_alerts_corrected, objId=test_currObjId,
            unnormalize_lcs=True, min_time=min_time, max_time=max_time,
            min_max_mags=_min_max_mags_corrected, prob_plot=False,
            whichclass=test_currCommonFinkclass,
            also_show_sigmapsf_and_rmse=True, show_separate_bands=False,
            show_only_rmse=True,
            fig_width=width*2/2, fig_height=height/2,
            show_legend=True if counter == 0 else False, legend_loc='best',
            show_corrected_mags=True  ##### New change
        )
        ax.set_title(f'Corrected | {test_currObjId} | {test_currCommonFinkclass} | AGN: {agn_indicator[counter]}')

        plt.show()
        plt.close()

        if show_counter == num_to_show:
            break

        show_counter += 1

In [ ]:
_decoded_lcs_test = np.load('evaluate_test_decoded_lcs.npz')
_decoded_lcs_test_corrected = np.load('evaluate_test_CORRECTED_decoded_lcs.npz')
_decoded_lcs_test

key = list(_decoded_lcs_test.keys())[10]
pred_xC, observed_dataC, observed_maskC, observed_timesC = _decoded_lcs_test_corrected[key]
pred_x, observed_data, observed_mask, observed_times = _decoded_lcs_test[key]

In [ ]:
_min_max_mags_corrected = np.load('min_max_mags_ftransfer_ztf_2025_05_31_430518_CORRECTED.npy')
_min_max_mags = np.load('min_max_mags.npy')
_objId = 'ZTF21aarfdjx'
row = _min_max_mags[np.where(_min_max_mags[:,0] == _objId)].squeeze()
rowC = _min_max_mags_corrected[np.where(_min_max_mags_corrected[:,0] == _objId)].squeeze()

unnormalize_mag(
    observed_data, observed_mask, float(row[1]), float(row[2])
)[:3], unnormalize_mag(
    observed_dataC, observed_maskC, float(rowC[1]), float(rowC[2])
)[:3]

In [ ]:
observed_data[:3], observed_dataC[:3]

In [ ]:
import pandas as pd
df = pd.read_parquet('ftransfer_ztf_2026-02-22_444163/').iloc[:1000]

from lc_correction.compute import *
from lc_correction.helpers import *

corrected = df.groupby(["objectId", "fid"]).apply(apply_correction_df)
corrected.reset_index(inplace=True)

corrected = corrected.drop_duplicates(subset=['objectId', 'candid', 'fid', 'jd'], keep='first')
corrected

In [ ]:
corrected[corrected['objectId'] == corrected['objectId'].unique()[4]]

In [ ]:
corrected.groupby(objectId')